In [1]:
# ============================================================
# CELL 1 — ENVIRONMENT, GPU AND PDF AUDIT
# ============================================================

from pathlib import Path
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 70)
print("BUILD4ALL FINAL EXPERIMENT — ENVIRONMENT AUDIT")
print("=" * 70)

# GPU check
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for gpu_id in range(torch.cuda.device_count()):
    print(f"GPU {gpu_id}:", torch.cuda.get_device_name(gpu_id))

# Find uploaded RAG PDFs
pdf_files = sorted(Path("/kaggle/input").rglob("*.pdf"))

print("\nRAG PDF count:", len(pdf_files))

for pdf_path in pdf_files:
    print("-", pdf_path)

# Stop early if setup is incomplete
assert torch.cuda.is_available(), "GPU is not active."
assert torch.cuda.device_count() == 2, "Select GPU T4 x2."
assert len(pdf_files) == 4, "Exactly 4 RAG PDFs are required."

print("\nCELL 1 COMPLETED SUCCESSFULLY")
print("GPU ready: True")
print("RAG documents ready: True")
print("Qwen loaded: False")
print("Generations performed: 0")

BUILD4ALL FINAL EXPERIMENT — ENVIRONMENT AUDIT
CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

RAG PDF count: 8
- /kaggle/input/datasets/lounaakoum/build4all-customer-regional-and-shipping-policy/Build4All_Customer_Regional_and_Shipping_Policy_v2.0.pdf
- /kaggle/input/datasets/lounaakoum/build4all-product-events-and-conversion-policy/Build4All_Product_Events_and_Conversion_Policy_v2.0.pdf
- /kaggle/input/datasets/lounaakoum/build4all-product-inventory-and-coupon-policy/Build4All_Product_Inventory_and_Coupon_Policy_v2.0.pdf
- /kaggle/input/datasets/lounaakoum/build4all-sales-and-order-analytics-policy/Build4All_Sales_and_Order_Analytics_Policy_v2.0.pdf
- /kaggle/input/datasets/lounaakoum/build4all-ui-artifacts/documents/Build4All_Customer_Regional_and_Shipping_Policy_v2.0.pdf
- /kaggle/input/datasets/lounaakoum/build4all-ui-artifacts/documents/Build4All_Product_Events_and_Conversion_Policy_v2.0.pdf
- /kaggle/input/datasets/lounaakoum/build4all-ui-artifacts/documents/B

AssertionError: Exactly 4 RAG PDFs are required.

In [2]:
# ============================================================
# CELL 2 — INSTALL AND VALIDATE REQUIRED DEPENDENCIES
# ============================================================

%pip install --quiet \
    "bitsandbytes>=0.46.1" \
    "faiss-cpu>=1.8.0" \
    "pypdf>=5.0.0" \
    "psycopg2-binary>=2.9.9" \
    "sqlglot>=25.0.0" \
    "sentence-transformers>=3.0.0"

import bitsandbytes
import faiss
import pypdf
import psycopg2
import sqlglot
import sentence_transformers
import transformers

print("=" * 70)
print("DEPENDENCY VALIDATION")
print("=" * 70)

print("Transformers:", transformers.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)
print("FAISS ready:", hasattr(faiss, "IndexFlatIP"))
print("PyPDF ready:", hasattr(pypdf, "PdfReader"))
print("Psycopg2 ready:", hasattr(psycopg2, "connect"))
print("SQLGlot:", sqlglot.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)

assert hasattr(faiss, "IndexFlatIP")
assert hasattr(pypdf, "PdfReader")
assert hasattr(psycopg2, "connect")

print("\nCELL 2 COMPLETED SUCCESSFULLY")
print("Qwen loaded: False")
print("Database connected: False")
print("Generations performed: 0")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 48.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 84.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 96.7 MB/s eta 0:00:00:00:01
Note: you may need to restart the kernel to use updated packages.
DEPENDENCY VALIDATION
Transformers: 5.0.0
BitsAndBytes: 0.50.2
FAISS ready: True
PyPDF ready: True
Psycopg2 ready: True
SQLGlot: 25.20.2
Sentence Transformers: 5.4.1

CELL 2 COMPLETED SUCCESSFULLY
Qwen loaded: False
Database connected: False
Generations performed: 0


In [7]:
# ============================================================
# CELL 3 — FROZEN EXPERIMENT CONFIGURATION
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json

PROJECT_ROOT = Path("/kaggle/working/build4all_final_experiment")

PATHS = {
    "root": PROJECT_ROOT,
    "config": PROJECT_ROOT / "config",
    "data": PROJECT_ROOT / "data",
    "documents": PROJECT_ROOT / "documents",
    "indexes": PROJECT_ROOT / "indexes",
    "predictions": PROJECT_ROOT / "predictions",
    "controller": PROJECT_ROOT / "controller",
    "reports": PROJECT_ROOT / "reports",
    "checkpoints": PROJECT_ROOT / "checkpoints",
}

for path in PATHS.values():
    path.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "project_name": "Build4All Qwen3 RAG Controller Final Experiment",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seed": 42,

    "model": {
        "model_id": "Qwen/Qwen3-8B",
        "quantization": "4-bit NF4",
        "do_sample": False,
        "enable_thinking": False,
        "max_new_tokens": 512,
    },

    "database": {
        "provider": "Neon PostgreSQL",
        "schema": "build4all_research",
        "secret_name": "NEON_DATABASE_URL",
        "read_only": True,
        "statement_timeout_seconds": 30,
    },

    "languages": [
        "English",
        "Arabic",
        "French",
        "Lebanese Arabizi",
    ],

    "rag": {
        "embedding_model": "intfloat/multilingual-e5-base",
        "retrieval_type": "dense",
        "index_type": "FAISS IndexFlatIP",
        "top_k": 4,
        "max_context_characters": 4000,
        "normalized_embeddings": True,
    },

    "experiment": {
        "systems": [
            "Qwen Alone",
            "Qwen + RAG",
            "Adaptive Controller",
        ],
        "target_question_count": 600,
        "split_method": "semantic-family group split",
        "train_ratio": 0.60,
        "validation_ratio": 0.20,
        "final_test_ratio": 0.20,
        "primary_metric": "execution accuracy",
        "minimum_target_accuracy": 0.75,
        "final_test_locked": True,
    },
}

config_path = PATHS["config"] / "experiment_config.json"

with config_path.open("w", encoding="utf-8") as file:
    json.dump(CONFIG, file, indent=2, ensure_ascii=False)

assert CONFIG["model"]["do_sample"] is False
assert CONFIG["database"]["read_only"] is True
assert CONFIG["experiment"]["final_test_locked"] is True
assert sum([
    CONFIG["experiment"]["train_ratio"],
    CONFIG["experiment"]["validation_ratio"],
    CONFIG["experiment"]["final_test_ratio"],
]) == 1.0

print("=" * 70)
print("FROZEN EXPERIMENT CONFIGURATION")
print("=" * 70)

print("Model:", CONFIG["model"]["model_id"])
print("Database schema:", CONFIG["database"]["schema"])
print("Languages:", CONFIG["languages"])
print("Systems:", CONFIG["experiment"]["systems"])
print("Target questions:", CONFIG["experiment"]["target_question_count"])
print("Minimum target accuracy:", CONFIG["experiment"]["minimum_target_accuracy"])
print("Final test locked:", CONFIG["experiment"]["final_test_locked"])
print("Configuration saved:", config_path)

print("\nCELL 3 COMPLETED SUCCESSFULLY")
print("Qwen loaded: False")
print("Database connected: False")
print("Generations performed: 0")

FROZEN EXPERIMENT CONFIGURATION
Model: Qwen/Qwen3-8B
Database schema: build4all_research
Languages: ['English', 'Arabic', 'French', 'Lebanese Arabizi']
Systems: ['Qwen Alone', 'Qwen + RAG', 'Adaptive Controller']
Target questions: 600
Minimum target accuracy: 0.75
Final test locked: True
Configuration saved: /kaggle/working/build4all_final_experiment/config/experiment_config.json

CELL 3 COMPLETED SUCCESSFULLY
Qwen loaded: False
Database connected: False
Generations performed: 0


In [8]:
# ============================================================
# CELL 4 — SECURE NEON CONNECTION AND SCHEMA AUDIT
# ============================================================

from kaggle_secrets import UserSecretsClient
import psycopg2
import pandas as pd

SECRET_NAME = CONFIG["database"]["secret_name"]
SCHEMA_NAME = CONFIG["database"]["schema"]

# ------------------------------------------------------------
# 1. Read Neon URL securely from Kaggle Secrets
# ------------------------------------------------------------

try:
    secrets_client = UserSecretsClient()
    #connection URL men Kaggle Secrets.
    NEON_DATABASE_URL = secrets_client.get_secret(SECRET_NAME)

except Exception as error:
    raise RuntimeError(
        "NEON_DATABASE_URL was not found or is not enabled."
    ) from error


# ------------------------------------------------------------
# 2. Create a reusable read-only connection
# ------------------------------------------------------------

def get_db_connection():

    connection = psycopg2.connect(
        NEON_DATABASE_URL,
        sslmode="require",
        connect_timeout=15,
        application_name="build4all_final_experiment",
        options="-c default_transaction_read_only=on",
    )

    # The Neon pooler accepts statement_timeout after connection.
    with connection.cursor() as cursor:
        cursor.execute(
            "SET statement_timeout TO 30000;"
        )

    return connection


# ------------------------------------------------------------
# 3. Define the expected final tables
# ------------------------------------------------------------

expected_tables = {
    "categories",
    "coupons",
    "customers",
    "shipping_methods",
    "tax_rules",
    "products",
    "orders",
    "order_items",
    "payments",
    "product_events",
}


# ------------------------------------------------------------
# 4. Test connection and inspect schema
# ------------------------------------------------------------

with get_db_connection() as connection:

    with connection.cursor() as cursor:

        cursor.execute("""
            SELECT
                current_database(),
                current_user,
                current_setting('transaction_read_only');
        """)

        database_name, database_user, read_only = (
            cursor.fetchone()
        )

        cursor.execute("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = %s
              AND table_type = 'BASE TABLE'
            ORDER BY table_name;
        """, (SCHEMA_NAME,))

        discovered_tables = {
            row[0]
            for row in cursor.fetchall()
        }

        cursor.execute("""
            SELECT
                table_name,
                column_name,
                data_type
            FROM information_schema.columns
            WHERE table_schema = %s
            ORDER BY table_name, ordinal_position;
        """, (SCHEMA_NAME,))

        schema_rows = cursor.fetchall()


# ------------------------------------------------------------
# 5. Store the schema audit
# ------------------------------------------------------------

schema_audit = pd.DataFrame(
    schema_rows,
    columns=[
        "table_name",
        "column_name",
        "data_type",
    ],
)

missing_tables = (
    expected_tables - discovered_tables
)

unexpected_tables = (
    discovered_tables - expected_tables
)


# ------------------------------------------------------------
# 6. Display safe audit results
# ------------------------------------------------------------

print("=" * 70)
print("NEON CONNECTION AND SCHEMA AUDIT")
print("=" * 70)

print("Database:", database_name)
print("Database user:", database_user)
print("Read-only mode:", read_only)
print("Schema:", SCHEMA_NAME)
print("Table count:", len(discovered_tables))
print("Tables:", sorted(discovered_tables))
print("Missing tables:", sorted(missing_tables))
print("Unexpected tables:", sorted(unexpected_tables))

print("\nCOLUMN COUNT BY TABLE")

print(
    schema_audit
    .groupby("table_name")
    .size()
    .rename("column_count")
    .to_string()
)


# ------------------------------------------------------------
# 7. Stop if the database is incomplete or unsafe
# ------------------------------------------------------------

assert read_only == "on", (
    "Database connection is not read-only."
)

assert not missing_tables, (
    f"Missing tables: {sorted(missing_tables)}"
)

assert len(schema_audit) > 0, (
    "No schema columns were discovered."
)


print("\nCELL 4 COMPLETED SUCCESSFULLY")
print("Database connected: True")
print("Schema validated: True")
print("Qwen loaded: False")
print("Generations performed: 0")

NEON CONNECTION AND SCHEMA AUDIT
Database: neondb
Database user: neondb_owner
Read-only mode: on
Schema: build4all_research
Table count: 10
Tables: ['categories', 'coupons', 'customers', 'order_items', 'orders', 'payments', 'product_events', 'products', 'shipping_methods', 'tax_rules']
Missing tables: []
Unexpected tables: []

COLUMN COUNT BY TABLE
table_name
categories           3
coupons             10
customers            5
order_items          5
orders              11
payments             6
product_events       5
products            11
shipping_methods     7
tax_rules            7

CELL 4 COMPLETED SUCCESSFULLY
Database connected: True
Schema validated: True
Qwen loaded: False
Generations performed: 0


In [9]:
# ============================================================
# CELL 5 — EXACT DATABASE SCHEMA AND DATA VOCABULARY
# ============================================================

from psycopg2 import sql
import json

# ------------------------------------------------------------
# 1. Get columns
# ------------------------------------------------------------

with get_db_connection() as connection:
    with connection.cursor() as cursor:

        cursor.execute("""
            SELECT
                table_name,
                column_name,
                data_type,
                is_nullable
            FROM information_schema.columns
            WHERE table_schema = %s
            ORDER BY table_name, ordinal_position;
        """, (SCHEMA_NAME,))

        column_rows = cursor.fetchall()

        # Primary keys
        cursor.execute("""
            SELECT
                tc.table_name,
                kcu.column_name
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            WHERE tc.table_schema = %s
              AND tc.constraint_type = 'PRIMARY KEY'
            ORDER BY tc.table_name, kcu.ordinal_position;
        """, (SCHEMA_NAME,))

        primary_key_rows = cursor.fetchall()

        # Foreign keys
        cursor.execute("""
            SELECT
                tc.table_name,
                kcu.column_name,
                ccu.table_name AS referenced_table,
                ccu.column_name AS referenced_column
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            JOIN information_schema.constraint_column_usage ccu
              ON tc.constraint_name = ccu.constraint_name
             AND tc.table_schema = ccu.table_schema
            WHERE tc.table_schema = %s
              AND tc.constraint_type = 'FOREIGN KEY'
            ORDER BY tc.table_name, kcu.column_name;
        """, (SCHEMA_NAME,))

        foreign_key_rows = cursor.fetchall()

        # Row counts
        table_row_counts = {}

        for table_name in sorted(discovered_tables):
            cursor.execute(
                sql.SQL("SELECT COUNT(*) FROM {}.{}").format(
                    sql.Identifier(SCHEMA_NAME),
                    sql.Identifier(table_name),
                )
            )

            table_row_counts[table_name] = (
                cursor.fetchone()[0]
            )

        # Controlled categorical values
        vocabulary_queries = {
            "orders.status": """
                SELECT DISTINCT status
                FROM build4all_research.orders
                ORDER BY status
            """,

            "payments.payment_status": """
    SELECT DISTINCT payment_status
    FROM build4all_research.payments
    ORDER BY payment_status
""",

            "payments.payment_method": """
                SELECT DISTINCT payment_method
                FROM build4all_research.payments
                ORDER BY payment_method
            """,

            "products.status": """
                SELECT DISTINCT status
                FROM build4all_research.products
                ORDER BY status
            """,

            "product_events.event_type": """
                SELECT DISTINCT event_type
                FROM build4all_research.product_events
                ORDER BY event_type
            """,
        }

        controlled_vocabulary = {}

        for name, query in vocabulary_queries.items():
            cursor.execute(query)

            controlled_vocabulary[name] = [
                row[0]
                for row in cursor.fetchall()
            ]


# ------------------------------------------------------------
# 2. Build a compact schema description for Qwen
# ------------------------------------------------------------

primary_keys = {
    (table_name, column_name)
    for table_name, column_name in primary_key_rows
}

schema_lines = []

current_table = None

for table_name, column_name, data_type, nullable in column_rows:

    if table_name != current_table:
        if current_table is not None:
            schema_lines.append("")

        schema_lines.append(
            f"TABLE {SCHEMA_NAME}.{table_name}"
        )

        current_table = table_name

    key_label = (
        " PRIMARY KEY"
        if (table_name, column_name) in primary_keys
        else ""
    )

    null_label = (
        " NULL"
        if nullable == "YES"
        else " NOT NULL"
    )

    schema_lines.append(
        f"- {column_name}: {data_type}"
        f"{key_label}{null_label}"
    )

schema_lines.append("")
schema_lines.append("FOREIGN KEYS")

for (
    table_name,
    column_name,
    referenced_table,
    referenced_column,
) in foreign_key_rows:

    schema_lines.append(
        f"- {SCHEMA_NAME}.{table_name}.{column_name} "
        f"-> {SCHEMA_NAME}.{referenced_table}."
        f"{referenced_column}"
    )

DATABASE_SCHEMA_TEXT = "\n".join(schema_lines)


# ------------------------------------------------------------
# 3. Save the audited schema
# ------------------------------------------------------------

schema_text_path = (
    PATHS["data"] / "database_schema.txt"
)

schema_json_path = (
    PATHS["data"] / "database_audit.json"
)

schema_text_path.write_text(
    DATABASE_SCHEMA_TEXT,
    encoding="utf-8",
)

schema_json_path.write_text(
    json.dumps(
        {
            "schema": SCHEMA_NAME,
            "row_counts": table_row_counts,
            "controlled_vocabulary": controlled_vocabulary,
            "primary_keys": primary_key_rows,
            "foreign_keys": foreign_key_rows,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 4. Display audit
# ------------------------------------------------------------

print("=" * 70)
print("EXACT DATABASE SCHEMA")
print("=" * 70)
print(DATABASE_SCHEMA_TEXT)

print("\n" + "=" * 70)
print("TABLE ROW COUNTS")
print("=" * 70)

for table_name, row_count in table_row_counts.items():
    print(f"{table_name:<20}: {row_count}")

print("\n" + "=" * 70)
print("CONTROLLED VOCABULARY")
print("=" * 70)

for field_name, values in controlled_vocabulary.items():
    print(f"{field_name}: {values}")

assert all(
    row_count > 0
    for row_count in table_row_counts.values()
), "At least one table is empty."

print("\nCELL 5 COMPLETED SUCCESSFULLY")
print("Exact schema saved:", schema_text_path)
print("Database audit saved:", schema_json_path)
print("Qwen loaded: False")
print("Generations performed: 0")

EXACT DATABASE SCHEMA
TABLE build4all_research.categories
- category_id: integer PRIMARY KEY NOT NULL
- name: character varying NOT NULL
- parent_category_id: integer NULL

TABLE build4all_research.coupons
- coupon_id: integer PRIMARY KEY NOT NULL
- code: character varying NOT NULL
- discount_type: character varying NOT NULL
- discount_value: numeric NOT NULL
- minimum_order_amount: numeric NOT NULL
- maximum_discount_amount: numeric NULL
- valid_from: timestamp with time zone NULL
- valid_to: timestamp with time zone NULL
- maximum_uses: integer NULL
- active: boolean NOT NULL

TABLE build4all_research.customers
- customer_id: integer PRIMARY KEY NOT NULL
- customer_code: character varying NOT NULL
- city: character varying NULL
- region: character varying NULL
- created_at: timestamp with time zone NOT NULL

TABLE build4all_research.order_items
- order_item_id: integer PRIMARY KEY NOT NULL
- order_id: integer NOT NULL
- product_id: integer NOT NULL
- quantity: integer NOT NULL
- unit

In [10]:
# ============================================================
# CELL 6 — EXTRACT AND AUDIT THE FOUR RAG PDFs
# ============================================================

from pypdf import PdfReader
from pathlib import Path
import json
import shutil

pdf_files = sorted(
    Path("/kaggle/input").rglob("*.pdf")
)

raw_documents = []
document_audit = []

for document_id, pdf_path in enumerate(pdf_files):

    reader = PdfReader(str(pdf_path))
    page_texts = []

    for page_number, page in enumerate(
        reader.pages,
        start=1,
    ):
        extracted_text = (
            page.extract_text() or ""
        ).strip()

        page_texts.append(extracted_text)

        raw_documents.append({
            "document_id": document_id,
            "document_name": pdf_path.stem,
            "source_path": str(pdf_path),
            "page_number": page_number,
            "text": extracted_text,
        })

    full_text = "\n\n".join(page_texts)

    document_audit.append({
        "document_id": document_id,
        "document_name": pdf_path.stem,
        "page_count": len(reader.pages),
        "character_count": len(full_text),
        "empty_pages": sum(
            not text.strip()
            for text in page_texts
        ),
    })

    # Keep a working copy with a stable filename
    destination = (
        PATHS["documents"] / pdf_path.name
    )

    shutil.copy2(
        pdf_path,
        destination,
    )


raw_documents_path = (
    PATHS["documents"] / "extracted_pdf_pages.json"
)

raw_documents_path.write_text(
    json.dumps(
        raw_documents,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 70)
print("RAG PDF EXTRACTION AUDIT")
print("=" * 70)

for audit in document_audit:
    print(
        f"\nDocument: {audit['document_name']}"
    )
    print(
        f"Pages: {audit['page_count']}"
    )
    print(
        f"Characters: {audit['character_count']}"
    )
    print(
        f"Empty pages: {audit['empty_pages']}"
    )

print("\n" + "=" * 70)
print("TEXT PREVIEW")
print("=" * 70)

for document in raw_documents:
    preview = (
        document["text"][:250]
        .replace("\n", " ")
    )

    print(
        f"\n{document['document_name']} "
        f"— Page {document['page_number']}"
    )
    print(preview)

assert len(pdf_files) == 4
assert len(raw_documents) > 0

assert all(
    audit["empty_pages"] == 0
    for audit in document_audit
), "At least one PDF page has no extractable text."

assert all(
    audit["character_count"] > 500
    for audit in document_audit
), "At least one PDF contains insufficient text."

print("\nCELL 6 COMPLETED SUCCESSFULLY")
print("PDF files processed:", len(pdf_files))
print("Extracted pages:", len(raw_documents))
print("Extraction saved:", raw_documents_path)
print("RAG index built: False")
print("Qwen loaded: False")

RAG PDF EXTRACTION AUDIT

Document: Build4All_Customer_Regional_and_Shipping_Policy_v2.0
Pages: 2
Characters: 4155
Empty pages: 0

Document: Build4All_Product_Events_and_Conversion_Policy_v2.0
Pages: 2
Characters: 4119
Empty pages: 0

Document: Build4All_Product_Inventory_and_Coupon_Policy_v2.0
Pages: 2
Characters: 4287
Empty pages: 0

Document: Build4All_Sales_and_Order_Analytics_Policy_v2.0
Pages: 3
Characters: 6009
Empty pages: 0

Document: Build4All_Customer_Regional_and_Shipping_Policy_v2.0
Pages: 2
Characters: 4155
Empty pages: 0

Document: Build4All_Product_Events_and_Conversion_Policy_v2.0
Pages: 2
Characters: 4119
Empty pages: 0

Document: Build4All_Product_Inventory_and_Coupon_Policy_v2.0
Pages: 2
Characters: 4287
Empty pages: 0

Document: Build4All_Sales_and_Order_Analytics_Policy_v2.0
Pages: 3
Characters: 6009
Empty pages: 0

TEXT PREVIEW

Build4All_Customer_Regional_and_Shipping_Policy_v2.0 — Page 1
Build4All Customer Regional and Shipping Policy Controlled knowledge sourc

AssertionError: 

In [11]:
# ============================================================
# CELL 7 — SENTENCE-SAFE SEMANTIC CHUNKING
# ============================================================

import re
import json

CHUNK_TARGET_CHARACTERS = 900
CHUNK_OVERLAP_CHARACTERS = 150


def clean_extracted_text(text):
    """
    Normalize PDF whitespace without changing its meaning.
    """

    text = text.replace("\u00a0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def split_into_sentences(text):
    """
    Split text at safe sentence or heading boundaries.
    Words are never cut in the middle.
    """

    text = clean_extracted_text(text)

    units = re.split(
        r"(?<=[.!?])\s+|\n+",
        text,
    )

    return [
        unit.strip()
        for unit in units
        if unit.strip()
    ]


def create_chunks(
    text,
    target_characters=900,
    overlap_characters=150,
):
    """
    Build sentence-safe chunks with limited semantic overlap.
    """

    sentences = split_into_sentences(text)
    chunks = []

    current_sentences = []
    current_length = 0

    for sentence in sentences:

        sentence_length = len(sentence) + 1

        if (
            current_sentences
            and current_length + sentence_length
            > target_characters
        ):
            chunk_text = " ".join(
                current_sentences
            ).strip()

            chunks.append(chunk_text)

            # Keep complete sentences for overlap
            overlap_sentences = []
            overlap_length = 0

            for previous_sentence in reversed(
                current_sentences
            ):
                if (
                    overlap_length
                    + len(previous_sentence)
                    > overlap_characters
                ):
                    break

                overlap_sentences.insert(
                    0,
                    previous_sentence,
                )

                overlap_length += (
                    len(previous_sentence) + 1
                )

            current_sentences = overlap_sentences
            current_length = sum(
                len(item) + 1
                for item in current_sentences
            )

        current_sentences.append(sentence)
        current_length += sentence_length

    if current_sentences:
        chunks.append(
            " ".join(current_sentences).strip()
        )

    return chunks


rag_chunks = []
global_chunk_id = 0

for document in raw_documents:

    page_chunks = create_chunks(
        document["text"],
        target_characters=(
            CHUNK_TARGET_CHARACTERS
        ),
        overlap_characters=(
            CHUNK_OVERLAP_CHARACTERS
        ),
    )

    for page_chunk_index, chunk_text in enumerate(
        page_chunks
    ):
        rag_chunks.append({
            "chunk_id": global_chunk_id,
            "document_id": document["document_id"],
            "document_name": document["document_name"],
            "page_number": document["page_number"],
            "page_chunk_index": page_chunk_index,
            "text": chunk_text,
            "character_count": len(chunk_text),
        })

        global_chunk_id += 1


chunks_path = (
    PATHS["documents"] / "rag_chunks.json"
)

chunks_path.write_text(
    json.dumps(
        rag_chunks,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 70)
print("RAG CHUNKING AUDIT")
print("=" * 70)

print("Source pages:", len(raw_documents))
print("Total chunks:", len(rag_chunks))
print(
    "Minimum chunk size:",
    min(chunk["character_count"] for chunk in rag_chunks),
)
print(
    "Maximum chunk size:",
    max(chunk["character_count"] for chunk in rag_chunks),
)
print(
    "Average chunk size:",
    round(
        sum(
            chunk["character_count"]
            for chunk in rag_chunks
        ) / len(rag_chunks),
        2,
    ),
)

print("\nCHUNKS BY DOCUMENT")

for document_name in sorted({
    chunk["document_name"]
    for chunk in rag_chunks
}):
    count = sum(
        chunk["document_name"] == document_name
        for chunk in rag_chunks
    )

    print(f"{document_name}: {count}")


print("\nFIRST CHUNK PREVIEW")
print("-" * 70)
print(rag_chunks[0]["text"][:700])

assert len(rag_chunks) >= len(raw_documents)
assert all(
    chunk["text"].strip()
    for chunk in rag_chunks
)
assert all(
    chunk["character_count"] <= 1200
    for chunk in rag_chunks
)

print("\nCELL 7 COMPLETED SUCCESSFULLY")
print("Chunks saved:", chunks_path)
print("RAG index built: False")
print("Qwen loaded: False")

RAG CHUNKING AUDIT
Source pages: 18
Total chunks: 52
Minimum chunk size: 341
Maximum chunk size: 899
Average chunk size: 783.15

CHUNKS BY DOCUMENT
Build4All_Customer_Regional_and_Shipping_Policy_v2.0: 12
Build4All_Product_Events_and_Conversion_Policy_v2.0: 12
Build4All_Product_Inventory_and_Coupon_Policy_v2.0: 12
Build4All_Sales_and_Order_Analytics_Policy_v2.0: 16

FIRST CHUNK PREVIEW
----------------------------------------------------------------------
Build4All Customer Regional and Shipping Policy Controlled knowledge source for multilingual Text to SQL retrieval Document ID B4A-DOC-CUSTOMER-002 Version 2.0 Authoritative schema build4all_research Status Controlled research source Scope Customers, regions, registrations, repeat purchasing, and shipping methods Purpose and authority This document defines customer, regional, registration, repeat-purchase, and shipping-method meanings for the frozen build4all_research PostgreSQL schema. It is authoritative for the controlled retrieval

In [12]:
# ============================================================
# CELL 8 — BUILD THE MULTILINGUAL FAISS RAG INDEX
# ============================================================

from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import json
import time

EMBEDDING_MODEL_ID = (
    CONFIG["rag"]["embedding_model"]
)

print(
    "Loading embedding model:",
    EMBEDDING_MODEL_ID,
)

# Keep embeddings on CPU to preserve GPU memory for Qwen
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_ID,
    device="cpu",
)

passage_texts = [
    "passage: " + chunk["text"]
    for chunk in rag_chunks
]

embedding_start = time.perf_counter()

chunk_embeddings = embedding_model.encode(
    passage_texts,
    batch_size=16,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

embedding_time = (
    time.perf_counter() - embedding_start
)

embedding_dimension = (
    chunk_embeddings.shape[1]
)

rag_index = faiss.IndexFlatIP(
    embedding_dimension
)

rag_index.add(chunk_embeddings)


# ------------------------------------------------------------
# Retrieval function
# ------------------------------------------------------------

def retrieve_chunks(question, top_k=None):

    if top_k is None:
        top_k = CONFIG["rag"]["top_k"]

    query_embedding = embedding_model.encode(
        ["query: " + question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")

    scores, indices = rag_index.search(
        query_embedding,
        top_k,
    )

    results = []

    for rank, (index_position, score) in enumerate(
        zip(indices[0], scores[0]),
        start=1,
    ):
        chunk = rag_chunks[
            int(index_position)
        ].copy()

        chunk["rank"] = rank
        chunk["retrieval_score"] = float(score)

        results.append(chunk)

    return results


# ------------------------------------------------------------
# Save index and embeddings
# ------------------------------------------------------------

index_path = (
    PATHS["indexes"] / "build4all_rag.faiss"
)

embeddings_path = (
    PATHS["indexes"] / "chunk_embeddings.npy"
)

faiss.write_index(
    rag_index,
    str(index_path),
)

np.save(
    embeddings_path,
    chunk_embeddings,
)


# ------------------------------------------------------------
# Multilingual retrieval smoke tests
# ------------------------------------------------------------

retrieval_tests = [
    "How is recognized revenue calculated?",
    "كيف يتم حساب الإيرادات المعترف بها؟",
    "Comment calculer le taux de conversion vue-achat ?",
    "Kif mn7seb el active inventory value?",
]

print("=" * 70)
print("MULTILINGUAL RETRIEVAL SMOKE TEST")
print("=" * 70)

for question in retrieval_tests:

    results = retrieve_chunks(
        question,
        top_k=2,
    )

    print("\nQUESTION:", question)

    for result in results:
        print(
            f"Rank {result['rank']} | "
            f"Score {result['retrieval_score']:.4f} | "
            f"{result['document_name']} | "
            f"Page {result['page_number']}"
        )


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert rag_index.ntotal == len(rag_chunks)
assert chunk_embeddings.shape[0] == len(rag_chunks)
assert np.isfinite(chunk_embeddings).all()
assert index_path.exists()
assert embeddings_path.exists()

print("\nCELL 8 COMPLETED SUCCESSFULLY")
print("Indexed chunks:", rag_index.ntotal)
print("Embedding dimension:", embedding_dimension)
print(
    "Embedding time:",
    round(embedding_time, 2),
    "seconds",
)
print("FAISS index saved:", index_path)
print("Embedding model device: CPU")
print("Qwen loaded: False")

Loading embedding model: intfloat/multilingual-e5-base


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

MULTILINGUAL RETRIEVAL SMOKE TEST

QUESTION: How is recognized revenue calculated?
Rank 1 | Score 0.8510 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 1
Rank 2 | Score 0.8510 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 1

QUESTION: كيف يتم حساب الإيرادات المعترف بها؟
Rank 1 | Score 0.7690 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 1
Rank 2 | Score 0.7690 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 1

QUESTION: Comment calculer le taux de conversion vue-achat ?
Rank 1 | Score 0.8591 | Build4All_Product_Events_and_Conversion_Policy_v2.0 | Page 1
Rank 2 | Score 0.8591 | Build4All_Product_Events_and_Conversion_Policy_v2.0 | Page 1

QUESTION: Kif mn7seb el active inventory value?
Rank 1 | Score 0.8274 | Build4All_Product_Inventory_and_Coupon_Policy_v2.0 | Page 1
Rank 2 | Score 0.8274 | Build4All_Product_Inventory_and_Coupon_Policy_v2.0 | Page 1

CELL 8 COMPLETED SUCCESSFULLY
Indexed chunks: 52
Embedding dimension: 768
Embedding time: 17.68 

In [13]:
# ============================================================
# CELL 9 — LOAD QWEN3-8B IN 4-BIT
# ============================================================

import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

MODEL_ID = CONFIG["model"]["model_id"]

# ------------------------------------------------------------
# 1. Configure 4-bit quantization
# ------------------------------------------------------------

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ------------------------------------------------------------
# 2. Load tokenizer
# ------------------------------------------------------------

print("Loading Qwen tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# ------------------------------------------------------------
# 3. Load Qwen
# ------------------------------------------------------------

print("Loading Qwen3-8B in 4-bit...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    max_memory={
        0: "14GiB",
        1: "14GiB",
        "cpu": "28GiB",
    },
)

model.eval()

# ------------------------------------------------------------
# 4. Validate model state
# ------------------------------------------------------------

model_devices = sorted({
    str(device)
    for device in model.hf_device_map.values()
})

print("=" * 70)
print("QWEN3-8B MODEL AUDIT")
print("=" * 70)

print("Model:", MODEL_ID)
print("Training mode:", model.training)
print("4-bit loaded:", getattr(model, "is_loaded_in_4bit", False))
print("Device map values:", model_devices)

for gpu_id in range(torch.cuda.device_count()):

    allocated_gb = (
        torch.cuda.memory_allocated(gpu_id)
        / 1024**3
    )

    reserved_gb = (
        torch.cuda.memory_reserved(gpu_id)
        / 1024**3
    )

    print(
        f"GPU {gpu_id} allocated: "
        f"{allocated_gb:.2f} GB"
    )

    print(
        f"GPU {gpu_id} reserved: "
        f"{reserved_gb:.2f} GB"
    )

assert model.training is False
assert getattr(
    model,
    "is_loaded_in_4bit",
    False,
), "Qwen was not loaded in 4-bit."

print("\nCELL 9 COMPLETED SUCCESSFULLY")
print("Qwen loaded: True")
print("Qwen generations performed: 0")
print("RAG index ready: True")
print("Database ready: True")

Loading Qwen tokenizer...


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

QWEN3-8B MODEL AUDIT
Model: Qwen/Qwen3-8B
Training mode: False
4-bit loaded: True
Device map values: ['0', '1']
GPU 0 allocated: 1.44 GB
GPU 0 reserved: 1.75 GB
GPU 1 allocated: 4.24 GB
GPU 1 reserved: 4.86 GB

CELL 9 COMPLETED SUCCESSFULLY
Qwen loaded: True
Qwen generations performed: 0
RAG index ready: True
Database ready: True


In [16]:
# ============================================================
# CELL 10 — FROZEN QWEN AND RAG GENERATION FUNCTIONS
# ============================================================

import re
import time
import torch

# ------------------------------------------------------------
# 1. Build controlled vocabulary text
# ------------------------------------------------------------

CONTROLLED_VOCABULARY_TEXT = "\n".join(
    f"- {field}: {values}"
    for field, values
    in controlled_vocabulary.items()
)


# ------------------------------------------------------------
# 2. Build RAG context
# ------------------------------------------------------------

def build_rag_context(question):

    retrieved = retrieve_chunks(
        question,
        top_k=CONFIG["rag"]["top_k"],
    )

    context_parts = []
    current_length = 0

    for result in retrieved:

        chunk_block = (
            f"[Source: {result['document_name']}, "
            f"page {result['page_number']}, "
            f"score {result['retrieval_score']:.4f}]\n"
            f"{result['text']}"
        )

        if (
            current_length + len(chunk_block)
            > CONFIG["rag"]["max_context_characters"]
        ):
            break

        context_parts.append(chunk_block)
        current_length += len(chunk_block)

    return (
        "\n\n".join(context_parts),
        retrieved,
    )


# ------------------------------------------------------------
# 3. Extract SQL safely from Qwen response
# ------------------------------------------------------------

def extract_sql(raw_response):

    response = raw_response.strip()

    fenced_match = re.search(
        r"```(?:sql)?\s*(.*?)```",
        response,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if fenced_match:
        response = fenced_match.group(1).strip()

    sql_start = re.search(
        r"\b(SELECT|WITH)\b",
        response,
        flags=re.IGNORECASE,
    )

    if not sql_start:
        return ""

    response = response[
        sql_start.start():
    ].strip()

    if ";" in response:
        response = (
            response.split(";", 1)[0].strip()
            + ";"
        )

    return response


# ------------------------------------------------------------
# 4. Frozen Text-to-SQL generation function
# ------------------------------------------------------------

def generate_sql(
    question,
    use_rag=False,
):

    rag_context = ""
    retrieved_chunks = []

    if use_rag:
        rag_context, retrieved_chunks = (
            build_rag_context(question)
        )

    system_prompt = """
You are a multilingual PostgreSQL Text-to-SQL system.

Generate exactly one read-only PostgreSQL query that answers
the user's business question.

Mandatory rules:
1. Use only the provided database schema and columns.
2. Use the build4all_research schema explicitly.
3. Generate only SELECT or WITH queries.
4. Never generate INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, TRUNCATE, GRANT, REVOKE, COPY, or CALL.
5. Follow the controlled vocabulary exactly.
6. Do not invent tables, columns, filters, joins, or values.
7. Return SQL only, with no explanation or Markdown.
""".strip()

    user_parts = [
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULARY_TEXT,
    ]

    if use_rag:
        user_parts.extend([
            "",
            "RETRIEVED BUSINESS POLICY:",
            rag_context,
            "",
            "Use the retrieved policy only when it is relevant.",
        ])

    user_parts.extend([
        "",
        "BUSINESS QUESTION:",
        question,
        "",
        "POSTGRESQL QUERY:",
    ])

    user_prompt = "\n".join(user_parts)

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=(
                CONFIG["model"]["max_new_tokens"]
            ),
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generation_time = (
        time.perf_counter() - start_time
    )

    new_tokens = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:,
    ]

    raw_response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    ).strip()

    generated_sql = extract_sql(
        raw_response
    )

    return {
        "question": question,
        "system": (
            "qwen_with_rag"
            if use_rag
            else "qwen_alone"
        ),
        "sql": generated_sql,
        "raw_response": raw_response,
        "latency_seconds": round(
            generation_time,
            4,
        ),
        "retrieved_chunks": retrieved_chunks,
        "rag_context_characters": len(
            rag_context
        ),
    }


print("=" * 70)
print("FROZEN GENERATION PIPELINES")
print("=" * 70)
print("Qwen Alone ready: True")
print("Qwen + RAG ready: True")
print("Same model: True")
print("Same database schema: True")
print("Same generation settings: True")
print("Only difference: retrieved business policy")
print("Generations performed: 0")

print("\nCELL 10 COMPLETED SUCCESSFULLY")

FROZEN GENERATION PIPELINES
Qwen Alone ready: True
Qwen + RAG ready: True
Same model: True
Same database schema: True
Same generation settings: True
Only difference: retrieved business policy
Generations performed: 0

CELL 10 COMPLETED SUCCESSFULLY


In [17]:
# ============================================================
# CELL 11 — SQL SAFETY, EXECUTION AND RESULT NORMALIZATION
# ============================================================

from decimal import Decimal
from datetime import date, datetime
import sqlglot
from sqlglot import expressions as exp


ALLOWED_TABLES = set(discovered_tables)

FORBIDDEN_EXPRESSIONS = (
    exp.Insert,
    exp.Update,
    exp.Delete,
    exp.Drop,
    exp.Create,
    exp.Alter,
    exp.Command,
    exp.Merge,
    exp.TruncateTable,
)


# ------------------------------------------------------------
# 1. Validate generated SQL
# ------------------------------------------------------------

def validate_readonly_sql(sql_text):

    if not sql_text or not sql_text.strip():
        return {
            "safe": False,
            "reason": "Empty SQL",
            "parsed": None,
        }

    try:
        statements = [
            statement
            for statement in sqlglot.parse(
                sql_text,
                read="postgres",
            )
            if statement is not None
        ]

    except Exception as error:
        return {
            "safe": False,
            "reason": f"SQL parse error: {error}",
            "parsed": None,
        }

    if len(statements) != 1:
        return {
            "safe": False,
            "reason": "Exactly one SQL statement is required",
            "parsed": None,
        }

    parsed = statements[0]

    if not isinstance(
        parsed,
        (
            exp.Select,
            exp.Union,
            exp.Intersect,
            exp.Except,
        ),
    ):
        return {
            "safe": False,
            "reason": (
                "Only SELECT or WITH queries are allowed"
            ),
            "parsed": parsed,
        }

    for forbidden_type in FORBIDDEN_EXPRESSIONS:
        if parsed.find(forbidden_type):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden SQL operation: "
                    f"{forbidden_type.__name__}"
                ),
                "parsed": parsed,
            }

    referenced_tables = []

    for table in parsed.find_all(exp.Table):

        table_name = table.name
        table_schema = table.db

        referenced_tables.append({
            "schema": table_schema,
            "table": table_name,
        })

        if table_name not in ALLOWED_TABLES:
            return {
                "safe": False,
                "reason": (
                    f"Unknown or forbidden table: "
                    f"{table_name}"
                ),
                "parsed": parsed,
            }

        if (
            table_schema
            and table_schema != SCHEMA_NAME
        ):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden schema: "
                    f"{table_schema}"
                ),
                "parsed": parsed,
            }

    if not referenced_tables:
        return {
            "safe": False,
            "reason": "No approved database table referenced",
            "parsed": parsed,
        }

    return {
        "safe": True,
        "reason": "Read-only SQL validated",
        "parsed": parsed,
        "referenced_tables": referenced_tables,
    }


# ------------------------------------------------------------
# 2. Normalize database values
# ------------------------------------------------------------

def normalize_value(value):

    if isinstance(value, Decimal):
        return round(float(value), 6)

    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, float):
        return round(value, 6)

    if isinstance(value, bytes):
        return value.decode(
            "utf-8",
            errors="replace",
        )

    return value


def normalize_result(columns, rows):

    normalized_rows = [
        tuple(
            normalize_value(value)
            for value in row
        )
        for row in rows
    ]

    # Result comparison should not fail because of row order
    sorted_rows = sorted(
        normalized_rows,
        key=lambda row: repr(row),
    )

    return {
        "columns": [
            str(column).lower()
            for column in columns
        ],
        "rows": sorted_rows,
    }


# ------------------------------------------------------------
# 3. Execute only validated SQL
# ------------------------------------------------------------

def execute_safe_sql(sql_text):

    validation = validate_readonly_sql(
        sql_text
    )

    if not validation["safe"]:
        return {
            "executed": False,
            "safe": False,
            "error": validation["reason"],
            "columns": [],
            "rows": [],
            "normalized_result": None,
        }

    try:
        with get_db_connection() as connection:

            with connection.cursor() as cursor:

                cursor.execute(sql_text)

                columns = [
                    description.name
                    for description in cursor.description
                ]

                rows = cursor.fetchall()

        return {
            "executed": True,
            "safe": True,
            "error": None,
            "columns": columns,
            "rows": rows,
            "normalized_result": normalize_result(
                columns,
                rows,
            ),
        }

    except Exception as error:
        return {
            "executed": False,
            "safe": True,
            "error": str(error),
            "columns": [],
            "rows": [],
            "normalized_result": None,
        }


# ------------------------------------------------------------
# 4. Safety smoke tests
# ------------------------------------------------------------

safe_test_sql = """
SELECT status, COUNT(*) AS order_count
FROM build4all_research.orders
GROUP BY status
ORDER BY status;
"""

unsafe_test_sql = """
DELETE FROM build4all_research.orders;
"""

unknown_table_test_sql = """
SELECT *
FROM private_customer_passwords;
"""

safe_validation = validate_readonly_sql(
    safe_test_sql
)

unsafe_validation = validate_readonly_sql(
    unsafe_test_sql
)

unknown_validation = validate_readonly_sql(
    unknown_table_test_sql
)

safe_execution = execute_safe_sql(
    safe_test_sql
)


print("=" * 70)
print("SQL SAFETY AND EXECUTION AUDIT")
print("=" * 70)

print(
    "Safe SELECT accepted:",
    safe_validation["safe"],
)

print(
    "DELETE rejected:",
    not unsafe_validation["safe"],
)

print(
    "Unknown table rejected:",
    not unknown_validation["safe"],
)

print(
    "Safe SELECT executed:",
    safe_execution["executed"],
)

print(
    "Returned rows:",
    len(safe_execution["rows"]),
)

print(
    "Execution error:",
    safe_execution["error"],
)


assert safe_validation["safe"] is True
assert unsafe_validation["safe"] is False
assert unknown_validation["safe"] is False
assert safe_execution["executed"] is True

print("\nCELL 11 COMPLETED SUCCESSFULLY")
print("SQL validator ready: True")
print("Read-only execution ready: True")
print("Qwen generations performed: 0")

SQL SAFETY AND EXECUTION AUDIT
Safe SELECT accepted: True
DELETE rejected: True
Unknown table rejected: True
Safe SELECT executed: True
Returned rows: 6
Execution error: None

CELL 11 COMPLETED SUCCESSFULLY
SQL validator ready: True
Read-only execution ready: True
Qwen generations performed: 0


In [13]:
# ============================================================
# CELL 12 — FIRST QWEN-ALONE TEXT-TO-SQL SMOKE TEST
# ============================================================

SMOKE_TEST_QUESTION = (
    "Show the number of orders in each order status."
)

qwen_alone_smoke = generate_sql(
    question=SMOKE_TEST_QUESTION,
    use_rag=False,
)

qwen_alone_smoke_execution = (
    execute_safe_sql(
        qwen_alone_smoke["sql"]
    )
)


print("=" * 70)
print("QWEN ALONE — FIRST SMOKE TEST")
print("=" * 70)

print("Question:")
print(qwen_alone_smoke["question"])

print("\nGenerated SQL:")
print(qwen_alone_smoke["sql"])

print("\nRaw response:")
print(qwen_alone_smoke["raw_response"])

print(
    "\nGeneration latency:",
    qwen_alone_smoke["latency_seconds"],
    "seconds",
)

print(
    "SQL safe:",
    qwen_alone_smoke_execution["safe"],
)

print(
    "SQL executed:",
    qwen_alone_smoke_execution["executed"],
)

print(
    "Execution error:",
    qwen_alone_smoke_execution["error"],
)

print("\nDatabase result:")

if qwen_alone_smoke_execution["executed"]:

    print(
        qwen_alone_smoke_execution["columns"]
    )

    for row in (
        qwen_alone_smoke_execution["rows"]
    ):
        print(row)


assert qwen_alone_smoke["system"] == (
    "qwen_alone"
)

assert (
    qwen_alone_smoke_execution["safe"]
    is True
)

assert (
    qwen_alone_smoke_execution["executed"]
    is True
)


print("\nCELL 12 COMPLETED SUCCESSFULLY")
print("Qwen Alone generations performed: 1")
print("RAG used: False")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QWEN ALONE — FIRST SMOKE TEST
Question:
Show the number of orders in each order status.

Generated SQL:
SELECT status, COUNT(*) AS order_count
FROM build4all_research.orders
GROUP BY status;

Raw response:
SELECT status, COUNT(*) AS order_count
FROM build4all_research.orders
GROUP BY status;

Generation latency: 5.4126 seconds
SQL safe: True
SQL executed: True
Execution error: None

Database result:
['status', 'order_count']
('completed', 2896)
('refunded', 238)
('pending', 322)
('shipped', 694)
('cancelled', 343)
('processing', 507)

CELL 12 COMPLETED SUCCESSFULLY
Qwen Alone generations performed: 1
RAG used: False


In [14]:
# ============================================================
# CELL 13 — QWEN + RAG SMOKE TEST AND DIRECT COMPARISON
# ============================================================

qwen_rag_smoke = generate_sql(
    question=SMOKE_TEST_QUESTION,
    use_rag=True,
)

qwen_rag_smoke_execution = (
    execute_safe_sql(
        qwen_rag_smoke["sql"]
    )
)


print("=" * 70)
print("QWEN + RAG — SMOKE TEST")
print("=" * 70)

print("Question:")
print(qwen_rag_smoke["question"])

print("\nRetrieved evidence:")

for chunk in qwen_rag_smoke[
    "retrieved_chunks"
]:
    print(
        f"Rank {chunk['rank']} | "
        f"Score {chunk['retrieval_score']:.4f} | "
        f"{chunk['document_name']} | "
        f"Page {chunk['page_number']}"
    )

print(
    "\nRAG context characters:",
    qwen_rag_smoke[
        "rag_context_characters"
    ],
)

print("\nGenerated SQL:")
print(qwen_rag_smoke["sql"])

print(
    "\nGeneration latency:",
    qwen_rag_smoke["latency_seconds"],
    "seconds",
)

print(
    "SQL safe:",
    qwen_rag_smoke_execution["safe"],
)

print(
    "SQL executed:",
    qwen_rag_smoke_execution["executed"],
)

print(
    "Execution error:",
    qwen_rag_smoke_execution["error"],
)

print("\nDatabase result:")

if qwen_rag_smoke_execution["executed"]:

    print(
        qwen_rag_smoke_execution["columns"]
    )

    for row in (
        qwen_rag_smoke_execution["rows"]
    ):
        print(row)


results_equal = (
    qwen_alone_smoke_execution[
        "normalized_result"
    ]
    ==
    qwen_rag_smoke_execution[
        "normalized_result"
    ]
)


print("\n" + "=" * 70)
print("DIRECT SMOKE-TEST COMPARISON")
print("=" * 70)

print(
    "Qwen Alone latency:",
    qwen_alone_smoke["latency_seconds"],
)

print(
    "Qwen + RAG latency:",
    qwen_rag_smoke["latency_seconds"],
)

print(
    "Database results equal:",
    results_equal,
)


assert qwen_rag_smoke["system"] == (
    "qwen_with_rag"
)

assert (
    qwen_rag_smoke_execution["safe"]
    is True
)

assert (
    qwen_rag_smoke_execution["executed"]
    is True
)

assert results_equal is True


print("\nCELL 13 COMPLETED SUCCESSFULLY")
print("Qwen Alone smoke generations: 1")
print("Qwen + RAG smoke generations: 1")
print("Final benchmark started: False")

QWEN + RAG — SMOKE TEST
Question:
Show the number of orders in each order status.

Retrieved evidence:
Rank 1 | Score 0.8227 | Build4All_Customer_Regional_and_Shipping_Policy_v2.0 | Page 2
Rank 2 | Score 0.8179 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 1
Rank 3 | Score 0.8104 | Build4All_Sales_and_Order_Analytics_Policy_v2.0 | Page 2
Rank 4 | Score 0.8088 | Build4All_Customer_Regional_and_Shipping_Policy_v2.0 | Page 1

RAG context characters: 3462

Generated SQL:
SELECT status, COUNT(*) AS order_count
FROM build4all_research.orders
GROUP BY status;

Generation latency: 5.2687 seconds
SQL safe: True
SQL executed: True
Execution error: None

Database result:
['status', 'order_count']
('completed', 2896)
('refunded', 238)
('pending', 322)
('shipped', 694)
('cancelled', 343)
('processing', 507)

DIRECT SMOKE-TEST COMPARISON
Qwen Alone latency: 5.4126
Qwen + RAG latency: 5.2687
Database results equal: True

CELL 13 COMPLETED SUCCESSFULLY
Qwen Alone smoke generations: 1
Qwen +

In [18]:
# ============================================================
# CELL 14 — UPGRADE VALIDATOR FOR SAFE WITH / CTE QUERIES
# ============================================================

def validate_readonly_sql(sql_text):

    if not sql_text or not sql_text.strip():
        return {
            "safe": False,
            "reason": "Empty SQL",
            "parsed": None,
        }

    try:
        statements = [
            statement
            for statement in sqlglot.parse(
                sql_text,
                read="postgres",
            )
            if statement is not None
        ]

    except Exception as error:
        return {
            "safe": False,
            "reason": f"SQL parse error: {error}",
            "parsed": None,
        }

    if len(statements) != 1:
        return {
            "safe": False,
            "reason": (
                "Exactly one SQL statement is required"
            ),
            "parsed": None,
        }

    parsed = statements[0]

    if not isinstance(
        parsed,
        (
            exp.Select,
            exp.Union,
            exp.Intersect,
            exp.Except,
        ),
    ):
        return {
            "safe": False,
            "reason": (
                "Only SELECT or WITH queries are allowed"
            ),
            "parsed": parsed,
        }

    for forbidden_type in FORBIDDEN_EXPRESSIONS:
        if parsed.find(forbidden_type):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden SQL operation: "
                    f"{forbidden_type.__name__}"
                ),
                "parsed": parsed,
            }

    # Collect temporary table names defined by WITH clauses
    cte_names = {
        cte.alias_or_name
        for cte in parsed.find_all(exp.CTE)
        if cte.alias_or_name
    }

    referenced_tables = []

    for table in parsed.find_all(exp.Table):

        table_name = table.name
        table_schema = table.db

        # CTE names are valid temporary query relations
        if (
            table_name in cte_names
            and not table_schema
        ):
            continue

        referenced_tables.append({
            "schema": table_schema,
            "table": table_name,
        })

        if table_name not in ALLOWED_TABLES:
            return {
                "safe": False,
                "reason": (
                    f"Unknown or forbidden table: "
                    f"{table_name}"
                ),
                "parsed": parsed,
            }

        if (
            table_schema
            and table_schema != SCHEMA_NAME
        ):
            return {
                "safe": False,
                "reason": (
                    f"Forbidden schema: "
                    f"{table_schema}"
                ),
                "parsed": parsed,
            }

    if not referenced_tables:
        return {
            "safe": False,
            "reason": (
                "No approved database table referenced"
            ),
            "parsed": parsed,
        }

    return {
        "safe": True,
        "reason": "Read-only SQL validated",
        "parsed": parsed,
        "referenced_tables": referenced_tables,
        "cte_names": sorted(cte_names),
    }


# ------------------------------------------------------------
# Test a real read-only CTE
# ------------------------------------------------------------

cte_test_sql = """
WITH recognized_customer_orders AS (
    SELECT
        customer_id,
        COUNT(*) AS recognized_order_count
    FROM build4all_research.orders
    WHERE status IN ('completed', 'shipped')
    GROUP BY customer_id
)
SELECT COUNT(*) AS repeat_customer_count
FROM recognized_customer_orders
WHERE recognized_order_count > 1;
"""

cte_validation = validate_readonly_sql(
    cte_test_sql
)

cte_execution = execute_safe_sql(
    cte_test_sql
)

print("=" * 70)
print("CTE SAFETY VALIDATION")
print("=" * 70)

print("CTE safe:", cte_validation["safe"])
print(
    "Detected CTE names:",
    cte_validation.get("cte_names"),
)
print("CTE executed:", cte_execution["executed"])
print("Execution error:", cte_execution["error"])
print("Result:", cte_execution["rows"])

assert cte_validation["safe"] is True
assert cte_execution["executed"] is True
assert "recognized_customer_orders" in (
    cte_validation["cte_names"]
)

print("\nCELL 14 COMPLETED SUCCESSFULLY")
print("SELECT validation ready: True")
print("WITH validation ready: True")
print("Final benchmark started: False")

CTE SAFETY VALIDATION
CTE safe: True
Detected CTE names: ['recognized_customer_orders']
CTE executed: True
Execution error: None
Result: [(393,)]

CELL 14 COMPLETED SUCCESSFULLY
SELECT validation ready: True
WITH validation ready: True
Final benchmark started: False


In [19]:
# ============================================================
# CELL 15 — CANONICAL INTENTS AND VERIFIED GOLD SQL
# ============================================================

import json

REFERENCE_DATE = "2026-08-31"

intent_rows = [
    (
        "B4A-001",
        "recognized_revenue_by_region",
        "sales",
        True,
        "What is the recognized order revenue by customer region?",
        """
        SELECT c.region,
               ROUND(SUM(o.total_amount), 2) AS recognized_revenue
        FROM build4all_research.orders AS o
        JOIN build4all_research.customers AS c
          ON c.customer_id = o.customer_id
        WHERE o.status IN ('completed', 'shipped')
        GROUP BY c.region
        ORDER BY recognized_revenue DESC, c.region;
        """,
    ),
    (
        "B4A-002",
        "monthly_recognized_revenue",
        "sales",
        True,
        "What is the recognized order revenue for each month?",
        """
        SELECT DATE_TRUNC('month', order_date) AS revenue_month,
               ROUND(SUM(total_amount), 2) AS recognized_revenue
        FROM build4all_research.orders
        WHERE status IN ('completed', 'shipped')
        GROUP BY DATE_TRUNC('month', order_date)
        ORDER BY revenue_month;
        """,
    ),
    (
        "B4A-003",
        "gross_merchandise_value",
        "sales",
        True,
        "What is the gross merchandise value of non-cancelled orders?",
        """
        SELECT ROUND(SUM(subtotal), 2) AS gross_merchandise_value
        FROM build4all_research.orders
        WHERE status <> 'cancelled';
        """,
    ),
    (
        "B4A-004",
        "net_merchandise_sales",
        "sales",
        True,
        "What are the net merchandise sales for recognized orders?",
        """
        SELECT ROUND(
                   SUM(subtotal - discount_amount),
                   2
               ) AS net_merchandise_sales
        FROM build4all_research.orders
        WHERE status IN ('completed', 'shipped');
        """,
    ),
    (
        "B4A-005",
        "average_recognized_order_value",
        "sales",
        True,
        "What is the average order value for recognized sales?",
        """
        SELECT ROUND(
                   AVG(total_amount),
                   2
               ) AS average_order_value
        FROM build4all_research.orders
        WHERE status IN ('completed', 'shipped');
        """,
    ),
    (
        "B4A-006",
        "orders_by_status",
        "orders",
        False,
        "How many orders are in each order status?",
        """
        SELECT status,
               COUNT(*) AS order_count
        FROM build4all_research.orders
        GROUP BY status
        ORDER BY order_count DESC, status;
        """,
    ),
    (
        "B4A-007",
        "operational_order_backlog",
        "orders",
        True,
        "How many orders are currently in the operational backlog?",
        """
        SELECT COUNT(*) AS backlog_order_count
        FROM build4all_research.orders
        WHERE status IN ('pending', 'processing');
        """,
    ),
    (
        "B4A-008",
        "order_cancellation_rate",
        "orders",
        False,
        "What percentage of orders were cancelled?",
        """
        SELECT ROUND(
                   100.0 * COUNT(*) FILTER (
                       WHERE status = 'cancelled'
                   ) / NULLIF(COUNT(*), 0),
                   2
               ) AS cancellation_rate_percent
        FROM build4all_research.orders;
        """,
    ),
    (
        "B4A-009",
        "refunded_order_count",
        "orders",
        False,
        "How many orders have a refunded order status?",
        """
        SELECT COUNT(*) AS refunded_order_count
        FROM build4all_research.orders
        WHERE status = 'refunded';
        """,
    ),
    (
        "B4A-010",
        "daily_order_volume",
        "orders",
        False,
        "What is the daily order volume during August 2026?",
        """
        SELECT order_date::date AS order_day,
               COUNT(*) AS order_count
        FROM build4all_research.orders
        WHERE order_date >= TIMESTAMPTZ '2026-08-01 00:00:00+00'
          AND order_date < TIMESTAMPTZ '2026-09-01 00:00:00+00'
        GROUP BY order_date::date
        ORDER BY order_day;
        """,
    ),
    (
        "B4A-011",
        "paid_collections_by_method",
        "payments",
        True,
        "What is the collected paid amount by payment method?",
        """
        SELECT payment_method,
               ROUND(SUM(paid_amount), 2) AS collected_amount
        FROM build4all_research.payments
        WHERE payment_status = 'paid'
        GROUP BY payment_method
        ORDER BY collected_amount DESC, payment_method;
        """,
    ),
    (
        "B4A-012",
        "outstanding_payment_amount",
        "payments",
        True,
        "What is the outstanding amount for pending payments?",
        """
        SELECT ROUND(
                   SUM(paid_amount),
                   2
               ) AS outstanding_payment_amount
        FROM build4all_research.payments
        WHERE payment_status = 'pending';
        """,
    ),
    (
        "B4A-013",
        "failed_payment_rate",
        "payments",
        False,
        "What percentage of payment records failed?",
        """
        SELECT ROUND(
                   100.0 * COUNT(*) FILTER (
                       WHERE payment_status = 'failed'
                   ) / NULLIF(COUNT(*), 0),
                   2
               ) AS failed_payment_rate_percent
        FROM build4all_research.payments;
        """,
    ),
    (
        "B4A-014",
        "refunded_payment_amount",
        "payments",
        False,
        "What is the total amount associated with refunded payments?",
        """
        SELECT ROUND(
                   SUM(paid_amount),
                   2
               ) AS refunded_payment_amount
        FROM build4all_research.payments
        WHERE payment_status = 'refunded';
        """,
    ),
    (
        "B4A-015",
        "paid_payment_order_reconciliation",
        "payments",
        True,
        "How many paid payments do not match their related order total?",
        """
        SELECT COUNT(*) AS mismatched_paid_payment_count
        FROM build4all_research.payments AS p
        JOIN build4all_research.orders AS o
          ON o.order_id = p.order_id
        WHERE p.payment_status = 'paid'
          AND ABS(p.paid_amount - o.total_amount) > 0.01;
        """,
    ),
    (
        "B4A-016",
        "coupon_usage_count",
        "coupons",
        True,
        "How many orders used each coupon?",
        """
        SELECT c.code,
               COUNT(o.order_id) AS orders_using_coupon
        FROM build4all_research.coupons AS c
        LEFT JOIN build4all_research.orders AS o
          ON o.coupon_id = c.coupon_id
        GROUP BY c.coupon_id, c.code
        ORDER BY orders_using_coupon DESC, c.code;
        """,
    ),
    (
        "B4A-017",
        "recognized_discount_by_coupon",
        "coupons",
        True,
        "What recognized discount amount is associated with each coupon?",
        """
        SELECT c.code,
               ROUND(
                   SUM(o.discount_amount),
                   2
               ) AS recognized_discount_amount
        FROM build4all_research.coupons AS c
        JOIN build4all_research.orders AS o
          ON o.coupon_id = c.coupon_id
        WHERE o.status IN ('completed', 'shipped')
        GROUP BY c.coupon_id, c.code
        ORDER BY recognized_discount_amount DESC, c.code;
        """,
    ),
    (
        "B4A-018",
        "active_valid_coupons",
        "coupons",
        True,
        "Which coupons are active and valid on the reporting reference date?",
        """
        SELECT code,
               discount_type,
               discount_value,
               valid_from,
               valid_to
        FROM build4all_research.coupons
        WHERE active = TRUE
          AND (
              valid_from IS NULL
              OR valid_from <= TIMESTAMPTZ '2026-08-31 23:59:59+00'
          )
          AND (
              valid_to IS NULL
              OR valid_to >= TIMESTAMPTZ '2026-08-31 00:00:00+00'
          )
        ORDER BY code;
        """,
    ),
    (
        "B4A-019",
        "products_by_status",
        "products",
        False,
        "How many products are in each catalog status?",
        """
        SELECT status,
               COUNT(*) AS product_count
        FROM build4all_research.products
        GROUP BY status
        ORDER BY product_count DESC, status;
        """,
    ),
    (
        "B4A-020",
        "low_stock_active_products",
        "products",
        True,
        "Which active products have fewer than ten units in stock?",
        """
        SELECT sku,
               name,
               stock
        FROM build4all_research.products
        WHERE status = 'active'
          AND stock < 10
        ORDER BY stock, sku;
        """,
    ),
    (
        "B4A-021",
        "active_inventory_value",
        "products",
        True,
        "What is the inventory value of active products?",
        """
        SELECT ROUND(
                   SUM(
                       stock * COALESCE(sale_price, price)
                   ),
                   2
               ) AS active_inventory_value
        FROM build4all_research.products
        WHERE status = 'active';
        """,
    ),
    (
        "B4A-022",
        "recognized_product_sales_by_category",
        "products",
        True,
        "What are recognized product sales by category?",
        """
        SELECT c.name AS category_name,
               ROUND(
                   SUM(oi.quantity * oi.unit_price),
                   2
               ) AS product_sales
        FROM build4all_research.order_items AS oi
        JOIN build4all_research.orders AS o
          ON o.order_id = oi.order_id
        JOIN build4all_research.products AS p
          ON p.product_id = oi.product_id
        JOIN build4all_research.categories AS c
          ON c.category_id = p.category_id
        WHERE o.status IN ('completed', 'shipped')
        GROUP BY c.category_id, c.name
        ORDER BY product_sales DESC, category_name;
        """,
    ),
    (
        "B4A-023",
        "purchasing_customers_by_region",
        "customers",
        True,
        "How many purchasing customers are in each region?",
        """
        SELECT c.region,
               COUNT(
                   DISTINCT c.customer_id
               ) AS purchasing_customer_count
        FROM build4all_research.customers AS c
        JOIN build4all_research.orders AS o
          ON o.customer_id = c.customer_id
        WHERE o.status IN ('completed', 'shipped')
        GROUP BY c.region
        ORDER BY purchasing_customer_count DESC, c.region;
        """,
    ),
    (
        "B4A-024",
        "monthly_customer_registrations",
        "customers",
        False,
        "How many customers registered in each month?",
        """
        SELECT DATE_TRUNC(
                   'month',
                   created_at
               ) AS registration_month,
               COUNT(*) AS registered_customer_count
        FROM build4all_research.customers
        GROUP BY DATE_TRUNC('month', created_at)
        ORDER BY registration_month;
        """,
    ),
    (
        "B4A-025",
        "repeat_purchasing_customers",
        "customers",
        True,
        "How many customers placed more than one recognized order?",
        """
        WITH recognized_customer_orders AS (
            SELECT customer_id,
                   COUNT(*) AS recognized_order_count
            FROM build4all_research.orders
            WHERE status IN ('completed', 'shipped')
            GROUP BY customer_id
        )
        SELECT COUNT(*) AS repeat_purchasing_customer_count
        FROM recognized_customer_orders
        WHERE recognized_order_count > 1;
        """,
    ),
    (
        "B4A-026",
        "recognized_shipping_amount_by_method",
        "shipping",
        True,
        "What recognized shipping amount is associated with each shipping method?",
        """
        SELECT sm.name AS shipping_method,
               ROUND(
                   SUM(o.shipping_amount),
                   2
               ) AS recognized_shipping_amount
        FROM build4all_research.orders AS o
        JOIN build4all_research.shipping_methods AS sm
          ON sm.shipping_method_id = o.shipping_method_id
        WHERE o.status IN ('completed', 'shipped')
        GROUP BY sm.shipping_method_id, sm.name
        ORDER BY recognized_shipping_amount DESC, shipping_method;
        """,
    ),
    (
        "B4A-027",
        "orders_by_shipping_method",
        "shipping",
        True,
        "How many orders used each shipping method?",
        """
        SELECT sm.name AS shipping_method,
               COUNT(o.order_id) AS order_count
        FROM build4all_research.shipping_methods AS sm
        LEFT JOIN build4all_research.orders AS o
          ON o.shipping_method_id = sm.shipping_method_id
        GROUP BY sm.shipping_method_id, sm.name
        ORDER BY order_count DESC, shipping_method;
        """,
    ),
    (
        "B4A-028",
        "effective_tax_rate",
        "tax",
        True,
        "What is the effective tax rate for recognized orders?",
        """
        SELECT ROUND(
                   100.0 * SUM(tax_amount)
                   / NULLIF(
                       SUM(subtotal - discount_amount),
                       0
                   ),
                   2
               ) AS effective_tax_rate_percent
        FROM build4all_research.orders
        WHERE status IN ('completed', 'shipped');
        """,
    ),
    (
        "B4A-029",
        "product_events_by_type",
        "events",
        False,
        "How many product events occurred for each event type?",
        """
        SELECT event_type,
               COUNT(*) AS event_count
        FROM build4all_research.product_events
        GROUP BY event_type
        ORDER BY event_count DESC, event_type;
        """,
    ),
    (
        "B4A-030",
        "view_to_purchase_conversion_rate",
        "events",
        True,
        "What is the product view-to-purchase conversion rate?",
        """
        SELECT ROUND(
                   100.0 * COUNT(*) FILTER (
                       WHERE event_type = 'purchase'
                   )
                   / NULLIF(
                       COUNT(*) FILTER (
                           WHERE event_type = 'view'
                       ),
                       0
                   ),
                   2
               ) AS view_to_purchase_conversion_rate_percent
        FROM build4all_research.product_events;
        """,
    ),
]


CANONICAL_INTENTS = []

for (
    intent_id,
    intent_name,
    domain,
    requires_policy,
    question,
    gold_sql,
) in intent_rows:

    CANONICAL_INTENTS.append({
        "intent_id": intent_id,
        "intent_name": intent_name,
        "domain": domain,
        "requires_policy": requires_policy,
        "canonical_question": question,
        "gold_sql": gold_sql.strip(),
    })


# ------------------------------------------------------------
# Execute and validate every gold query
# ------------------------------------------------------------

validated_gold_intents = []

for intent in CANONICAL_INTENTS:

    validation = validate_readonly_sql(
        intent["gold_sql"]
    )

    execution = execute_safe_sql(
        intent["gold_sql"]
    )

    if not validation["safe"]:
        raise RuntimeError(
            f"{intent['intent_id']} failed safety: "
            f"{validation['reason']}"
        )

    if not execution["executed"]:
        raise RuntimeError(
            f"{intent['intent_id']} failed execution: "
            f"{execution['error']}"
        )

    validated_intent = intent.copy()

    validated_intent["gold_columns"] = (
        execution["normalized_result"]["columns"]
    )

    validated_intent["gold_rows"] = (
        execution["normalized_result"]["rows"]
    )

    validated_gold_intents.append(
        validated_intent
    )


gold_catalog_path = (
    PATHS["data"] / "canonical_gold_intents.json"
)

gold_catalog_path.write_text(
    json.dumps(
        validated_gold_intents,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 70)
print("CANONICAL GOLD INTENT VALIDATION")
print("=" * 70)

print(
    "Canonical intent count:",
    len(validated_gold_intents),
)

print(
    "Policy-dependent intents:",
    sum(
        intent["requires_policy"]
        for intent in validated_gold_intents
    ),
)

print(
    "Schema-only intents:",
    sum(
        not intent["requires_policy"]
        for intent in validated_gold_intents
    ),
)

print("\nINTENTS BY DOMAIN")

domain_counts = {}

for intent in validated_gold_intents:
    domain = intent["domain"]

    domain_counts[domain] = (
        domain_counts.get(domain, 0) + 1
    )

for domain, count in sorted(
    domain_counts.items()
):
    print(f"{domain:<15}: {count}")

print("\nGOLD RESULT ROW COUNTS")

for intent in validated_gold_intents:
    print(
        f"{intent['intent_id']} "
        f"{intent['intent_name']:<42} "
        f"{len(intent['gold_rows'])} row(s)"
    )


assert len(validated_gold_intents) == 30
assert all(
    len(intent["gold_rows"]) >= 1
    for intent in validated_gold_intents
)

print("\nCELL 15 COMPLETED SUCCESSFULLY")
print("All gold SQL safe: True")
print("All gold SQL executable: True")
print("Gold catalog saved:", gold_catalog_path)
print("Qwen benchmark generations performed: 0")

CANONICAL GOLD INTENT VALIDATION
Canonical intent count: 30
Policy-dependent intents: 21
Schema-only intents: 9

INTENTS BY DOMAIN
coupons        : 3
customers      : 3
events         : 2
orders         : 5
payments       : 5
products       : 4
sales          : 5
shipping       : 2
tax            : 1

GOLD RESULT ROW COUNTS
B4A-001 recognized_revenue_by_region               6 row(s)
B4A-002 monthly_recognized_revenue                 31 row(s)
B4A-003 gross_merchandise_value                    1 row(s)
B4A-004 net_merchandise_sales                      1 row(s)
B4A-005 average_recognized_order_value             1 row(s)
B4A-006 orders_by_status                           6 row(s)
B4A-007 operational_order_backlog                  1 row(s)
B4A-008 order_cancellation_rate                    1 row(s)
B4A-009 refunded_order_count                       1 row(s)
B4A-010 daily_order_volume                         31 row(s)
B4A-011 paid_collections_by_method                 5 row(s)
B4A-012 outs

In [20]:
# ============================================================
# CELL 16 — LOW-SCALE BENCHMARK SPLIT BLUEPRINT
# ============================================================

import json
from collections import Counter

# All four language versions of the same intent
# must remain inside the same split.

VALIDATION_INTENTS = {
    "B4A-003",  # Sales policy
    "B4A-008",  # Order rate
    "B4A-015",  # Payment reconciliation
    "B4A-020",  # Product policy
    "B4A-027",  # Shipping policy
    "B4A-029",  # Event count
}

FINAL_TEST_INTENTS = {
    "B4A-005",  # Sales policy
    "B4A-010",  # Date filtering
    "B4A-013",  # Payment rate
    "B4A-018",  # Coupon validity
    "B4A-023",  # Distinct customers
    "B4A-030",  # Event conversion
}

TRAIN_INTENTS = {
    intent["intent_id"]
    for intent in validated_gold_intents
} - VALIDATION_INTENTS - FINAL_TEST_INTENTS


def get_intent_split(intent_id):

    if intent_id in VALIDATION_INTENTS:
        return "validation"

    if intent_id in FINAL_TEST_INTENTS:
        return "final_test"

    return "train"


intent_split_manifest = []

for intent in validated_gold_intents:

    intent_split_manifest.append({
        "intent_id": intent["intent_id"],
        "intent_name": intent["intent_name"],
        "domain": intent["domain"],
        "requires_policy": intent["requires_policy"],
        "split": get_intent_split(
            intent["intent_id"]
        ),
        "expected_language_versions": 4,
    })


split_counts = Counter(
    row["split"]
    for row in intent_split_manifest
)

expected_question_counts = {
    split_name: intent_count * 4
    for split_name, intent_count
    in split_counts.items()
}


stage_config = {
    "stage": "low_scale_decision_experiment",
    "intent_count": 30,
    "languages_per_intent": 4,
    "question_count": 120,
    "train_intents": len(TRAIN_INTENTS),
    "validation_intents": len(
        VALIDATION_INTENTS
    ),
    "final_test_intents": len(
        FINAL_TEST_INTENTS
    ),
    "translation_versions_kept_together": True,
    "final_test_locked": True,
    "future_high_scale_target": (
        CONFIG["experiment"][
            "target_question_count"
        ]
    ),
}

split_manifest_path = (
    PATHS["data"]
    / "low_scale_split_manifest.json"
)

split_manifest_path.write_text(
    json.dumps(
        {
            "stage_config": stage_config,
            "intent_splits": (
                intent_split_manifest
            ),
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


print("=" * 70)
print("LOW-SCALE BENCHMARK SPLIT BLUEPRINT")
print("=" * 70)

print("Current question count: 120")
print(
    "Future high-scale target:",
    stage_config["future_high_scale_target"],
)

print("\nINTENT COUNTS")

for split_name in [
    "train",
    "validation",
    "final_test",
]:
    print(
        f"{split_name:<12}: "
        f"{split_counts[split_name]} intents"
    )

print("\nEXPECTED QUESTION COUNTS")

for split_name in [
    "train",
    "validation",
    "final_test",
]:
    print(
        f"{split_name:<12}: "
        f"{expected_question_counts[split_name]} "
        f"questions"
    )

print("\nFINAL TEST INTENTS")

for row in intent_split_manifest:
    if row["split"] == "final_test":
        print(
            row["intent_id"],
            row["intent_name"],
        )


all_ids = (
    TRAIN_INTENTS
    | VALIDATION_INTENTS
    | FINAL_TEST_INTENTS
)

assert len(all_ids) == 30
assert len(TRAIN_INTENTS) == 18
assert len(VALIDATION_INTENTS) == 6
assert len(FINAL_TEST_INTENTS) == 6

assert not (
    TRAIN_INTENTS
    & VALIDATION_INTENTS
)

assert not (
    TRAIN_INTENTS
    & FINAL_TEST_INTENTS
)

assert not (
    VALIDATION_INTENTS
    & FINAL_TEST_INTENTS
)

assert expected_question_counts == {
    "train": 72,
    "validation": 24,
    "final_test": 24,
}

print("\nCELL 16 COMPLETED SUCCESSFULLY")
print("Translation leakage possible: False")
print("Final test locked: True")
print("Final benchmark generations performed: 0")

LOW-SCALE BENCHMARK SPLIT BLUEPRINT
Current question count: 120
Future high-scale target: 600

INTENT COUNTS
train       : 18 intents
validation  : 6 intents
final_test  : 6 intents

EXPECTED QUESTION COUNTS
train       : 72 questions
validation  : 24 questions
final_test  : 24 questions

FINAL TEST INTENTS
B4A-005 average_recognized_order_value
B4A-010 daily_order_volume
B4A-013 failed_payment_rate
B4A-018 active_valid_coupons
B4A-023 purchasing_customers_by_region
B4A-030 view_to_purchase_conversion_rate

CELL 16 COMPLETED SUCCESSFULLY
Translation leakage possible: False
Final test locked: True
Final benchmark generations performed: 0


In [21]:
# ============================================================
# CELL 17 — BUILD THE 120-QUESTION MULTILINGUAL BENCHMARK
# ============================================================

import pandas as pd
import json
import re

FRENCH_QUESTIONS = [
    "Quel est le chiffre d'affaires reconnu des commandes par région client ?",
    "Quel est le chiffre d'affaires reconnu des commandes pour chaque mois ?",
    "Quelle est la valeur brute des marchandises des commandes non annulées ?",
    "Quel est le montant net des ventes de marchandises pour les commandes reconnues ?",
    "Quelle est la valeur moyenne des commandes pour les ventes reconnues ?",
    "Combien de commandes existe-t-il dans chaque statut ?",
    "Combien de commandes se trouvent actuellement dans le carnet opérationnel ?",
    "Quel pourcentage des commandes a été annulé ?",
    "Combien de commandes ont le statut remboursé ?",
    "Quel est le volume quotidien des commandes pendant août 2026 ?",
    "Quel est le montant payé collecté par mode de paiement ?",
    "Quel est le montant restant pour les paiements en attente ?",
    "Quel pourcentage des enregistrements de paiement a échoué ?",
    "Quel est le montant total associé aux paiements remboursés ?",
    "Combien de paiements payés ne correspondent pas au total de leur commande ?",
    "Combien de commandes ont utilisé chaque coupon ?",
    "Quel montant de remise reconnu est associé à chaque coupon ?",
    "Quels coupons sont actifs et valides à la date de référence du rapport ?",
    "Combien de produits existe-t-il dans chaque statut du catalogue ?",
    "Quels produits actifs ont moins de dix unités en stock ?",
    "Quelle est la valeur du stock des produits actifs ?",
    "Quelles sont les ventes reconnues de produits par catégorie ?",
    "Combien de clients acheteurs existe-t-il dans chaque région ?",
    "Combien de clients se sont inscrits chaque mois ?",
    "Combien de clients ont passé plus d'une commande reconnue ?",
    "Quel montant de livraison reconnu est associé à chaque méthode de livraison ?",
    "Combien de commandes ont utilisé chaque méthode de livraison ?",
    "Quel est le taux d'imposition effectif des commandes reconnues ?",
    "Combien d'événements produit ont eu lieu pour chaque type d'événement ?",
    "Quel est le taux de conversion des vues de produits en achats ?",
]

ARABIC_QUESTIONS = [
    "ما قيمة إيرادات الطلبات المعترف بها حسب منطقة العميل؟",
    "ما قيمة إيرادات الطلبات المعترف بها لكل شهر؟",
    "ما القيمة الإجمالية للبضائع في الطلبات غير الملغاة؟",
    "ما صافي مبيعات البضائع للطلبات المعترف بها؟",
    "ما متوسط قيمة الطلب للمبيعات المعترف بها؟",
    "كم عدد الطلبات في كل حالة من حالات الطلب؟",
    "كم عدد الطلبات الموجودة حاليا في قائمة العمليات المتأخرة؟",
    "ما النسبة المئوية للطلبات التي ألغيت؟",
    "كم عدد الطلبات التي تحمل حالة مسترد؟",
    "ما حجم الطلبات اليومي خلال شهر أغسطس 2026؟",
    "ما المبلغ المدفوع المحصل حسب طريقة الدفع؟",
    "ما المبلغ المستحق للمدفوعات المعلقة؟",
    "ما النسبة المئوية لسجلات الدفع الفاشلة؟",
    "ما إجمالي المبلغ المرتبط بالمدفوعات المستردة؟",
    "كم عدد المدفوعات المسددة التي لا تطابق إجمالي الطلب المرتبط بها؟",
    "كم عدد الطلبات التي استخدمت كل قسيمة؟",
    "ما مبلغ الخصم المعترف به المرتبط بكل قسيمة؟",
    "ما القسائم النشطة والصالحة في التاريخ المرجعي للتقرير؟",
    "كم عدد المنتجات في كل حالة من حالات الكتالوج؟",
    "ما المنتجات النشطة التي تحتوي على أقل من عشر وحدات في المخزون؟",
    "ما قيمة مخزون المنتجات النشطة؟",
    "ما قيمة مبيعات المنتجات المعترف بها حسب الفئة؟",
    "كم عدد العملاء المشترين في كل منطقة؟",
    "كم عدد العملاء الذين سجلوا في كل شهر؟",
    "كم عدد العملاء الذين قدموا أكثر من طلب معترف به؟",
    "ما مبلغ الشحن المعترف به المرتبط بكل طريقة شحن؟",
    "كم عدد الطلبات التي استخدمت كل طريقة شحن؟",
    "ما معدل الضريبة الفعلي للطلبات المعترف بها؟",
    "كم عدد أحداث المنتجات التي وقعت لكل نوع من الأحداث؟",
    "ما معدل التحويل من مشاهدة المنتج إلى شرائه؟",
]

ARABIZI_QUESTIONS = [
    "Adde el recognized order revenue 7asab region taba3 el customer?",
    "Adde el recognized order revenue bi kel chaher?",
    "Chou hiye el gross merchandise value lal orders li mesh cancelled?",
    "Adde net merchandise sales lal recognized orders?",
    "Chou howwe average order value lal recognized sales?",
    "Adde fi orders bi kel order status?",
    "Adde fi orders halla2 bel operational backlog?",
    "Adde nesbet el orders li tcancellet?",
    "Adde fi orders status taba3a refunded?",
    "Chou howwe daily order volume bi August 2026?",
    "Adde el paid amount el collected 7asab payment method?",
    "Adde el outstanding amount lal pending payments?",
    "Adde nesbet payment records li failed?",
    "Adde total amount taba3 el refunded payments?",
    "Adde fi paid payments ma byet2ab2o ma3 total taba3 el order?",
    "Adde orders esta3malo kel coupon?",
    "Adde el recognized discount amount taba3 kel coupon?",
    "Aya coupons active w valid bi reporting reference date?",
    "Adde fi products bi kel catalog status?",
    "Aya active products 3enda stock a2al men 10 units?",
    "Adde inventory value taba3 el active products?",
    "Adde el recognized product sales 7asab category?",
    "Adde fi purchasing customers bi kel region?",
    "Adde customers tsajjalo bi kel chaher?",
    "Adde customers 3emlo aktar men recognized order wa7ad?",
    "Adde el recognized shipping amount taba3 kel shipping method?",
    "Adde orders esta3malo kel shipping method?",
    "Chou howwe effective tax rate lal recognized orders?",
    "Adde product events saro la kel event type?",
    "Chou howwe product view-to-purchase conversion rate?",
]


# ------------------------------------------------------------
# 1. Validate translation counts
# ------------------------------------------------------------

assert len(FRENCH_QUESTIONS) == 30
assert len(ARABIC_QUESTIONS) == 30
assert len(ARABIZI_QUESTIONS) == 30
assert len(validated_gold_intents) == 30


# ------------------------------------------------------------
# 2. Assemble all four language versions
# ------------------------------------------------------------

benchmark_records = []

for index, intent in enumerate(
    validated_gold_intents
):

    language_questions = {
        "english": (
            intent["canonical_question"]
        ),
        "arabic": (
            ARABIC_QUESTIONS[index]
        ),
        "french": (
            FRENCH_QUESTIONS[index]
        ),
        "lebanese_arabizi": (
            ARABIZI_QUESTIONS[index]
        ),
    }

    for language, question in (
        language_questions.items()
    ):
        benchmark_records.append({
            "question_id": (
                f"{intent['intent_id']}-"
                f"{language.upper()}"
            ),
            "intent_id": intent["intent_id"],
            "intent_name": intent["intent_name"],
            "domain": intent["domain"],
            "requires_policy": (
                intent["requires_policy"]
            ),
            "language": language,
            "question": question,
            "split": get_intent_split(
                intent["intent_id"]
            ),
            "gold_sql": intent["gold_sql"],
            "gold_columns": (
                intent["gold_columns"]
            ),
            "gold_rows": intent["gold_rows"],
        })


multilingual_benchmark_df = pd.DataFrame(
    benchmark_records
)


# ------------------------------------------------------------
# 3. Quality and leakage validation
# ------------------------------------------------------------

arabic_pattern = re.compile(
    r"[\u0600-\u06FF]"
)

arabic_rows = multilingual_benchmark_df[
    multilingual_benchmark_df["language"]
    == "arabic"
]

arabizi_rows = multilingual_benchmark_df[
    multilingual_benchmark_df["language"]
    == "lebanese_arabizi"
]

translation_split_counts = (
    multilingual_benchmark_df
    .groupby("intent_id")["split"]
    .nunique()
)

language_counts = (
    multilingual_benchmark_df[
        "language"
    ].value_counts()
)

split_question_counts = (
    multilingual_benchmark_df[
        "split"
    ].value_counts()
)

assert len(multilingual_benchmark_df) == 120
assert multilingual_benchmark_df[
    "question_id"
].is_unique

assert multilingual_benchmark_df[
    "question"
].str.strip().ne("").all()

assert all(
    count == 30
    for count in language_counts.values
)

assert all(
    count == 1
    for count in translation_split_counts.values
)

assert arabic_rows["question"].apply(
    lambda text: bool(
        arabic_pattern.search(text)
    )
).all()

assert not arabizi_rows["question"].apply(
    lambda text: bool(
        arabic_pattern.search(text)
    )
).any()

assert split_question_counts.to_dict() == {
    "train": 72,
    "validation": 24,
    "final_test": 24,
}


# ------------------------------------------------------------
# 4. Save benchmark
# ------------------------------------------------------------

benchmark_csv_path = (
    PATHS["data"]
    / "multilingual_low_scale_benchmark.csv"
)

benchmark_json_path = (
    PATHS["data"]
    / "multilingual_low_scale_benchmark.json"
)

csv_columns = [
    "question_id",
    "intent_id",
    "intent_name",
    "domain",
    "requires_policy",
    "language",
    "question",
    "split",
    "gold_sql",
]

multilingual_benchmark_df[
    csv_columns
].to_csv(
    benchmark_csv_path,
    index=False,
    encoding="utf-8-sig",
)

benchmark_json_path.write_text(
    json.dumps(
        benchmark_records,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 5. Display audit
# ------------------------------------------------------------

print("=" * 70)
print("MULTILINGUAL LOW-SCALE BENCHMARK")
print("=" * 70)

print(
    "Total questions:",
    len(multilingual_benchmark_df),
)

print("\nQUESTIONS BY LANGUAGE")
print(language_counts.to_string())

print("\nQUESTIONS BY SPLIT")
print(split_question_counts.to_string())

print(
    "\nUnique intents:",
    multilingual_benchmark_df[
        "intent_id"
    ].nunique(),
)

print(
    "Translations crossing splits:",
    int(
        (
            translation_split_counts > 1
        ).sum()
    ),
)

print("\nQUESTION PREVIEW")

print(
    multilingual_benchmark_df[
        [
            "question_id",
            "language",
            "split",
            "question",
        ]
    ].head(8).to_string(index=False)
)

print("\nCELL 17 COMPLETED SUCCESSFULLY")
print("Benchmark saved:", benchmark_csv_path)
print("Final test locked: True")
print("Qwen benchmark generations performed: 0")

MULTILINGUAL LOW-SCALE BENCHMARK
Total questions: 120

QUESTIONS BY LANGUAGE
language
english             30
arabic              30
french              30
lebanese_arabizi    30

QUESTIONS BY SPLIT
split
train         72
validation    24
final_test    24

Unique intents: 30
Translations crossing splits: 0

QUESTION PREVIEW
             question_id         language split                                                                 question
         B4A-001-ENGLISH          english train                 What is the recognized order revenue by customer region?
          B4A-001-ARABIC           arabic train                    ما قيمة إيرادات الطلبات المعترف بها حسب منطقة العميل؟
          B4A-001-FRENCH           french train Quel est le chiffre d'affaires reconnu des commandes par région client ?
B4A-001-LEBANESE_ARABIZI lebanese_arabizi train         Adde el recognized order revenue 7asab region taba3 el customer?
         B4A-002-ENGLISH          english train                     Wh

In [22]:
# ================================================================
# CELL 18 — EXECUTION ACCURACY EVALUATION FRAMEWORK
# ================================================================

from decimal import Decimal
from datetime import date, datetime
import json
import math
import os
import pandas as pd


# ------------------------------------------------
# 1. Normalize database values
# ------------------------------------------------
def normalize_evaluation_value(value):
    if value is None:
        return None

    if isinstance(value, bool):
        return value

    if isinstance(value, Decimal):
        if value == value.to_integral():
            return str(value.quantize(Decimal("1")))

        normalized = format(value.normalize(), "f")
        return normalized.rstrip("0").rstrip(".") if "." in normalized else normalized

    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"

        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"

        return format(value, ".10g")

    if isinstance(value, (datetime, date)):
        return value.isoformat()

    if isinstance(value, (int, str)):
        return str(value)

    return str(value)


def normalize_evaluation_rows(rows):
    normalized_rows = []

    for row in rows:
        normalized_row = tuple(
            normalize_evaluation_value(value)
            for value in row
        )
        normalized_rows.append(normalized_row)

    # Row order should not affect execution accuracy.
    return sorted(
        normalized_rows,
        key=lambda row: json.dumps(
            row,
            ensure_ascii=False,
            sort_keys=True,
            default=str
        )
    )


# ------------------------------------------------
# 2. Execute evaluated SQL safely
# ------------------------------------------------
def execute_for_evaluation(sql):
    validation = validate_readonly_sql(sql)

    if not validation["safe"]:
        return {
            "executed": False,
            "safe": False,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": None
        }

    connection = None

    try:
        connection = get_db_connection()

        with connection.cursor() as cursor:
            cursor.execute("SET statement_timeout TO 30000")
            cursor.execute(sql)

            columns = [
                description.name
                for description in cursor.description
            ]

            rows = cursor.fetchall()

        connection.rollback()

        return {
            "executed": True,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": columns,
            "rows": rows,
            "error": None
        }

    except Exception as error:
        if connection is not None:
            connection.rollback()

        return {
            "executed": False,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": f"{type(error).__name__}: {error}"
        }

    finally:
        if connection is not None:
            connection.close()


# ------------------------------------------------
# 3. Compare generated result with gold result
# ------------------------------------------------
def evaluate_generated_sql(generated_sql, gold_rows):
    execution = execute_for_evaluation(generated_sql)

    predicted_rows = normalize_evaluation_rows(
        execution["rows"]
    )

    expected_rows = normalize_evaluation_rows(
        gold_rows
    )

    same_column_count = True

    if predicted_rows and expected_rows:
        same_column_count = (
            len(predicted_rows[0]) ==
            len(expected_rows[0])
        )

    execution_correct = (
        execution["executed"]
        and same_column_count
        and predicted_rows == expected_rows
    )

    return {
        "execution_correct": execution_correct,
        "sql_safe": execution["safe"],
        "sql_executed": execution["executed"],
        "execution_error": execution["error"],
        "predicted_row_count": len(predicted_rows),
        "gold_row_count": len(expected_rows),
        "predicted_rows": predicted_rows,
        "gold_rows": expected_rows
    }


# ------------------------------------------------
# 4. Validate evaluator using every gold query
# ------------------------------------------------
required_columns = {
    "question_id",
    "intent_id",
    "language",
    "split",
    "question",
    "gold_sql",
    "gold_rows"
}

missing_columns = (
    required_columns -
    set(multilingual_benchmark_df.columns)
)

assert not missing_columns, (
    f"Benchmark is missing columns: {sorted(missing_columns)}"
)

gold_self_validation = []

for intent_id, intent_group in multilingual_benchmark_df.groupby(
    "intent_id",
    sort=True
):
    benchmark_row = intent_group.iloc[0]

    result = evaluate_generated_sql(
        benchmark_row["gold_sql"],
        benchmark_row["gold_rows"]
    )

    gold_self_validation.append({
        "intent_id": intent_id,
        "execution_correct": result["execution_correct"],
        "sql_safe": result["sql_safe"],
        "sql_executed": result["sql_executed"],
        "execution_error": result["execution_error"],
        "predicted_row_count": result["predicted_row_count"],
        "gold_row_count": result["gold_row_count"]
    })

gold_self_validation_df = pd.DataFrame(
    gold_self_validation
)

failed_gold_checks = gold_self_validation_df[
    ~gold_self_validation_df["execution_correct"]
]

assert failed_gold_checks.empty, (
    "Gold self-validation failed:\n"
    + failed_gold_checks.to_string(index=False)
)


# ------------------------------------------------
# 5. Save evaluator audit
# ------------------------------------------------
evaluation_audit_path = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "data",
    "execution_evaluator_audit.csv"
)

gold_self_validation_df.to_csv(
    evaluation_audit_path,
    index=False
)


# ------------------------------------------------
# 6. Final audit
# ------------------------------------------------
print("=" * 70)
print("EXECUTION ACCURACY EVALUATOR")
print("=" * 70)
print("Canonical intents tested:", len(gold_self_validation_df))
print(
    "Safe gold SQL:",
    int(gold_self_validation_df["sql_safe"].sum())
)
print(
    "Executable gold SQL:",
    int(gold_self_validation_df["sql_executed"].sum())
)
print(
    "Exact result matches:",
    int(gold_self_validation_df["execution_correct"].sum())
)
print("Column aliases used for scoring: False")
print("Row order affects scoring: False")
print("Database values normalized: True")
print("Failed checks:", len(failed_gold_checks))
print()
print("CELL 18 COMPLETED SUCCESSFULLY")
print("Execution evaluator ready: True")
print("Evaluator audit saved:", evaluation_audit_path)
print("Qwen benchmark generations performed: 0")
print("Final test opened: False")

EXECUTION ACCURACY EVALUATOR
Canonical intents tested: 30
Safe gold SQL: 30
Executable gold SQL: 30
Exact result matches: 30
Column aliases used for scoring: False
Row order affects scoring: False
Database values normalized: True
Failed checks: 0

CELL 18 COMPLETED SUCCESSFULLY
Execution evaluator ready: True
Evaluator audit saved: /kaggle/working/build4all_final_experiment/data/execution_evaluator_audit.csv
Qwen benchmark generations performed: 0
Final test opened: False


In [23]:
# Find the SQL validation function created in earlier cells
validation_functions = sorted([
    name
    for name, value in globals().items()
    if callable(value)
    and ("valid" in name.lower() or "safe" in name.lower())
])

print("Available validation functions:")
for name in validation_functions:
    print("-", name)

Available validation functions:
- execute_safe_sql
- validate_readonly_sql


In [24]:
import inspect

pipeline_functions = sorted([
    name
    for name, value in globals().items()
    if callable(value)
    and any(
        keyword in name.lower()
        for keyword in ["qwen", "generate", "rag", "retrieve"]
    )
])

print("Available pipeline functions:")

for name in pipeline_functions:
    function = globals()[name]

    try:
        signature = inspect.signature(function)
    except Exception:
        signature = "(signature unavailable)"

    print(f"- {name}{signature}")

Available pipeline functions:
- build_rag_context(question)
- evaluate_generated_sql(generated_sql, gold_rows)
- generate_sql(question, use_rag=False)
- retrieve_chunks(question, top_k=None)


In [25]:
# ================================================================
# CELL 19 — QWEN ALONE TRAIN + VALIDATION BENCHMARK
# ================================================================

import json
import os
import time
import pandas as pd


BENCHMARK_ROOT = "/kaggle/working/build4all_final_experiment"

qwen_alone_checkpoint_path = os.path.join(
    BENCHMARK_ROOT,
    "results",
    "qwen_alone_train_validation.jsonl"
)

qwen_alone_csv_path = os.path.join(
    BENCHMARK_ROOT,
    "results",
    "qwen_alone_train_validation.csv"
)

os.makedirs(
    os.path.dirname(qwen_alone_checkpoint_path),
    exist_ok=True
)


# ------------------------------------------------
# 1. Use train and validation only
# ------------------------------------------------
qwen_alone_benchmark_df = multilingual_benchmark_df[
    multilingual_benchmark_df["split"].isin(
        ["train", "validation"]
    )
].copy()

assert len(qwen_alone_benchmark_df) == 96
assert "final_test" not in qwen_alone_benchmark_df["split"].unique()


# ------------------------------------------------
# 2. Read completed checkpoint rows
# ------------------------------------------------
completed_results = []
completed_question_ids = set()

if os.path.exists(qwen_alone_checkpoint_path):
    with open(
        qwen_alone_checkpoint_path,
        "r",
        encoding="utf-8"
    ) as checkpoint_file:
        for line in checkpoint_file:
            line = line.strip()

            if not line:
                continue

            try:
                saved_result = json.loads(line)
                completed_results.append(saved_result)
                completed_question_ids.add(
                    saved_result["question_id"]
                )
            except json.JSONDecodeError:
                print(
                    "Ignored one incomplete checkpoint line."
                )

print("=" * 70)
print("QWEN ALONE — TRAIN + VALIDATION BENCHMARK")
print("=" * 70)
print("Target questions:", len(qwen_alone_benchmark_df))
print("Previously completed:", len(completed_question_ids))
print("Remaining:", (
    len(qwen_alone_benchmark_df) -
    len(completed_question_ids)
))
print("RAG used: False")
print("Final test included: False")
print()


# ------------------------------------------------
# 3. Flexible generation-output reader
# ------------------------------------------------
def unpack_generation_output(generation_output):
    if isinstance(generation_output, str):
        return {
            "generated_sql": generation_output,
            "raw_response": generation_output,
            "reported_latency": None
        }

    if not isinstance(generation_output, dict):
        raise TypeError(
            "generate_sql returned an unsupported type: "
            f"{type(generation_output).__name__}"
        )

    generated_sql = None

    for key in [
        "sql",
        "generated_sql",
        "extracted_sql"
    ]:
        if generation_output.get(key):
            generated_sql = generation_output[key]
            break

    if generated_sql is None:
        raise KeyError(
            "No generated SQL field was found. "
            f"Available keys: {list(generation_output.keys())}"
        )

    raw_response = generation_output.get(
        "raw_response",
        generation_output.get(
            "response",
            generated_sql
        )
    )

    reported_latency = generation_output.get(
        "generation_latency",
        generation_output.get(
            "latency_seconds",
            generation_output.get("latency")
        )
    )

    return {
        "generated_sql": generated_sql,
        "raw_response": raw_response,
        "reported_latency": reported_latency
    }


# ------------------------------------------------
# 4. Run Qwen Alone with checkpointing
# ------------------------------------------------
total_questions = len(qwen_alone_benchmark_df)

for position, (_, benchmark_row) in enumerate(
    qwen_alone_benchmark_df.iterrows(),
    start=1
):
    question_id = benchmark_row["question_id"]

    if question_id in completed_question_ids:
        continue

    started_at = time.perf_counter()

    try:
        generation_output = generate_sql(
            benchmark_row["question"],
            use_rag=False
        )

        measured_latency = (
            time.perf_counter() -
            started_at
        )

        unpacked_output = unpack_generation_output(
            generation_output
        )

        generated_sql = unpacked_output["generated_sql"]

        evaluation = evaluate_generated_sql(
            generated_sql,
            benchmark_row["gold_rows"]
        )

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": benchmark_row["split"],
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Qwen Alone",
            "rag_used": False,
            "generated_sql": generated_sql,
            "raw_response": str(
                unpacked_output["raw_response"]
            ),
            "generation_latency_seconds": (
                float(
                    unpacked_output["reported_latency"]
                )
                if unpacked_output["reported_latency"]
                is not None
                else float(measured_latency)
            ),
            "sql_safe": bool(
                evaluation["sql_safe"]
            ),
            "sql_executed": bool(
                evaluation["sql_executed"]
            ),
            "execution_correct": bool(
                evaluation["execution_correct"]
            ),
            "execution_error": evaluation[
                "execution_error"
            ],
            "predicted_row_count": int(
                evaluation["predicted_row_count"]
            ),
            "gold_row_count": int(
                evaluation["gold_row_count"]
            ),
            "benchmark_error": None
        }

    except Exception as error:
        measured_latency = (
            time.perf_counter() -
            started_at
        )

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": benchmark_row["split"],
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Qwen Alone",
            "rag_used": False,
            "generated_sql": None,
            "raw_response": None,
            "generation_latency_seconds": float(
                measured_latency
            ),
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": None,
            "predicted_row_count": 0,
            "gold_row_count": len(
                benchmark_row["gold_rows"]
            ),
            "benchmark_error": (
                f"{type(error).__name__}: {error}"
            )
        }

    with open(
        qwen_alone_checkpoint_path,
        "a",
        encoding="utf-8"
    ) as checkpoint_file:
        checkpoint_file.write(
            json.dumps(
                result_record,
                ensure_ascii=False,
                default=str
            ) + "\n"
        )

    completed_results.append(result_record)
    completed_question_ids.add(question_id)

    correct_so_far = sum(
        bool(result["execution_correct"])
        for result in completed_results
    )

    completed_count = len(completed_results)
    running_accuracy = (
        correct_so_far / completed_count
    )

    print(
        f"[{completed_count:03d}/{total_questions}] "
        f"{question_id} | "
        f"{benchmark_row['language']} | "
        f"correct={result_record['execution_correct']} | "
        f"running_accuracy={running_accuracy:.2%} | "
        f"latency={result_record['generation_latency_seconds']:.2f}s"
    )


# ------------------------------------------------
# 5. Build final results table
# ------------------------------------------------
qwen_alone_results_df = pd.DataFrame(
    completed_results
)

qwen_alone_results_df = (
    qwen_alone_results_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last"
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(qwen_alone_results_df) == 96

qwen_alone_results_df.to_csv(
    qwen_alone_csv_path,
    index=False
)


# ------------------------------------------------
# 6. Accuracy summaries
# ------------------------------------------------
overall_accuracy = (
    qwen_alone_results_df[
        "execution_correct"
    ].mean()
)

accuracy_by_split = (
    qwen_alone_results_df
    .groupby("split")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

accuracy_by_language = (
    qwen_alone_results_df
    .groupby("language")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

safe_rate = qwen_alone_results_df[
    "sql_safe"
].mean()

execution_rate = qwen_alone_results_df[
    "sql_executed"
].mean()

average_latency = qwen_alone_results_df[
    "generation_latency_seconds"
].mean()

benchmark_error_count = (
    qwen_alone_results_df[
        "benchmark_error"
    ].notna().sum()
)


# ------------------------------------------------
# 7. Final report
# ------------------------------------------------
print()
print("=" * 70)
print("QWEN ALONE — TRAIN + VALIDATION RESULTS")
print("=" * 70)
print(
    f"Overall execution accuracy: "
    f"{overall_accuracy:.2%}"
)
print(f"Correct questions: {
    int(qwen_alone_results_df['execution_correct'].sum())
}/{len(qwen_alone_results_df)}")
print(f"Safe SQL rate: {safe_rate:.2%}")
print(f"Execution success rate: {execution_rate:.2%}")
print(f"Average generation latency: {average_latency:.4f} seconds")
print("Benchmark errors:", benchmark_error_count)

print()
print("ACCURACY BY SPLIT")
print(accuracy_by_split.to_string())

print()
print("ACCURACY BY LANGUAGE")
print(accuracy_by_language.to_string())

print()
print("CELL 19 COMPLETED SUCCESSFULLY")
print("Qwen Alone benchmark generations:", len(qwen_alone_results_df))
print("RAG used: False")
print("Final test opened: False")
print("Results saved:", qwen_alone_csv_path)
print("Checkpoint saved:", qwen_alone_checkpoint_path)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


QWEN ALONE — TRAIN + VALIDATION BENCHMARK
Target questions: 96
Previously completed: 0
Remaining: 96
RAG used: False
Final test included: False

[001/96] B4A-001-ENGLISH | english | correct=False | running_accuracy=0.00% | latency=11.04s
[002/96] B4A-001-ARABIC | arabic | correct=False | running_accuracy=0.00% | latency=8.51s
[003/96] B4A-001-FRENCH | french | correct=False | running_accuracy=0.00% | latency=7.80s
[004/96] B4A-001-LEBANESE_ARABIZI | lebanese_arabizi | correct=False | running_accuracy=0.00% | latency=8.92s
[005/96] B4A-002-ENGLISH | english | correct=False | running_accuracy=0.00% | latency=8.96s
[006/96] B4A-002-ARABIC | arabic | correct=False | running_accuracy=0.00% | latency=10.38s
[007/96] B4A-002-FRENCH | french | correct=False | running_accuracy=0.00% | latency=10.47s
[008/96] B4A-002-LEBANESE_ARABIZI | lebanese_arabizi | correct=False | running_accuracy=0.00% | latency=7.65s
[009/96] B4A-003-ENGLISH | english | correct=False | running_accuracy=0.00% | latency=4.

In [26]:
from pathlib import Path
from IPython.display import HTML, display
import base64
import pandas as pd

csv_path = Path(
    "/kaggle/working/build4all_final_experiment/results/"
    "qwen_alone_train_validation.csv"
)

if not csv_path.exists():
    print("Entore Cell 19 y5las — el CSV ba3do mch jehez.")
else:
    df = pd.read_csv(csv_path)

    if len(df) != 96 or df["question_id"].nunique() != 96:
        print(f"El results ba3da mch kemle: {len(df)}/96 rows.")
    else:
        encoded = base64.b64encode(csv_path.read_bytes()).decode("ascii")

        display(HTML(f"""
        <a href="data:text/csv;base64,{encoded}"
           download="qwen_alone_train_validation.csv"
           style="display:inline-block;background:#168c8d;color:white;
                  padding:15px 25px;border-radius:10px;
                  font-size:17px;font-weight:bold;text-decoration:none;">
            ⬇ Download Qwen Alone Results — 96 Questions
        </a>
        """))

        print("Ready: 96 questions. Ekbse el button la download.")

Ready: 96 questions. Ekbse el button la download.


In [24]:
# ================================================================
# CELL 20 — QWEN ALONE ERROR DIAGNOSIS
# ================================================================

import pandas as pd

diagnostic_df = qwen_alone_results_df.copy()

diagnostic_df["failure_type"] = "correct"

diagnostic_df.loc[
    diagnostic_df["benchmark_error"].notna(),
    "failure_type"
] = "benchmark_error"

diagnostic_df.loc[
    diagnostic_df["benchmark_error"].isna()
    & ~diagnostic_df["sql_safe"],
    "failure_type"
] = "unsafe_sql"

diagnostic_df.loc[
    diagnostic_df["benchmark_error"].isna()
    & diagnostic_df["sql_safe"]
    & ~diagnostic_df["sql_executed"],
    "failure_type"
] = "execution_error"

diagnostic_df.loc[
    diagnostic_df["benchmark_error"].isna()
    & diagnostic_df["sql_executed"]
    & ~diagnostic_df["execution_correct"],
    "failure_type"
] = "executed_wrong_result"


print("=" * 70)
print("QWEN ALONE ERROR DIAGNOSIS")
print("=" * 70)

print("\nFAILURE TYPES")
print(
    diagnostic_df["failure_type"]
    .value_counts()
    .to_string()
)

print("\nRESULTS BY INTENT")
intent_diagnosis = (
    diagnostic_df
    .groupby(["intent_id", "intent_name"], dropna=False)
    .agg(
        correct=("execution_correct", "sum"),
        total=("question_id", "count"),
        executed=("sql_executed", "sum")
    )
    .reset_index()
)

intent_diagnosis["accuracy"] = (
    intent_diagnosis["correct"]
    / intent_diagnosis["total"]
)

print(intent_diagnosis.to_string(index=False))


print("\nEXECUTION ERRORS")
execution_errors = diagnostic_df[
    diagnostic_df["failure_type"] == "execution_error"
][[
    "question_id",
    "language",
    "generated_sql",
    "execution_error"
]]

if execution_errors.empty:
    print("None")
else:
    print(execution_errors.to_string(index=False))


print("\nFIRST 12 EXECUTED BUT WRONG QUERIES")

wrong_result_examples = diagnostic_df[
    diagnostic_df["failure_type"] ==
    "executed_wrong_result"
][[
    "question_id",
    "intent_id",
    "language",
    "question",
    "generated_sql"
]].head(12)

print(
    wrong_result_examples.to_string(
        index=False
    )
)


diagnostic_path = (
    "/kaggle/working/build4all_final_experiment/"
    "results/qwen_alone_error_diagnosis.csv"
)

diagnostic_df.to_csv(
    diagnostic_path,
    index=False
)

print("\nCELL 20 COMPLETED SUCCESSFULLY")
print("New generations performed: 0")
print("Final test opened: False")
print("Diagnosis saved:", diagnostic_path)

QWEN ALONE ERROR DIAGNOSIS

FAILURE TYPES
failure_type
executed_wrong_result    54
correct                  27
execution_error          15

RESULTS BY INTENT
intent_id                          intent_name  correct  total  executed  accuracy
  B4A-001         recognized_revenue_by_region        0      4         4      0.00
  B4A-002           monthly_recognized_revenue        0      4         3      0.00
  B4A-003              gross_merchandise_value        1      4         3      0.25
  B4A-004                net_merchandise_sales        0      4         4      0.00
  B4A-006                     orders_by_status        3      4         4      0.75
  B4A-007            operational_order_backlog        0      4         4      0.00
  B4A-008              order_cancellation_rate        2      4         4      0.50
  B4A-009                 refunded_order_count        4      4         4      1.00
  B4A-011           paid_collections_by_method        4      4         4      1.00
  B4A-012   

In [25]:
# ================================================================
# CELL 21 — FIX SEARCH PATH AND RESCORE QWEN ALONE
# ================================================================

import json
import os
import pandas as pd


# ------------------------------------------------
# 1. Correct evaluation execution environment
# ------------------------------------------------
def execute_for_evaluation(sql):
    validation = validate_readonly_sql(sql)

    if not validation["safe"]:
        return {
            "executed": False,
            "safe": False,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": None
        }

    connection = None

    try:
        connection = get_db_connection()

        with connection.cursor() as cursor:
            cursor.execute(
                "SET statement_timeout TO 30000"
            )

            # Allow both:
            # FROM orders
            # FROM build4all_research.orders
            cursor.execute(
                "SET search_path TO "
                "build4all_research, public"
            )

            cursor.execute(sql)

            columns = [
                description.name
                for description in cursor.description
            ]

            rows = cursor.fetchall()

        connection.rollback()

        return {
            "executed": True,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": columns,
            "rows": rows,
            "error": None
        }

    except Exception as error:
        if connection is not None:
            connection.rollback()

        return {
            "executed": False,
            "safe": True,
            "validation_reason": validation.get("reason"),
            "columns": [],
            "rows": [],
            "error": f"{type(error).__name__}: {error}"
        }

    finally:
        if connection is not None:
            connection.close()


# ------------------------------------------------
# 2. Match every result with benchmark gold rows
# ------------------------------------------------
benchmark_lookup = (
    multilingual_benchmark_df
    .set_index("question_id")
)

rescored_records = []

for _, saved_result in qwen_alone_results_df.iterrows():
    question_id = saved_result["question_id"]
    generated_sql = saved_result["generated_sql"]

    updated_result = saved_result.to_dict()

    if (
        generated_sql is None
        or pd.isna(generated_sql)
        or not str(generated_sql).strip()
    ):
        updated_result.update({
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": (
                "No generated SQL available."
            ),
            "predicted_row_count": 0
        })

    else:
        gold_rows = benchmark_lookup.loc[
            question_id,
            "gold_rows"
        ]

        evaluation = evaluate_generated_sql(
            str(generated_sql),
            gold_rows
        )

        updated_result.update({
            "sql_safe": bool(
                evaluation["sql_safe"]
            ),
            "sql_executed": bool(
                evaluation["sql_executed"]
            ),
            "execution_correct": bool(
                evaluation["execution_correct"]
            ),
            "execution_error": evaluation[
                "execution_error"
            ],
            "predicted_row_count": int(
                evaluation["predicted_row_count"]
            ),
            "gold_row_count": int(
                evaluation["gold_row_count"]
            )
        })

    rescored_records.append(updated_result)


qwen_alone_results_df = pd.DataFrame(
    rescored_records
).sort_values(
    "question_id"
).reset_index(drop=True)


# ------------------------------------------------
# 3. Replace flawed checkpoint with corrected scores
# ------------------------------------------------
qwen_alone_results_df.to_csv(
    qwen_alone_csv_path,
    index=False
)

with open(
    qwen_alone_checkpoint_path,
    "w",
    encoding="utf-8"
) as checkpoint_file:
    for result_record in qwen_alone_results_df.to_dict(
        orient="records"
    ):
        checkpoint_file.write(
            json.dumps(
                result_record,
                ensure_ascii=False,
                default=str
            ) + "\n"
        )


# ------------------------------------------------
# 4. Corrected summaries
# ------------------------------------------------
corrected_accuracy = qwen_alone_results_df[
    "execution_correct"
].mean()

corrected_execution_rate = qwen_alone_results_df[
    "sql_executed"
].mean()

corrected_by_split = (
    qwen_alone_results_df
    .groupby("split")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

corrected_by_language = (
    qwen_alone_results_df
    .groupby("language")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

remaining_execution_errors = (
    ~qwen_alone_results_df["sql_executed"]
).sum()


print("=" * 70)
print("QWEN ALONE — CORRECTED RESCORING")
print("=" * 70)
print("New Qwen generations performed: 0")
print("Previously reported accuracy: 28.12%")
print(
    f"Corrected execution accuracy: "
    f"{corrected_accuracy:.2%}"
)
print(
    "Correct questions:",
    int(
        qwen_alone_results_df[
            "execution_correct"
        ].sum()
    ),
    "/",
    len(qwen_alone_results_df)
)
print(
    f"Corrected execution success rate: "
    f"{corrected_execution_rate:.2%}"
)
print(
    "Remaining execution errors:",
    int(remaining_execution_errors)
)

print()
print("CORRECTED ACCURACY BY SPLIT")
print(corrected_by_split.to_string())

print()
print("CORRECTED ACCURACY BY LANGUAGE")
print(corrected_by_language.to_string())

print()
print("CELL 21 COMPLETED SUCCESSFULLY")
print("Search path corrected: True")
print("Scores recomputed from existing SQL: True")
print("Final test opened: False")
print("Results updated:", qwen_alone_csv_path)
print("Checkpoint updated:", qwen_alone_checkpoint_path)

QWEN ALONE — CORRECTED RESCORING
New Qwen generations performed: 0
Previously reported accuracy: 28.12%
Corrected execution accuracy: 32.29%
Correct questions: 31 / 96
Corrected execution success rate: 100.00%
Remaining execution errors: 0

CORRECTED ACCURACY BY SPLIT
            sum  count      mean
split                           
train        22     72  0.305556
validation    9     24  0.375000

CORRECTED ACCURACY BY LANGUAGE
                  sum  count      mean
language                              
arabic              9     24  0.375000
english             8     24  0.333333
french             10     24  0.416667
lebanese_arabizi    4     24  0.166667

CELL 21 COMPLETED SUCCESSFULLY
Search path corrected: True
Scores recomputed from existing SQL: True
Final test opened: False
Results updated: /kaggle/working/build4all_final_experiment/results/qwen_alone_train_validation.csv
Checkpoint updated: /kaggle/working/build4all_final_experiment/results/qwen_alone_train_validation.jsonl


In [26]:
# ================================================================
# CELL 22 — QWEN + RAG TRAIN + VALIDATION BENCHMARK
# ================================================================

import json
import os
import time
import pandas as pd


BENCHMARK_ROOT = "/kaggle/working/build4all_final_experiment"

rag_checkpoint_path = os.path.join(
    BENCHMARK_ROOT,
    "results",
    "qwen_rag_train_validation.jsonl"
)

rag_csv_path = os.path.join(
    BENCHMARK_ROOT,
    "results",
    "qwen_rag_train_validation.csv"
)

os.makedirs(
    os.path.dirname(rag_checkpoint_path),
    exist_ok=True
)


# ------------------------------------------------
# 1. Train + validation only
# ------------------------------------------------
rag_benchmark_df = multilingual_benchmark_df[
    multilingual_benchmark_df["split"].isin(
        ["train", "validation"]
    )
].copy()

assert len(rag_benchmark_df) == 96
assert "final_test" not in rag_benchmark_df["split"].unique()


# ------------------------------------------------
# 2. Load checkpoint if available
# ------------------------------------------------
rag_completed_results = []
rag_completed_ids = set()

if os.path.exists(rag_checkpoint_path):
    with open(
        rag_checkpoint_path,
        "r",
        encoding="utf-8"
    ) as checkpoint_file:
        for line in checkpoint_file:
            line = line.strip()

            if not line:
                continue

            try:
                saved_result = json.loads(line)
                rag_completed_results.append(saved_result)
                rag_completed_ids.add(
                    saved_result["question_id"]
                )
            except json.JSONDecodeError:
                print(
                    "Ignored one incomplete checkpoint line."
                )


print("=" * 70)
print("QWEN + RAG — TRAIN + VALIDATION BENCHMARK")
print("=" * 70)
print("Target questions:", len(rag_benchmark_df))
print("Previously completed:", len(rag_completed_ids))
print(
    "Remaining:",
    len(rag_benchmark_df) - len(rag_completed_ids)
)
print("RAG used: True")
print("Final test included: False")
print()


# ------------------------------------------------
# 3. Run Qwen + RAG
# ------------------------------------------------
total_questions = len(rag_benchmark_df)

for _, benchmark_row in rag_benchmark_df.iterrows():
    question_id = benchmark_row["question_id"]

    if question_id in rag_completed_ids:
        continue

    started_at = time.perf_counter()

    try:
        generation_output = generate_sql(
            benchmark_row["question"],
            use_rag=True
        )

        measured_latency = (
            time.perf_counter() -
            started_at
        )

        unpacked_output = unpack_generation_output(
            generation_output
        )

        generated_sql = unpacked_output[
            "generated_sql"
        ]

        evaluation = evaluate_generated_sql(
            generated_sql,
            benchmark_row["gold_rows"]
        )

        if isinstance(generation_output, dict):
            retrieved_chunks_output = (
                generation_output.get(
                    "retrieved_chunks",
                    generation_output.get(
                        "retrieval",
                        []
                    )
                )
            )

            rag_context = generation_output.get(
                "rag_context",
                generation_output.get(
                    "context",
                    ""
                )
            )
        else:
            retrieved_chunks_output = []
            rag_context = ""

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": benchmark_row["split"],
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Qwen + RAG",
            "rag_used": True,
            "generated_sql": generated_sql,
            "raw_response": str(
                unpacked_output["raw_response"]
            ),
            "generation_latency_seconds": (
                float(
                    unpacked_output[
                        "reported_latency"
                    ]
                )
                if unpacked_output[
                    "reported_latency"
                ] is not None
                else float(measured_latency)
            ),
            "rag_context_characters": len(
                str(rag_context)
            ),
            "retrieved_chunks": (
                retrieved_chunks_output
            ),
            "sql_safe": bool(
                evaluation["sql_safe"]
            ),
            "sql_executed": bool(
                evaluation["sql_executed"]
            ),
            "execution_correct": bool(
                evaluation["execution_correct"]
            ),
            "execution_error": evaluation[
                "execution_error"
            ],
            "predicted_row_count": int(
                evaluation["predicted_row_count"]
            ),
            "gold_row_count": int(
                evaluation["gold_row_count"]
            ),
            "benchmark_error": None
        }

    except Exception as error:
        measured_latency = (
            time.perf_counter() -
            started_at
        )

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": benchmark_row["split"],
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Qwen + RAG",
            "rag_used": True,
            "generated_sql": None,
            "raw_response": None,
            "generation_latency_seconds": float(
                measured_latency
            ),
            "rag_context_characters": 0,
            "retrieved_chunks": [],
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": None,
            "predicted_row_count": 0,
            "gold_row_count": len(
                benchmark_row["gold_rows"]
            ),
            "benchmark_error": (
                f"{type(error).__name__}: {error}"
            )
        }

    with open(
        rag_checkpoint_path,
        "a",
        encoding="utf-8"
    ) as checkpoint_file:
        checkpoint_file.write(
            json.dumps(
                result_record,
                ensure_ascii=False,
                default=str
            ) + "\n"
        )

    rag_completed_results.append(result_record)
    rag_completed_ids.add(question_id)

    completed_count = len(rag_completed_results)

    correct_so_far = sum(
        bool(result["execution_correct"])
        for result in rag_completed_results
    )

    running_accuracy = (
        correct_so_far / completed_count
    )

    print(
        f"[{completed_count:03d}/{total_questions}] "
        f"{question_id} | "
        f"{benchmark_row['language']} | "
        f"correct={result_record['execution_correct']} | "
        f"running_accuracy={running_accuracy:.2%} | "
        f"latency={result_record['generation_latency_seconds']:.2f}s"
    )


# ------------------------------------------------
# 4. Build and save results
# ------------------------------------------------
qwen_rag_results_df = pd.DataFrame(
    rag_completed_results
)

qwen_rag_results_df = (
    qwen_rag_results_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last"
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(qwen_rag_results_df) == 96

qwen_rag_results_df.to_csv(
    rag_csv_path,
    index=False
)


# ------------------------------------------------
# 5. Summaries
# ------------------------------------------------
rag_accuracy = qwen_rag_results_df[
    "execution_correct"
].mean()

rag_safe_rate = qwen_rag_results_df[
    "sql_safe"
].mean()

rag_execution_rate = qwen_rag_results_df[
    "sql_executed"
].mean()

rag_average_latency = qwen_rag_results_df[
    "generation_latency_seconds"
].mean()

rag_by_split = (
    qwen_rag_results_df
    .groupby("split")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

rag_by_language = (
    qwen_rag_results_df
    .groupby("language")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

qwen_alone_accuracy = qwen_alone_results_df[
    "execution_correct"
].mean()

accuracy_difference = (
    rag_accuracy - qwen_alone_accuracy
)


# ------------------------------------------------
# 6. Final comparison
# ------------------------------------------------
print()
print("=" * 70)
print("QWEN + RAG — TRAIN + VALIDATION RESULTS")
print("=" * 70)
print(
    f"Qwen Alone accuracy: "
    f"{qwen_alone_accuracy:.2%}"
)
print(
    f"Qwen + RAG accuracy: "
    f"{rag_accuracy:.2%}"
)
print(
    f"RAG difference: "
    f"{accuracy_difference:+.2%}"
)
print(
    "Correct questions:",
    int(
        qwen_rag_results_df[
            "execution_correct"
        ].sum()
    ),
    "/",
    len(qwen_rag_results_df)
)
print(f"Safe SQL rate: {rag_safe_rate:.2%}")
print(
    f"Execution success rate: "
    f"{rag_execution_rate:.2%}"
)
print(
    f"Average generation latency: "
    f"{rag_average_latency:.4f} seconds"
)

print()
print("ACCURACY BY SPLIT")
print(rag_by_split.to_string())

print()
print("ACCURACY BY LANGUAGE")
print(rag_by_language.to_string())

print()
print("CELL 22 COMPLETED SUCCESSFULLY")
print(
    "Qwen + RAG benchmark generations:",
    len(qwen_rag_results_df)
)
print("Final test opened: False")
print("Results saved:", rag_csv_path)
print("Checkpoint saved:", rag_checkpoint_path)

QWEN + RAG — TRAIN + VALIDATION BENCHMARK
Target questions: 96
Previously completed: 0
Remaining: 96
RAG used: True
Final test included: False

[001/96] B4A-001-ENGLISH | english | correct=True | running_accuracy=100.00% | latency=13.11s
[002/96] B4A-001-ARABIC | arabic | correct=True | running_accuracy=100.00% | latency=11.54s
[003/96] B4A-001-FRENCH | french | correct=True | running_accuracy=100.00% | latency=11.94s
[004/96] B4A-001-LEBANESE_ARABIZI | lebanese_arabizi | correct=True | running_accuracy=100.00% | latency=10.10s
[005/96] B4A-002-ENGLISH | english | correct=False | running_accuracy=80.00% | latency=13.33s
[006/96] B4A-002-ARABIC | arabic | correct=True | running_accuracy=83.33% | latency=11.99s
[007/96] B4A-002-FRENCH | french | correct=False | running_accuracy=71.43% | latency=14.99s
[008/96] B4A-002-LEBANESE_ARABIZI | lebanese_arabizi | correct=False | running_accuracy=62.50% | latency=7.07s
[009/96] B4A-003-ENGLISH | english | correct=True | running_accuracy=66.67% | 

In [27]:
# ================================================================
# CELL 23 — PAIRWISE COMPLEMENTARITY AND CONTROLLER CEILING
# ================================================================

import os
import pandas as pd


CONTROLLER_ROOT = (
    "/kaggle/working/build4all_final_experiment"
)

controller_analysis_path = os.path.join(
    CONTROLLER_ROOT,
    "results",
    "controller_pairwise_analysis.csv"
)


# ------------------------------------------------
# 1. Select required columns
# ------------------------------------------------
alone_columns = [
    "question_id",
    "intent_id",
    "intent_name",
    "language",
    "split",
    "policy_dependent",
    "question",
    "execution_correct",
    "generation_latency_seconds"
]

rag_columns = [
    "question_id",
    "execution_correct",
    "generation_latency_seconds"
]

alone_available_columns = [
    column
    for column in alone_columns
    if column in qwen_alone_results_df.columns
]

rag_available_columns = [
    column
    for column in rag_columns
    if column in qwen_rag_results_df.columns
]


# ------------------------------------------------
# 2. Merge both systems
# ------------------------------------------------
pairwise_df = (
    qwen_alone_results_df[
        alone_available_columns
    ]
    .merge(
        qwen_rag_results_df[
            rag_available_columns
        ],
        on="question_id",
        how="inner",
        suffixes=("_alone", "_rag"),
        validate="one_to_one"
    )
)

assert len(pairwise_df) == 96
assert "final_test" not in pairwise_df["split"].unique()


# ------------------------------------------------
# 3. Assign comparison categories
# ------------------------------------------------
pairwise_df["alone_correct"] = (
    pairwise_df["execution_correct_alone"]
    .astype(bool)
)

pairwise_df["rag_correct"] = (
    pairwise_df["execution_correct_rag"]
    .astype(bool)
)

pairwise_df["comparison_group"] = "both_wrong"

pairwise_df.loc[
    pairwise_df["alone_correct"]
    & pairwise_df["rag_correct"],
    "comparison_group"
] = "both_correct"

pairwise_df.loc[
    pairwise_df["alone_correct"]
    & ~pairwise_df["rag_correct"],
    "comparison_group"
] = "qwen_alone_only"

pairwise_df.loc[
    ~pairwise_df["alone_correct"]
    & pairwise_df["rag_correct"],
    "comparison_group"
] = "rag_only"

pairwise_df["oracle_correct"] = (
    pairwise_df["alone_correct"]
    | pairwise_df["rag_correct"]
)

pairwise_df["oracle_choice"] = "neither"

pairwise_df.loc[
    pairwise_df["comparison_group"] ==
    "both_correct",
    "oracle_choice"
] = "Qwen Alone"

pairwise_df.loc[
    pairwise_df["comparison_group"] ==
    "qwen_alone_only",
    "oracle_choice"
] = "Qwen Alone"

pairwise_df.loc[
    pairwise_df["comparison_group"] ==
    "rag_only",
    "oracle_choice"
] = "Qwen + RAG"


# ------------------------------------------------
# 4. Compute controller ceiling
# ------------------------------------------------
alone_accuracy = pairwise_df[
    "alone_correct"
].mean()

rag_accuracy = pairwise_df[
    "rag_correct"
].mean()

oracle_accuracy = pairwise_df[
    "oracle_correct"
].mean()

oracle_gain_over_rag = (
    oracle_accuracy - rag_accuracy
)

comparison_counts = (
    pairwise_df["comparison_group"]
    .value_counts()
    .reindex(
        [
            "both_correct",
            "qwen_alone_only",
            "rag_only",
            "both_wrong"
        ],
        fill_value=0
    )
)

oracle_by_split = (
    pairwise_df
    .groupby("split")["oracle_correct"]
    .agg(["sum", "count", "mean"])
)

oracle_by_language = (
    pairwise_df
    .groupby("language")["oracle_correct"]
    .agg(["sum", "count", "mean"])
)


# ------------------------------------------------
# 5. Intent-level complementarity
# ------------------------------------------------
intent_comparison = (
    pairwise_df
    .groupby(
        ["intent_id", "intent_name"],
        dropna=False
    )
    .agg(
        alone_correct=("alone_correct", "sum"),
        rag_correct=("rag_correct", "sum"),
        oracle_correct=("oracle_correct", "sum"),
        total=("question_id", "count")
    )
    .reset_index()
)

intent_comparison["alone_accuracy"] = (
    intent_comparison["alone_correct"]
    / intent_comparison["total"]
)

intent_comparison["rag_accuracy"] = (
    intent_comparison["rag_correct"]
    / intent_comparison["total"]
)

intent_comparison["oracle_accuracy"] = (
    intent_comparison["oracle_correct"]
    / intent_comparison["total"]
)


# ------------------------------------------------
# 6. Save analysis
# ------------------------------------------------
pairwise_df.to_csv(
    controller_analysis_path,
    index=False
)


# ------------------------------------------------
# 7. Report
# ------------------------------------------------
print("=" * 70)
print("CONTROLLER COMPLEMENTARITY ANALYSIS")
print("=" * 70)
print(f"Qwen Alone accuracy: {alone_accuracy:.2%}")
print(f"Qwen + RAG accuracy: {rag_accuracy:.2%}")
print(
    f"Oracle controller ceiling: "
    f"{oracle_accuracy:.2%}"
)
print(
    f"Maximum possible gain over always-on RAG: "
    f"{oracle_gain_over_rag:+.2%}"
)

print()
print("PAIRWISE OUTCOMES")
print(comparison_counts.to_string())

print()
print("ORACLE ACCURACY BY SPLIT")
print(oracle_by_split.to_string())

print()
print("ORACLE ACCURACY BY LANGUAGE")
print(oracle_by_language.to_string())

print()
print("INTENT-LEVEL COMPARISON")
print(
    intent_comparison.to_string(
        index=False
    )
)

print()
print("QUESTIONS WHERE QWEN ALONE BEAT RAG")

alone_only_examples = pairwise_df[
    pairwise_df["comparison_group"] ==
    "qwen_alone_only"
][[
    "question_id",
    "intent_id",
    "language",
    "question"
]]

if alone_only_examples.empty:
    print("None")
else:
    print(
        alone_only_examples.to_string(
            index=False
        )
    )

print()
print("CELL 23 COMPLETED SUCCESSFULLY")
print("New generations performed: 0")
print("Final test opened: False")
print(
    "Controller analysis saved:",
    controller_analysis_path
)

CONTROLLER COMPLEMENTARITY ANALYSIS
Qwen Alone accuracy: 32.29%
Qwen + RAG accuracy: 63.54%
Oracle controller ceiling: 66.67%
Maximum possible gain over always-on RAG: +3.12%

PAIRWISE OUTCOMES
comparison_group
both_correct       28
qwen_alone_only     3
rag_only           33
both_wrong         32

ORACLE ACCURACY BY SPLIT
            sum  count      mean
split                           
train        46     72  0.638889
validation   18     24  0.750000

ORACLE ACCURACY BY LANGUAGE
                  sum  count      mean
language                              
arabic             17     24  0.708333
english            15     24  0.625000
french             17     24  0.708333
lebanese_arabizi   15     24  0.625000

INTENT-LEVEL COMPARISON
intent_id                          intent_name  alone_correct  rag_correct  oracle_correct  total  alone_accuracy  rag_accuracy  oracle_accuracy
  B4A-001         recognized_revenue_by_region              0            4               4      4            0

In [29]:
# ================================================================
# CELL 24 — RAG SYSTEMATIC FAILURE DIAGNOSIS
# ================================================================

import os
import pandas as pd


DIAGNOSIS_ROOT = (
    "/kaggle/working/build4all_final_experiment"
)

rag_failure_path = os.path.join(
    DIAGNOSIS_ROOT,
    "results",
    "rag_systematic_failure_diagnosis.csv"
)


# ------------------------------------------------
# 1. Attach gold SQL to RAG results
# ------------------------------------------------
gold_sql_lookup = (
    multilingual_benchmark_df[
        [
            "question_id",
            "gold_sql",
            "gold_rows"
        ]
    ]
    .drop_duplicates("question_id")
)

rag_diagnosis_df = (
    qwen_rag_results_df
    .merge(
        gold_sql_lookup,
        on="question_id",
        how="left",
        validate="one_to_one"
    )
)


# ------------------------------------------------
# 2. Intent-level RAG performance
# ------------------------------------------------
rag_intent_summary = (
    rag_diagnosis_df
    .groupby(
        ["intent_id", "intent_name"],
        dropna=False
    )
    .agg(
        correct=("execution_correct", "sum"),
        total=("question_id", "count"),
        executed=("sql_executed", "sum"),
        average_latency=(
            "generation_latency_seconds",
            "mean"
        )
    )
    .reset_index()
)

rag_intent_summary["accuracy"] = (
    rag_intent_summary["correct"]
    / rag_intent_summary["total"]
)

problem_intents = rag_intent_summary[
    rag_intent_summary["accuracy"] < 1.0
].copy()


# ------------------------------------------------
# 3. Retrieval formatter
# ------------------------------------------------
def format_retrieval_result(retrieved_items):
    formatted_items = []

    for rank, item in enumerate(
        retrieved_items,
        start=1
    ):
        if isinstance(item, dict):
            document = (
                item.get("document")
                or item.get("document_name")
                or item.get("source")
                or item.get("doc_name")
                or "unknown"
            )

            page = (
                item.get("page")
                or item.get("page_number")
                or "unknown"
            )

            score = item.get(
                "score",
                item.get(
                    "similarity",
                    "unknown"
                )
            )

            text = (
                item.get("text")
                or item.get("content")
                or item.get("chunk")
                or ""
            )

        else:
            document = "unknown"
            page = "unknown"
            score = "unknown"
            text = str(item)

        formatted_items.append({
            "rank": rank,
            "document": document,
            "page": page,
            "score": score,
            "preview": str(text)[:220]
        })

    return formatted_items


# ------------------------------------------------
# 4. Inspect one failed example per intent
# ------------------------------------------------
failure_records = []

print("=" * 70)
print("RAG SYSTEMATIC FAILURE DIAGNOSIS")
print("=" * 70)

print("\nPROBLEM INTENTS")
print(
    problem_intents[
        [
            "intent_id",
            "intent_name",
            "correct",
            "total",
            "accuracy"
        ]
    ].to_string(index=False)
)

for _, intent_row in problem_intents.iterrows():
    intent_id = intent_row["intent_id"]

    intent_failures = rag_diagnosis_df[
        (rag_diagnosis_df["intent_id"] == intent_id)
        & (~rag_diagnosis_df["execution_correct"])
    ]

    if intent_failures.empty:
        continue

    english_failure = intent_failures[
        intent_failures["language"] == "english"
    ]

    if not english_failure.empty:
        example = english_failure.iloc[0]
    else:
        example = intent_failures.iloc[0]

    retrieved_items = retrieve_chunks(
        example["question"],
        top_k=4
    )

    formatted_retrieval = format_retrieval_result(
        retrieved_items
    )

    retrieved_documents = " | ".join(
        str(item["document"])
        for item in formatted_retrieval
    )

    failure_records.append({
        "intent_id": intent_id,
        "intent_name": example["intent_name"],
        "question_id": example["question_id"],
        "language": example["language"],
        "question": example["question"],
        "generated_sql": example["generated_sql"],
        "gold_sql": example["gold_sql"],
        "sql_executed": example["sql_executed"],
        "execution_error": example["execution_error"],
        "retrieved_documents": retrieved_documents,
        "retrieval_details": formatted_retrieval
    })

    print()
    print("-" * 70)
    print(
        f"{intent_id} — "
        f"{example['intent_name']}"
    )
    print("-" * 70)

    print("Question:")
    print(example["question"])

    print("\nGenerated SQL:")
    print(example["generated_sql"])

    print("\nGold SQL:")
    print(example["gold_sql"])

    print("\nRetrieved evidence:")

    for item in formatted_retrieval:
        print(
            f"Rank {item['rank']} | "
            f"Score {item['score']} | "
            f"{item['document']} | "
            f"Page {item['page']}"
        )
        print(
            "Preview:",
            item["preview"]
        )


# ------------------------------------------------
# 5. Save diagnosis
# ------------------------------------------------
rag_failure_diagnosis_df = pd.DataFrame(
    failure_records
)

rag_failure_diagnosis_df[
    "retrieval_details"
] = rag_failure_diagnosis_df[
    "retrieval_details"
].apply(
    lambda value: str(value)
)

rag_failure_diagnosis_df.to_csv(
    rag_failure_path,
    index=False
)


print()
print("=" * 70)
print("CELL 24 COMPLETED SUCCESSFULLY")
print("=" * 70)
print(
    "Problem intents:",
    len(problem_intents)
)
print(
    "Failed examples inspected:",
    len(rag_failure_diagnosis_df)
)
print("New Qwen generations performed: 0")
print("Final test opened: False")
print("Diagnosis saved:", rag_failure_path)

RAG SYSTEMATIC FAILURE DIAGNOSIS

PROBLEM INTENTS
intent_id                          intent_name  correct  total  accuracy
  B4A-002           monthly_recognized_revenue        1      4      0.25
  B4A-003              gross_merchandise_value        3      4      0.75
  B4A-007            operational_order_backlog        3      4      0.75
  B4A-008              order_cancellation_rate        0      4      0.00
  B4A-011           paid_collections_by_method        3      4      0.75
  B4A-016                   coupon_usage_count        0      4      0.00
  B4A-017        recognized_discount_by_coupon        0      4      0.00
  B4A-020            low_stock_active_products        3      4      0.75
  B4A-022 recognized_product_sales_by_category        0      4      0.00
  B4A-024       monthly_customer_registrations        0      4      0.00
  B4A-025          repeat_purchasing_customers        0      4      0.00
  B4A-026 recognized_shipping_amount_by_method        3      4      0.75
 

In [30]:
# Inspect the current frozen RAG pipeline before optimization
import inspect

functions_to_inspect = [
    generate_sql,
    build_rag_context
]

for function in functions_to_inspect:
    print("=" * 70)
    print("FUNCTION:", function.__name__)
    print("=" * 70)

    try:
        print(inspect.getsource(function))
    except Exception as error:
        print(
            "Could not read source:",
            type(error).__name__,
            str(error)
        )

    print()


print("=" * 70)
print("POSSIBLE PROMPT AND SCHEMA VARIABLES")
print("=" * 70)

for name, value in sorted(globals().items()):
    lowered_name = name.lower()

    if any(
        keyword in lowered_name
        for keyword in [
            "prompt",
            "schema_text",
            "database_schema",
            "system_message"
        ]
    ):
        if isinstance(value, str):
            print(
                f"{name}: string with "
                f"{len(value)} characters"
            )
        else:
            print(
                f"{name}: "
                f"{type(value).__name__}"
            )

FUNCTION: generate_sql
def generate_sql(
    question,
    use_rag=False,
):

    rag_context = ""
    retrieved_chunks = []

    if use_rag:
        rag_context, retrieved_chunks = (
            build_rag_context(question)
        )

    system_prompt = """
You are a multilingual PostgreSQL Text-to-SQL system.

Generate exactly one read-only PostgreSQL query that answers
the user's business question.

Mandatory rules:
1. Use only the provided database schema and columns.
2. Use the build4all_research schema explicitly.
3. Generate only SELECT or WITH queries.
4. Never generate INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, TRUNCATE, GRANT, REVOKE, COPY, or CALL.
5. Follow the controlled vocabulary exactly.
6. Do not invent tables, columns, filters, joins, or values.
7. Return SQL only, with no explanation or Markdown.
""".strip()

    user_parts = [
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULA

In [31]:
# ================================================================
# CELL 25 — OPTIMIZED RAG PIPELINE AND TARGETED TRAIN SMOKE TEST
# ================================================================

import os
import time
import pandas as pd
import torch


OPTIMIZED_RAG_TOP_K = 6
OPTIMIZED_RAG_CONTEXT_LIMIT = 5000


# ------------------------------------------------
# 1. Optimized context builder
# ------------------------------------------------
def build_optimized_rag_context(question):
    retrieved = retrieve_chunks(
        question,
        top_k=OPTIMIZED_RAG_TOP_K
    )

    context_parts = []
    current_length = 0
    included_chunks = []

    for result in retrieved:
        chunk_block = (
            f"[Source: {result['document_name']}, "
            f"page {result['page_number']}, "
            f"score "
            f"{result['retrieval_score']:.4f}]\n"
            f"{result['text']}"
        )

        additional_length = (
            len(chunk_block)
            + (2 if context_parts else 0)
        )

        if (
            current_length + additional_length
            > OPTIMIZED_RAG_CONTEXT_LIMIT
        ):
            continue

        context_parts.append(chunk_block)
        included_chunks.append(result)
        current_length += additional_length

    return (
        "\n\n".join(context_parts),
        included_chunks
    )


# ------------------------------------------------
# 2. Optimized generation function
# ------------------------------------------------
def generate_sql_optimized_rag(question):
    rag_context, retrieved_chunks = (
        build_optimized_rag_context(question)
    )

    system_prompt = """
You are a multilingual PostgreSQL Text-to-SQL system.

Generate exactly one read-only PostgreSQL query that returns
the exact database result requested by the business question.

SECURITY RULES:
1. Use only the provided schema, tables, columns, and values.
2. Use the build4all_research schema explicitly for base tables.
3. Generate only one SELECT or WITH query.
4. Never generate a modifying or administrative SQL operation.
5. Return SQL only, without Markdown or explanation.

BUSINESS-POLICY RULES:
6. Treat the retrieved business policy as authoritative when it
   directly defines the requested metric.
7. Do not combine rules from unrelated retrieved passages.
8. Never invent a date range, status filter, validity filter,
   active filter, join, threshold, or grouping.
9. Apply a date or reference-time filter only when the question
   or the directly relevant policy explicitly requires it.
10. Distinguish carefully between:
    - all orders and recognized orders;
    - order status and payment status;
    - current product value and historical sales;
    - coupon usage and coupon validity;
    - amounts, counts, rates, and percentages.

RESULT-SHAPE RULES:
11. Return only the business dimensions and metrics requested.
    Do not add IDs or extra descriptive columns unless required.
12. For a result grouped by a named business entity, return its
    human-readable name or code when the schema provides one.
13. Follow any ordering rule stated by the relevant policy.
14. Use ROUND(numeric_expression, 2) for monetary totals, rates,
    percentages, and averages when the policy defines a numeric
    analytical metric.
15. Do not replace a requested monthly DATE_TRUNC result with
    separate year and month columns.
16. Do not apply DISTINCT before counting repeated occurrences.
17. When counting every category, coupon, or shipping method,
    preserve zero-count entities with the correct LEFT JOIN.
18. Parenthesize mixed AND and OR conditions explicitly.
19. Do not name a CTE the same as an existing database table.
20. Before returning, silently verify tables, columns, joins,
    filters, grouping, output columns, and aggregation formula.
""".strip()

    user_prompt = "\n".join([
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULARY_TEXT,
        "",
        "RETRIEVED BUSINESS POLICY:",
        rag_context,
        "",
        "INSTRUCTIONS:",
        (
            "Identify only the passage that directly defines "
            "the requested metric. Follow it literally and "
            "ignore unrelated retrieved rules."
        ),
        (
            "Do not copy an example blindly. Construct the "
            "query from the schema and relevant definition."
        ),
        "",
        "BUSINESS QUESTION:",
        question,
        "",
        "POSTGRESQL QUERY:"
    ])

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=CONFIG[
                "model"
            ]["max_new_tokens"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generation_time = (
        time.perf_counter() - start_time
    )

    new_tokens = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:
    ]

    raw_response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True
    ).strip()

    generated_sql = extract_sql(raw_response)

    return {
        "question": question,
        "system": "qwen_optimized_rag",
        "sql": generated_sql,
        "raw_response": raw_response,
        "latency_seconds": round(
            generation_time,
            4
        ),
        "retrieved_chunks": retrieved_chunks,
        "rag_context_characters": len(
            rag_context
        )
    }


# ------------------------------------------------
# 3. Target difficult TRAIN intents only
# ------------------------------------------------
target_intents = [
    "B4A-002",
    "B4A-007",
    "B4A-008",
    "B4A-016",
    "B4A-017",
    "B4A-022",
    "B4A-024",
    "B4A-025",
    "B4A-027"
]

targeted_train_df = multilingual_benchmark_df[
    (
        multilingual_benchmark_df[
            "intent_id"
        ].isin(target_intents)
    )
    & (
        multilingual_benchmark_df[
            "split"
        ] == "train"
    )
    & (
        multilingual_benchmark_df[
            "language"
        ] == "english"
    )
].copy()

assert "final_test" not in targeted_train_df["split"].unique()
assert "validation" not in targeted_train_df["split"].unique()

print("=" * 70)
print("OPTIMIZED RAG — TARGETED TRAIN SMOKE TEST")
print("=" * 70)
print(
    "Target examples:",
    len(targeted_train_df)
)
print("Languages:", ["english"])
print("Split:", ["train"])
print("Final test opened: False")
print()


# ------------------------------------------------
# 4. Generate and evaluate targeted examples
# ------------------------------------------------
optimized_smoke_results = []

for _, benchmark_row in targeted_train_df.iterrows():
    generation = generate_sql_optimized_rag(
        benchmark_row["question"]
    )

    evaluation = evaluate_generated_sql(
        generation["sql"],
        benchmark_row["gold_rows"]
    )

    old_rag_match = qwen_rag_results_df[
        qwen_rag_results_df["question_id"]
        == benchmark_row["question_id"]
    ]

    old_rag_correct = (
        bool(
            old_rag_match.iloc[0][
                "execution_correct"
            ]
        )
        if not old_rag_match.empty
        else False
    )

    record = {
        "question_id": benchmark_row["question_id"],
        "intent_id": benchmark_row["intent_id"],
        "intent_name": benchmark_row.get(
            "intent_name",
            None
        ),
        "question": benchmark_row["question"],
        "old_rag_correct": old_rag_correct,
        "optimized_rag_correct": bool(
            evaluation["execution_correct"]
        ),
        "sql_safe": bool(
            evaluation["sql_safe"]
        ),
        "sql_executed": bool(
            evaluation["sql_executed"]
        ),
        "generated_sql": generation["sql"],
        "gold_sql": benchmark_row["gold_sql"],
        "latency_seconds": generation[
            "latency_seconds"
        ],
        "context_characters": generation[
            "rag_context_characters"
        ],
        "execution_error": evaluation[
            "execution_error"
        ]
    }

    optimized_smoke_results.append(record)

    print("-" * 70)
    print(
        benchmark_row["question_id"],
        benchmark_row["intent_name"]
    )
    print("Old RAG correct:", old_rag_correct)
    print(
        "Optimized RAG correct:",
        evaluation["execution_correct"]
    )
    print("Generated SQL:")
    print(generation["sql"])
    print()


optimized_smoke_df = pd.DataFrame(
    optimized_smoke_results
)


# ------------------------------------------------
# 5. Save targeted results
# ------------------------------------------------
optimized_smoke_path = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "results",
    "optimized_rag_targeted_train_smoke.csv"
)

optimized_smoke_df.to_csv(
    optimized_smoke_path,
    index=False
)


# ------------------------------------------------
# 6. Summary
# ------------------------------------------------
old_correct = int(
    optimized_smoke_df[
        "old_rag_correct"
    ].sum()
)

optimized_correct = int(
    optimized_smoke_df[
        "optimized_rag_correct"
    ].sum()
)

target_count = len(optimized_smoke_df)

print("=" * 70)
print("TARGETED OPTIMIZATION SUMMARY")
print("=" * 70)
print(
    f"Original RAG: "
    f"{old_correct}/{target_count}"
)
print(
    f"Optimized RAG: "
    f"{optimized_correct}/{target_count}"
)
print(
    "Net corrections:",
    optimized_correct - old_correct
)
print()
print("CELL 25 COMPLETED SUCCESSFULLY")
print("Optimized RAG function ready: True")
print("Validation used for tuning: False")
print("Final test opened: False")
print("Smoke results saved:", optimized_smoke_path)

OPTIMIZED RAG — TARGETED TRAIN SMOKE TEST
Target examples: 7
Languages: ['english']
Split: ['train']
Final test opened: False

----------------------------------------------------------------------
B4A-002-ENGLISH monthly_recognized_revenue
Old RAG correct: False
Optimized RAG correct: True
Generated SQL:
SELECT 
    DATE_TRUNC('month', orders.order_date) AS month,
    SUM(orders.total_amount) AS recognized_order_revenue
FROM 
    build4all_research.orders
WHERE 
    orders.status = 'completed' OR orders.status = 'shipped'
GROUP BY 
    DATE_TRUNC('month', orders.order_date)
ORDER BY 
    month;

----------------------------------------------------------------------
B4A-007-ENGLISH operational_order_backlog
Old RAG correct: True
Optimized RAG correct: True
Generated SQL:
SELECT COUNT(*) 
FROM build4all_research.orders 
WHERE status = 'pending' OR status = 'processing';

----------------------------------------------------------------------
B4A-016-ENGLISH coupon_usage_count
Old RAG cor

In [32]:
# ================================================================
# CELL 26 — DOMAIN-RESTRICTED RAG RETRIEVAL TEST
# ================================================================

import os
import pandas as pd


DOMAIN_CANDIDATE_K = 12
DOMAIN_CONTEXT_K = 6
DOMAIN_CONTEXT_LIMIT = 5000


# ------------------------------------------------
# 1. Domain-restricted context builder
# ------------------------------------------------
def build_optimized_rag_context(question):
    candidates = retrieve_chunks(
        question,
        top_k=DOMAIN_CANDIDATE_K
    )

    if not candidates:
        return "", []

    # The highest-ranked document defines the query domain.
    selected_document = candidates[0][
        "document_name"
    ]

    domain_candidates = [
        result
        for result in candidates
        if result["document_name"]
        == selected_document
    ]

    selected_chunks = domain_candidates[
        :DOMAIN_CONTEXT_K
    ]

    context_parts = []
    included_chunks = []
    current_length = 0

    for result in selected_chunks:
        chunk_block = (
            f"[Source: {result['document_name']}, "
            f"page {result['page_number']}, "
            f"score "
            f"{result['retrieval_score']:.4f}]\n"
            f"{result['text']}"
        )

        added_length = (
            len(chunk_block)
            + (2 if context_parts else 0)
        )

        if (
            current_length + added_length
            > DOMAIN_CONTEXT_LIMIT
        ):
            continue

        context_parts.append(chunk_block)
        included_chunks.append(result)
        current_length += added_length

    return (
        "\n\n".join(context_parts),
        included_chunks
    )


# ------------------------------------------------
# 2. Re-run same targeted train examples
# ------------------------------------------------
domain_rag_results = []

print("=" * 70)
print("DOMAIN-RESTRICTED RAG — TARGETED TRAIN TEST")
print("=" * 70)
print("Target examples:", len(targeted_train_df))
print("Validation used: False")
print("Final test opened: False")
print()

for _, benchmark_row in targeted_train_df.iterrows():
    generation = generate_sql_optimized_rag(
        benchmark_row["question"]
    )

    evaluation = evaluate_generated_sql(
        generation["sql"],
        benchmark_row["gold_rows"]
    )

    v1_match = optimized_smoke_df[
        optimized_smoke_df["question_id"]
        == benchmark_row["question_id"]
    ]

    v1_correct = (
        bool(
            v1_match.iloc[0][
                "optimized_rag_correct"
            ]
        )
        if not v1_match.empty
        else False
    )

    retrieved_documents = [
        result["document_name"]
        for result in generation[
            "retrieved_chunks"
        ]
    ]

    record = {
        "question_id": benchmark_row["question_id"],
        "intent_id": benchmark_row["intent_id"],
        "intent_name": benchmark_row.get(
            "intent_name",
            None
        ),
        "question": benchmark_row["question"],
        "optimized_v1_correct": v1_correct,
        "domain_rag_correct": bool(
            evaluation["execution_correct"]
        ),
        "sql_safe": bool(
            evaluation["sql_safe"]
        ),
        "sql_executed": bool(
            evaluation["sql_executed"]
        ),
        "generated_sql": generation["sql"],
        "gold_sql": benchmark_row["gold_sql"],
        "selected_document": (
            retrieved_documents[0]
            if retrieved_documents
            else None
        ),
        "retrieved_document_count": len(
            set(retrieved_documents)
        ),
        "retrieved_chunk_count": len(
            retrieved_documents
        ),
        "latency_seconds": generation[
            "latency_seconds"
        ],
        "execution_error": evaluation[
            "execution_error"
        ]
    }

    domain_rag_results.append(record)

    print("-" * 70)
    print(
        benchmark_row["question_id"],
        benchmark_row["intent_name"]
    )
    print(
        "Selected document:",
        record["selected_document"]
    )
    print(
        "Optimized V1 correct:",
        v1_correct
    )
    print(
        "Domain RAG correct:",
        evaluation["execution_correct"]
    )
    print("Generated SQL:")
    print(generation["sql"])
    print()


domain_rag_smoke_df = pd.DataFrame(
    domain_rag_results
)

domain_smoke_path = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "results",
    "domain_restricted_rag_targeted_train.csv"
)

domain_rag_smoke_df.to_csv(
    domain_smoke_path,
    index=False
)


# ------------------------------------------------
# 3. Compare optimization versions
# ------------------------------------------------
v1_correct_count = int(
    domain_rag_smoke_df[
        "optimized_v1_correct"
    ].sum()
)

domain_correct_count = int(
    domain_rag_smoke_df[
        "domain_rag_correct"
    ].sum()
)

target_count = len(domain_rag_smoke_df)

regressions = domain_rag_smoke_df[
    domain_rag_smoke_df[
        "optimized_v1_correct"
    ]
    & ~domain_rag_smoke_df[
        "domain_rag_correct"
    ]
]

new_corrections = domain_rag_smoke_df[
    ~domain_rag_smoke_df[
        "optimized_v1_correct"
    ]
    & domain_rag_smoke_df[
        "domain_rag_correct"
    ]
]


print("=" * 70)
print("DOMAIN-RESTRICTED RETRIEVAL SUMMARY")
print("=" * 70)
print(
    f"Optimized RAG V1: "
    f"{v1_correct_count}/{target_count}"
)
print(
    f"Domain-restricted RAG: "
    f"{domain_correct_count}/{target_count}"
)
print(
    "New corrections:",
    len(new_corrections)
)
print(
    "Regressions:",
    len(regressions)
)

if not new_corrections.empty:
    print("\nNEWLY CORRECTED QUESTIONS")
    print(
        new_corrections[
            [
                "question_id",
                "intent_name"
            ]
        ].to_string(index=False)
    )

if not regressions.empty:
    print("\nREGRESSIONS")
    print(
        regressions[
            [
                "question_id",
                "intent_name"
            ]
        ].to_string(index=False)
    )

print()
print("CELL 26 COMPLETED SUCCESSFULLY")
print("New generations performed:", target_count)
print("Validation used for tuning: False")
print("Final test opened: False")
print("Results saved:", domain_smoke_path)

DOMAIN-RESTRICTED RAG — TARGETED TRAIN TEST
Target examples: 7
Validation used: False
Final test opened: False

----------------------------------------------------------------------
B4A-002-ENGLISH monthly_recognized_revenue
Selected document: Build4All_Sales_and_Order_Analytics_Policy_v2.0
Optimized V1 correct: True
Domain RAG correct: True
Generated SQL:
SELECT 
    DATE_TRUNC('month', orders.order_date) AS month,
    SUM(orders.total_amount) AS recognized_order_revenue
FROM 
    build4all_research.orders
WHERE 
    orders.status = 'completed' OR orders.status = 'shipped'
GROUP BY 
    DATE_TRUNC('month', orders.order_date)
ORDER BY 
    month;

----------------------------------------------------------------------
B4A-007-ENGLISH operational_order_backlog
Selected document: Build4All_Sales_and_Order_Analytics_Policy_v2.0
Optimized V1 correct: True
Domain RAG correct: True
Generated SQL:
SELECT COUNT(*) 
FROM build4all_research.orders 
WHERE status IN ('pending', 'processing');

---

In [33]:
# ================================================================
# CELL 27 — OPTIMIZED RAG V2 WITH ANALYTICAL SQL TEMPLATES
# ================================================================

import os
import time
import pandas as pd
import torch


def generate_sql_optimized_rag_v2(question):
    rag_context, retrieved_chunks = (
        build_optimized_rag_context(question)
    )

    system_prompt = """
You are a multilingual PostgreSQL Text-to-SQL system.

Return exactly one read-only PostgreSQL SELECT or WITH query.
Return SQL only, without explanations or Markdown.

SECURITY:
- Use only the provided schema, columns, and controlled values.
- Fully qualify base tables with build4all_research.
- Never use a modifying or administrative SQL command.

POLICY INTERPRETATION:
- Find the retrieved passage that directly defines the metric.
- Treat that passage as authoritative.
- Ignore unrelated retrieved rules.
- Never invent filters, date ranges, joins, or thresholds.
- Apply recognized-order, active, valid, or date filters only
  when the requested metric explicitly requires them.
- Distinguish order status from payment status.

OUTPUT CONTRACT:
- Return only the requested business dimension and metric.
- Do not add technical IDs or descriptive attributes unless the
  question explicitly requests them.
- When grouping by a coupon, product, category, shipping method,
  or other named entity, display its human-readable code or name.
- Do not change the requested result into a different breakdown.
- Monetary totals, averages, rates, and percentages should use
  ROUND(expression, 2).
- Use clear parentheses or IN (...) for multiple status values.

GENERAL ANALYTICAL SQL PATTERNS:
- Monthly grouping:
  DATE_TRUNC('month', the_correct_timestamp_column).
  Return one month column, not separate year/month columns.
- Repeat-entity counting:
  first GROUP BY the entity and COUNT its occurrences;
  then count groups whose occurrence count is greater than one.
  Never use DISTINCT before counting repeat occurrences.
- Count by every reference entity:
  begin with the reference table and LEFT JOIN the event table,
  so entities with zero events remain present.
- A filter on a LEFT JOIN that should preserve zero rows belongs
  in the ON clause, not the WHERE clause.
- If the question asks how many events used each entity, do not
  apply activity, validity, or recognized-status filters unless
  the relevant metric definition explicitly requires them.
- Do not name a CTE the same as a physical table.
- Do not select extra identifiers merely to make grouping easier.

Before returning SQL, silently verify:
1. requested metric formula;
2. correct timestamp;
3. required and forbidden filters;
4. joins and operator precedence;
5. grouping grain;
6. output column count and meaning.
""".strip()

    user_prompt = "\n".join([
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULARY_TEXT,
        "",
        "RETRIEVED BUSINESS POLICY:",
        rag_context,
        "",
        "BUSINESS QUESTION:",
        question,
        "",
        "Construct the smallest query that returns exactly the",
        "requested dimension and metric.",
        "",
        "POSTGRESQL QUERY:"
    ])

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt"
    ).to(model.device)

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=CONFIG[
                "model"
            ]["max_new_tokens"],
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generation_time = (
        time.perf_counter() - start_time
    )

    new_tokens = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:
    ]

    raw_response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True
    ).strip()

    return {
        "question": question,
        "system": "qwen_optimized_rag_v2",
        "sql": extract_sql(raw_response),
        "raw_response": raw_response,
        "latency_seconds": round(
            generation_time,
            4
        ),
        "retrieved_chunks": retrieved_chunks,
        "rag_context_characters": len(
            rag_context
        )
    }


# ------------------------------------------------
# Test the same seven TRAIN examples
# ------------------------------------------------
v2_results = []

print("=" * 70)
print("OPTIMIZED RAG V2 — TARGETED TRAIN TEST")
print("=" * 70)
print("Target examples:", len(targeted_train_df))
print("Validation used: False")
print("Final test opened: False")
print()

for _, benchmark_row in targeted_train_df.iterrows():
    generation = generate_sql_optimized_rag_v2(
        benchmark_row["question"]
    )

    evaluation = evaluate_generated_sql(
        generation["sql"],
        benchmark_row["gold_rows"]
    )

    domain_match = domain_rag_smoke_df[
        domain_rag_smoke_df["question_id"]
        == benchmark_row["question_id"]
    ]

    previous_correct = bool(
        domain_match.iloc[0]["domain_rag_correct"]
    )

    record = {
        "question_id": benchmark_row["question_id"],
        "intent_id": benchmark_row["intent_id"],
        "intent_name": benchmark_row.get(
            "intent_name",
            None
        ),
        "question": benchmark_row["question"],
        "previous_correct": previous_correct,
        "v2_correct": bool(
            evaluation["execution_correct"]
        ),
        "sql_safe": bool(evaluation["sql_safe"]),
        "sql_executed": bool(
            evaluation["sql_executed"]
        ),
        "generated_sql": generation["sql"],
        "gold_sql": benchmark_row["gold_sql"],
        "latency_seconds": generation[
            "latency_seconds"
        ],
        "execution_error": evaluation[
            "execution_error"
        ]
    }

    v2_results.append(record)

    print("-" * 70)
    print(
        benchmark_row["question_id"],
        benchmark_row["intent_name"]
    )
    print("Previous correct:", previous_correct)
    print(
        "V2 correct:",
        evaluation["execution_correct"]
    )
    print("Generated SQL:")
    print(generation["sql"])
    print()


optimized_v2_smoke_df = pd.DataFrame(v2_results)

v2_smoke_path = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "results",
    "optimized_rag_v2_targeted_train.csv"
)

optimized_v2_smoke_df.to_csv(
    v2_smoke_path,
    index=False
)


previous_correct_count = int(
    optimized_v2_smoke_df[
        "previous_correct"
    ].sum()
)

v2_correct_count = int(
    optimized_v2_smoke_df[
        "v2_correct"
    ].sum()
)

new_corrections = optimized_v2_smoke_df[
    ~optimized_v2_smoke_df["previous_correct"]
    & optimized_v2_smoke_df["v2_correct"]
]

regressions = optimized_v2_smoke_df[
    optimized_v2_smoke_df["previous_correct"]
    & ~optimized_v2_smoke_df["v2_correct"]
]


print("=" * 70)
print("OPTIMIZED RAG V2 SUMMARY")
print("=" * 70)
print(
    f"Previous version: "
    f"{previous_correct_count}/"
    f"{len(optimized_v2_smoke_df)}"
)
print(
    f"Optimized V2: "
    f"{v2_correct_count}/"
    f"{len(optimized_v2_smoke_df)}"
)
print("New corrections:", len(new_corrections))
print("Regressions:", len(regressions))

if not new_corrections.empty:
    print("\nNEW CORRECTIONS")
    print(
        new_corrections[
            ["question_id", "intent_name"]
        ].to_string(index=False)
    )

if not regressions.empty:
    print("\nREGRESSIONS")
    print(
        regressions[
            ["question_id", "intent_name"]
        ].to_string(index=False)
    )

print()
print("CELL 27 COMPLETED SUCCESSFULLY")
print(
    "New generations performed:",
    len(optimized_v2_smoke_df)
)
print("Validation used for tuning: False")
print("Final test opened: False")
print("Results saved:", v2_smoke_path)

OPTIMIZED RAG V2 — TARGETED TRAIN TEST
Target examples: 7
Validation used: False
Final test opened: False

----------------------------------------------------------------------
B4A-002-ENGLISH monthly_recognized_revenue
Previous correct: True
V2 correct: True
Generated SQL:
SELECT 
    DATE_TRUNC('month', orders.order_date) AS month,
    SUM(orders.total_amount) AS recognized_order_revenue
FROM 
    build4all_research.orders
WHERE 
    orders.status = 'completed' OR orders.status = 'shipped'
GROUP BY 
    DATE_TRUNC('month', orders.order_date)
ORDER BY 
    month;

----------------------------------------------------------------------
B4A-007-ENGLISH operational_order_backlog
Previous correct: True
V2 correct: True
Generated SQL:
SELECT COUNT(*) 
FROM build4all_research.orders 
WHERE status IN ('pending', 'processing');

----------------------------------------------------------------------
B4A-016-ENGLISH coupon_usage_count
Previous correct: False
V2 correct: False
Generated SQL:
SEL

In [34]:
# ================================================================
# CELL 28 — OPTIMIZED RAG V2 FULL TRAIN BENCHMARK
# ================================================================

import json
import os
import time
import pandas as pd


RESULTS_ROOT = (
    "/kaggle/working/build4all_final_experiment/results"
)

v2_train_checkpoint_path = os.path.join(
    RESULTS_ROOT,
    "optimized_rag_v2_train.jsonl"
)

v2_train_csv_path = os.path.join(
    RESULTS_ROOT,
    "optimized_rag_v2_train.csv"
)

os.makedirs(RESULTS_ROOT, exist_ok=True)


# ------------------------------------------------
# 1. Train split only
# ------------------------------------------------
v2_train_benchmark_df = (
    multilingual_benchmark_df[
        multilingual_benchmark_df["split"] == "train"
    ]
    .copy()
    .sort_values("question_id")
)

assert len(v2_train_benchmark_df) == 72
assert set(v2_train_benchmark_df["split"]) == {"train"}


# ------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------
v2_completed_results = []
v2_completed_ids = set()

if os.path.exists(v2_train_checkpoint_path):
    with open(
        v2_train_checkpoint_path,
        "r",
        encoding="utf-8"
    ) as checkpoint_file:
        for line in checkpoint_file:
            line = line.strip()

            if not line:
                continue

            try:
                saved_result = json.loads(line)

                v2_completed_results.append(
                    saved_result
                )

                v2_completed_ids.add(
                    saved_result["question_id"]
                )

            except json.JSONDecodeError:
                print(
                    "Ignored one incomplete checkpoint line."
                )


print("=" * 70)
print("OPTIMIZED RAG V2 — FULL TRAIN BENCHMARK")
print("=" * 70)
print("Target questions:", len(v2_train_benchmark_df))
print("Previously completed:", len(v2_completed_ids))
print(
    "Remaining:",
    len(v2_train_benchmark_df) -
    len(v2_completed_ids)
)
print("Validation included: False")
print("Final test included: False")
print()


# ------------------------------------------------
# 3. Generate, execute, and checkpoint
# ------------------------------------------------
for _, benchmark_row in v2_train_benchmark_df.iterrows():
    question_id = benchmark_row["question_id"]

    if question_id in v2_completed_ids:
        continue

    started_at = time.perf_counter()

    try:
        generation = generate_sql_optimized_rag_v2(
            benchmark_row["question"]
        )

        measured_latency = (
            time.perf_counter() - started_at
        )

        evaluation = evaluate_generated_sql(
            generation["sql"],
            benchmark_row["gold_rows"]
        )

        retrieved_documents = [
            result["document_name"]
            for result in generation[
                "retrieved_chunks"
            ]
        ]

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": benchmark_row["split"],
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Optimized RAG V2",
            "rag_used": True,
            "generated_sql": generation["sql"],
            "raw_response": generation[
                "raw_response"
            ],
            "generation_latency_seconds": float(
                generation.get(
                    "latency_seconds",
                    measured_latency
                )
            ),
            "rag_context_characters": int(
                generation[
                    "rag_context_characters"
                ]
            ),
            "selected_document": (
                retrieved_documents[0]
                if retrieved_documents
                else None
            ),
            "retrieved_chunk_count": len(
                generation["retrieved_chunks"]
            ),
            "sql_safe": bool(
                evaluation["sql_safe"]
            ),
            "sql_executed": bool(
                evaluation["sql_executed"]
            ),
            "execution_correct": bool(
                evaluation["execution_correct"]
            ),
            "execution_error": evaluation[
                "execution_error"
            ],
            "predicted_row_count": int(
                evaluation["predicted_row_count"]
            ),
            "gold_row_count": int(
                evaluation["gold_row_count"]
            ),
            "benchmark_error": None
        }

    except Exception as error:
        measured_latency = (
            time.perf_counter() - started_at
        )

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": "train",
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Optimized RAG V2",
            "rag_used": True,
            "generated_sql": None,
            "raw_response": None,
            "generation_latency_seconds": float(
                measured_latency
            ),
            "rag_context_characters": 0,
            "selected_document": None,
            "retrieved_chunk_count": 0,
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": None,
            "predicted_row_count": 0,
            "gold_row_count": len(
                benchmark_row["gold_rows"]
            ),
            "benchmark_error": (
                f"{type(error).__name__}: {error}"
            )
        }

    with open(
        v2_train_checkpoint_path,
        "a",
        encoding="utf-8"
    ) as checkpoint_file:
        checkpoint_file.write(
            json.dumps(
                result_record,
                ensure_ascii=False,
                default=str
            ) + "\n"
        )

    v2_completed_results.append(result_record)
    v2_completed_ids.add(question_id)

    completed_count = len(v2_completed_results)

    correct_count = sum(
        bool(result["execution_correct"])
        for result in v2_completed_results
    )

    running_accuracy = (
        correct_count / completed_count
    )

    print(
        f"[{completed_count:03d}/72] "
        f"{question_id} | "
        f"{benchmark_row['language']} | "
        f"correct="
        f"{result_record['execution_correct']} | "
        f"running_accuracy="
        f"{running_accuracy:.2%} | "
        f"latency="
        f"{result_record['generation_latency_seconds']:.2f}s"
    )


# ------------------------------------------------
# 4. Build and save complete train results
# ------------------------------------------------
optimized_rag_v2_train_df = pd.DataFrame(
    v2_completed_results
)

optimized_rag_v2_train_df = (
    optimized_rag_v2_train_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last"
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(optimized_rag_v2_train_df) == 72

optimized_rag_v2_train_df.to_csv(
    v2_train_csv_path,
    index=False
)


# ------------------------------------------------
# 5. Compare original RAG against V2
# ------------------------------------------------
original_rag_train = qwen_rag_results_df[
    qwen_rag_results_df["split"] == "train"
][[
    "question_id",
    "execution_correct",
    "generation_latency_seconds"
]].copy()

comparison_df = original_rag_train.merge(
    optimized_rag_v2_train_df[[
        "question_id",
        "intent_id",
        "intent_name",
        "language",
        "execution_correct",
        "generation_latency_seconds"
    ]],
    on="question_id",
    how="inner",
    suffixes=("_original", "_v2"),
    validate="one_to_one"
)

assert len(comparison_df) == 72

original_accuracy = comparison_df[
    "execution_correct_original"
].mean()

v2_accuracy = comparison_df[
    "execution_correct_v2"
].mean()

new_corrections = comparison_df[
    ~comparison_df["execution_correct_original"]
    & comparison_df["execution_correct_v2"]
]

regressions = comparison_df[
    comparison_df["execution_correct_original"]
    & ~comparison_df["execution_correct_v2"]
]

net_gain = (
    len(new_corrections) - len(regressions)
)

v2_by_language = (
    optimized_rag_v2_train_df
    .groupby("language")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

v2_by_intent = (
    optimized_rag_v2_train_df
    .groupby(
        ["intent_id", "intent_name"],
        dropna=False
    )["execution_correct"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)


# ------------------------------------------------
# 6. Report
# ------------------------------------------------
print()
print("=" * 70)
print("OPTIMIZED RAG V2 — TRAIN RESULTS")
print("=" * 70)
print(
    f"Original RAG train accuracy: "
    f"{original_accuracy:.2%}"
)
print(
    f"Optimized RAG V2 train accuracy: "
    f"{v2_accuracy:.2%}"
)
print(
    f"Accuracy difference: "
    f"{v2_accuracy - original_accuracy:+.2%}"
)
print("New corrections:", len(new_corrections))
print("Regressions:", len(regressions))
print("Net corrected questions:", net_gain)

print()
print("V2 ACCURACY BY LANGUAGE")
print(v2_by_language.to_string())

print()
print("V2 ACCURACY BY INTENT")
print(v2_by_intent.to_string(index=False))

if not regressions.empty:
    print()
    print("REGRESSIONS")
    print(
        regressions[[
            "question_id",
            "intent_name",
            "language"
        ]].to_string(index=False)
    )

print()
print("CELL 28 COMPLETED SUCCESSFULLY")
print(
    "Optimized RAG V2 train generations:",
    len(optimized_rag_v2_train_df)
)
print("Validation opened: False")
print("Final test opened: False")
print("Results saved:", v2_train_csv_path)
print(
    "Checkpoint saved:",
    v2_train_checkpoint_path
)

OPTIMIZED RAG V2 — FULL TRAIN BENCHMARK
Target questions: 72
Previously completed: 0
Remaining: 72
Validation included: False
Final test included: False

[001/72] B4A-001-ARABIC | arabic | correct=True | running_accuracy=100.00% | latency=13.59s
[002/72] B4A-001-ENGLISH | english | correct=True | running_accuracy=100.00% | latency=13.65s
[003/72] B4A-001-FRENCH | french | correct=True | running_accuracy=100.00% | latency=13.91s
[004/72] B4A-001-LEBANESE_ARABIZI | lebanese_arabizi | correct=True | running_accuracy=100.00% | latency=13.60s
[005/72] B4A-002-ARABIC | arabic | correct=True | running_accuracy=100.00% | latency=13.14s
[006/72] B4A-002-ENGLISH | english | correct=True | running_accuracy=100.00% | latency=13.20s
[007/72] B4A-002-FRENCH | french | correct=True | running_accuracy=100.00% | latency=13.15s
[008/72] B4A-002-LEBANESE_ARABIZI | lebanese_arabizi | correct=False | running_accuracy=87.50% | latency=8.64s
[009/72] B4A-004-ARABIC | arabic | correct=False | running_accuracy

In [35]:
# ================================================================
# CELL 29 — OPTIMIZED RAG V2 VALIDATION EVALUATION
# ================================================================

import json
import os
import time
import pandas as pd


RESULTS_ROOT = (
    "/kaggle/working/build4all_final_experiment/results"
)

v2_validation_checkpoint_path = os.path.join(
    RESULTS_ROOT,
    "optimized_rag_v2_validation.jsonl"
)

v2_validation_csv_path = os.path.join(
    RESULTS_ROOT,
    "optimized_rag_v2_validation.csv"
)


# ------------------------------------------------
# 1. Validation split only
# ------------------------------------------------
v2_validation_benchmark_df = (
    multilingual_benchmark_df[
        multilingual_benchmark_df["split"]
        == "validation"
    ]
    .copy()
    .sort_values("question_id")
)

assert len(v2_validation_benchmark_df) == 24
assert set(
    v2_validation_benchmark_df["split"]
) == {"validation"}


# ------------------------------------------------
# 2. Load checkpoint
# ------------------------------------------------
validation_results = []
validation_completed_ids = set()

if os.path.exists(v2_validation_checkpoint_path):
    with open(
        v2_validation_checkpoint_path,
        "r",
        encoding="utf-8"
    ) as checkpoint_file:
        for line in checkpoint_file:
            line = line.strip()

            if not line:
                continue

            try:
                saved_result = json.loads(line)
                validation_results.append(saved_result)
                validation_completed_ids.add(
                    saved_result["question_id"]
                )
            except json.JSONDecodeError:
                print(
                    "Ignored one incomplete checkpoint line."
                )


print("=" * 70)
print("OPTIMIZED RAG V2 — VALIDATION EVALUATION")
print("=" * 70)
print(
    "Target questions:",
    len(v2_validation_benchmark_df)
)
print(
    "Previously completed:",
    len(validation_completed_ids)
)
print(
    "Remaining:",
    len(v2_validation_benchmark_df)
    - len(validation_completed_ids)
)
print("Final test included: False")
print()


# ------------------------------------------------
# 3. Run validation
# ------------------------------------------------
for _, benchmark_row in (
    v2_validation_benchmark_df.iterrows()
):
    question_id = benchmark_row["question_id"]

    if question_id in validation_completed_ids:
        continue

    started_at = time.perf_counter()

    try:
        generation = generate_sql_optimized_rag_v2(
            benchmark_row["question"]
        )

        measured_latency = (
            time.perf_counter() - started_at
        )

        evaluation = evaluate_generated_sql(
            generation["sql"],
            benchmark_row["gold_rows"]
        )

        retrieved_documents = [
            result["document_name"]
            for result in generation[
                "retrieved_chunks"
            ]
        ]

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": "validation",
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Optimized RAG V2",
            "rag_used": True,
            "generated_sql": generation["sql"],
            "raw_response": generation[
                "raw_response"
            ],
            "generation_latency_seconds": float(
                generation.get(
                    "latency_seconds",
                    measured_latency
                )
            ),
            "rag_context_characters": int(
                generation[
                    "rag_context_characters"
                ]
            ),
            "selected_document": (
                retrieved_documents[0]
                if retrieved_documents
                else None
            ),
            "retrieved_chunk_count": len(
                generation["retrieved_chunks"]
            ),
            "sql_safe": bool(
                evaluation["sql_safe"]
            ),
            "sql_executed": bool(
                evaluation["sql_executed"]
            ),
            "execution_correct": bool(
                evaluation["execution_correct"]
            ),
            "execution_error": evaluation[
                "execution_error"
            ],
            "predicted_row_count": int(
                evaluation["predicted_row_count"]
            ),
            "gold_row_count": int(
                evaluation["gold_row_count"]
            ),
            "benchmark_error": None
        }

    except Exception as error:
        measured_latency = (
            time.perf_counter() - started_at
        )

        result_record = {
            "question_id": question_id,
            "intent_id": benchmark_row["intent_id"],
            "intent_name": benchmark_row.get(
                "intent_name",
                None
            ),
            "language": benchmark_row["language"],
            "split": "validation",
            "policy_dependent": bool(
                benchmark_row.get(
                    "policy_dependent",
                    False
                )
            ),
            "question": benchmark_row["question"],
            "system": "Optimized RAG V2",
            "rag_used": True,
            "generated_sql": None,
            "raw_response": None,
            "generation_latency_seconds": float(
                measured_latency
            ),
            "rag_context_characters": 0,
            "selected_document": None,
            "retrieved_chunk_count": 0,
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": None,
            "predicted_row_count": 0,
            "gold_row_count": len(
                benchmark_row["gold_rows"]
            ),
            "benchmark_error": (
                f"{type(error).__name__}: {error}"
            )
        }

    with open(
        v2_validation_checkpoint_path,
        "a",
        encoding="utf-8"
    ) as checkpoint_file:
        checkpoint_file.write(
            json.dumps(
                result_record,
                ensure_ascii=False,
                default=str
            ) + "\n"
        )

    validation_results.append(result_record)
    validation_completed_ids.add(question_id)

    completed_count = len(validation_results)

    correct_count = sum(
        bool(result["execution_correct"])
        for result in validation_results
    )

    running_accuracy = (
        correct_count / completed_count
    )

    print(
        f"[{completed_count:02d}/24] "
        f"{question_id} | "
        f"{benchmark_row['language']} | "
        f"correct="
        f"{result_record['execution_correct']} | "
        f"running_accuracy="
        f"{running_accuracy:.2%}"
    )


# ------------------------------------------------
# 4. Save results
# ------------------------------------------------
optimized_rag_v2_validation_df = pd.DataFrame(
    validation_results
)

optimized_rag_v2_validation_df = (
    optimized_rag_v2_validation_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last"
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(
    optimized_rag_v2_validation_df
) == 24

optimized_rag_v2_validation_df.to_csv(
    v2_validation_csv_path,
    index=False
)


# ------------------------------------------------
# 5. Compare original RAG and V2
# ------------------------------------------------
original_validation_df = qwen_rag_results_df[
    qwen_rag_results_df["split"]
    == "validation"
][[
    "question_id",
    "execution_correct"
]].copy()

validation_comparison_df = (
    original_validation_df
    .merge(
        optimized_rag_v2_validation_df[[
            "question_id",
            "intent_id",
            "intent_name",
            "language",
            "execution_correct"
        ]],
        on="question_id",
        how="inner",
        suffixes=("_original", "_v2"),
        validate="one_to_one"
    )
)

assert len(validation_comparison_df) == 24

original_validation_accuracy = (
    validation_comparison_df[
        "execution_correct_original"
    ].mean()
)

v2_validation_accuracy = (
    validation_comparison_df[
        "execution_correct_v2"
    ].mean()
)

validation_new_corrections = (
    validation_comparison_df[
        ~validation_comparison_df[
            "execution_correct_original"
        ]
        & validation_comparison_df[
            "execution_correct_v2"
        ]
    ]
)

validation_regressions = (
    validation_comparison_df[
        validation_comparison_df[
            "execution_correct_original"
        ]
        & ~validation_comparison_df[
            "execution_correct_v2"
        ]
    ]
)

validation_by_language = (
    optimized_rag_v2_validation_df
    .groupby("language")["execution_correct"]
    .agg(["sum", "count", "mean"])
)

validation_by_intent = (
    optimized_rag_v2_validation_df
    .groupby(
        ["intent_id", "intent_name"],
        dropna=False
    )["execution_correct"]
    .agg(["sum", "count", "mean"])
    .reset_index()
)


# ------------------------------------------------
# 6. Report
# ------------------------------------------------
print()
print("=" * 70)
print("OPTIMIZED RAG V2 — VALIDATION RESULTS")
print("=" * 70)
print(
    f"Original RAG validation accuracy: "
    f"{original_validation_accuracy:.2%}"
)
print(
    f"Optimized RAG V2 validation accuracy: "
    f"{v2_validation_accuracy:.2%}"
)
print(
    f"Validation difference: "
    f"{v2_validation_accuracy - original_validation_accuracy:+.2%}"
)
print(
    "New corrections:",
    len(validation_new_corrections)
)
print(
    "Regressions:",
    len(validation_regressions)
)

print()
print("VALIDATION ACCURACY BY LANGUAGE")
print(validation_by_language.to_string())

print()
print("VALIDATION ACCURACY BY INTENT")
print(validation_by_intent.to_string(index=False))

if not validation_regressions.empty:
    print()
    print("VALIDATION REGRESSIONS")
    print(
        validation_regressions[[
            "question_id",
            "intent_name",
            "language"
        ]].to_string(index=False)
    )

print()
print("CELL 29 COMPLETED SUCCESSFULLY")
print(
    "Optimized RAG V2 validation generations:",
    len(optimized_rag_v2_validation_df)
)
print("Final test opened: False")
print(
    "Validation results saved:",
    v2_validation_csv_path
)
print(
    "Checkpoint saved:",
    v2_validation_checkpoint_path
)

OPTIMIZED RAG V2 — VALIDATION EVALUATION
Target questions: 24
Previously completed: 0
Remaining: 24
Final test included: False

[01/24] B4A-003-ARABIC | arabic | correct=False | running_accuracy=0.00%
[02/24] B4A-003-ENGLISH | english | correct=True | running_accuracy=50.00%
[03/24] B4A-003-FRENCH | french | correct=True | running_accuracy=66.67%
[04/24] B4A-003-LEBANESE_ARABIZI | lebanese_arabizi | correct=True | running_accuracy=75.00%
[05/24] B4A-008-ARABIC | arabic | correct=True | running_accuracy=80.00%
[06/24] B4A-008-ENGLISH | english | correct=False | running_accuracy=66.67%
[07/24] B4A-008-FRENCH | french | correct=False | running_accuracy=57.14%
[08/24] B4A-008-LEBANESE_ARABIZI | lebanese_arabizi | correct=False | running_accuracy=50.00%
[09/24] B4A-015-ARABIC | arabic | correct=True | running_accuracy=55.56%
[10/24] B4A-015-ENGLISH | english | correct=True | running_accuracy=60.00%
[11/24] B4A-015-FRENCH | french | correct=True | running_accuracy=63.64%
[12/24] B4A-015-LEBA

In [36]:
# ================================================================
# CELL 30 — THREE-SYSTEM CONTROLLER CEILING
# ================================================================

import os
import pandas as pd


RESULTS_ROOT = (
    "/kaggle/working/build4all_final_experiment/results"
)

three_system_analysis_path = os.path.join(
    RESULTS_ROOT,
    "three_system_controller_analysis.csv"
)


# ------------------------------------------------
# 1. Combine optimized V2 train and validation
# ------------------------------------------------
optimized_rag_v2_results_df = pd.concat(
    [
        optimized_rag_v2_train_df,
        optimized_rag_v2_validation_df
    ],
    ignore_index=True
)

optimized_rag_v2_results_df = (
    optimized_rag_v2_results_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last"
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(optimized_rag_v2_results_df) == 96
assert "final_test" not in set(
    optimized_rag_v2_results_df["split"]
)


# ------------------------------------------------
# 2. Prepare question metadata
# ------------------------------------------------
metadata_columns = [
    "question_id",
    "intent_id",
    "intent_name",
    "language",
    "split",
    "question"
]

if (
    "policy_dependent"
    in multilingual_benchmark_df.columns
):
    metadata_columns.append(
        "policy_dependent"
    )

base_metadata = multilingual_benchmark_df[
    multilingual_benchmark_df["split"].isin(
        ["train", "validation"]
    )
][metadata_columns].copy()


# ------------------------------------------------
# 3. Merge all three systems
# ------------------------------------------------
three_system_df = (
    base_metadata
    .merge(
        qwen_alone_results_df[[
            "question_id",
            "execution_correct",
            "generation_latency_seconds"
        ]],
        on="question_id",
        how="inner",
        validate="one_to_one"
    )
    .rename(columns={
        "execution_correct": "alone_correct",
        "generation_latency_seconds": (
            "alone_latency"
        )
    })
    .merge(
        qwen_rag_results_df[[
            "question_id",
            "execution_correct",
            "generation_latency_seconds"
        ]],
        on="question_id",
        how="inner",
        validate="one_to_one"
    )
    .rename(columns={
        "execution_correct": (
            "original_rag_correct"
        ),
        "generation_latency_seconds": (
            "original_rag_latency"
        )
    })
    .merge(
        optimized_rag_v2_results_df[[
            "question_id",
            "execution_correct",
            "generation_latency_seconds"
        ]],
        on="question_id",
        how="inner",
        validate="one_to_one"
    )
    .rename(columns={
        "execution_correct": "v2_rag_correct",
        "generation_latency_seconds": (
            "v2_rag_latency"
        )
    })
)

assert len(three_system_df) == 96
assert "final_test" not in set(
    three_system_df["split"]
)


# ------------------------------------------------
# 4. Normalize correctness fields
# ------------------------------------------------
correctness_columns = [
    "alone_correct",
    "original_rag_correct",
    "v2_rag_correct"
]

for column in correctness_columns:
    three_system_df[column] = (
        three_system_df[column].astype(bool)
    )


# ------------------------------------------------
# 5. Compute oracle controller
# ------------------------------------------------
three_system_df["oracle_correct"] = (
    three_system_df["alone_correct"]
    | three_system_df["original_rag_correct"]
    | three_system_df["v2_rag_correct"]
)

three_system_df["correct_system_count"] = (
    three_system_df[
        correctness_columns
    ].sum(axis=1)
)


def choose_fastest_correct_system(row):
    candidates = []

    if row["alone_correct"]:
        candidates.append((
            "Qwen Alone",
            float(row["alone_latency"])
        ))

    if row["original_rag_correct"]:
        candidates.append((
            "Original RAG",
            float(row["original_rag_latency"])
        ))

    if row["v2_rag_correct"]:
        candidates.append((
            "Optimized RAG V2",
            float(row["v2_rag_latency"])
        ))

    if not candidates:
        return "No correct system"

    candidates.sort(
        key=lambda item: item[1]
    )

    return candidates[0][0]


three_system_df["oracle_choice"] = (
    three_system_df.apply(
        choose_fastest_correct_system,
        axis=1
    )
)


# ------------------------------------------------
# 6. System accuracy
# ------------------------------------------------
system_accuracies = {
    "Qwen Alone": three_system_df[
        "alone_correct"
    ].mean(),

    "Original RAG": three_system_df[
        "original_rag_correct"
    ].mean(),

    "Optimized RAG V2": three_system_df[
        "v2_rag_correct"
    ].mean(),

    "Oracle Controller": three_system_df[
        "oracle_correct"
    ].mean()
}

oracle_by_split = (
    three_system_df
    .groupby("split")["oracle_correct"]
    .agg(["sum", "count", "mean"])
)

oracle_by_language = (
    three_system_df
    .groupby("language")["oracle_correct"]
    .agg(["sum", "count", "mean"])
)

oracle_choices = (
    three_system_df[
        "oracle_choice"
    ].value_counts()
)


# ------------------------------------------------
# 7. Analyze recoveries and failures
# ------------------------------------------------
v2_recoverable_errors = three_system_df[
    ~three_system_df["v2_rag_correct"]
    & (
        three_system_df["alone_correct"]
        | three_system_df[
            "original_rag_correct"
        ]
    )
]

all_systems_wrong = three_system_df[
    ~three_system_df["oracle_correct"]
]

only_alone_correct = three_system_df[
    three_system_df["alone_correct"]
    & ~three_system_df[
        "original_rag_correct"
    ]
    & ~three_system_df[
        "v2_rag_correct"
    ]
]

only_original_rag_correct = three_system_df[
    ~three_system_df["alone_correct"]
    & three_system_df[
        "original_rag_correct"
    ]
    & ~three_system_df[
        "v2_rag_correct"
    ]
]

only_v2_correct = three_system_df[
    ~three_system_df["alone_correct"]
    & ~three_system_df[
        "original_rag_correct"
    ]
    & three_system_df["v2_rag_correct"]
]


# ------------------------------------------------
# 8. Save analysis
# ------------------------------------------------
three_system_df.to_csv(
    three_system_analysis_path,
    index=False
)


# ------------------------------------------------
# 9. Report
# ------------------------------------------------
print("=" * 70)
print("THREE-SYSTEM CONTROLLER CEILING")
print("=" * 70)

for system_name, accuracy in (
    system_accuracies.items()
):
    print(
        f"{system_name}: "
        f"{accuracy:.2%}"
    )

print()
print(
    "Recoverable V2 mistakes:",
    len(v2_recoverable_errors)
)
print(
    "Questions all systems missed:",
    len(all_systems_wrong)
)

print()
print("EXCLUSIVE CORRECT SYSTEM")
print(
    "Only Qwen Alone:",
    len(only_alone_correct)
)
print(
    "Only Original RAG:",
    len(only_original_rag_correct)
)
print(
    "Only Optimized RAG V2:",
    len(only_v2_correct)
)

print()
print("FASTEST CORRECT ORACLE CHOICES")
print(oracle_choices.to_string())

print()
print("ORACLE ACCURACY BY SPLIT")
print(oracle_by_split.to_string())

print()
print("ORACLE ACCURACY BY LANGUAGE")
print(oracle_by_language.to_string())

print()
print("ALL-SYSTEM FAILURES BY INTENT")

if all_systems_wrong.empty:
    print("None")
else:
    print(
        all_systems_wrong
        .groupby([
            "intent_id",
            "intent_name"
        ])
        .size()
        .to_string()
    )

print()
print("RECOVERABLE V2 ERRORS")

if v2_recoverable_errors.empty:
    print("None")
else:
    print(
        v2_recoverable_errors[[
            "question_id",
            "intent_name",
            "language",
            "alone_correct",
            "original_rag_correct",
            "v2_rag_correct"
        ]].to_string(index=False)
    )

print()
print("CELL 30 COMPLETED SUCCESSFULLY")
print("New generations performed: 0")
print("Final test opened: False")
print(
    "Three-system analysis saved:",
    three_system_analysis_path
)

THREE-SYSTEM CONTROLLER CEILING
Qwen Alone: 32.29%
Original RAG: 63.54%
Optimized RAG V2: 73.96%
Oracle Controller: 82.29%

Recoverable V2 mistakes: 8
Questions all systems missed: 17

EXCLUSIVE CORRECT SYSTEM
Only Qwen Alone: 1
Only Original RAG: 6
Only Optimized RAG V2: 15

FASTEST CORRECT ORACLE CHOICES
oracle_choice
Original RAG         32
Qwen Alone           31
No correct system    17
Optimized RAG V2     16

ORACLE ACCURACY BY SPLIT
            sum  count      mean
split                           
train        59     72  0.819444
validation   20     24  0.833333

ORACLE ACCURACY BY LANGUAGE
                  sum  count      mean
language                              
arabic             20     24  0.833333
english            21     24  0.875000
french             21     24  0.875000
lebanese_arabizi   17     24  0.708333

ALL-SYSTEM FAILURES BY INTENT
intent_id  intent_name                         
B4A-002    monthly_recognized_revenue              1
B4A-003    gross_merchandise_

In [37]:
# ============================================================
# CELL 31 — BUILD THE ADAPTIVE CONTROLLER DATASET
# ============================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Prepare the three systems' existing results
# ------------------------------------------------------------

def prepare_system_results(
    dataframe,
    system_prefix,
):
    required_columns = {
        "question_id",
        "execution_correct",
    }

    missing_columns = (
        required_columns
        - set(dataframe.columns)
    )

    if missing_columns:
        raise ValueError(
            f"{system_prefix} missing columns: "
            f"{sorted(missing_columns)}"
        )

    selected_columns = [
        "question_id",
        "execution_correct",
    ]

    if (
        "generation_latency_seconds"
        in dataframe.columns
    ):
        selected_columns.append(
            "generation_latency_seconds"
        )

    prepared = dataframe[
        selected_columns
    ].copy()

    prepared = prepared.rename(
        columns={
            "execution_correct": (
                f"{system_prefix}_correct"
            ),
            "generation_latency_seconds": (
                f"{system_prefix}_latency"
            ),
        }
    )

    return prepared
          

   


alone_controller_df = prepare_system_results(
    qwen_alone_results_df,
    "alone",
)

original_rag_controller_df = (
    prepare_system_results(
        qwen_rag_results_df,
        "original_rag",
    )
)

v2_controller_df = prepare_system_results(
    optimized_rag_v2_results_df,
    "v2",
)


# ------------------------------------------------------------
# 2. Start from train + validation questions only
# ------------------------------------------------------------

controller_base_df = (
    multilingual_benchmark_df[
        multilingual_benchmark_df[
            "split"
        ].isin([
            "train",
            "validation",
        ])
    ][
        [
            "question_id",
            "intent_id",
            "intent_name",
            "split",
            "language",
            "question",
        ]
    ]
    .copy()
)

controller_base_df = (
    controller_base_df
    .merge(
        alone_controller_df,
        on="question_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        original_rag_controller_df,
        on="question_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        v2_controller_df,
        on="question_id",
        how="inner",
        validate="one_to_one",
    )
)

for correctness_column in [
    "alone_correct",
    "original_rag_correct",
    "v2_correct",
]:
    controller_base_df[
        correctness_column
    ] = (
        controller_base_df[
            correctness_column
        ]
        .fillna(False)
        .astype(bool)
    )


# ------------------------------------------------------------
# 3. Define the controller training target
#
# V2 is the default route.
# We switch away from V2 only when another system corrected
# a V2 mistake.
# ------------------------------------------------------------

def choose_training_route(row):

    if row["v2_correct"]:
        return "optimized_rag_v2"

    if row["original_rag_correct"]:
        return "original_rag"

    if row["alone_correct"]:
        return "qwen_alone"

    # No system answered correctly.
    # Keep the strongest general system as default.
    return "optimized_rag_v2"


controller_base_df[
    "controller_target"
] = controller_base_df.apply(
    choose_training_route,
    axis=1,
)

controller_base_df[
    "all_systems_wrong"
] = ~(
    controller_base_df[
        "alone_correct"
    ]
    | controller_base_df[
        "original_rag_correct"
    ]
    | controller_base_df[
        "v2_correct"
    ]
)


# ------------------------------------------------------------
# 4. Question features available before SQL generation
# ------------------------------------------------------------

QUESTION_CUE_GROUPS = {
    "cue_month": [
        "month",
        "monthly",
        "mois",
        "mensuel",
        "شهر",
        "شهري",
        "chaher",
    ],
    "cue_rate": [
        "rate",
        "percentage",
        "percent",
        "pourcentage",
        "taux",
        "نسبة",
        "nesbe",
    ],
    "cue_count": [
        "how many",
        "count",
        "number",
        "combien",
        "كم",
        "عدد",
        "adde",
    ],
    "cue_each_or_by": [
        " each ",
        " by ",
        " per ",
        " par ",
        " chaque ",
        "كل",
        "حسب",
        "7asab",
        "kel",
    ],
    "cue_recognized": [
        "recognized",
        "reconnu",
        "reconnue",
        "معترف",
    ],
    "cue_coupon": [
        "coupon",
        "coupons",
        "قسيمة",
    ],
    "cue_customer": [
        "customer",
        "customers",
        "client",
        "clients",
        "عميل",
        "عملاء",
        "زبون",
    ],
    "cue_shipping": [
        "shipping",
        "livraison",
        "expedition",
        "شحن",
    ],
    "cue_payment": [
        "payment",
        "payments",
        "paid",
        "paiement",
        "payé",
        "دفع",
        "مدفوع",
    ],
    "cue_product": [
        "product",
        "products",
        "produit",
        "produits",
        "منتج",
    ],
    "cue_order": [
        "order",
        "orders",
        "commande",
        "commandes",
        "طلب",
        "طلبات",
    ],
    "cue_status": [
        "status",
        "statut",
        "حالة",
    ],
    "cue_active": [
        "active",
        "actif",
        "actifs",
        "نشط",
    ],
}


def contains_arabic_script(text):
    return bool(
        re.search(
            r"[\u0600-\u06FF]",
            str(text),
        )
    )


def extract_controller_features(row):

    question = str(
        row["question"]
    ).strip()

    normalized_question = (
        " "
        + question.lower()
        + " "
    )

    retrieved = retrieve_chunks(
        question,
        top_k=12,
    )

    retrieval_scores = [
        float(
            result["retrieval_score"]
        )
        for result in retrieved
    ]

    document_names = [
        str(
            result["document_name"]
        )
        for result in retrieved
    ]

    if retrieval_scores:
        top_score = retrieval_scores[0]
        second_score = (
            retrieval_scores[1]
            if len(retrieval_scores) > 1
            else retrieval_scores[0]
        )
        mean_score = float(
            np.mean(retrieval_scores)
        )
        score_std = float(
            np.std(retrieval_scores)
        )
    else:
        top_score = 0.0
        second_score = 0.0
        mean_score = 0.0
        score_std = 0.0

    top_document = (
        document_names[0]
        if document_names
        else "no_document"
    )

    unique_documents = len(
        set(document_names[:6])
    )

    top_document_count = sum(
        document_name == top_document
        for document_name
        in document_names[:6]
    )

    features = {
        "question_id": row["question_id"],
        "intent_id": row["intent_id"],
        "intent_name": row["intent_name"],
        "split": row["split"],
        "language": row["language"],
        "question": question,

        "question_characters": len(
            question
        ),
        "question_words": len(
            question.split()
        ),
        "contains_arabic_script": int(
            contains_arabic_script(
                question
            )
        ),
        "contains_digit": int(
            bool(
                re.search(
                    r"\d",
                    question,
                )
            )
        ),
        "question_mark": int(
            "?" in question
            or "؟" in question
        ),

        "retrieval_top_score": top_score,
        "retrieval_second_score": second_score,
        "retrieval_margin": (
            top_score - second_score
        ),
        "retrieval_mean_score": mean_score,
        "retrieval_score_std": score_std,
        "retrieval_unique_documents": (
            unique_documents
        ),
        "top_document_count_top6": (
            top_document_count
        ),
        "top_document": top_document,
    }

    for feature_name, keywords in (
        QUESTION_CUE_GROUPS.items()
    ):
        features[feature_name] = int(
            any(
                keyword.lower()
                in normalized_question
                for keyword in keywords
            )
        )

    return features


controller_feature_rows = []

for row_number, row in (
    controller_base_df
    .reset_index(drop=True)
    .iterrows()
):
    controller_feature_rows.append(
        extract_controller_features(
            row
        )
    )

    if (
        (row_number + 1) % 12 == 0
        or row_number + 1
        == len(controller_base_df)
    ):
        print(
            f"Features prepared: "
            f"{row_number + 1}/"
            f"{len(controller_base_df)}"
        )


controller_features_df = pd.DataFrame(
    controller_feature_rows
)

controller_dataset_df = (
    controller_features_df
    .merge(
        controller_base_df[
            [
                "question_id",
                "alone_correct",
                "original_rag_correct",
                "v2_correct",
                "all_systems_wrong",
                "controller_target",
            ]
        ],
        on="question_id",
        how="inner",
        validate="one_to_one",
    )
)

controller_train_df = (
    controller_dataset_df[
        controller_dataset_df[
            "split"
        ] == "train"
    ]
    .reset_index(drop=True)
)

controller_validation_df = (
    controller_dataset_df[
        controller_dataset_df[
            "split"
        ] == "validation"
    ]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Save the prepared controller data
# ------------------------------------------------------------

controller_results_directory = Path(
    "/kaggle/working/"
    "build4all_final_experiment/"
    "results"
)

controller_results_directory.mkdir(
    parents=True,
    exist_ok=True,
)

controller_dataset_path = (
    controller_results_directory
    / "adaptive_controller_dataset.csv"
)

controller_train_path = (
    controller_results_directory
    / "adaptive_controller_train.csv"
)

controller_validation_path = (
    controller_results_directory
    / "adaptive_controller_validation.csv"
)

controller_dataset_df.to_csv(
    controller_dataset_path,
    index=False,
)

controller_train_df.to_csv(
    controller_train_path,
    index=False,
)

controller_validation_df.to_csv(
    controller_validation_path,
    index=False,
)


# ------------------------------------------------------------
# 6. Audit
# ------------------------------------------------------------

print()
print("=" * 70)
print("ADAPTIVE CONTROLLER DATASET READY")
print("=" * 70)

print(
    "Total questions:",
    len(controller_dataset_df),
)

print(
    "Train questions:",
    len(controller_train_df),
)

print(
    "Validation questions:",
    len(controller_validation_df),
)

print(
    "Final test opened:",
    False,
)

print()
print("CONTROLLER TARGETS BY SPLIT")

print(
    controller_dataset_df.groupby(
        [
            "split",
            "controller_target",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("ALL-SYSTEM FAILURES BY SPLIT")

print(
    controller_dataset_df.groupby(
        "split"
    )[
        "all_systems_wrong"
    ]
    .sum()
)

print()
print("TOP RETRIEVED DOCUMENTS")

print(
    controller_dataset_df[
        "top_document"
    ]
    .value_counts()
)

print()
print("Saved:", controller_dataset_path)
print("Saved:", controller_train_path)
print("Saved:", controller_validation_path)

Features prepared: 12/96
Features prepared: 24/96
Features prepared: 36/96
Features prepared: 48/96
Features prepared: 60/96
Features prepared: 72/96
Features prepared: 84/96
Features prepared: 96/96

ADAPTIVE CONTROLLER DATASET READY
Total questions: 96
Train questions: 72
Validation questions: 24
Final test opened: False

CONTROLLER TARGETS BY SPLIT
controller_target  optimized_rag_v2  original_rag  qwen_alone
split                                                        
train                            67             5           0
validation                       21             2           1

ALL-SYSTEM FAILURES BY SPLIT
split
train         13
validation     4
Name: all_systems_wrong, dtype: int64

TOP RETRIEVED DOCUMENTS
top_document
Build4All_Sales_and_Order_Analytics_Policy_v2.0         44
Build4All_Customer_Regional_and_Shipping_Policy_v2.0    23
Build4All_Product_Inventory_and_Coupon_Policy_v2.0      20
Build4All_Product_Events_and_Conversion_Policy_v2.0      9
Name: count, dty

In [38]:
# CELL 31A — INSPECT RESULT COLUMN NAMES

print("QWEN ALONE COLUMNS:")
print(qwen_alone_results_df.columns.tolist())

print("\nORIGINAL RAG COLUMNS:")
print(qwen_rag_results_df.columns.tolist())

print("\nOPTIMIZED RAG V2 COLUMNS:")
print(optimized_rag_v2_results_df.columns.tolist())

QWEN ALONE COLUMNS:
['question_id', 'intent_id', 'intent_name', 'language', 'split', 'policy_dependent', 'question', 'system', 'rag_used', 'generated_sql', 'raw_response', 'generation_latency_seconds', 'sql_safe', 'sql_executed', 'execution_correct', 'execution_error', 'predicted_row_count', 'gold_row_count', 'benchmark_error']

ORIGINAL RAG COLUMNS:
['question_id', 'intent_id', 'intent_name', 'language', 'split', 'policy_dependent', 'question', 'system', 'rag_used', 'generated_sql', 'raw_response', 'generation_latency_seconds', 'rag_context_characters', 'retrieved_chunks', 'sql_safe', 'sql_executed', 'execution_correct', 'execution_error', 'predicted_row_count', 'gold_row_count', 'benchmark_error']

OPTIMIZED RAG V2 COLUMNS:
['question_id', 'intent_id', 'intent_name', 'language', 'split', 'policy_dependent', 'question', 'system', 'rag_used', 'generated_sql', 'raw_response', 'generation_latency_seconds', 'rag_context_characters', 'selected_document', 'retrieved_chunk_count', 'sql_safe'

In [39]:
# ============================================================
# CELL 31B — CORRECT CONTROLLER TARGET USING FASTEST CORRECT
# ============================================================

import numpy as np


# ------------------------------------------------------------
# 1. Attach system latencies to the controller dataset
# ------------------------------------------------------------

latency_columns_df = controller_base_df[
    [
        "question_id",
        "alone_latency",
        "original_rag_latency",
        "v2_latency",
    ]
].copy()

controller_dataset_df = (
    controller_dataset_df
    .drop(
        columns=[
            column
            for column in [
                "alone_latency",
                "original_rag_latency",
                "v2_latency",
            ]
            if column
            in controller_dataset_df.columns
        ]
    )
    .merge(
        latency_columns_df,
        on="question_id",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 2. Select the fastest system among correct systems
# ------------------------------------------------------------

def choose_fastest_correct_route(row):

    candidates = []

    if row["alone_correct"]:
        candidates.append(
            (
                "qwen_alone",
                row["alone_latency"],
            )
        )

    if row["original_rag_correct"]:
        candidates.append(
            (
                "original_rag",
                row["original_rag_latency"],
            )
        )

    if row["v2_correct"]:
        candidates.append(
            (
                "optimized_rag_v2",
                row["v2_latency"],
            )
        )

    if not candidates:
        # No system was correct.
        # Use the strongest general system as default.
        return "optimized_rag_v2"

    cleaned_candidates = []

    for route_name, latency in candidates:

        if pd.isna(latency):
            latency = np.inf

        cleaned_candidates.append(
            (
                route_name,
                float(latency),
            )
        )

    return min(
        cleaned_candidates,
        key=lambda item: item[1],
    )[0]


controller_dataset_df[
    "controller_target"
] = controller_dataset_df.apply(
    choose_fastest_correct_route,
    axis=1,
)


# ------------------------------------------------------------
# 3. Recreate train and validation datasets
# ------------------------------------------------------------

controller_train_df = (
    controller_dataset_df[
        controller_dataset_df[
            "split"
        ] == "train"
    ]
    .reset_index(drop=True)
)

controller_validation_df = (
    controller_dataset_df[
        controller_dataset_df[
            "split"
        ] == "validation"
    ]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Save corrected datasets
# ------------------------------------------------------------

controller_dataset_df.to_csv(
    controller_dataset_path,
    index=False,
)

controller_train_df.to_csv(
    controller_train_path,
    index=False,
)

controller_validation_df.to_csv(
    controller_validation_path,
    index=False,
)


# ------------------------------------------------------------
# 5. Audit corrected controller targets
# ------------------------------------------------------------

print("=" * 70)
print("CONTROLLER TARGETS CORRECTED")
print("=" * 70)

print()
print("FASTEST-CORRECT TARGETS BY SPLIT")

print(
    controller_dataset_df.groupby(
        [
            "split",
            "controller_target",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("TARGETS EXCLUDING ALL-SYSTEM FAILURES")

print(
    controller_dataset_df[
        ~controller_dataset_df[
            "all_systems_wrong"
        ]
    ]
    .groupby(
        [
            "split",
            "controller_target",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print(
    "Train questions:",
    len(controller_train_df),
)

print(
    "Validation questions:",
    len(controller_validation_df),
)

print(
    "Final test opened:",
    False,
)

print()
print("Corrected files saved.")

CONTROLLER TARGETS CORRECTED

FASTEST-CORRECT TARGETS BY SPLIT
controller_target  optimized_rag_v2  original_rag  qwen_alone
split                                                        
train                            27            23          22
validation                        6             9           9

TARGETS EXCLUDING ALL-SYSTEM FAILURES
controller_target  optimized_rag_v2  original_rag  qwen_alone
split                                                        
train                            14            23          22
validation                        2             9           9

Train questions: 72
Validation questions: 24
Final test opened: False

Corrected files saved.


In [40]:
# ============================================================
# CELL 32 — TRAIN CORRECTNESS-FIRST ADAPTIVE CONTROLLER
# ============================================================

import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold


# ------------------------------------------------------------
# 1. Controller feature columns
# ------------------------------------------------------------

categorical_features = [
    "language",
    "top_document",
]

excluded_columns = {
    "question_id",
    "intent_id",
    "intent_name",
    "split",
    "question",
    "controller_target",
    "all_systems_wrong",
    "alone_correct",
    "original_rag_correct",
    "v2_correct",
    "alone_latency",
    "original_rag_latency",
    "v2_latency",
}

numeric_features = [
    column
    for column
    in controller_train_df.columns
    if (
        column not in excluded_columns
        and column
        not in categorical_features
        and pd.api.types.is_numeric_dtype(
            controller_train_df[column]
        )
    )
]

feature_columns = (
    categorical_features
    + numeric_features
)

route_definitions = {
    "qwen_alone": "alone_correct",
    "original_rag": (
        "original_rag_correct"
    ),
    "optimized_rag_v2": "v2_correct",
}

route_names = list(
    route_definitions.keys()
)


# ------------------------------------------------------------
# 2. Input data
# ------------------------------------------------------------

X_train = controller_train_df[
    feature_columns
].copy()

train_groups = controller_train_df[
    "intent_id"
].copy()

X_validation = controller_validation_df[
    feature_columns
].copy()


# ------------------------------------------------------------
# 3. Preprocessing
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
            categorical_features,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_features,
        ),
    ],
    remainder="drop",
)


# ------------------------------------------------------------
# 4. Candidate controller models
# ------------------------------------------------------------

candidate_models = {
    "logistic_regularized": (
        LogisticRegression(
            C=0.25,
            class_weight="balanced",
            max_iter=3000,
            random_state=42,
        )
    ),
    "logistic_standard": (
        LogisticRegression(
            C=1.0,
            class_weight="balanced",
            max_iter=3000,
            random_state=42,
        )
    ),
    "random_forest_shallow": (
        RandomForestClassifier(
            n_estimators=300,
            max_depth=4,
            min_samples_leaf=3,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )
    ),
}


# ------------------------------------------------------------
# 5. Group-aware cross-validation
#
# Translations of the same intent remain together.
# ------------------------------------------------------------

group_cv = GroupKFold(
    n_splits=6
)

candidate_summaries = []
candidate_oof_probabilities = {}

for candidate_name, classifier in (
    candidate_models.items()
):

    oof_probabilities = np.zeros(
        (
            len(controller_train_df),
            len(route_names),
        ),
        dtype=float,
    )

    for fold_number, (
        fit_indices,
        test_indices,
    ) in enumerate(
        group_cv.split(
            X_train,
            groups=train_groups,
        ),
        start=1,
    ):

        for route_index, route_name in (
            enumerate(route_names)
        ):

            correctness_column = (
                route_definitions[
                    route_name
                ]
            )

            y_route = (
                controller_train_df[
                    correctness_column
                ]
                .astype(int)
                .to_numpy()
            )

            route_pipeline = Pipeline(
                steps=[
                    (
                        "preprocessor",
                        preprocessor,
                    ),
                    (
                        "classifier",
                        classifier,
                    ),
                ]
            )

            route_pipeline.fit(
                X_train.iloc[
                    fit_indices
                ],
                y_route[
                    fit_indices
                ],
            )

            fold_probabilities = (
                route_pipeline.predict_proba(
                    X_train.iloc[
                        test_indices
                    ]
                )
            )

            positive_class_index = list(
                route_pipeline.named_steps[
                    "classifier"
                ].classes_
            ).index(1)

            oof_probabilities[
                test_indices,
                route_index,
            ] = fold_probabilities[
                :,
                positive_class_index,
            ]

    predicted_route_indices = (
        np.argmax(
            oof_probabilities,
            axis=1,
        )
    )

    predicted_routes = [
        route_names[index]
        for index
        in predicted_route_indices
    ]

    selected_correctness = []

    for row_index, predicted_route in (
        enumerate(predicted_routes)
    ):
        correctness_column = (
            route_definitions[
                predicted_route
            ]
        )

        selected_correctness.append(
            bool(
                controller_train_df.iloc[
                    row_index
                ][
                    correctness_column
                ]
            )
        )

    oof_accuracy = float(
        np.mean(
            selected_correctness
        )
    )

    route_distribution = (
        pd.Series(
            predicted_routes
        )
        .value_counts()
        .to_dict()
    )

    candidate_summaries.append({
        "candidate": candidate_name,
        "oof_controller_accuracy": (
            oof_accuracy
        ),
        "qwen_alone_choices": (
            route_distribution.get(
                "qwen_alone",
                0,
            )
        ),
        "original_rag_choices": (
            route_distribution.get(
                "original_rag",
                0,
            )
        ),
        "optimized_rag_v2_choices": (
            route_distribution.get(
                "optimized_rag_v2",
                0,
            )
        ),
    })

    candidate_oof_probabilities[
        candidate_name
    ] = oof_probabilities


controller_cv_results_df = (
    pd.DataFrame(
        candidate_summaries
    )
    .sort_values(
        [
            "oof_controller_accuracy",
            "optimized_rag_v2_choices",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

selected_controller_name = (
    controller_cv_results_df.iloc[
        0
    ]["candidate"]
)

selected_classifier = (
    candidate_models[
        selected_controller_name
    ]
)


# ------------------------------------------------------------
# 6. Train three correctness predictors on full train split
# ------------------------------------------------------------

adaptive_controller_models = {}

for route_name in route_names:

    correctness_column = (
        route_definitions[
            route_name
        ]
    )

    y_route = (
        controller_train_df[
            correctness_column
        ]
        .astype(int)
    )

    final_route_model = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor,
            ),
            (
                "classifier",
                selected_classifier,
            ),
        ]
    )

    final_route_model.fit(
        X_train,
        y_route,
    )

    adaptive_controller_models[
        route_name
    ] = final_route_model


# ------------------------------------------------------------
# 7. Controller prediction function
#
# Accuracy is the priority.
# Route order is only an exact-probability tie-break.
# ------------------------------------------------------------

CONTROLLER_TIE_PRIORITY = [
    "optimized_rag_v2",
    "original_rag",
    "qwen_alone",
]


def predict_controller_route(
    feature_dataframe,
):

    probability_table = {}

    for route_name in route_names:

        route_model = (
            adaptive_controller_models[
                route_name
            ]
        )

        classes = list(
            route_model.named_steps[
                "classifier"
            ].classes_
        )

        positive_index = (
            classes.index(1)
        )

        probability_table[
            route_name
        ] = (
            route_model.predict_proba(
                feature_dataframe[
                    feature_columns
                ]
            )[
                :,
                positive_index,
            ]
        )

    probability_df = pd.DataFrame(
        probability_table,
        index=feature_dataframe.index,
    )

    predicted_routes = []

    for row_index in probability_df.index:

        row_probabilities = (
            probability_df.loc[
                row_index
            ]
        )

        maximum_probability = (
            row_probabilities.max()
        )

        tied_routes = [
            route_name
            for route_name
            in CONTROLLER_TIE_PRIORITY
            if np.isclose(
                row_probabilities[
                    route_name
                ],
                maximum_probability,
                rtol=1e-10,
                atol=1e-12,
            )
        ]

        predicted_routes.append(
            tied_routes[0]
        )

    return (
        predicted_routes,
        probability_df,
    )


# ------------------------------------------------------------
# 8. Locked validation evaluation
# ------------------------------------------------------------

(
    validation_predicted_routes,
    validation_probability_df,
) = predict_controller_route(
    controller_validation_df
)

controller_validation_results_df = (
    controller_validation_df[
        [
            "question_id",
            "intent_id",
            "intent_name",
            "language",
            "alone_correct",
            "original_rag_correct",
            "v2_correct",
        ]
    ]
    .copy()
)

controller_validation_results_df[
    "controller_route"
] = validation_predicted_routes

controller_validation_results_df[
    "controller_correct"
] = [
    bool(
        controller_validation_df.iloc[
            row_index
        ][
            route_definitions[
                predicted_route
            ]
        ]
    )
    for row_index, predicted_route
    in enumerate(
        validation_predicted_routes
    )
]

for route_name in route_names:
    controller_validation_results_df[
        f"probability_{route_name}"
    ] = (
        validation_probability_df[
            route_name
        ]
        .to_numpy()
    )


actual_controller_validation_accuracy = (
    controller_validation_results_df[
        "controller_correct"
    ]
    .mean()
)

v2_validation_accuracy = (
    controller_validation_df[
        "v2_correct"
    ]
    .mean()
)

validation_oracle_accuracy = (
    controller_validation_df[
        [
            "alone_correct",
            "original_rag_correct",
            "v2_correct",
        ]
    ]
    .any(axis=1)
    .mean()
)


# ------------------------------------------------------------
# 9. Save controller and results
# ------------------------------------------------------------

controller_model_path = (
    controller_results_directory
    / "adaptive_controller_models.joblib"
)

controller_cv_path = (
    controller_results_directory
    / "adaptive_controller_cv_results.csv"
)

controller_validation_result_path = (
    controller_results_directory
    / "adaptive_controller_validation_results.csv"
)

joblib.dump(
    {
        "selected_model": (
            selected_controller_name
        ),
        "feature_columns": (
            feature_columns
        ),
        "categorical_features": (
            categorical_features
        ),
        "numeric_features": (
            numeric_features
        ),
        "route_definitions": (
            route_definitions
        ),
        "tie_priority": (
            CONTROLLER_TIE_PRIORITY
        ),
        "models": (
            adaptive_controller_models
        ),
    },
    controller_model_path,
)

controller_cv_results_df.to_csv(
    controller_cv_path,
    index=False,
)

controller_validation_results_df.to_csv(
    controller_validation_result_path,
    index=False,
)


# ------------------------------------------------------------
# 10. Results
# ------------------------------------------------------------

print("=" * 70)
print("CORRECTNESS-FIRST ADAPTIVE CONTROLLER")
print("=" * 70)

print()
print("GROUP-AWARE TRAIN CROSS-VALIDATION")
print(
    controller_cv_results_df.to_string(
        index=False
    )
)

print()
print(
    "Selected controller model:",
    selected_controller_name,
)

print()
print("VALIDATION RESULTS")

print(
    "Optimized RAG V2:",
    f"{v2_validation_accuracy:.2%}",
)

print(
    "Actual controller:",
    f"{actual_controller_validation_accuracy:.2%}",
)

print(
    "Validation difference:",
    f"{actual_controller_validation_accuracy - v2_validation_accuracy:+.2%}",
)

print(
    "Oracle ceiling:",
    f"{validation_oracle_accuracy:.2%}",
)

print()
print("CONTROLLER ROUTE DISTRIBUTION")

print(
    controller_validation_results_df[
        "controller_route"
    ]
    .value_counts()
)

print()
print("CONTROLLER ACCURACY BY LANGUAGE")

print(
    controller_validation_results_df.groupby(
        "language"
    )[
        "controller_correct"
    ]
    .agg([
        "sum",
        "count",
        "mean",
    ])
)

print()
print("CONTROLLER ERRORS")

print(
    controller_validation_results_df[
        ~controller_validation_results_df[
            "controller_correct"
        ]
    ][
        [
            "question_id",
            "intent_name",
            "language",
            "controller_route",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("Final test opened:", False)
print("Saved:", controller_model_path)
print("Saved:", controller_validation_result_path)

CORRECTNESS-FIRST ADAPTIVE CONTROLLER

GROUP-AWARE TRAIN CROSS-VALIDATION
            candidate  oof_controller_accuracy  qwen_alone_choices  original_rag_choices  optimized_rag_v2_choices
random_forest_shallow                 0.680556                   1                    23                        48
 logistic_regularized                 0.666667                   6                    22                        44
    logistic_standard                 0.652778                   6                    23                        43

Selected controller model: random_forest_shallow

VALIDATION RESULTS
Optimized RAG V2: 70.83%
Actual controller: 70.83%
Validation difference: +0.00%
Oracle ceiling: 83.33%

CONTROLLER ROUTE DISTRIBUTION
controller_route
optimized_rag_v2    24
Name: count, dtype: int64

CONTROLLER ACCURACY BY LANGUAGE
                  sum  count      mean
language                              
arabic              5      6  0.833333
english             5      6  0.833333
french

In [41]:
# CELL 32A — FIND AVAILABLE EMBEDDING OBJECTS

embedding_candidates = {}

for variable_name, variable_value in globals().items():

    if (
        hasattr(variable_value, "encode")
        and callable(
            getattr(
                variable_value,
                "encode",
                None,
            )
        )
    ):
        embedding_candidates[
            variable_name
        ] = type(
            variable_value
        ).__name__

print("AVAILABLE EMBEDDING OBJECTS:")

for variable_name, class_name in (
    embedding_candidates.items()
):
    print(
        f"{variable_name}: {class_name}"
    )

RuntimeError: dictionary changed size during iteration

In [43]:
# CELL 32A — FIND AVAILABLE EMBEDDING OBJECTS (FIXED)

embedding_candidates = {}

global_variables_snapshot = list(
    globals().items()
)

for (
    variable_name,
    variable_value,
) in global_variables_snapshot:

    if (
        hasattr(
            variable_value,
            "encode",
        )
        and callable(
            getattr(
                variable_value,
                "encode",
                None,
            )
        )
    ):
        embedding_candidates[
            variable_name
        ] = type(
            variable_value
        ).__name__

print("AVAILABLE EMBEDDING OBJECTS:")

for (
    variable_name,
    class_name,
) in embedding_candidates.items():

    print(
        f"{variable_name}: {class_name}"
    )

AVAILABLE EMBEDDING OBJECTS:
__name__: str
__doc__: str
__: str
___: str
_i: str
_ii: str
_iii: str
_i1: str
_i2: str
_i3: str
_i4: str
_i5: str
SECRET_NAME: str
SCHEMA_NAME: str
NEON_DATABASE_URL: str
database_name: str
database_user: str
read_only: str
_i6: str
table_name: str
name: str
query: str
current_table: str
column_name: str
data_type: str
nullable: str
key_label: str
null_label: str
referenced_table: str
referenced_column: str
DATABASE_SCHEMA_TEXT: str
field_name: str
_i7: str
extracted_text: str
full_text: str
preview: str
_i8: str
chunk_text: str
document_name: str
_i9: str
SentenceTransformer: ABCMeta
EMBEDDING_MODEL_ID: str
embedding_model: SentenceTransformer
question: str
_i10: str
MODEL_ID: str
tokenizer: Qwen2Tokenizer
_i11: str
CONTROLLED_VOCABULARY_TEXT: str
_i12: str
safe_test_sql: str
unsafe_test_sql: str
unknown_table_test_sql: str
_i13: str
SMOKE_TEST_QUESTION: str
_i14: str
_i15: str
cte_test_sql: str
_i16: str
REFERENCE_DATE: str
intent_id: str
intent_name: s

In [44]:
# ============================================================
# CELL 33 — SEMANTIC CORRECTNESS-FIRST CONTROLLER
# ============================================================

import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold


# ------------------------------------------------------------
# 1. Encode question meaning using multilingual E5
# ------------------------------------------------------------

def encode_controller_questions(
    questions,
):
    formatted_questions = [
        f"query: {str(question).strip()}"
        for question in questions
    ]

    return embedding_model.encode(
        formatted_questions,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )


semantic_train_embeddings = (
    encode_controller_questions(
        controller_train_df[
            "question"
        ].tolist()
    )
)

semantic_validation_embeddings = (
    encode_controller_questions(
        controller_validation_df[
            "question"
        ].tolist()
    )
)


# ------------------------------------------------------------
# 2. Correctness targets for the three routes
# ------------------------------------------------------------

SEMANTIC_ROUTE_NAMES = [
    "qwen_alone",
    "original_rag",
    "optimized_rag_v2",
]

semantic_correctness_columns = {
    "qwen_alone": "alone_correct",
    "original_rag": (
        "original_rag_correct"
    ),
    "optimized_rag_v2": "v2_correct",
}

semantic_train_correctness = np.column_stack([
    controller_train_df[
        semantic_correctness_columns[
            route_name
        ]
    ]
    .astype(float)
    .to_numpy()
    for route_name
    in SEMANTIC_ROUTE_NAMES
])

semantic_validation_correctness = (
    np.column_stack([
        controller_validation_df[
            semantic_correctness_columns[
                route_name
            ]
        ]
        .astype(float)
        .to_numpy()
        for route_name
        in SEMANTIC_ROUTE_NAMES
    ])
)

semantic_route_priors = (
    semantic_train_correctness.mean(
        axis=0
    )
)


# ------------------------------------------------------------
# 3. Semantic KNN probability estimator
# ------------------------------------------------------------

def semantic_route_probabilities(
    reference_embeddings,
    reference_correctness,
    query_embeddings,
    neighbor_count,
    similarity_power,
    prior_strength,
    route_priors,
):
    similarity_matrix = (
        query_embeddings
        @ reference_embeddings.T
    )

    effective_k = min(
        neighbor_count,
        len(reference_embeddings),
    )

    nearest_indices = np.argsort(
        -similarity_matrix,
        axis=1,
    )[
        :,
        :effective_k,
    ]

    probability_rows = []

    for query_index in range(
        len(query_embeddings)
    ):
        neighbor_indices = (
            nearest_indices[
                query_index
            ]
        )

        neighbor_similarities = (
            similarity_matrix[
                query_index,
                neighbor_indices,
            ]
        )

        # Transform cosine similarity into positive weights.
        weights = np.clip(
            (
                neighbor_similarities
                + 1.0
            )
            / 2.0,
            1e-6,
            1.0,
        ) ** similarity_power

        neighbor_correctness = (
            reference_correctness[
                neighbor_indices
            ]
        )

        weighted_success = (
            weights[:, None]
            * neighbor_correctness
        ).sum(axis=0)

        weight_total = weights.sum()

        prior_weight = (
            prior_strength
            * effective_k
        )

        probabilities = (
            weighted_success
            + prior_weight
            * route_priors
        ) / (
            weight_total
            + prior_weight
        )

        probability_rows.append(
            probabilities
        )

    return np.vstack(
        probability_rows
    )


# ------------------------------------------------------------
# 4. Select configuration using train-only grouped CV
# ------------------------------------------------------------

semantic_candidates = []

for neighbor_count in [
    3,
    5,
    8,
    12,
    16,
]:
    for similarity_power in [
        1.0,
        2.0,
        4.0,
    ]:
        for prior_strength in [
            0.0,
            0.25,
            0.5,
            1.0,
        ]:
            semantic_candidates.append({
                "neighbor_count": (
                    neighbor_count
                ),
                "similarity_power": (
                    similarity_power
                ),
                "prior_strength": (
                    prior_strength
                ),
            })


semantic_group_cv = GroupKFold(
    n_splits=6
)

semantic_cv_rows = []

for configuration in semantic_candidates:

    oof_probabilities = np.zeros(
        (
            len(controller_train_df),
            len(SEMANTIC_ROUTE_NAMES),
        ),
        dtype=float,
    )

    for (
        reference_indices,
        held_out_indices,
    ) in semantic_group_cv.split(
        semantic_train_embeddings,
        groups=controller_train_df[
            "intent_id"
        ],
    ):
        fold_priors = (
            semantic_train_correctness[
                reference_indices
            ]
            .mean(axis=0)
        )

        oof_probabilities[
            held_out_indices
        ] = semantic_route_probabilities(
            reference_embeddings=(
                semantic_train_embeddings[
                    reference_indices
                ]
            ),
            reference_correctness=(
                semantic_train_correctness[
                    reference_indices
                ]
            ),
            query_embeddings=(
                semantic_train_embeddings[
                    held_out_indices
                ]
            ),
            neighbor_count=(
                configuration[
                    "neighbor_count"
                ]
            ),
            similarity_power=(
                configuration[
                    "similarity_power"
                ]
            ),
            prior_strength=(
                configuration[
                    "prior_strength"
                ]
            ),
            route_priors=fold_priors,
        )

    predicted_route_indices = (
        np.argmax(
            oof_probabilities,
            axis=1,
        )
    )

    selected_correctness = (
        semantic_train_correctness[
            np.arange(
                len(controller_train_df)
            ),
            predicted_route_indices,
        ]
    )

    route_counts = np.bincount(
        predicted_route_indices,
        minlength=len(
            SEMANTIC_ROUTE_NAMES
        ),
    )

    semantic_cv_rows.append({
        **configuration,
        "oof_controller_accuracy": (
            selected_correctness.mean()
        ),
        "qwen_alone_choices": int(
            route_counts[0]
        ),
        "original_rag_choices": int(
            route_counts[1]
        ),
        "optimized_rag_v2_choices": int(
            route_counts[2]
        ),
    })


semantic_controller_cv_df = (
    pd.DataFrame(
        semantic_cv_rows
    )
    .sort_values(
        [
            "oof_controller_accuracy",
            "prior_strength",
            "neighbor_count",
        ],
        ascending=[
            False,
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)

best_semantic_configuration = (
    semantic_controller_cv_df.iloc[
        0
    ].to_dict()
)


# ------------------------------------------------------------
# 5. Predict validation routes
# ------------------------------------------------------------

semantic_validation_probabilities = (
    semantic_route_probabilities(
        reference_embeddings=(
            semantic_train_embeddings
        ),
        reference_correctness=(
            semantic_train_correctness
        ),
        query_embeddings=(
            semantic_validation_embeddings
        ),
        neighbor_count=int(
            best_semantic_configuration[
                "neighbor_count"
            ]
        ),
        similarity_power=float(
            best_semantic_configuration[
                "similarity_power"
            ]
        ),
        prior_strength=float(
            best_semantic_configuration[
                "prior_strength"
            ]
        ),
        route_priors=(
            semantic_route_priors
        ),
    )
)

semantic_validation_route_indices = (
    np.argmax(
        semantic_validation_probabilities,
        axis=1,
    )
)

semantic_validation_routes = [
    SEMANTIC_ROUTE_NAMES[
        route_index
    ]
    for route_index
    in semantic_validation_route_indices
]

semantic_validation_selected_correctness = (
    semantic_validation_correctness[
        np.arange(
            len(controller_validation_df)
        ),
        semantic_validation_route_indices,
    ]
)

semantic_validation_accuracy = float(
    semantic_validation_selected_correctness.mean()
)

v2_validation_accuracy = float(
    controller_validation_df[
        "v2_correct"
    ].mean()
)

semantic_validation_oracle = float(
    semantic_validation_correctness
    .max(axis=1)
    .mean()
)


# ------------------------------------------------------------
# 6. Build validation result table
# ------------------------------------------------------------

semantic_controller_validation_df = (
    controller_validation_df[
        [
            "question_id",
            "intent_id",
            "intent_name",
            "language",
            "alone_correct",
            "original_rag_correct",
            "v2_correct",
        ]
    ]
    .copy()
)

semantic_controller_validation_df[
    "controller_route"
] = semantic_validation_routes

semantic_controller_validation_df[
    "controller_correct"
] = (
    semantic_validation_selected_correctness
    .astype(bool)
)

for route_index, route_name in enumerate(
    SEMANTIC_ROUTE_NAMES
):
    semantic_controller_validation_df[
        f"probability_{route_name}"
    ] = (
        semantic_validation_probabilities[
            :,
            route_index,
        ]
    )


# ------------------------------------------------------------
# 7. Save semantic controller
# ------------------------------------------------------------

semantic_controller_path = (
    controller_results_directory
    / "semantic_adaptive_controller.joblib"
)

semantic_cv_path = (
    controller_results_directory
    / "semantic_controller_cv_results.csv"
)

semantic_validation_path = (
    controller_results_directory
    / "semantic_controller_validation_results.csv"
)

joblib.dump(
    {
        "route_names": (
            SEMANTIC_ROUTE_NAMES
        ),
        "correctness_columns": (
            semantic_correctness_columns
        ),
        "configuration": (
            best_semantic_configuration
        ),
        "train_embeddings": (
            semantic_train_embeddings
        ),
        "train_correctness": (
            semantic_train_correctness
        ),
        "route_priors": (
            semantic_route_priors
        ),
        "embedding_model_id": (
            EMBEDDING_MODEL_ID
        ),
        "accuracy_priority": True,
        "latency_used": False,
    },
    semantic_controller_path,
)

semantic_controller_cv_df.to_csv(
    semantic_cv_path,
    index=False,
)

semantic_controller_validation_df.to_csv(
    semantic_validation_path,
    index=False,
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print("=" * 70)
print("SEMANTIC CORRECTNESS-FIRST CONTROLLER")
print("=" * 70)

print()
print("BEST TRAIN-ONLY CONFIGURATION")

print(
    semantic_controller_cv_df.head(
        10
    ).to_string(
        index=False
    )
)

print()
print(
    "Selected neighbor count:",
    int(
        best_semantic_configuration[
            "neighbor_count"
        ]
    ),
)

print(
    "Selected similarity power:",
    best_semantic_configuration[
        "similarity_power"
    ],
)

print(
    "Selected prior strength:",
    best_semantic_configuration[
        "prior_strength"
    ],
)

print()
print("VALIDATION RESULTS")

print(
    "Optimized RAG V2:",
    f"{v2_validation_accuracy:.2%}",
)

print(
    "Semantic controller:",
    f"{semantic_validation_accuracy:.2%}",
)

print(
    "Difference over V2:",
    f"{semantic_validation_accuracy - v2_validation_accuracy:+.2%}",
)

print(
    "Oracle ceiling:",
    f"{semantic_validation_oracle:.2%}",
)

print()
print("ROUTE DISTRIBUTION")

print(
    semantic_controller_validation_df[
        "controller_route"
    ]
    .value_counts()
)

print()
print("ACCURACY BY LANGUAGE")

print(
    semantic_controller_validation_df.groupby(
        "language"
    )[
        "controller_correct"
    ]
    .agg([
        "sum",
        "count",
        "mean",
    ])
)

print()
print("ROUTING DECISIONS")

print(
    semantic_controller_validation_df[
        [
            "question_id",
            "controller_route",
            "controller_correct",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("Latency used for routing:", False)
print("Final test opened:", False)
print("Saved:", semantic_controller_path)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

SEMANTIC CORRECTNESS-FIRST CONTROLLER

BEST TRAIN-ONLY CONFIGURATION
 neighbor_count  similarity_power  prior_strength  oof_controller_accuracy  qwen_alone_choices  original_rag_choices  optimized_rag_v2_choices
             16               1.0            0.00                 0.763889                   0                    11                        61
             16               2.0            0.00                 0.763889                   0                    11                        61
             16               4.0            0.00                 0.763889                   0                    11                        61
             16               1.0            0.25                 0.763889                   0                     9                        63
             16               2.0            0.25                 0.763889                   0                     9                        63
             16               4.0            0.25                 0.76388

In [45]:
# ============================================================
# CELL 34 — THREE-SYSTEM EXECUTION CONSENSUS AUDIT
# ============================================================

import json
import numpy as np
import pandas as pd
from collections import Counter


# ------------------------------------------------------------
# 1. Normalize executed database results
# ------------------------------------------------------------

def make_result_signature(
    execution_result,
):
    if execution_result is None:
        return None

    try:
        normalized_rows = []

        for row in execution_result:
            normalized_row = []

            for value in row:

                if isinstance(
                    value,
                    float,
                ):
                    normalized_value = round(
                        value,
                        8,
                    )
                elif value is None:
                    normalized_value = None
                else:
                    normalized_value = str(
                        value
                    )

                normalized_row.append(
                    normalized_value
                )

            normalized_rows.append(
                normalized_row
            )

        normalized_rows = sorted(
            normalized_rows,
            key=lambda row: json.dumps(
                row,
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            ),
        )

        return json.dumps(
            normalized_rows,
            ensure_ascii=False,
            sort_keys=True,
            default=str,
        )

    except Exception:
        return None


# ------------------------------------------------------------
# 2. Execute existing SQL only — no new generations
# ------------------------------------------------------------

system_result_frames = {
    "qwen_alone": (
        qwen_alone_results_df
    ),
    "original_rag": (
        qwen_rag_results_df
    ),
    "optimized_rag_v2": (
        optimized_rag_v2_results_df
    ),
}

consensus_rows = []

benchmark_subset = (
    multilingual_benchmark_df[
        multilingual_benchmark_df[
            "split"
        ].isin([
            "train",
            "validation",
        ])
    ]
    .copy()
    .reset_index(drop=True)
)

for row_number, benchmark_row in (
    benchmark_subset.iterrows()
):
    question_id = (
        benchmark_row[
            "question_id"
        ]
    )

    system_signatures = {}
    system_correctness = {}
    system_executed = {}

    for (
        system_name,
        result_dataframe,
    ) in system_result_frames.items():

        matching_rows = result_dataframe[
            result_dataframe[
                "question_id"
            ] == question_id
        ]

        if len(matching_rows) != 1:
            raise ValueError(
                f"{system_name}: expected one "
                f"row for {question_id}, "
                f"found {len(matching_rows)}"
            )

        system_row = (
            matching_rows.iloc[0]
        )

        generated_sql = (
            system_row[
                "generated_sql"
            ]
        )

        execution_result = None
        execution_succeeded = False

        try:
            execution_result = (
                execute_for_evaluation(
                    generated_sql
                )
            )
            execution_succeeded = True
        except Exception:
            execution_result = None
            execution_succeeded = False

        system_signatures[
            system_name
        ] = make_result_signature(
            execution_result
        )

        system_correctness[
            system_name
        ] = bool(
            system_row[
                "execution_correct"
            ]
        )

        system_executed[
            system_name
        ] = (
            execution_succeeded
        )

    valid_signatures = {
        system_name: signature
        for (
            system_name,
            signature,
        ) in system_signatures.items()
        if signature is not None
    }

    signature_counts = Counter(
        valid_signatures.values()
    )

    consensus_signature = None

    if signature_counts:
        most_common_signature, count = (
            signature_counts.most_common(
                1
            )[0]
        )

        if count >= 2:
            consensus_signature = (
                most_common_signature
            )

    agreeing_systems = [
        system_name
        for (
            system_name,
            signature,
        ) in valid_signatures.items()
        if (
            consensus_signature is not None
            and signature
            == consensus_signature
        )
    ]

    if consensus_signature is not None:

        # If V2 agrees with consensus, keep V2.
        # Otherwise use the first agreeing RAG route,
        # then Qwen Alone.
        consensus_priority = [
            "optimized_rag_v2",
            "original_rag",
            "qwen_alone",
        ]

        selected_route = next(
            route_name
            for route_name
            in consensus_priority
            if route_name
            in agreeing_systems
        )

        decision_type = "consensus"

    else:
        selected_route = (
            "optimized_rag_v2"
        )
        decision_type = (
            "v2_no_consensus_fallback"
        )

    consensus_rows.append({
        "question_id": question_id,
        "intent_id": (
            benchmark_row[
                "intent_id"
            ]
        ),
        "intent_name": (
            benchmark_row[
                "intent_name"
            ]
        ),
        "split": (
            benchmark_row[
                "split"
            ]
        ),
        "language": (
            benchmark_row[
                "language"
            ]
        ),
        "decision_type": (
            decision_type
        ),
        "selected_route": (
            selected_route
        ),
        "agreeing_system_count": (
            len(agreeing_systems)
        ),
        "agreeing_systems": (
            ", ".join(
                agreeing_systems
            )
        ),
        "controller_correct": (
            system_correctness[
                selected_route
            ]
        ),
        "alone_correct": (
            system_correctness[
                "qwen_alone"
            ]
        ),
        "original_rag_correct": (
            system_correctness[
                "original_rag"
            ]
        ),
        "v2_correct": (
            system_correctness[
                "optimized_rag_v2"
            ]
        ),
    })

    if (
        (row_number + 1) % 12 == 0
        or row_number + 1
        == len(benchmark_subset)
    ):
        print(
            f"Consensus audited: "
            f"{row_number + 1}/"
            f"{len(benchmark_subset)}"
        )


consensus_controller_df = pd.DataFrame(
    consensus_rows
)


# ------------------------------------------------------------
# 3. Evaluate train and validation separately
# ------------------------------------------------------------

consensus_summary_df = (
    consensus_controller_df.groupby(
        "split"
    )
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        controller_correct=(
            "controller_correct",
            "sum",
        ),
        controller_accuracy=(
            "controller_correct",
            "mean",
        ),
        v2_accuracy=(
            "v2_correct",
            "mean",
        ),
    )
    .reset_index()
)

consensus_summary_df[
    "difference_over_v2"
] = (
    consensus_summary_df[
        "controller_accuracy"
    ]
    - consensus_summary_df[
        "v2_accuracy"
    ]
)


# ------------------------------------------------------------
# 4. Save audit
# ------------------------------------------------------------

consensus_audit_path = (
    controller_results_directory
    / "execution_consensus_controller_audit.csv"
)

consensus_controller_df.to_csv(
    consensus_audit_path,
    index=False,
)


# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------

print()
print("=" * 70)
print("THREE-SYSTEM EXECUTION CONSENSUS AUDIT")
print("=" * 70)

print()
print(
    consensus_summary_df.to_string(
        index=False
    )
)

print()
print("DECISION TYPES BY SPLIT")

print(
    consensus_controller_df.groupby(
        [
            "split",
            "decision_type",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("SELECTED ROUTES BY SPLIT")

print(
    consensus_controller_df.groupby(
        [
            "split",
            "selected_route",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("VALIDATION DECISIONS")

print(
    consensus_controller_df[
        consensus_controller_df[
            "split"
        ] == "validation"
    ][
        [
            "question_id",
            "decision_type",
            "selected_route",
            "agreeing_system_count",
            "controller_correct",
            "v2_correct",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("New generations performed:", 0)
print("Latency used for selection:", False)
print("Final test opened:", False)
print("Saved:", consensus_audit_path)

Consensus audited: 12/96
Consensus audited: 24/96
Consensus audited: 36/96
Consensus audited: 48/96
Consensus audited: 60/96
Consensus audited: 72/96
Consensus audited: 84/96
Consensus audited: 96/96

THREE-SYSTEM EXECUTION CONSENSUS AUDIT

     split  questions  controller_correct  controller_accuracy  v2_accuracy  difference_over_v2
     train         72                  54             0.750000     0.750000                 0.0
validation         24                  17             0.708333     0.708333                 0.0

DECISION TYPES BY SPLIT
decision_type  consensus
split                   
train                 72
validation            24

SELECTED ROUTES BY SPLIT
selected_route  optimized_rag_v2
split                           
train                         72
validation                    24

VALIDATION DECISIONS
             question_id decision_type   selected_route  agreeing_system_count  controller_correct  v2_correct
         B4A-003-ENGLISH     consensus optimized_rag_v2

In [46]:
# CELL 34A — INSPECT EXECUTION RETURN STRUCTURE

diagnostic_question_id = (
    "B4A-008-FRENCH"
)

print(
    "QUESTION:",
    diagnostic_question_id,
)

for system_name, result_dataframe in {
    "qwen_alone": qwen_alone_results_df,
    "original_rag": qwen_rag_results_df,
    "optimized_rag_v2": (
        optimized_rag_v2_results_df
    ),
}.items():

    system_row = result_dataframe[
        result_dataframe[
            "question_id"
        ] == diagnostic_question_id
    ].iloc[0]

    generated_sql = system_row[
        "generated_sql"
    ]

    print()
    print("=" * 60)
    print("SYSTEM:", system_name)
    print(
        "RECORDED CORRECT:",
        system_row[
            "execution_correct"
        ],
    )
    print("SQL:")
    print(generated_sql)

    try:
        execution_output = (
            execute_for_evaluation(
                generated_sql
            )
        )

        print(
            "RETURN TYPE:",
            type(
                execution_output
            ),
        )

        print(
            "RETURN VALUE:",
            repr(
                execution_output
            )[:3000],
        )

        if isinstance(
            execution_output,
            dict,
        ):
            print(
                "DICT KEYS:",
                list(
                    execution_output.keys()
                ),
            )

        elif isinstance(
            execution_output,
            tuple,
        ):
            print(
                "TUPLE LENGTH:",
                len(
                    execution_output
                ),
            )

            for item_index, item in enumerate(
                execution_output
            ):
                print(
                    f"ITEM {item_index} TYPE:",
                    type(item),
                )
                print(
                    f"ITEM {item_index} VALUE:",
                    repr(item)[:1000],
                )

    except Exception as error:
        print(
            "EXECUTION ERROR:",
            type(error).__name__,
            str(error),
        )

QUESTION: B4A-008-FRENCH

SYSTEM: qwen_alone
RECORDED CORRECT: True
SQL:
SELECT (COUNT(DISTINCT CASE WHEN status = 'cancelled' THEN order_id END) * 100.0 / COUNT(DISTINCT order_id)) AS cancellation_percentage
FROM build4all_research.orders;
RETURN TYPE: <class 'dict'>
RETURN VALUE: {'executed': True, 'safe': True, 'validation_reason': 'Read-only SQL validated', 'columns': ['cancellation_percentage'], 'rows': [(Decimal('6.8600000000000000'),)], 'error': None}
DICT KEYS: ['executed', 'safe', 'validation_reason', 'columns', 'rows', 'error']

SYSTEM: original_rag
RECORDED CORRECT: False
SQL:
SELECT 
    ROUND(100.0 * COUNT(CASE WHEN status = 'cancelled' THEN 1 END) / NULLIF(COUNT(*), 0), 2) AS cancellation_rate_percent
FROM 
    build4all_research.orders
WHERE 
    order_date >= '2026-08-01 00:00:00 UTC'
    AND order_date < '2026-09-01 00:00:00 UTC';
RETURN TYPE: <class 'dict'>
RETURN VALUE: {'executed': True, 'safe': True, 'validation_reason': 'Read-only SQL validated', 'columns': ['canc

In [47]:
# ============================================================
# CELL 34 — CORRECTED THREE-SYSTEM EXECUTION CONSENSUS AUDIT
# ============================================================

import json
import numpy as np
import pandas as pd

from collections import Counter
from decimal import Decimal
from datetime import date, datetime
from pathlib import Path


# ------------------------------------------------------------
# 1. Normalize individual database values
# ------------------------------------------------------------

def normalize_database_value(value):

    if value is None:
        return None

    if isinstance(
        value,
        (bool, np.bool_),
    ):
        return str(
            bool(value)
        ).lower()

    if isinstance(
        value,
        Decimal,
    ):
        normalized_value = (
            value.normalize()
        )

        if normalized_value == 0:
            normalized_value = Decimal(
                "0"
            )

        return format(
            normalized_value,
            "f",
        )

    if isinstance(
        value,
        (int, np.integer),
    ):
        return str(
            int(value)
        )

    if isinstance(
        value,
        (float, np.floating),
    ):
        return format(
            float(value),
            ".10g",
        )

    if isinstance(
        value,
        (date, datetime),
    ):
        return value.isoformat()

    return str(
        value
    )


# ------------------------------------------------------------
# 2. Create a signature from actual database rows
# ------------------------------------------------------------

def make_result_signature(
    execution_output,
):
    if not isinstance(
        execution_output,
        dict,
    ):
        return None

    if not execution_output.get(
        "executed",
        False,
    ):
        return None

    if execution_output.get(
        "error"
    ) is not None:
        return None

    rows = execution_output.get(
        "rows"
    )

    if rows is None:
        return None

    normalized_rows = []

    for row in rows:

        normalized_row = [
            normalize_database_value(
                value
            )
            for value in row
        ]

        normalized_rows.append(
            normalized_row
        )

    # Ignore output row order.
    normalized_rows = sorted(
        normalized_rows,
        key=lambda row: json.dumps(
            row,
            ensure_ascii=False,
            default=str,
        ),
    )

    # Column aliases are ignored,
    # but row values and column count remain.
    return json.dumps(
        normalized_rows,
        ensure_ascii=False,
        default=str,
    )


# ------------------------------------------------------------
# 3. Existing system results
# ------------------------------------------------------------

system_result_frames = {
    "qwen_alone": (
        qwen_alone_results_df
    ),
    "original_rag": (
        qwen_rag_results_df
    ),
    "optimized_rag_v2": (
        optimized_rag_v2_results_df
    ),
}

benchmark_subset = (
    multilingual_benchmark_df[
        multilingual_benchmark_df[
            "split"
        ].isin([
            "train",
            "validation",
        ])
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Execute existing SQL and compare actual rows
# ------------------------------------------------------------

consensus_rows = []

for row_number, benchmark_row in (
    benchmark_subset.iterrows()
):

    question_id = (
        benchmark_row[
            "question_id"
        ]
    )

    system_signatures = {}
    system_correctness = {}
    system_execution_success = {}

    for (
        system_name,
        result_dataframe,
    ) in system_result_frames.items():

        matching_rows = (
            result_dataframe[
                result_dataframe[
                    "question_id"
                ] == question_id
            ]
        )

        if len(matching_rows) != 1:
            raise ValueError(
                f"{system_name}: expected "
                f"one row for {question_id}, "
                f"found {len(matching_rows)}"
            )

        system_row = (
            matching_rows.iloc[0]
        )

        generated_sql = (
            system_row[
                "generated_sql"
            ]
        )

        try:
            execution_output = (
                execute_for_evaluation(
                    generated_sql
                )
            )
        except Exception as error:
            execution_output = {
                "executed": False,
                "safe": False,
                "rows": None,
                "error": str(error),
            }

        system_signatures[
            system_name
        ] = make_result_signature(
            execution_output
        )

        system_execution_success[
            system_name
        ] = bool(
            isinstance(
                execution_output,
                dict,
            )
            and execution_output.get(
                "executed",
                False,
            )
            and execution_output.get(
                "error"
            ) is None
        )

        system_correctness[
            system_name
        ] = bool(
            system_row[
                "execution_correct"
            ]
        )

    valid_signatures = {
        system_name: signature
        for (
            system_name,
            signature,
        ) in system_signatures.items()
        if signature is not None
    }

    signature_counts = Counter(
        valid_signatures.values()
    )

    consensus_signature = None
    consensus_count = 0

    if signature_counts:

        (
            most_common_signature,
            most_common_count,
        ) = signature_counts.most_common(
            1
        )[0]

        if most_common_count >= 2:
            consensus_signature = (
                most_common_signature
            )
            consensus_count = (
                most_common_count
            )

    agreeing_systems = [
        system_name
        for (
            system_name,
            signature,
        ) in valid_signatures.items()
        if (
            consensus_signature is not None
            and signature
            == consensus_signature
        )
    ]

    # --------------------------------------------------------
    # Correctness-first selection policy
    #
    # If at least two systems agree:
    #   choose V2 when it belongs to consensus;
    #   otherwise choose Original RAG;
    #   otherwise choose Qwen Alone.
    #
    # If there is no consensus:
    #   fall back to strongest system, V2.
    # --------------------------------------------------------

    if consensus_signature is not None:

        consensus_priority = [
            "optimized_rag_v2",
            "original_rag",
            "qwen_alone",
        ]

        selected_route = next(
            route_name
            for route_name
            in consensus_priority
            if route_name
            in agreeing_systems
        )

        decision_type = "consensus"

    else:
        selected_route = (
            "optimized_rag_v2"
        )

        decision_type = (
            "v2_no_consensus_fallback"
        )

    consensus_rows.append({
        "question_id": question_id,
        "intent_id": (
            benchmark_row[
                "intent_id"
            ]
        ),
        "intent_name": (
            benchmark_row[
                "intent_name"
            ]
        ),
        "split": (
            benchmark_row[
                "split"
            ]
        ),
        "language": (
            benchmark_row[
                "language"
            ]
        ),
        "decision_type": (
            decision_type
        ),
        "selected_route": (
            selected_route
        ),
        "agreeing_system_count": (
            consensus_count
        ),
        "agreeing_systems": (
            ", ".join(
                agreeing_systems
            )
        ),
        "controller_correct": (
            system_correctness[
                selected_route
            ]
        ),
        "alone_correct": (
            system_correctness[
                "qwen_alone"
            ]
        ),
        "original_rag_correct": (
            system_correctness[
                "original_rag"
            ]
        ),
        "v2_correct": (
            system_correctness[
                "optimized_rag_v2"
            ]
        ),
        "alone_executed": (
            system_execution_success[
                "qwen_alone"
            ]
        ),
        "original_rag_executed": (
            system_execution_success[
                "original_rag"
            ]
        ),
        "v2_executed": (
            system_execution_success[
                "optimized_rag_v2"
            ]
        ),
        "alone_signature": (
            system_signatures[
                "qwen_alone"
            ]
        ),
        "original_rag_signature": (
            system_signatures[
                "original_rag"
            ]
        ),
        "v2_signature": (
            system_signatures[
                "optimized_rag_v2"
            ]
        ),
    })

    if (
        (row_number + 1) % 12 == 0
        or row_number + 1
        == len(benchmark_subset)
    ):
        print(
            f"Consensus audited: "
            f"{row_number + 1}/"
            f"{len(benchmark_subset)}"
        )


consensus_controller_df = pd.DataFrame(
    consensus_rows
)


# ------------------------------------------------------------
# 5. Train and validation summaries
# ------------------------------------------------------------

consensus_summary_df = (
    consensus_controller_df.groupby(
        "split"
    )
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        controller_correct=(
            "controller_correct",
            "sum",
        ),
        controller_accuracy=(
            "controller_correct",
            "mean",
        ),
        v2_accuracy=(
            "v2_correct",
            "mean",
        ),
        consensus_questions=(
            "decision_type",
            lambda values: (
                values
                == "consensus"
            ).sum(),
        ),
        no_consensus_questions=(
            "decision_type",
            lambda values: (
                values
                == (
                    "v2_no_consensus_fallback"
                )
            ).sum(),
        ),
    )
    .reset_index()
)

consensus_summary_df[
    "difference_over_v2"
] = (
    consensus_summary_df[
        "controller_accuracy"
    ]
    - consensus_summary_df[
        "v2_accuracy"
    ]
)


# ------------------------------------------------------------
# 6. Identify corrections and regressions against V2
# ------------------------------------------------------------

consensus_controller_df[
    "corrected_v2_error"
] = (
    consensus_controller_df[
        "controller_correct"
    ]
    & ~consensus_controller_df[
        "v2_correct"
    ]
)

consensus_controller_df[
    "regressed_from_v2"
] = (
    ~consensus_controller_df[
        "controller_correct"
    ]
    & consensus_controller_df[
        "v2_correct"
    ]
)


# ------------------------------------------------------------
# 7. Save corrected audit
# ------------------------------------------------------------

controller_results_directory = Path(
    "/kaggle/working/"
    "build4all_final_experiment/"
    "results"
)

controller_results_directory.mkdir(
    parents=True,
    exist_ok=True,
)

consensus_audit_path = (
    controller_results_directory
    / (
        "corrected_execution_"
        "consensus_controller_audit.csv"
    )
)

consensus_controller_df.to_csv(
    consensus_audit_path,
    index=False,
)


# ------------------------------------------------------------
# 8. Report
# ------------------------------------------------------------

print()
print("=" * 70)
print("CORRECTED THREE-SYSTEM EXECUTION CONSENSUS AUDIT")
print("=" * 70)

print()
print("SUMMARY")

print(
    consensus_summary_df.to_string(
        index=False
    )
)

print()
print("DECISION TYPES BY SPLIT")

print(
    consensus_controller_df.groupby(
        [
            "split",
            "decision_type",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("SELECTED ROUTES BY SPLIT")

print(
    consensus_controller_df.groupby(
        [
            "split",
            "selected_route",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("CORRECTIONS OVER V2 BY SPLIT")

print(
    consensus_controller_df.groupby(
        "split"
    )[
        "corrected_v2_error"
    ]
    .sum()
)

print()
print("REGRESSIONS FROM V2 BY SPLIT")

print(
    consensus_controller_df.groupby(
        "split"
    )[
        "regressed_from_v2"
    ]
    .sum()
)

print()
print("VALIDATION ROUTING DECISIONS")

print(
    consensus_controller_df[
        consensus_controller_df[
            "split"
        ] == "validation"
    ][
        [
            "question_id",
            "decision_type",
            "selected_route",
            "agreeing_system_count",
            "agreeing_systems",
            "controller_correct",
            "v2_correct",
            "corrected_v2_error",
            "regressed_from_v2",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("New generations performed:", 0)
print("Latency used for selection:", False)
print("Final test opened:", False)
print("Saved:", consensus_audit_path)

Consensus audited: 12/96
Consensus audited: 24/96
Consensus audited: 36/96
Consensus audited: 48/96
Consensus audited: 60/96
Consensus audited: 72/96
Consensus audited: 84/96
Consensus audited: 96/96

CORRECTED THREE-SYSTEM EXECUTION CONSENSUS AUDIT

SUMMARY
     split  questions  controller_correct  controller_accuracy  v2_accuracy  consensus_questions  no_consensus_questions  difference_over_v2
     train         72                  54             0.750000     0.750000                   49                      23                 0.0
validation         24                  17             0.708333     0.708333                   16                       8                 0.0

DECISION TYPES BY SPLIT
decision_type  consensus  v2_no_consensus_fallback
split                                             
train                 49                        23
validation            16                         8

SELECTED ROUTES BY SPLIT
selected_route  optimized_rag_v2  original_rag
split           

In [48]:
# ============================================================
# CELL 35 — BUILD BINARY CONTROLLER LABELS
# Qwen Alone vs Optimized RAG V2
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Create correctness-first binary target
#
# Target:
#   0 = Qwen Alone
#   1 = Optimized RAG V2
#
# If both are correct:
#   choose the lower recorded generation latency.
#
# If both are wrong:
#   exclude the example from controller training.
# ------------------------------------------------------------

def build_binary_controller_label(
    row,
):
    alone_correct = bool(
        row["alone_correct"]
    )

    v2_correct = bool(
        row["v2_correct"]
    )

    alone_latency = row.get(
        "alone_latency",
        np.nan,
    )

    v2_latency = row.get(
        "v2_latency",
        np.nan,
    )

    # Qwen Alone is the only correct route.
    if (
        alone_correct
        and not v2_correct
    ):
        return pd.Series({
            "binary_target": 0,
            "target_route": (
                "qwen_alone"
            ),
            "target_reason": (
                "qwen_only_correct"
            ),
            "controller_trainable": (
                True
            ),
        })

    # Optimized RAG V2 is the only correct route.
    if (
        v2_correct
        and not alone_correct
    ):
        return pd.Series({
            "binary_target": 1,
            "target_route": (
                "optimized_rag_v2"
            ),
            "target_reason": (
                "v2_only_correct"
            ),
            "controller_trainable": (
                True
            ),
        })

    # Both systems are correct.
    # Latency is used only here.
    if (
        alone_correct
        and v2_correct
    ):
        if (
            pd.notna(alone_latency)
            and pd.notna(v2_latency)
        ):
            if (
                float(alone_latency)
                <= float(v2_latency)
            ):
                selected_target = 0
                selected_route = (
                    "qwen_alone"
                )
                selected_reason = (
                    "both_correct_"
                    "qwen_faster"
                )
            else:
                selected_target = 1
                selected_route = (
                    "optimized_rag_v2"
                )
                selected_reason = (
                    "both_correct_"
                    "v2_faster"
                )

        else:
            # If latency is missing,
            # prefer Qwen Alone because it
            # avoids retrieval.
            selected_target = 0
            selected_route = (
                "qwen_alone"
            )
            selected_reason = (
                "both_correct_"
                "latency_missing_"
                "qwen_default"
            )

        return pd.Series({
            "binary_target": (
                selected_target
            ),
            "target_route": (
                selected_route
            ),
            "target_reason": (
                selected_reason
            ),
            "controller_trainable": (
                True
            ),
        })

    # Both systems are wrong.
    # There is no correct routing label.
    return pd.Series({
        "binary_target": np.nan,
        "target_route": (
            "no_correct_route"
        ),
        "target_reason": (
            "both_wrong_excluded"
        ),
        "controller_trainable": (
            False
        ),
    })


# ------------------------------------------------------------
# 2. Apply labeling policy
# ------------------------------------------------------------

binary_controller_dataset_df = (
    controller_dataset_df.copy()
)

binary_label_df = (
    binary_controller_dataset_df.apply(
        build_binary_controller_label,
        axis=1,
    )
)

binary_controller_dataset_df = pd.concat(
    [
        binary_controller_dataset_df,
        binary_label_df,
    ],
    axis=1,
)

binary_controller_dataset_df[
    "controller_trainable"
] = (
    binary_controller_dataset_df[
        "controller_trainable"
    ].astype(bool)
)


# ------------------------------------------------------------
# 3. Create train and validation partitions
# ------------------------------------------------------------

binary_controller_train_df = (
    binary_controller_dataset_df[
        (
            binary_controller_dataset_df[
                "split"
            ] == "train"
        )
        & binary_controller_dataset_df[
            "controller_trainable"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

binary_controller_validation_df = (
    binary_controller_dataset_df[
        (
            binary_controller_dataset_df[
                "split"
            ] == "validation"
        )
    ]
    .copy()
    .reset_index(drop=True)
)

binary_controller_train_df[
    "binary_target"
] = (
    binary_controller_train_df[
        "binary_target"
    ].astype(int)
)


# ------------------------------------------------------------
# 4. Calculate binary oracle ceiling
# ------------------------------------------------------------

binary_controller_dataset_df[
    "binary_oracle_correct"
] = (
    binary_controller_dataset_df[
        "alone_correct"
    ]
    | binary_controller_dataset_df[
        "v2_correct"
    ]
)

binary_oracle_summary_df = (
    binary_controller_dataset_df.groupby(
        "split"
    )
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        qwen_alone_accuracy=(
            "alone_correct",
            "mean",
        ),
        optimized_rag_v2_accuracy=(
            "v2_correct",
            "mean",
        ),
        binary_oracle_accuracy=(
            "binary_oracle_correct",
            "mean",
        ),
        excluded_both_wrong=(
            "controller_trainable",
            lambda values: (
                ~values
            ).sum(),
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# 5. Save binary controller data
# ------------------------------------------------------------

binary_dataset_path = (
    controller_results_directory
    / "binary_controller_dataset.csv"
)

binary_train_path = (
    controller_results_directory
    / "binary_controller_train.csv"
)

binary_validation_path = (
    controller_results_directory
    / "binary_controller_validation.csv"
)

binary_controller_dataset_df.to_csv(
    binary_dataset_path,
    index=False,
)

binary_controller_train_df.to_csv(
    binary_train_path,
    index=False,
)

binary_controller_validation_df.to_csv(
    binary_validation_path,
    index=False,
)


# ------------------------------------------------------------
# 6. Audit
# ------------------------------------------------------------

print("=" * 70)
print("BINARY CONTROLLER LABELS READY")
print("=" * 70)

print()
print("BINARY ROUTES BY SPLIT")

print(
    binary_controller_dataset_df[
        binary_controller_dataset_df[
            "controller_trainable"
        ]
    ]
    .groupby(
        [
            "split",
            "target_route",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("LABEL REASONS BY SPLIT")

print(
    binary_controller_dataset_df.groupby(
        [
            "split",
            "target_reason",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
)

print()
print("BINARY ORACLE CEILING")

display(
    binary_oracle_summary_df.style.format({
        "qwen_alone_accuracy": (
            "{:.2%}"
        ),
        "optimized_rag_v2_accuracy": (
            "{:.2%}"
        ),
        "binary_oracle_accuracy": (
            "{:.2%}"
        ),
    })
)

print()
print(
    "Trainable train examples:",
    len(binary_controller_train_df),
)

print(
    "Total validation examples:",
    len(binary_controller_validation_df),
)

print(
    "Latency used only when both correct:",
    True,
)

print(
    "Original RAG included as route:",
    False,
)

print(
    "Final test opened:",
    False,
)

print()
print("Saved:", binary_dataset_path)
print("Saved:", binary_train_path)
print("Saved:", binary_validation_path)

BINARY CONTROLLER LABELS READY

BINARY ROUTES BY SPLIT
target_route  optimized_rag_v2  qwen_alone
split                                     
train                       33          22
validation                   9           9

LABEL REASONS BY SPLIT
target_reason  both_correct_qwen_faster  both_wrong_excluded  \
split                                                          
train                                21                   17   
validation                            8                    6   

target_reason  qwen_only_correct  v2_only_correct  
split                                              
train                          1               33  
validation                     1                9  

BINARY ORACLE CEILING


,split,questions,qwen_alone_accuracy,optimized_rag_v2_accuracy,binary_oracle_accuracy,excluded_both_wrong
0,train,72,30.56%,75.00%,76.39%,17
1,validation,24,37.50%,70.83%,75.00%,6



Trainable train examples: 55
Total validation examples: 24
Latency used only when both correct: True
Original RAG included as route: False
Final test opened: False

Saved: /kaggle/working/build4all_final_experiment/results/binary_controller_dataset.csv
Saved: /kaggle/working/build4all_final_experiment/results/binary_controller_train.csv
Saved: /kaggle/working/build4all_final_experiment/results/binary_controller_validation.csv


In [49]:
# ============================================================
# CELL 36 — BINARY LOGISTIC REGRESSION CONTROLLER
# Qwen Alone vs Optimized RAG V2
# ============================================================

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold


# ------------------------------------------------------------
# 1. Features available before generation
# ------------------------------------------------------------

binary_categorical_features = [
    "language",
    "top_document",
]

binary_excluded_features = {
    "question_id",
    "intent_id",
    "intent_name",
    "split",
    "question",

    "controller_target",
    "binary_target",
    "target_route",
    "target_reason",
    "controller_trainable",

    "all_systems_wrong",
    "binary_oracle_correct",

    "alone_correct",
    "original_rag_correct",
    "v2_correct",

    # Latency is used for target creation and evaluation,
    # but never as an input feature.
    "alone_latency",
    "original_rag_latency",
    "v2_latency",
}

binary_numeric_features = [
    column
    for column
    in binary_controller_dataset_df.columns
    if (
        column
        not in binary_excluded_features
        and column
        not in binary_categorical_features
        and pd.api.types.is_numeric_dtype(
            binary_controller_dataset_df[
                column
            ]
        )
    )
]

binary_feature_columns = (
    binary_categorical_features
    + binary_numeric_features
)


# ------------------------------------------------------------
# 2. Full train data
#
# Group CV holds complete intents out together.
# Only trainable rows are used to fit the classifier.
# All held-out rows are used to measure system accuracy.
# ------------------------------------------------------------

binary_full_train_df = (
    binary_controller_dataset_df[
        binary_controller_dataset_df[
            "split"
        ] == "train"
    ]
    .copy()
    .reset_index(drop=True)
)

X_binary_full_train = (
    binary_full_train_df[
        binary_feature_columns
    ]
)

binary_train_groups = (
    binary_full_train_df[
        "intent_id"
    ]
)


# ------------------------------------------------------------
# 3. Candidate configurations
#
# Probability means:
#   P(Optimized RAG V2 should be selected)
# ------------------------------------------------------------

binary_candidate_configurations = []

for regularization_c in [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    1.00,
    2.00,
]:
    for class_weight_name in [
        "none",
        "balanced",
    ]:
        for decision_threshold in [
            0.30,
            0.35,
            0.40,
            0.45,
            0.50,
            0.55,
            0.60,
            0.65,
            0.70,
        ]:
            binary_candidate_configurations.append({
                "regularization_c": (
                    regularization_c
                ),
                "class_weight_name": (
                    class_weight_name
                ),
                "decision_threshold": (
                    decision_threshold
                ),
            })


# ------------------------------------------------------------
# 4. Group-aware out-of-fold predictions
# ------------------------------------------------------------

binary_group_cv = GroupKFold(
    n_splits=6
)

binary_cv_rows = []

for configuration in (
    binary_candidate_configurations
):

    oof_v2_probabilities = np.zeros(
        len(binary_full_train_df),
        dtype=float,
    )

    for (
        fit_indices,
        held_out_indices,
    ) in binary_group_cv.split(
        X_binary_full_train,
        groups=binary_train_groups,
    ):

        fit_frame = (
            binary_full_train_df.iloc[
                fit_indices
            ]
        )

        fit_trainable_mask = (
            fit_frame[
                "controller_trainable"
            ].to_numpy()
        )

        eligible_fit_indices = (
            fit_indices[
                fit_trainable_mask
            ]
        )

        X_fold_fit = (
            X_binary_full_train.iloc[
                eligible_fit_indices
            ]
        )

        y_fold_fit = (
            binary_full_train_df.iloc[
                eligible_fit_indices
            ][
                "binary_target"
            ]
            .astype(int)
        )

        fold_preprocessor = (
            ColumnTransformer(
                transformers=[
                    (
                        "categorical",
                        OneHotEncoder(
                            handle_unknown=(
                                "ignore"
                            ),
                        ),
                        binary_categorical_features,
                    ),
                    (
                        "numeric",
                        StandardScaler(),
                        binary_numeric_features,
                    ),
                ],
                remainder="drop",
            )
        )

        selected_class_weight = (
            None
            if configuration[
                "class_weight_name"
            ] == "none"
            else "balanced"
        )

        fold_controller = Pipeline(
            steps=[
                (
                    "preprocessor",
                    fold_preprocessor,
                ),
                (
                    "classifier",
                    LogisticRegression(
                        C=configuration[
                            "regularization_c"
                        ],
                        class_weight=(
                            selected_class_weight
                        ),
                        max_iter=5000,
                        solver="liblinear",
                        random_state=42,
                    ),
                ),
            ]
        )

        fold_controller.fit(
            X_fold_fit,
            y_fold_fit,
        )

        fold_classes = list(
            fold_controller.named_steps[
                "classifier"
            ].classes_
        )

        v2_class_index = (
            fold_classes.index(1)
        )

        oof_v2_probabilities[
            held_out_indices
        ] = (
            fold_controller.predict_proba(
                X_binary_full_train.iloc[
                    held_out_indices
                ]
            )[
                :,
                v2_class_index,
            ]
        )

    oof_predicted_v2 = (
        oof_v2_probabilities
        >= configuration[
            "decision_threshold"
        ]
    )

    oof_selected_correctness = np.where(
        oof_predicted_v2,
        binary_full_train_df[
            "v2_correct"
        ].to_numpy(
            dtype=bool
        ),
        binary_full_train_df[
            "alone_correct"
        ].to_numpy(
            dtype=bool
        ),
    )

    oof_controller_accuracy = float(
        oof_selected_correctness.mean()
    )

    both_correct_mask = (
        binary_full_train_df[
            "alone_correct"
        ].to_numpy(
            dtype=bool
        )
        & binary_full_train_df[
            "v2_correct"
        ].to_numpy(
            dtype=bool
        )
    )

    selected_latencies = np.where(
        oof_predicted_v2,
        binary_full_train_df[
            "v2_latency"
        ].to_numpy(
            dtype=float
        ),
        binary_full_train_df[
            "alone_latency"
        ].to_numpy(
            dtype=float
        ),
    )

    if both_correct_mask.any():
        both_correct_mean_latency = float(
            np.nanmean(
                selected_latencies[
                    both_correct_mask
                ]
            )
        )
    else:
        both_correct_mean_latency = np.inf

    binary_cv_rows.append({
        **configuration,
        "oof_controller_accuracy": (
            oof_controller_accuracy
        ),
        "qwen_alone_choices": int(
            (~oof_predicted_v2).sum()
        ),
        "optimized_rag_v2_choices": int(
            oof_predicted_v2.sum()
        ),
        "both_correct_mean_latency": (
            both_correct_mean_latency
        ),
    })


# ------------------------------------------------------------
# 5. Select configuration
#
# Primary: highest accuracy.
# Secondary: lowest latency when both systems are correct.
# ------------------------------------------------------------

binary_logistic_cv_df = (
    pd.DataFrame(
        binary_cv_rows
    )
    .sort_values(
        [
            "oof_controller_accuracy",
            "both_correct_mean_latency",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

best_binary_configuration = (
    binary_logistic_cv_df.iloc[
        0
    ].to_dict()
)


# ------------------------------------------------------------
# 6. Train final logistic regression on all trainable rows
# ------------------------------------------------------------

final_binary_preprocessor = (
    ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(
                    handle_unknown="ignore",
                ),
                binary_categorical_features,
            ),
            (
                "numeric",
                StandardScaler(),
                binary_numeric_features,
            ),
        ],
        remainder="drop",
    )
)

final_class_weight = (
    None
    if best_binary_configuration[
        "class_weight_name"
    ] == "none"
    else "balanced"
)

binary_logistic_controller = Pipeline(
    steps=[
        (
            "preprocessor",
            final_binary_preprocessor,
        ),
        (
            "classifier",
            LogisticRegression(
                C=float(
                    best_binary_configuration[
                        "regularization_c"
                    ]
                ),
                class_weight=(
                    final_class_weight
                ),
                max_iter=5000,
                solver="liblinear",
                random_state=42,
            ),
        ),
    ]
)

binary_logistic_controller.fit(
    binary_controller_train_df[
        binary_feature_columns
    ],
    binary_controller_train_df[
        "binary_target"
    ].astype(int),
)

binary_controller_threshold = float(
    best_binary_configuration[
        "decision_threshold"
    ]
)


# ------------------------------------------------------------
# 7. Evaluate on validation
# ------------------------------------------------------------

X_binary_validation = (
    binary_controller_validation_df[
        binary_feature_columns
    ]
)

binary_classes = list(
    binary_logistic_controller.named_steps[
        "classifier"
    ].classes_
)

binary_v2_class_index = (
    binary_classes.index(1)
)

validation_v2_probabilities = (
    binary_logistic_controller.predict_proba(
        X_binary_validation
    )[
        :,
        binary_v2_class_index,
    ]
)

validation_predicted_v2 = (
    validation_v2_probabilities
    >= binary_controller_threshold
)

validation_routes = np.where(
    validation_predicted_v2,
    "optimized_rag_v2",
    "qwen_alone",
)

validation_controller_correct = np.where(
    validation_predicted_v2,
    binary_controller_validation_df[
        "v2_correct"
    ].to_numpy(
        dtype=bool
    ),
    binary_controller_validation_df[
        "alone_correct"
    ].to_numpy(
        dtype=bool
    ),
)

validation_selected_latency = np.where(
    validation_predicted_v2,
    binary_controller_validation_df[
        "v2_latency"
    ].to_numpy(
        dtype=float
    ),
    binary_controller_validation_df[
        "alone_latency"
    ].to_numpy(
        dtype=float
    ),
)


# ------------------------------------------------------------
# 8. Validation result table
# ------------------------------------------------------------

binary_logistic_validation_df = (
    binary_controller_validation_df[
        [
            "question_id",
            "intent_id",
            "intent_name",
            "language",
            "alone_correct",
            "v2_correct",
            "alone_latency",
            "v2_latency",
        ]
    ]
    .copy()
)

binary_logistic_validation_df[
    "v2_probability"
] = validation_v2_probabilities

binary_logistic_validation_df[
    "selected_route"
] = validation_routes

binary_logistic_validation_df[
    "controller_correct"
] = validation_controller_correct

binary_logistic_validation_df[
    "selected_latency"
] = validation_selected_latency


# ------------------------------------------------------------
# 9. Summary metrics
# ------------------------------------------------------------

binary_logistic_validation_accuracy = float(
    validation_controller_correct.mean()
)

binary_v2_validation_accuracy = float(
    binary_controller_validation_df[
        "v2_correct"
    ].mean()
)

binary_alone_validation_accuracy = float(
    binary_controller_validation_df[
        "alone_correct"
    ].mean()
)

binary_validation_oracle_accuracy = float(
    (
        binary_controller_validation_df[
            "alone_correct"
        ]
        | binary_controller_validation_df[
            "v2_correct"
        ]
    ).mean()
)


# ------------------------------------------------------------
# 10. Save controller
# ------------------------------------------------------------

binary_controller_model_path = (
    controller_results_directory
    / "binary_logistic_controller.joblib"
)

binary_controller_cv_path = (
    controller_results_directory
    / "binary_logistic_controller_cv.csv"
)

binary_controller_validation_path = (
    controller_results_directory
    / "binary_logistic_controller_validation.csv"
)

joblib.dump(
    {
        "model": (
            binary_logistic_controller
        ),
        "threshold": (
            binary_controller_threshold
        ),
        "feature_columns": (
            binary_feature_columns
        ),
        "categorical_features": (
            binary_categorical_features
        ),
        "numeric_features": (
            binary_numeric_features
        ),
        "routes": {
            0: "qwen_alone",
            1: "optimized_rag_v2",
        },
        "correctness_priority": True,
        "latency_tiebreak_only": True,
        "original_rag_route": False,
    },
    binary_controller_model_path,
)

binary_logistic_cv_df.to_csv(
    binary_controller_cv_path,
    index=False,
)

binary_logistic_validation_df.to_csv(
    binary_controller_validation_path,
    index=False,
)


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print("=" * 70)
print("BINARY LOGISTIC REGRESSION CONTROLLER")
print("=" * 70)

print()
print("TOP TRAIN-ONLY CV CONFIGURATIONS")

print(
    binary_logistic_cv_df.head(
        10
    ).to_string(
        index=False
    )
)

print()
print("SELECTED CONFIGURATION")

print(
    "C:",
    best_binary_configuration[
        "regularization_c"
    ],
)

print(
    "Class weight:",
    best_binary_configuration[
        "class_weight_name"
    ],
)

print(
    "Decision threshold:",
    binary_controller_threshold,
)

print()
print("VALIDATION RESULTS")

print(
    "Qwen Alone:",
    f"{binary_alone_validation_accuracy:.2%}",
)

print(
    "Optimized RAG V2:",
    f"{binary_v2_validation_accuracy:.2%}",
)

print(
    "Binary controller:",
    f"{binary_logistic_validation_accuracy:.2%}",
)

print(
    "Difference over V2:",
    f"{binary_logistic_validation_accuracy - binary_v2_validation_accuracy:+.2%}",
)

print(
    "Binary oracle ceiling:",
    f"{binary_validation_oracle_accuracy:.2%}",
)

print()
print("ROUTE DISTRIBUTION")

print(
    binary_logistic_validation_df[
        "selected_route"
    ]
    .value_counts()
)

print()
print("ACCURACY BY LANGUAGE")

print(
    binary_logistic_validation_df.groupby(
        "language"
    )[
        "controller_correct"
    ]
    .agg([
        "sum",
        "count",
        "mean",
    ])
)

print()
print("VALIDATION DECISIONS")

print(
    binary_logistic_validation_df[
        [
            "question_id",
            "v2_probability",
            "selected_route",
            "controller_correct",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("Correctness priority:", True)
print("Latency used only as tie-break:", True)
print("Original RAG used as route:", False)
print("Final test opened:", False)
print("Saved:", binary_controller_model_path)

BINARY LOGISTIC REGRESSION CONTROLLER

TOP TRAIN-ONLY CV CONFIGURATIONS
 regularization_c class_weight_name  decision_threshold  oof_controller_accuracy  qwen_alone_choices  optimized_rag_v2_choices  both_correct_mean_latency
             0.50          balanced                0.30                     0.75                  23                        49                   4.991210
             0.25              none                0.35                     0.75                  21                        51                   5.291805
             0.50              none                0.30                     0.75                  20                        52                   5.419229
             0.25              none                0.30                     0.75                  19                        53                   5.551271
             0.10              none                0.30                     0.75                  15                        57                   5.945395
    

In [50]:
# ============================================================
# CELL 37 — BINARY CONTROLLER EFFICIENCY AUDIT
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Calculate validation latency
# ------------------------------------------------------------

always_v2_total_latency = float(
    binary_logistic_validation_df[
        "v2_latency"
    ].sum()
)

controller_total_latency = float(
    binary_logistic_validation_df[
        "selected_latency"
    ].sum()
)

always_v2_average_latency = float(
    binary_logistic_validation_df[
        "v2_latency"
    ].mean()
)

controller_average_latency = float(
    binary_logistic_validation_df[
        "selected_latency"
    ].mean()
)

latency_saved_seconds = (
    always_v2_total_latency
    - controller_total_latency
)

latency_reduction_percent = (
    100.0
    * latency_saved_seconds
    / always_v2_total_latency
)

qwen_route_count = int(
    (
        binary_logistic_validation_df[
            "selected_route"
        ] == "qwen_alone"
    ).sum()
)

v2_route_count = int(
    (
        binary_logistic_validation_df[
            "selected_route"
        ] == "optimized_rag_v2"
    ).sum()
)

rag_avoidance_percent = (
    100.0
    * qwen_route_count
    / len(
        binary_logistic_validation_df
    )
)


# ------------------------------------------------------------
# 2. Detect corrections and regressions
# ------------------------------------------------------------

binary_logistic_validation_df[
    "corrected_v2_error"
] = (
    binary_logistic_validation_df[
        "controller_correct"
    ]
    & ~binary_logistic_validation_df[
        "v2_correct"
    ]
)

binary_logistic_validation_df[
    "regressed_from_v2"
] = (
    ~binary_logistic_validation_df[
        "controller_correct"
    ]
    & binary_logistic_validation_df[
        "v2_correct"
    ]
)

corrections_over_v2 = int(
    binary_logistic_validation_df[
        "corrected_v2_error"
    ].sum()
)

regressions_from_v2 = int(
    binary_logistic_validation_df[
        "regressed_from_v2"
    ].sum()
)


# ------------------------------------------------------------
# 3. Create final validation summary
# ------------------------------------------------------------

binary_controller_efficiency_summary = {
    "qwen_alone_accuracy": float(
        binary_controller_validation_df[
            "alone_correct"
        ].mean()
    ),
    "optimized_rag_v2_accuracy": float(
        binary_controller_validation_df[
            "v2_correct"
        ].mean()
    ),
    "binary_controller_accuracy": float(
        binary_logistic_validation_df[
            "controller_correct"
        ].mean()
    ),
    "binary_oracle_ceiling": float(
        (
            binary_controller_validation_df[
                "alone_correct"
            ]
            | binary_controller_validation_df[
                "v2_correct"
            ]
        ).mean()
    ),
    "qwen_alone_routes": (
        qwen_route_count
    ),
    "optimized_rag_v2_routes": (
        v2_route_count
    ),
    "rag_avoidance_percent": (
        rag_avoidance_percent
    ),
    "always_v2_total_latency": (
        always_v2_total_latency
    ),
    "controller_total_latency": (
        controller_total_latency
    ),
    "always_v2_average_latency": (
        always_v2_average_latency
    ),
    "controller_average_latency": (
        controller_average_latency
    ),
    "latency_saved_seconds": (
        latency_saved_seconds
    ),
    "latency_reduction_percent": (
        latency_reduction_percent
    ),
    "corrections_over_v2": (
        corrections_over_v2
    ),
    "regressions_from_v2": (
        regressions_from_v2
    ),
}


# ------------------------------------------------------------
# 4. Save
# ------------------------------------------------------------

efficiency_summary_path = (
    controller_results_directory
    / "binary_controller_efficiency_summary.json"
)

efficiency_decisions_path = (
    controller_results_directory
    / "binary_controller_efficiency_decisions.csv"
)

with open(
    efficiency_summary_path,
    "w",
    encoding="utf-8",
) as output_file:
    json.dump(
        binary_controller_efficiency_summary,
        output_file,
        indent=2,
    )

binary_logistic_validation_df.to_csv(
    efficiency_decisions_path,
    index=False,
)


# ------------------------------------------------------------
# 5. Report
# ------------------------------------------------------------

print("=" * 70)
print("BINARY CONTROLLER — ACCURACY AND EFFICIENCY")
print("=" * 70)

print()
print(
    "Qwen Alone accuracy:",
    f"{binary_controller_efficiency_summary['qwen_alone_accuracy']:.2%}",
)

print(
    "Optimized RAG V2 accuracy:",
    f"{binary_controller_efficiency_summary['optimized_rag_v2_accuracy']:.2%}",
)

print(
    "Binary controller accuracy:",
    f"{binary_controller_efficiency_summary['binary_controller_accuracy']:.2%}",
)

print(
    "Binary oracle ceiling:",
    f"{binary_controller_efficiency_summary['binary_oracle_ceiling']:.2%}",
)

print()
print(
    "Qwen Alone routes:",
    qwen_route_count,
)

print(
    "Optimized RAG V2 routes:",
    v2_route_count,
)

print(
    "RAG avoided:",
    f"{rag_avoidance_percent:.2f}%",
)

print()
print(
    "Always-V2 average latency:",
    f"{always_v2_average_latency:.4f} seconds",
)

print(
    "Controller average latency:",
    f"{controller_average_latency:.4f} seconds",
)

print(
    "Total latency saved:",
    f"{latency_saved_seconds:.4f} seconds",
)

print(
    "Latency reduction:",
    f"{latency_reduction_percent:.2f}%",
)

print()
print(
    "Corrections over V2:",
    corrections_over_v2,
)

print(
    "Regressions from V2:",
    regressions_from_v2,
)

print()
print("Correctness priority:", True)
print("Original RAG route:", False)
print("Final test opened:", False)
print("Saved:", efficiency_summary_path)

BINARY CONTROLLER — ACCURACY AND EFFICIENCY

Qwen Alone accuracy: 37.50%
Optimized RAG V2 accuracy: 70.83%
Binary controller accuracy: 70.83%
Binary oracle ceiling: 75.00%

Qwen Alone routes: 4
Optimized RAG V2 routes: 20
RAG avoided: 16.67%

Always-V2 average latency: 9.9132 seconds
Controller average latency: 9.3479 seconds
Total latency saved: 13.5662 seconds
Latency reduction: 5.70%

Corrections over V2: 0
Regressions from V2: 0

Correctness priority: True
Original RAG route: False
Final test opened: False
Saved: /kaggle/working/build4all_final_experiment/results/binary_controller_efficiency_summary.json


In [51]:
# ============================================================
# CELL 38 — LOCKED FINAL TEST
# Full roadmap:
# Qwen Alone vs Original RAG vs Optimized RAG V2
# vs Binary Logistic Controller
# ============================================================

import json
import time
import numpy as np
import pandas as pd

from pathlib import Path


# ------------------------------------------------------------
# 1. Freeze final experiment
# ------------------------------------------------------------

FINAL_TEST_OPENED = True

FINAL_CONTROLLER_CONFIGURATION = {
    "controller_type": (
        "binary_logistic_regression"
    ),
    "routes": [
        "qwen_alone",
        "optimized_rag_v2",
    ],
    "regularization_c": 0.5,
    "class_weight": "balanced",
    "decision_threshold": 0.30,
    "correctness_priority": True,
    "latency_tiebreak_only": True,
    "original_rag_comparison_only": True,
    "post_test_tuning_allowed": False,
}


# ------------------------------------------------------------
# 2. Final-test paths
# ------------------------------------------------------------

final_results_directory = Path(
    "/kaggle/working/"
    "build4all_final_experiment/"
    "results"
)

final_results_directory.mkdir(
    parents=True,
    exist_ok=True,
)

final_checkpoint_path = (
    final_results_directory
    / "locked_final_test_checkpoint.jsonl"
)

final_csv_path = (
    final_results_directory
    / "locked_final_test_results.csv"
)

final_summary_path = (
    final_results_directory
    / "locked_final_test_summary.json"
)


# ------------------------------------------------------------
# 3. Open final test exactly once
# ------------------------------------------------------------

final_test_df = (
    multilingual_benchmark_df[
        multilingual_benchmark_df[
            "split"
        ] == "final_test"
    ]
    .copy()
    .reset_index(drop=True)
)

if len(final_test_df) != 24:
    raise ValueError(
        "Expected 24 final-test questions, "
        f"found {len(final_test_df)}"
    )

if final_test_df[
    "question_id"
].duplicated().any():
    raise ValueError(
        "Duplicate final-test question IDs"
    )


# ------------------------------------------------------------
# 4. Resume safely if Kaggle stopped midway
# ------------------------------------------------------------

completed_final_rows = []

if final_checkpoint_path.exists():

    with open(
        final_checkpoint_path,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:

        for line in checkpoint_file:

            stripped_line = (
                line.strip()
            )

            if stripped_line:
                completed_final_rows.append(
                    json.loads(
                        stripped_line
                    )
                )

completed_question_ids = {
    row["question_id"]
    for row in completed_final_rows
}

remaining_final_df = (
    final_test_df[
        ~final_test_df[
            "question_id"
        ].isin(
            completed_question_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 70)
print("LOCKED FINAL TEST")
print("=" * 70)

print(
    "Target questions:",
    len(final_test_df),
)

print(
    "Previously completed:",
    len(completed_final_rows),
)

print(
    "Remaining:",
    len(remaining_final_df),
)

print(
    "Controller frozen:",
    True,
)

print(
    "Post-test tuning allowed:",
    False,
)

print(
    "Final test opened:",
    FINAL_TEST_OPENED,
)


# ------------------------------------------------------------
# 5. Helper: execute gold query
# ------------------------------------------------------------

def get_gold_rows_from_sql(
    gold_sql,
):
    gold_execution = (
        execute_for_evaluation(
            gold_sql
        )
    )

    if not isinstance(
        gold_execution,
        dict,
    ):
        raise ValueError(
            "Gold execution did not "
            "return a dictionary"
        )

    if not gold_execution.get(
        "executed",
        False,
    ):
        raise ValueError(
            "Gold SQL did not execute: "
            f"{gold_execution.get('error')}"
        )

    return gold_execution.get(
        "rows",
        [],
    )


# ------------------------------------------------------------
# 6. Helper: evaluate one generated query
# ------------------------------------------------------------

def evaluate_final_query(
    generated_sql,
    gold_rows,
):
    try:
        evaluation = (
            evaluate_generated_sql(
                generated_sql,
                gold_rows,
            )
        )

        if not isinstance(
            evaluation,
            dict,
        ):
            raise ValueError(
                "Evaluator did not return "
                "a dictionary"
            )

        return {
            "sql_safe": bool(
                evaluation.get(
                    "sql_safe",
                    evaluation.get(
                        "safe",
                        False,
                    ),
                )
            ),
            "sql_executed": bool(
                evaluation.get(
                    "sql_executed",
                    evaluation.get(
                        "executed",
                        False,
                    ),
                )
            ),
            "execution_correct": bool(
                evaluation.get(
                    "execution_correct",
                    evaluation.get(
                        "correct",
                        False,
                    ),
                )
            ),
            "execution_error": (
                evaluation.get(
                    "execution_error",
                    evaluation.get(
                        "error"
                    ),
                )
            ),
        }

    except Exception as error:

        return {
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": (
                f"{type(error).__name__}: "
                f"{str(error)}"
            ),
        }


# ------------------------------------------------------------
# 7. Helper: predict controller route before generation
# ------------------------------------------------------------

def predict_final_controller_route(
    benchmark_row,
):
    decision_start = (
        time.perf_counter()
    )

    feature_dictionary = (
        extract_controller_features(
            benchmark_row
        )
    )

    feature_frame = pd.DataFrame(
        [feature_dictionary]
    )

    v2_class_index = list(
        binary_logistic_controller
        .named_steps[
            "classifier"
        ]
        .classes_
    ).index(1)

    v2_probability = float(
        binary_logistic_controller
        .predict_proba(
            feature_frame[
                binary_feature_columns
            ]
        )[
            0,
            v2_class_index,
        ]
    )

    if (
        v2_probability
        >= binary_controller_threshold
    ):
        selected_route = (
            "optimized_rag_v2"
        )
    else:
        selected_route = (
            "qwen_alone"
        )

    decision_latency = (
        time.perf_counter()
        - decision_start
    )

    return {
        "selected_route": (
            selected_route
        ),
        "v2_probability": (
            v2_probability
        ),
        "decision_latency_seconds": (
            decision_latency
        ),
    }


# ------------------------------------------------------------
# 8. Run all final-test questions
# ------------------------------------------------------------

for row_index, benchmark_row in (
    remaining_final_df.iterrows()
):

    question_id = (
        benchmark_row[
            "question_id"
        ]
    )

    question = (
        benchmark_row[
            "question"
        ]
    )

    gold_sql = (
        benchmark_row[
            "gold_sql"
        ]
    )

    gold_rows = get_gold_rows_from_sql(
        gold_sql
    )

    # --------------------------------------------------------
    # Controller decides before any SQL generation
    # --------------------------------------------------------

    controller_decision = (
        predict_final_controller_route(
            benchmark_row
        )
    )

    selected_route = (
        controller_decision[
            "selected_route"
        ]
    )

    # --------------------------------------------------------
    # Qwen Alone
    # --------------------------------------------------------

    qwen_start = time.perf_counter()

    qwen_output = generate_sql(
        question,
        use_rag=False,
    )

    qwen_wall_latency = (
        time.perf_counter()
        - qwen_start
    )

    qwen_evaluation = (
        evaluate_final_query(
            qwen_output["sql"],
            gold_rows,
        )
    )

    # --------------------------------------------------------
    # Original/default RAG — comparison baseline only
    # --------------------------------------------------------

    original_rag_start = (
        time.perf_counter()
    )

    original_rag_output = generate_sql(
        question,
        use_rag=True,
    )

    original_rag_wall_latency = (
        time.perf_counter()
        - original_rag_start
    )

    original_rag_evaluation = (
        evaluate_final_query(
            original_rag_output["sql"],
            gold_rows,
        )
    )

    # --------------------------------------------------------
    # Optimized RAG V2
    # --------------------------------------------------------

    v2_start = time.perf_counter()

    v2_output = (
        generate_sql_optimized_rag_v2(
            question
        )
    )

    v2_wall_latency = (
        time.perf_counter()
        - v2_start
    )

    v2_evaluation = (
        evaluate_final_query(
            v2_output["sql"],
            gold_rows,
        )
    )

    # --------------------------------------------------------
    # Simulate deployed controller
    #
    # Experimental comparison generated all systems,
    # but deployed controller pays only:
    # decision time + selected route time.
    # --------------------------------------------------------

    if selected_route == "qwen_alone":

        controller_sql = (
            qwen_output["sql"]
        )

        controller_evaluation = (
            qwen_evaluation
        )

        selected_generation_latency = (
            qwen_output[
                "latency_seconds"
            ]
        )

        selected_wall_latency = (
            qwen_wall_latency
        )

    else:

        controller_sql = (
            v2_output["sql"]
        )

        controller_evaluation = (
            v2_evaluation
        )

        selected_generation_latency = (
            v2_output[
                "latency_seconds"
            ]
        )

        selected_wall_latency = (
            v2_wall_latency
        )

    controller_total_latency = (
        controller_decision[
            "decision_latency_seconds"
        ]
        + selected_wall_latency
    )

    result_row = {
        "question_id": (
            question_id
        ),
        "intent_id": (
            benchmark_row[
                "intent_id"
            ]
        ),
        "intent_name": (
            benchmark_row[
                "intent_name"
            ]
        ),
        "language": (
            benchmark_row[
                "language"
            ]
        ),
        "split": "final_test",
        "question": question,

        # Qwen Alone
        "qwen_alone_sql": (
            qwen_output["sql"]
        ),
        "qwen_alone_correct": (
            qwen_evaluation[
                "execution_correct"
            ]
        ),
        "qwen_alone_safe": (
            qwen_evaluation[
                "sql_safe"
            ]
        ),
        "qwen_alone_executed": (
            qwen_evaluation[
                "sql_executed"
            ]
        ),
        "qwen_alone_generation_latency": (
            qwen_output[
                "latency_seconds"
            ]
        ),
        "qwen_alone_wall_latency": (
            qwen_wall_latency
        ),

        # Original RAG
        "original_rag_sql": (
            original_rag_output[
                "sql"
            ]
        ),
        "original_rag_correct": (
            original_rag_evaluation[
                "execution_correct"
            ]
        ),
        "original_rag_safe": (
            original_rag_evaluation[
                "sql_safe"
            ]
        ),
        "original_rag_executed": (
            original_rag_evaluation[
                "sql_executed"
            ]
        ),
        "original_rag_generation_latency": (
            original_rag_output[
                "latency_seconds"
            ]
        ),
        "original_rag_wall_latency": (
            original_rag_wall_latency
        ),

        # Optimized RAG V2
        "v2_sql": (
            v2_output["sql"]
        ),
        "v2_correct": (
            v2_evaluation[
                "execution_correct"
            ]
        ),
        "v2_safe": (
            v2_evaluation[
                "sql_safe"
            ]
        ),
        "v2_executed": (
            v2_evaluation[
                "sql_executed"
            ]
        ),
        "v2_generation_latency": (
            v2_output[
                "latency_seconds"
            ]
        ),
        "v2_wall_latency": (
            v2_wall_latency
        ),

        # Controller
        "controller_route": (
            selected_route
        ),
        "controller_v2_probability": (
            controller_decision[
                "v2_probability"
            ]
        ),
        "controller_sql": (
            controller_sql
        ),
        "controller_correct": (
            controller_evaluation[
                "execution_correct"
            ]
        ),
        "controller_safe": (
            controller_evaluation[
                "sql_safe"
            ]
        ),
        "controller_executed": (
            controller_evaluation[
                "sql_executed"
            ]
        ),
        "controller_decision_latency": (
            controller_decision[
                "decision_latency_seconds"
            ]
        ),
        "controller_selected_generation_latency": (
            selected_generation_latency
        ),
        "controller_selected_wall_latency": (
            selected_wall_latency
        ),
        "controller_total_wall_latency": (
            controller_total_latency
        ),
    }

    with open(
        final_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint_file.write(
            json.dumps(
                result_row,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

    completed_final_rows.append(
        result_row
    )

    current_results_df = (
        pd.DataFrame(
            completed_final_rows
        )
    )

    running_controller_accuracy = (
        current_results_df[
            "controller_correct"
        ].mean()
    )

    print(
        f"[{len(completed_final_rows):02d}/"
        f"{len(final_test_df):02d}] "
        f"{question_id} | "
        f"route={selected_route} | "
        f"controller_correct="
        f"{controller_evaluation['execution_correct']} | "
        f"running_accuracy="
        f"{running_controller_accuracy:.2%}"
    )


# ------------------------------------------------------------
# 9. Load complete checkpoint
# ------------------------------------------------------------

locked_final_results_df = (
    pd.DataFrame(
        completed_final_rows
    )
    .drop_duplicates(
        subset=[
            "question_id"
        ],
        keep="last",
    )
    .sort_values(
        "question_id"
    )
    .reset_index(drop=True)
)

if len(locked_final_results_df) != 24:
    raise ValueError(
        "Final test is incomplete: "
        f"{len(locked_final_results_df)}/24"
    )

locked_final_results_df.to_csv(
    final_csv_path,
    index=False,
)


# ------------------------------------------------------------
# 10. Final accuracy metrics
# ------------------------------------------------------------

qwen_final_accuracy = float(
    locked_final_results_df[
        "qwen_alone_correct"
    ].mean()
)

original_rag_final_accuracy = float(
    locked_final_results_df[
        "original_rag_correct"
    ].mean()
)

v2_final_accuracy = float(
    locked_final_results_df[
        "v2_correct"
    ].mean()
)

controller_final_accuracy = float(
    locked_final_results_df[
        "controller_correct"
    ].mean()
)

binary_final_oracle_accuracy = float(
    (
        locked_final_results_df[
            "qwen_alone_correct"
        ]
        | locked_final_results_df[
            "v2_correct"
        ]
    ).mean()
)


# ------------------------------------------------------------
# 11. Final time metrics
# ------------------------------------------------------------

qwen_average_wall_latency = float(
    locked_final_results_df[
        "qwen_alone_wall_latency"
    ].mean()
)

original_rag_average_wall_latency = float(
    locked_final_results_df[
        "original_rag_wall_latency"
    ].mean()
)

v2_average_wall_latency = float(
    locked_final_results_df[
        "v2_wall_latency"
    ].mean()
)

controller_average_wall_latency = float(
    locked_final_results_df[
        "controller_total_wall_latency"
    ].mean()
)

controller_latency_difference = (
    controller_average_wall_latency
    - v2_average_wall_latency
)

controller_latency_change_percent = (
    100.0
    * controller_latency_difference
    / v2_average_wall_latency
)

qwen_route_count = int(
    (
        locked_final_results_df[
            "controller_route"
        ] == "qwen_alone"
    ).sum()
)

v2_route_count = int(
    (
        locked_final_results_df[
            "controller_route"
        ] == "optimized_rag_v2"
    ).sum()
)

rag_avoidance_percent = (
    100.0
    * qwen_route_count
    / len(
        locked_final_results_df
    )
)

controller_corrections = int(
    (
        locked_final_results_df[
            "controller_correct"
        ]
        & ~locked_final_results_df[
            "v2_correct"
        ]
    ).sum()
)

controller_regressions = int(
    (
        ~locked_final_results_df[
            "controller_correct"
        ]
        & locked_final_results_df[
            "v2_correct"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 12. Save locked summary
# ------------------------------------------------------------

locked_final_summary = {
    "questions": 24,
    "qwen_alone_accuracy": (
        qwen_final_accuracy
    ),
    "original_rag_accuracy": (
        original_rag_final_accuracy
    ),
    "optimized_rag_v2_accuracy": (
        v2_final_accuracy
    ),
    "binary_controller_accuracy": (
        controller_final_accuracy
    ),
    "binary_oracle_accuracy": (
        binary_final_oracle_accuracy
    ),
    "qwen_alone_average_wall_latency": (
        qwen_average_wall_latency
    ),
    "original_rag_average_wall_latency": (
        original_rag_average_wall_latency
    ),
    "optimized_rag_v2_average_wall_latency": (
        v2_average_wall_latency
    ),
    "controller_average_wall_latency": (
        controller_average_wall_latency
    ),
    "controller_latency_difference_seconds": (
        controller_latency_difference
    ),
    "controller_latency_change_percent": (
        controller_latency_change_percent
    ),
    "qwen_alone_routes": (
        qwen_route_count
    ),
    "optimized_rag_v2_routes": (
        v2_route_count
    ),
    "rag_avoidance_percent": (
        rag_avoidance_percent
    ),
    "controller_corrections_over_v2": (
        controller_corrections
    ),
    "controller_regressions_from_v2": (
        controller_regressions
    ),
    "configuration": (
        FINAL_CONTROLLER_CONFIGURATION
    ),
    "final_test_opened": True,
    "post_test_tuning_allowed": False,
}

with open(
    final_summary_path,
    "w",
    encoding="utf-8",
) as summary_file:
    json.dump(
        locked_final_summary,
        summary_file,
        indent=2,
    )


# ------------------------------------------------------------
# 13. Final report
# ------------------------------------------------------------

print()
print("=" * 70)
print("LOCKED FINAL TEST RESULTS")
print("=" * 70)

print()
print("FINAL ACCURACY")

print(
    "Qwen Alone:",
    f"{qwen_final_accuracy:.2%}",
)

print(
    "Original RAG:",
    f"{original_rag_final_accuracy:.2%}",
)

print(
    "Optimized RAG V2:",
    f"{v2_final_accuracy:.2%}",
)

print(
    "Binary Controller:",
    f"{controller_final_accuracy:.2%}",
)

print(
    "Binary Oracle Ceiling:",
    f"{binary_final_oracle_accuracy:.2%}",
)

print()
print("FULL WALL-CLOCK LATENCY")

print(
    "Qwen Alone average:",
    f"{qwen_average_wall_latency:.4f} s",
)

print(
    "Original RAG average:",
    f"{original_rag_average_wall_latency:.4f} s",
)

print(
    "Optimized RAG V2 average:",
    f"{v2_average_wall_latency:.4f} s",
)

print(
    "Controller average:",
    f"{controller_average_wall_latency:.4f} s",
)

print(
    "Controller difference vs V2:",
    f"{controller_latency_difference:+.4f} s",
)

print(
    "Controller latency change:",
    f"{controller_latency_change_percent:+.2f}%",
)

print()
print("CONTROLLER ROUTING")

print(
    "Qwen Alone routes:",
    qwen_route_count,
)

print(
    "Optimized RAG V2 routes:",
    v2_route_count,
)

print(
    "RAG avoided:",
    f"{rag_avoidance_percent:.2f}%",
)

print(
    "Corrections over V2:",
    controller_corrections,
)

print(
    "Regressions from V2:",
    controller_regressions,
)

print()
print("FINAL ACCURACY BY LANGUAGE")

print(
    locked_final_results_df.groupby(
        "language"
    )[
        [
            "qwen_alone_correct",
            "original_rag_correct",
            "v2_correct",
            "controller_correct",
        ]
    ]
    .mean()
)

print()
print("FINAL CONTROLLER DECISIONS")

print(
    locked_final_results_df[
        [
            "question_id",
            "language",
            "controller_route",
            "controller_v2_probability",
            "qwen_alone_correct",
            "v2_correct",
            "controller_correct",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("Final test opened:", True)
print("Controller frozen:", True)
print("Post-test tuning allowed:", False)
print("Results saved:", final_csv_path)
print("Summary saved:", final_summary_path)

LOCKED FINAL TEST
Target questions: 24
Previously completed: 0
Remaining: 24
Controller frozen: True
Post-test tuning allowed: False
Final test opened: True
[01/24] B4A-005-ENGLISH | route=optimized_rag_v2 | controller_correct=True | running_accuracy=100.00%
[02/24] B4A-005-ARABIC | route=optimized_rag_v2 | controller_correct=False | running_accuracy=50.00%
[03/24] B4A-005-FRENCH | route=optimized_rag_v2 | controller_correct=True | running_accuracy=66.67%
[04/24] B4A-005-LEBANESE_ARABIZI | route=optimized_rag_v2 | controller_correct=True | running_accuracy=75.00%
[05/24] B4A-010-ENGLISH | route=optimized_rag_v2 | controller_correct=False | running_accuracy=60.00%
[06/24] B4A-010-ARABIC | route=optimized_rag_v2 | controller_correct=False | running_accuracy=50.00%
[07/24] B4A-010-FRENCH | route=optimized_rag_v2 | controller_correct=False | running_accuracy=42.86%
[08/24] B4A-010-LEBANESE_ARABIZI | route=optimized_rag_v2 | controller_correct=False | running_accuracy=37.50%
[09/24] B4A-013

In [52]:
# ============================================================
# CELL 39 — OPTIMIZED RAG V2 FINAL-ERROR DIAGNOSIS
# Start of post-final V3 development
# ============================================================

import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Select V2 final-test failures
# ------------------------------------------------------------

v2_final_failures_df = (
    locked_final_results_df[
        ~locked_final_results_df[
            "v2_correct"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

final_gold_columns_df = (
    final_test_df[
        [
            "question_id",
            "question",
            "gold_sql",
        ]
    ]
    .copy()
)

v2_final_failures_df = (
    v2_final_failures_df
    .drop(
        columns=[
            column
            for column in [
                "question",
                "gold_sql",
            ]
            if column
            in v2_final_failures_df.columns
        ]
    )
    .merge(
        final_gold_columns_df,
        on="question_id",
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# 2. Retrieve V2 context again without new SQL generation
# ------------------------------------------------------------

diagnosis_rows = []

for row_index, failure_row in (
    v2_final_failures_df.iterrows()
):

    question = failure_row[
        "question"
    ]

    try:
        (
            optimized_context,
            optimized_chunks,
        ) = build_optimized_rag_context(
            question
        )

        retrieved_document_names = []

        retrieved_previews = []

        for chunk in optimized_chunks:

            document_name = str(
                chunk.get(
                    "document_name",
                    "unknown_document",
                )
            )

            retrieved_document_names.append(
                document_name
            )

            chunk_text = str(
                chunk.get(
                    "text",
                    "",
                )
            )

            retrieved_previews.append(
                chunk_text[:500]
            )

        selected_document = (
            retrieved_document_names[0]
            if retrieved_document_names
            else "none"
        )

        retrieval_preview = "\n\n".join(
            retrieved_previews[:3]
        )

    except Exception as error:

        selected_document = (
            "retrieval_error"
        )

        retrieval_preview = (
            f"{type(error).__name__}: "
            f"{str(error)}"
        )

        optimized_context = ""

        optimized_chunks = []

    diagnosis_rows.append({
        "question_id": (
            failure_row[
                "question_id"
            ]
        ),
        "intent_id": (
            failure_row[
                "intent_id"
            ]
        ),
        "intent_name": (
            failure_row[
                "intent_name"
            ]
        ),
        "language": (
            failure_row[
                "language"
            ]
        ),
        "question": question,
        "selected_document": (
            selected_document
        ),
        "retrieved_chunk_count": (
            len(optimized_chunks)
        ),
        "rag_context_characters": (
            len(optimized_context)
        ),
        "qwen_alone_correct": (
            failure_row[
                "qwen_alone_correct"
            ]
        ),
        "original_rag_correct": (
            failure_row[
                "original_rag_correct"
            ]
        ),
        "v2_generated_sql": (
            failure_row[
                "v2_sql"
            ]
        ),
        "gold_sql": (
            failure_row[
                "gold_sql"
            ]
        ),
        "retrieved_preview": (
            retrieval_preview
        ),
    })


v2_final_error_diagnosis_df = (
    pd.DataFrame(
        diagnosis_rows
    )
)


# ------------------------------------------------------------
# 3. Simple automatic failure indicators
# ------------------------------------------------------------

def detect_failure_indicators(
    generated_sql,
    gold_sql,
):
    generated = str(
        generated_sql
    ).lower()

    gold = str(
        gold_sql
    ).lower()

    indicators = []

    if (
        "current_date" in generated
        and "current_date" not in gold
    ):
        indicators.append(
            "invented_current_date"
        )

    if (
        "date_trunc" in gold
        and "date_trunc" not in generated
    ):
        indicators.append(
            "missing_date_trunc"
        )

    if (
        "order by" in gold
        and "order by" not in generated
    ):
        indicators.append(
            "missing_order_by"
        )

    if (
        "round(" in gold
        and "round(" not in generated
    ):
        indicators.append(
            "missing_round"
        )

    if (
        "count(*)" in gold
        and "count(distinct" in generated
    ):
        indicators.append(
            "count_distinct_mismatch"
        )

    if (
        "left join" in gold
        and "left join" not in generated
    ):
        indicators.append(
            "missing_left_join"
        )

    if (
        "group by" in gold
        and "group by" not in generated
    ):
        indicators.append(
            "missing_group_by"
        )

    if (
        "build4all_research."
        not in generated
    ):
        indicators.append(
            "missing_schema_prefix"
        )

    if (
        " and " in gold
        and " or " in generated
        and " in (" not in generated
    ):
        indicators.append(
            "possible_boolean_precedence"
        )

    if not indicators:
        indicators.append(
            "semantic_or_output_mismatch"
        )

    return ", ".join(
        indicators
    )


v2_final_error_diagnosis_df[
    "failure_indicators"
] = (
    v2_final_error_diagnosis_df.apply(
        lambda row: (
            detect_failure_indicators(
                row[
                    "v2_generated_sql"
                ],
                row[
                    "gold_sql"
                ],
            )
        ),
        axis=1,
    )
)


# ------------------------------------------------------------
# 4. Save diagnosis
# ------------------------------------------------------------

v2_final_diagnosis_path = (
    final_results_directory
    / "v2_final_error_diagnosis.csv"
)

v2_final_error_diagnosis_df.to_csv(
    v2_final_diagnosis_path,
    index=False,
)


# ------------------------------------------------------------
# 5. Report each failure
# ------------------------------------------------------------

print("=" * 70)
print("OPTIMIZED RAG V2 — FINAL ERROR DIAGNOSIS")
print("=" * 70)

print(
    "V2 final failures:",
    len(
        v2_final_error_diagnosis_df
    ),
)

print(
    "New SQL generations:",
    0,
)

print(
    "Current final results modified:",
    False,
)

print()
print("FAILURE SUMMARY")

print(
    v2_final_error_diagnosis_df[
        [
            "question_id",
            "intent_name",
            "language",
            "qwen_alone_correct",
            "original_rag_correct",
            "selected_document",
            "failure_indicators",
        ]
    ]
    .to_string(
        index=False
    )
)

for _, row in (
    v2_final_error_diagnosis_df
    .iterrows()
):

    print()
    print("-" * 70)

    print(
        row["question_id"],
        "—",
        row["intent_name"],
    )

    print("-" * 70)

    print("QUESTION:")
    print(row["question"])

    print()
    print("V2 GENERATED SQL:")
    print(
        row[
            "v2_generated_sql"
        ]
    )

    print()
    print("GOLD SQL:")
    print(row["gold_sql"])

    print()
    print(
        "FAILURE INDICATORS:",
        row[
            "failure_indicators"
        ],
    )

    print()
    print(
        "SELECTED DOCUMENT:",
        row[
            "selected_document"
        ],
    )

    print()
    print("RETRIEVED PREVIEW:")
    print(
        row[
            "retrieved_preview"
        ]
    )

print()
print("=" * 70)
print("CELL 39 COMPLETED")
print("=" * 70)

print(
    "Frozen final test preserved:",
    True,
)

print(
    "V3 development started:",
    True,
)

print(
    "A new untouched test will "
    "be required:",
    True,
)

print(
    "Saved:",
    v2_final_diagnosis_path,
)

OPTIMIZED RAG V2 — FINAL ERROR DIAGNOSIS
V2 final failures: 9
New SQL generations: 0
Current final results modified: False

FAILURE SUMMARY
             question_id                    intent_name         language  qwen_alone_correct  original_rag_correct                                    selected_document                            failure_indicators
          B4A-005-ARABIC average_recognized_order_value           arabic               False                 False  Build4All_Product_Events_and_Conversion_Policy_v2.0                   semantic_or_output_mismatch
          B4A-010-ARABIC             daily_order_volume           arabic               False                 False  Build4All_Product_Events_and_Conversion_Policy_v2.0                         invented_current_date
         B4A-010-ENGLISH             daily_order_volume          english                True                  True  Build4All_Product_Events_and_Conversion_Policy_v2.0                   semantic_or_output_mismatch
    

In [53]:
# ============================================================
# CELL 40 — OPTIMIZED RAG V3
# Multilingual domain routing + generation safeguards
# Targeted post-final development test
# ============================================================

import re
import time
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Controlled document names
# ------------------------------------------------------------

V3_DOCUMENTS = {
    "sales": (
        "Build4All_Sales_and_"
        "Order_Analytics_Policy_v2.0"
    ),
    "catalog": (
        "Build4All_Product_Inventory_"
        "and_Coupon_Policy_v2.0"
    ),
    "customer": (
        "Build4All_Customer_Regional_"
        "and_Shipping_Policy_v2.0"
    ),
    "events": (
        "Build4All_Product_Events_"
        "and_Conversion_Policy_v2.0"
    ),
}


# ------------------------------------------------------------
# 2. Multilingual domain keywords
# ------------------------------------------------------------

V3_DOMAIN_KEYWORDS = {
    "catalog": [
        "coupon",
        "coupons",
        "voucher",
        "discount code",
        "valid coupon",
        "active coupon",
        "product stock",
        "inventory",
        "sale price",
        "catalog",
        "produit",
        "stock",
        "inventaire",
        "coupon valide",
        "قسيمة",
        "قسائم",
        "كوبون",
        "مخزون",
        "منتج نشط",
    ],
    "customer": [
        "customer",
        "customers",
        "customer region",
        "purchasing customer",
        "repeat customer",
        "registration",
        "shipping method",
        "shipping amount",
        "client",
        "clients",
        "région",
        "region",
        "inscription",
        "livraison",
        "عميل",
        "عملاء",
        "زبون",
        "المنطقة",
        "تسجيل",
        "شحن",
        "customer bi",
        "customers bi",
        "kel region",
        "7asab region",
    ],
    "events": [
        "product event",
        "product events",
        "event type",
        "conversion",
        "purchase event",
        "view event",
        "événement produit",
        "conversion",
        "حدث المنتج",
        "أحداث المنتج",
        "تحويل",
    ],
    "sales": [
        "order",
        "orders",
        "revenue",
        "sales",
        "recognized",
        "payment",
        "payments",
        "paid",
        "refunded",
        "failed payment",
        "daily volume",
        "monthly revenue",
        "backlog",
        "tax rate",
        "cancellation",
        "commande",
        "commandes",
        "vente",
        "revenu",
        "paiement",
        "remboursé",
        "échoué",
        "quotidien",
        "طلب",
        "طلبات",
        "إيراد",
        "مبيعات",
        "مدفوعات",
        "دفع",
        "فشل",
        "يومي",
        "nesbet payment",
        "order volume",
    ],
}


# ------------------------------------------------------------
# 3. Query enrichment
# ------------------------------------------------------------

def enrich_v3_query(
    question,
):
    normalized_question = str(
        question
    ).lower()

    enrichment_terms = []

    enrichment_rules = {
        "daily order volume calendar day "
        "all order statuses explicit date range": [
            "daily order",
            "order volume",
            "volume quotidien",
            "حجم الطلبات اليومي",
        ],
        "failed payment rate percentage "
        "failed rows divided by all payment rows": [
            "failed payment",
            "payment records li failed",
            "paiements échoués",
            "المدفوعات الفاشلة",
        ],
        "recognized orders completed or shipped": [
            "recognized",
            "reconnu",
            "معترف",
        ],
        "active valid coupons reporting reference date": [
            "active w valid",
            "active and valid",
            "actifs et valides",
            "النشطة والصالحة",
        ],
        "purchasing customers by customer region "
        "at least one recognized order": [
            "purchasing customer",
            "purchasing customers",
            "clients acheteurs",
            "العملاء المشترون",
        ],
    }

    for (
        enrichment_text,
        triggers,
    ) in enrichment_rules.items():

        if any(
            trigger
            in normalized_question
            for trigger in triggers
        ):
            enrichment_terms.append(
                enrichment_text
            )

    if enrichment_terms:
        return (
            str(question)
            + " "
            + " ".join(
                enrichment_terms
            )
        )

    return str(question)


# ------------------------------------------------------------
# 4. Deterministic multilingual domain router
# ------------------------------------------------------------

def select_v3_domain(
    question,
):
    normalized_question = (
        str(question).lower()
    )

    domain_scores = {
        domain_name: sum(
            keyword
            in normalized_question
            for keyword
            in keywords
        )
        for (
            domain_name,
            keywords,
        ) in V3_DOMAIN_KEYWORDS.items()
    }

    priority = [
        "catalog",
        "customer",
        "events",
        "sales",
    ]

    best_score = max(
        domain_scores.values()
    )

    if best_score == 0:
        return None, domain_scores

    selected_domain = next(
        domain_name
        for domain_name
        in priority
        if domain_scores[
            domain_name
        ] == best_score
    )

    return (
        selected_domain,
        domain_scores,
    )


# ------------------------------------------------------------
# 5. V3 domain-restricted context
# ------------------------------------------------------------

def build_v3_rag_context(
    question,
):
    enriched_query = enrich_v3_query(
        question
    )

    selected_domain, domain_scores = (
        select_v3_domain(
            question
        )
    )

    retrieved_candidates = (
        retrieve_chunks(
            enriched_query,
            top_k=26,
        )
    )

    if selected_domain is not None:

        selected_document = (
            V3_DOCUMENTS[
                selected_domain
            ]
        )

        domain_candidates = [
            result
            for result
            in retrieved_candidates
            if result[
                "document_name"
            ] == selected_document
        ]

    else:
        selected_document = (
            retrieved_candidates[0][
                "document_name"
            ]
            if retrieved_candidates
            else "none"
        )

        domain_candidates = [
            result
            for result
            in retrieved_candidates
            if result[
                "document_name"
            ] == selected_document
        ]

    selected_chunks = (
        domain_candidates[:6]
    )

    context_parts = []
    current_characters = 0
    maximum_characters = 5500

    for result in selected_chunks:

        chunk_block = (
            f"[Source: "
            f"{result['document_name']}, "
            f"page "
            f"{result['page_number']}, "
            f"score "
            f"{result['retrieval_score']:.4f}]\n"
            f"{result['text']}"
        )

        if (
            current_characters
            + len(chunk_block)
            > maximum_characters
        ):
            break

        context_parts.append(
            chunk_block
        )

        current_characters += len(
            chunk_block
        )

    return {
        "context": "\n\n".join(
            context_parts
        ),
        "retrieved_chunks": (
            selected_chunks
        ),
        "selected_domain": (
            selected_domain
        ),
        "selected_document": (
            selected_document
        ),
        "domain_scores": (
            domain_scores
        ),
        "enriched_query": (
            enriched_query
        ),
    }


# ------------------------------------------------------------
# 6. V3 SQL generator
# ------------------------------------------------------------

def generate_sql_optimized_rag_v3(
    question,
):
    v3_retrieval = (
        build_v3_rag_context(
            question
        )
    )

    rag_context = (
        v3_retrieval[
            "context"
        ]
    )

    system_prompt = """
You are a multilingual PostgreSQL Text-to-SQL system.

Generate exactly one read-only PostgreSQL query that answers
the user's business question.

Mandatory rules:
1. Use only the provided database schema and columns.
2. Explicitly qualify tables with build4all_research.
3. Generate only one SELECT or WITH query.
4. Never invent tables, columns, dates, filters, joins,
   grouping levels, statuses, or business conditions.
5. Follow the retrieved business policy exactly.
6. A filter may be added only when the question or the
   retrieved policy explicitly requires it.
7. Do not apply a recognized-order status filter to a metric
   about all orders.
8. "Recognized orders" means status IN
   ('completed', 'shipped').
9. For a requested rate or percentage, calculate the full
   percentage, use a null-safe denominator, multiply by
   100.0, and ROUND to 2 decimal places.
10. For an explicit calendar month, use a half-open timestamp
    range: start inclusive and next-month start exclusive.
11. For daily reporting, group by the calendar day, not month.
12. Return the requested business fields. Do not add internal
    ID columns unless the question or policy requests them.
13. Preserve required zero-count groups with LEFT JOIN when
    the policy requires every category, coupon, or method.
14. Use parentheses or IN (...) for multi-value conditions.
15. Add deterministic ORDER BY when returning grouped or
    multi-row results.
16. Return SQL only, without explanation or Markdown.
""".strip()

    user_prompt = "\n".join([
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULARY_TEXT,
        "",
        "SELECTED BUSINESS DOMAIN:",
        str(
            v3_retrieval[
                "selected_domain"
            ]
        ),
        "",
        "RETRIEVED BUSINESS POLICY:",
        rag_context,
        "",
        "QUESTION INTERPRETATION SAFEGUARDS:",
        (
            "- Distinguish counts, amounts, averages, "
            "rates, and percentages."
        ),
        (
            "- Do not convert a general question into "
            "a current-date question."
        ),
        (
            "- Do not add completed/shipped filters "
            "unless recognized scope is requested."
        ),
        (
            "- Purchasing customer means a customer "
            "with at least one recognized order."
        ),
        (
            "- Repeat purchasing customer requires "
            "more than one recognized order."
        ),
        (
            "- Coupon validity reports must follow the "
            "policy reference timestamp and requested "
            "output fields."
        ),
        "",
        "BUSINESS QUESTION:",
        str(question),
        "",
        "POSTGRESQL QUERY:",
    ])

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    prompt_text = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(
        model.device
    )

    generation_start = (
        time.perf_counter()
    )

    with torch.inference_mode():

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=(
                CONFIG[
                    "model"
                ][
                    "max_new_tokens"
                ]
            ),
            do_sample=False,
            pad_token_id=(
                tokenizer.eos_token_id
            ),
        )

    generation_latency = (
        time.perf_counter()
        - generation_start
    )

    new_tokens = generated_ids[
        :,
        model_inputs[
            "input_ids"
        ].shape[1]:,
    ]

    raw_response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    ).strip()

    generated_sql = extract_sql(
        raw_response
    )

    return {
        "question": question,
        "system": (
            "optimized_rag_v3"
        ),
        "sql": generated_sql,
        "raw_response": raw_response,
        "latency_seconds": round(
            generation_latency,
            4,
        ),
        "rag_used": True,
        "selected_domain": (
            v3_retrieval[
                "selected_domain"
            ]
        ),
        "selected_document": (
            v3_retrieval[
                "selected_document"
            ]
        ),
        "domain_scores": (
            v3_retrieval[
                "domain_scores"
            ]
        ),
        "enriched_query": (
            v3_retrieval[
                "enriched_query"
            ]
        ),
        "retrieved_chunks": (
            v3_retrieval[
                "retrieved_chunks"
            ]
        ),
        "rag_context_characters": (
            len(rag_context)
        ),
    }


# ------------------------------------------------------------
# 7. Targeted test on the 9 diagnosed failures
# ------------------------------------------------------------

v3_targeted_rows = []

for test_index, failure_row in (
    v2_final_error_diagnosis_df
    .iterrows()
):

    question_id = (
        failure_row[
            "question_id"
        ]
    )

    question = (
        failure_row[
            "question"
        ]
    )

    gold_rows = get_gold_rows_from_sql(
        failure_row[
            "gold_sql"
        ]
    )

    total_start = (
        time.perf_counter()
    )

    v3_output = (
        generate_sql_optimized_rag_v3(
            question
        )
    )

    total_latency = (
        time.perf_counter()
        - total_start
    )

    v3_evaluation = (
        evaluate_final_query(
            v3_output[
                "sql"
            ],
            gold_rows,
        )
    )

    v3_targeted_rows.append({
        "question_id": (
            question_id
        ),
        "intent_name": (
            failure_row[
                "intent_name"
            ]
        ),
        "language": (
            failure_row[
                "language"
            ]
        ),
        "question": question,
        "selected_domain": (
            v3_output[
                "selected_domain"
            ]
        ),
        "selected_document": (
            v3_output[
                "selected_document"
            ]
        ),
        "v2_correct": False,
        "v3_correct": (
            v3_evaluation[
                "execution_correct"
            ]
        ),
        "v3_safe": (
            v3_evaluation[
                "sql_safe"
            ]
        ),
        "v3_executed": (
            v3_evaluation[
                "sql_executed"
            ]
        ),
        "v3_sql": (
            v3_output[
                "sql"
            ]
        ),
        "generation_latency": (
            v3_output[
                "latency_seconds"
            ]
        ),
        "total_latency": (
            total_latency
        ),
    })

    print(
        f"[{test_index + 1:02d}/"
        f"{len(v2_final_error_diagnosis_df):02d}] "
        f"{question_id} | "
        f"domain="
        f"{v3_output['selected_domain']} | "
        f"correct="
        f"{v3_evaluation['execution_correct']}"
    )


v3_targeted_results_df = (
    pd.DataFrame(
        v3_targeted_rows
    )
)

v3_targeted_accuracy = float(
    v3_targeted_results_df[
        "v3_correct"
    ].mean()
)

v3_new_corrections = int(
    v3_targeted_results_df[
        "v3_correct"
    ].sum()
)


# ------------------------------------------------------------
# 8. Save targeted V3 results
# ------------------------------------------------------------

v3_targeted_path = (
    final_results_directory
    / "optimized_rag_v3_targeted_failures.csv"
)

v3_targeted_results_df.to_csv(
    v3_targeted_path,
    index=False,
)


# ------------------------------------------------------------
# 9. Report
# ------------------------------------------------------------

print()
print("=" * 70)
print("OPTIMIZED RAG V3 — TARGETED DEVELOPMENT RESULTS")
print("=" * 70)

print(
    "Targeted V2 failures:",
    len(
        v3_targeted_results_df
    ),
)

print(
    "V3 corrections:",
    v3_new_corrections,
)

print(
    "Targeted V3 accuracy:",
    f"{v3_targeted_accuracy:.2%}",
)

print()
print("RESULTS")

print(
    v3_targeted_results_df[
        [
            "question_id",
            "language",
            "selected_domain",
            "v3_correct",
            "generation_latency",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("GENERATED SQL")

for _, result_row in (
    v3_targeted_results_df
    .iterrows()
):

    print()
    print("-" * 70)

    print(
        result_row[
            "question_id"
        ],
        "| correct=",
        result_row[
            "v3_correct"
        ],
    )

    print(
        result_row[
            "v3_sql"
        ]
    )

print()
print("=" * 70)

print(
    "Frozen final result modified:",
    False,
)

print(
    "This is post-final V3 development:",
    True,
)

print(
    "New untouched test required:",
    True,
)

print(
    "Saved:",
    v3_targeted_path,
)

[01/09] B4A-005-ARABIC | domain=sales | correct=True
[02/09] B4A-010-ARABIC | domain=sales | correct=True
[03/09] B4A-010-ENGLISH | domain=sales | correct=True
[04/09] B4A-010-FRENCH | domain=sales | correct=True
[05/09] B4A-010-LEBANESE_ARABIZI | domain=sales | correct=True
[06/09] B4A-013-LEBANESE_ARABIZI | domain=sales | correct=False
[07/09] B4A-018-ARABIC | domain=catalog | correct=True
[08/09] B4A-018-LEBANESE_ARABIZI | domain=catalog | correct=False
[09/09] B4A-023-LEBANESE_ARABIZI | domain=customer | correct=True

OPTIMIZED RAG V3 — TARGETED DEVELOPMENT RESULTS
Targeted V2 failures: 9
V3 corrections: 7
Targeted V3 accuracy: 77.78%

RESULTS
             question_id         language selected_domain  v3_correct  generation_latency
          B4A-005-ARABIC           arabic           sales        True              8.5478
          B4A-010-ARABIC           arabic           sales        True             17.0666
         B4A-010-ENGLISH          english           sales        True     

In [54]:
# ============================================================
# CELL 41 — OPTIMIZED RAG V3 FULL REGRESSION BENCHMARK
# 120 questions with checkpoint/resume
# ============================================================

import json
import time
import numpy as np
import pandas as pd

from pathlib import Path


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

v3_full_checkpoint_path = (
    final_results_directory
    / "optimized_rag_v3_full_checkpoint.jsonl"
)

v3_full_csv_path = (
    final_results_directory
    / "optimized_rag_v3_full_results.csv"
)

v3_full_summary_path = (
    final_results_directory
    / "optimized_rag_v3_full_summary.json"
)


# ------------------------------------------------------------
# 2. Build V2 reference across all 120 questions
# ------------------------------------------------------------

v2_train_validation_reference = (
    optimized_rag_v2_results_df[
        [
            "question_id",
            "execution_correct",
        ]
    ]
    .rename(
        columns={
            "execution_correct": (
                "v2_correct"
            )
        }
    )
)

v2_final_reference = (
    locked_final_results_df[
        [
            "question_id",
            "v2_correct",
        ]
    ]
)

v2_all_reference_df = (
    pd.concat(
        [
            v2_train_validation_reference,
            v2_final_reference,
        ],
        ignore_index=True,
    )
    .drop_duplicates(
        subset=[
            "question_id"
        ],
        keep="last",
    )
)

if len(v2_all_reference_df) != 120:
    raise ValueError(
        "Expected 120 V2 reference rows, "
        f"found {len(v2_all_reference_df)}"
    )


# ------------------------------------------------------------
# 3. Full 120-question benchmark
# ------------------------------------------------------------

v3_full_benchmark_df = (
    multilingual_benchmark_df
    .merge(
        v2_all_reference_df,
        on="question_id",
        how="left",
        validate="one_to_one",
    )
    .copy()
    .reset_index(drop=True)
)

if len(v3_full_benchmark_df) != 120:
    raise ValueError(
        "Expected 120 benchmark questions, "
        f"found {len(v3_full_benchmark_df)}"
    )

if (
    v3_full_benchmark_df[
        "v2_correct"
    ].isna().any()
):
    raise ValueError(
        "Missing V2 reference scores"
    )


# ------------------------------------------------------------
# 4. Load existing V3 checkpoint
# ------------------------------------------------------------

v3_completed_rows = []

if v3_full_checkpoint_path.exists():

    with open(
        v3_full_checkpoint_path,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:

        for line in checkpoint_file:

            line = line.strip()

            if line:
                v3_completed_rows.append(
                    json.loads(line)
                )


# ------------------------------------------------------------
# 5. Reuse the 9 targeted V3 generations
# ------------------------------------------------------------

existing_v3_ids = {
    row["question_id"]
    for row in v3_completed_rows
}

for _, targeted_row in (
    v3_targeted_results_df
    .iterrows()
):

    question_id = (
        targeted_row[
            "question_id"
        ]
    )

    if question_id in existing_v3_ids:
        continue

    benchmark_row = (
        v3_full_benchmark_df[
            v3_full_benchmark_df[
                "question_id"
            ] == question_id
        ]
        .iloc[0]
    )

    reused_row = {
        "question_id": question_id,
        "intent_id": (
            benchmark_row[
                "intent_id"
            ]
        ),
        "intent_name": (
            benchmark_row[
                "intent_name"
            ]
        ),
        "language": (
            benchmark_row[
                "language"
            ]
        ),
        "split": (
            benchmark_row[
                "split"
            ]
        ),
        "question": (
            benchmark_row[
                "question"
            ]
        ),
        "selected_domain": (
            targeted_row[
                "selected_domain"
            ]
        ),
        "selected_document": (
            targeted_row[
                "selected_document"
            ]
        ),
        "v2_correct": bool(
            benchmark_row[
                "v2_correct"
            ]
        ),
        "v3_correct": bool(
            targeted_row[
                "v3_correct"
            ]
        ),
        "v3_safe": bool(
            targeted_row[
                "v3_safe"
            ]
        ),
        "v3_executed": bool(
            targeted_row[
                "v3_executed"
            ]
        ),
        "v3_sql": (
            targeted_row[
                "v3_sql"
            ]
        ),
        "generation_latency": float(
            targeted_row[
                "generation_latency"
            ]
        ),
        "total_latency": float(
            targeted_row[
                "total_latency"
            ]
        ),
        "reused_targeted_generation": True,
    }

    with open(
        v3_full_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint_file.write(
            json.dumps(
                reused_row,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

    v3_completed_rows.append(
        reused_row
    )

    existing_v3_ids.add(
        question_id
    )


# ------------------------------------------------------------
# 6. Determine remaining questions
# ------------------------------------------------------------

completed_v3_question_ids = {
    row["question_id"]
    for row in v3_completed_rows
}

remaining_v3_df = (
    v3_full_benchmark_df[
        ~v3_full_benchmark_df[
            "question_id"
        ].isin(
            completed_v3_question_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 70)
print("OPTIMIZED RAG V3 — FULL REGRESSION BENCHMARK")
print("=" * 70)

print(
    "Target questions:",
    len(v3_full_benchmark_df),
)

print(
    "Previously completed or reused:",
    len(v3_completed_rows),
)

print(
    "Remaining generations:",
    len(remaining_v3_df),
)

print(
    "V2 frozen reference available:",
    True,
)


# ------------------------------------------------------------
# 7. Run remaining V3 generations
# ------------------------------------------------------------

for _, benchmark_row in (
    remaining_v3_df.iterrows()
):

    question_id = (
        benchmark_row[
            "question_id"
        ]
    )

    question = (
        benchmark_row[
            "question"
        ]
    )

    gold_rows = get_gold_rows_from_sql(
        benchmark_row[
            "gold_sql"
        ]
    )

    total_start = (
        time.perf_counter()
    )

    v3_output = (
        generate_sql_optimized_rag_v3(
            question
        )
    )

    total_latency = (
        time.perf_counter()
        - total_start
    )

    v3_evaluation = (
        evaluate_final_query(
            v3_output[
                "sql"
            ],
            gold_rows,
        )
    )

    result_row = {
        "question_id": question_id,
        "intent_id": (
            benchmark_row[
                "intent_id"
            ]
        ),
        "intent_name": (
            benchmark_row[
                "intent_name"
            ]
        ),
        "language": (
            benchmark_row[
                "language"
            ]
        ),
        "split": (
            benchmark_row[
                "split"
            ]
        ),
        "question": question,
        "selected_domain": (
            v3_output[
                "selected_domain"
            ]
        ),
        "selected_document": (
            v3_output[
                "selected_document"
            ]
        ),
        "v2_correct": bool(
            benchmark_row[
                "v2_correct"
            ]
        ),
        "v3_correct": bool(
            v3_evaluation[
                "execution_correct"
            ]
        ),
        "v3_safe": bool(
            v3_evaluation[
                "sql_safe"
            ]
        ),
        "v3_executed": bool(
            v3_evaluation[
                "sql_executed"
            ]
        ),
        "v3_sql": (
            v3_output[
                "sql"
            ]
        ),
        "generation_latency": (
            v3_output[
                "latency_seconds"
            ]
        ),
        "total_latency": (
            total_latency
        ),
        "reused_targeted_generation": (
            False
        ),
    }

    with open(
        v3_full_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as checkpoint_file:
        checkpoint_file.write(
            json.dumps(
                result_row,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

    v3_completed_rows.append(
        result_row
    )

    completed_count = len(
        v3_completed_rows
    )

    current_v3_accuracy = float(
        pd.DataFrame(
            v3_completed_rows
        )[
            "v3_correct"
        ].mean()
    )

    print(
        f"[{completed_count:03d}/120] "
        f"{question_id} | "
        f"domain="
        f"{v3_output['selected_domain']} | "
        f"correct="
        f"{v3_evaluation['execution_correct']} | "
        f"running_accuracy="
        f"{current_v3_accuracy:.2%}"
    )


# ------------------------------------------------------------
# 8. Finalize full V3 results
# ------------------------------------------------------------

optimized_rag_v3_full_df = (
    pd.DataFrame(
        v3_completed_rows
    )
    .drop_duplicates(
        subset=[
            "question_id"
        ],
        keep="last",
    )
    .sort_values(
        "question_id"
    )
    .reset_index(drop=True)
)

if len(optimized_rag_v3_full_df) != 120:
    raise ValueError(
        "V3 benchmark incomplete: "
        f"{len(optimized_rag_v3_full_df)}/120"
    )

optimized_rag_v3_full_df[
    "corrected_v2_error"
] = (
    optimized_rag_v3_full_df[
        "v3_correct"
    ]
    & ~optimized_rag_v3_full_df[
        "v2_correct"
    ]
)

optimized_rag_v3_full_df[
    "regressed_from_v2"
] = (
    ~optimized_rag_v3_full_df[
        "v3_correct"
    ]
    & optimized_rag_v3_full_df[
        "v2_correct"
    ]
)

optimized_rag_v3_full_df.to_csv(
    v3_full_csv_path,
    index=False,
)


# ------------------------------------------------------------
# 9. Global metrics
# ------------------------------------------------------------

v2_full_accuracy = float(
    optimized_rag_v3_full_df[
        "v2_correct"
    ].mean()
)

v3_full_accuracy = float(
    optimized_rag_v3_full_df[
        "v3_correct"
    ].mean()
)

v3_corrections = int(
    optimized_rag_v3_full_df[
        "corrected_v2_error"
    ].sum()
)

v3_regressions = int(
    optimized_rag_v3_full_df[
        "regressed_from_v2"
    ].sum()
)

v3_net_corrections = (
    v3_corrections
    - v3_regressions
)


# ------------------------------------------------------------
# 10. Split and language summaries
# ------------------------------------------------------------

v3_accuracy_by_split_df = (
    optimized_rag_v3_full_df.groupby(
        "split"
    )
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        v2_accuracy=(
            "v2_correct",
            "mean",
        ),
        v3_accuracy=(
            "v3_correct",
            "mean",
        ),
        corrections=(
            "corrected_v2_error",
            "sum",
        ),
        regressions=(
            "regressed_from_v2",
            "sum",
        ),
    )
    .reset_index()
)

v3_accuracy_by_split_df[
    "difference"
] = (
    v3_accuracy_by_split_df[
        "v3_accuracy"
    ]
    - v3_accuracy_by_split_df[
        "v2_accuracy"
    ]
)

v3_accuracy_by_language_df = (
    optimized_rag_v3_full_df.groupby(
        "language"
    )
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        v2_accuracy=(
            "v2_correct",
            "mean",
        ),
        v3_accuracy=(
            "v3_correct",
            "mean",
        ),
        corrections=(
            "corrected_v2_error",
            "sum",
        ),
        regressions=(
            "regressed_from_v2",
            "sum",
        ),
    )
    .reset_index()
)

v3_accuracy_by_language_df[
    "difference"
] = (
    v3_accuracy_by_language_df[
        "v3_accuracy"
    ]
    - v3_accuracy_by_language_df[
        "v2_accuracy"
    ]
)


# ------------------------------------------------------------
# 11. Save summary
# ------------------------------------------------------------

v3_full_summary = {
    "questions": 120,
    "v2_accuracy": (
        v2_full_accuracy
    ),
    "v3_accuracy": (
        v3_full_accuracy
    ),
    "accuracy_difference": (
        v3_full_accuracy
        - v2_full_accuracy
    ),
    "corrections": (
        v3_corrections
    ),
    "regressions": (
        v3_regressions
    ),
    "net_corrections": (
        v3_net_corrections
    ),
    "target_75_reached": bool(
        v3_full_accuracy >= 0.75
    ),
    "post_final_development": True,
    "new_untouched_test_required": True,
}

with open(
    v3_full_summary_path,
    "w",
    encoding="utf-8",
) as summary_file:
    json.dump(
        v3_full_summary,
        summary_file,
        indent=2,
    )


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print()
print("=" * 70)
print("OPTIMIZED RAG V3 — FULL RESULTS")
print("=" * 70)

print(
    "V2 full accuracy:",
    f"{v2_full_accuracy:.2%}",
)

print(
    "V3 full accuracy:",
    f"{v3_full_accuracy:.2%}",
)

print(
    "Accuracy difference:",
    f"{v3_full_accuracy - v2_full_accuracy:+.2%}",
)

print(
    "Corrections:",
    v3_corrections,
)

print(
    "Regressions:",
    v3_regressions,
)

print(
    "Net corrections:",
    v3_net_corrections,
)

print(
    "75% target reached:",
    v3_full_accuracy >= 0.75,
)

print()
print("ACCURACY BY SPLIT")

display(
    v3_accuracy_by_split_df.style.format({
        "v2_accuracy": "{:.2%}",
        "v3_accuracy": "{:.2%}",
        "difference": "{:+.2%}",
    })
)

print()
print("ACCURACY BY LANGUAGE")

display(
    v3_accuracy_by_language_df.style.format({
        "v2_accuracy": "{:.2%}",
        "v3_accuracy": "{:.2%}",
        "difference": "{:+.2%}",
    })
)

print()
print("V3 REGRESSIONS")

print(
    optimized_rag_v3_full_df[
        optimized_rag_v3_full_df[
            "regressed_from_v2"
        ]
    ][
        [
            "question_id",
            "intent_name",
            "language",
            "selected_domain",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("V3 CORRECTIONS")

print(
    optimized_rag_v3_full_df[
        optimized_rag_v3_full_df[
            "corrected_v2_error"
        ]
    ][
        [
            "question_id",
            "intent_name",
            "language",
            "selected_domain",
        ]
    ]
    .to_string(
        index=False
    )
)

print()
print("Frozen final result modified:", False)
print("Post-final V3 development:", True)
print("New untouched test required:", True)
print("Saved:", v3_full_csv_path)
print("Summary saved:", v3_full_summary_path)

OPTIMIZED RAG V3 — FULL REGRESSION BENCHMARK
Target questions: 120
Previously completed or reused: 9
Remaining generations: 111
V2 frozen reference available: True
[010/120] B4A-001-ENGLISH | domain=sales | correct=True | running_accuracy=80.00%
[011/120] B4A-001-ARABIC | domain=sales | correct=False | running_accuracy=72.73%
[012/120] B4A-001-FRENCH | domain=customer | correct=True | running_accuracy=75.00%
[013/120] B4A-001-LEBANESE_ARABIZI | domain=sales | correct=True | running_accuracy=76.92%
[014/120] B4A-002-ENGLISH | domain=sales | correct=False | running_accuracy=71.43%
[015/120] B4A-002-ARABIC | domain=sales | correct=False | running_accuracy=66.67%
[016/120] B4A-002-FRENCH | domain=sales | correct=False | running_accuracy=62.50%
[017/120] B4A-002-LEBANESE_ARABIZI | domain=sales | correct=False | running_accuracy=58.82%
[018/120] B4A-003-ENGLISH | domain=sales | correct=True | running_accuracy=61.11%
[019/120] B4A-003-ARABIC | domain=sales | correct=True | running_accuracy=63

,split,questions,v2_accuracy,v3_accuracy,corrections,regressions,difference
0,final_test,24,62.50%,75.00%,7,4,+12.50%
1,train,72,75.00%,58.33%,4,16,-16.67%
2,validation,24,70.83%,66.67%,5,6,-4.17%



ACCURACY BY LANGUAGE


,language,questions,v2_accuracy,v3_accuracy,corrections,regressions,difference
0,arabic,30,63.33%,56.67%,6,8,-6.67%
1,english,30,86.67%,66.67%,2,8,-20.00%
2,french,30,80.00%,70.00%,4,7,-10.00%
3,lebanese_arabizi,30,56.67%,60.00%,4,3,+3.33%



V3 REGRESSIONS
             question_id                          intent_name         language selected_domain
          B4A-001-ARABIC         recognized_revenue_by_region           arabic           sales
          B4A-002-ARABIC           monthly_recognized_revenue           arabic           sales
         B4A-002-ENGLISH           monthly_recognized_revenue          english           sales
          B4A-002-FRENCH           monthly_recognized_revenue           french           sales
          B4A-004-FRENCH                net_merchandise_sales           french           sales
B4A-006-LEBANESE_ARABIZI                     orders_by_status lebanese_arabizi           sales
B4A-011-LEBANESE_ARABIZI           paid_collections_by_method lebanese_arabizi           sales
          B4A-013-ARABIC                  failed_payment_rate           arabic           sales
         B4A-013-ENGLISH                  failed_payment_rate          english           sales
         B4A-017-ENGLISH        re

In [55]:
# ============================================================
# CELL 42 — FIND SAFE V3 IMPROVEMENTS FOR ONE FINAL RAG
# Selection uses train + validation only
# ============================================================

import os
import pandas as pd

v3_results_path = (
    "/kaggle/working/build4all_final_experiment/results/"
    "optimized_rag_v3_full_results.csv"
)

v3_full_df = pd.read_csv(v3_results_path)

required_columns = {
    "question_id",
    "split",
    "intent_name",
    "language",
    "v2_correct",
    "v3_correct",
}

missing_columns = required_columns - set(v3_full_df.columns)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# Convert correctness columns safely to Boolean
for column in ["v2_correct", "v3_correct"]:
    if v3_full_df[column].dtype != bool:
        v3_full_df[column] = (
            v3_full_df[column]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )

v3_full_df["v3_correction"] = (
    (~v3_full_df["v2_correct"]) &
    (v3_full_df["v3_correct"])
)

v3_full_df["v3_regression"] = (
    (v3_full_df["v2_correct"]) &
    (~v3_full_df["v3_correct"])
)

# Only train + validation may influence the design
development_df = v3_full_df[
    v3_full_df["split"].isin(["train", "validation"])
].copy()

intent_audit = (
    development_df
    .groupby("intent_name", as_index=False)
    .agg(
        questions=("question_id", "count"),
        v2_correct=("v2_correct", "sum"),
        v3_correct=("v3_correct", "sum"),
        corrections=("v3_correction", "sum"),
        regressions=("v3_regression", "sum"),
    )
)

intent_audit["v2_accuracy"] = (
    100.0 * intent_audit["v2_correct"] /
    intent_audit["questions"]
)

intent_audit["v3_accuracy"] = (
    100.0 * intent_audit["v3_correct"] /
    intent_audit["questions"]
)

intent_audit["accuracy_difference"] = (
    intent_audit["v3_accuracy"] -
    intent_audit["v2_accuracy"]
)

# Conservative rule:
# accept a V3 improvement only if it corrected at least one
# development example and caused zero development regressions.
safe_improvements_df = intent_audit[
    (intent_audit["corrections"] >= 1) &
    (intent_audit["regressions"] == 0) &
    (intent_audit["v3_accuracy"] > intent_audit["v2_accuracy"])
].copy()

safe_improvements_df = safe_improvements_df.sort_values(
    ["accuracy_difference", "corrections"],
    ascending=False
).reset_index(drop=True)

safe_intents = safe_improvements_df["intent_name"].tolist()

print("=" * 70)
print("SAFE IMPROVEMENTS FOR ONE FINAL OPTIMIZED RAG")
print("=" * 70)

print("\nSelection data:")
print("Train + validation only")

print("\nSafe intents:")
if safe_intents:
    for intent in safe_intents:
        print(f"- {intent}")
else:
    print("No V3 intent passed the conservative rule.")

print("\nSAFE IMPROVEMENT AUDIT")
display(
    safe_improvements_df[
        [
            "intent_name",
            "questions",
            "v2_accuracy",
            "v3_accuracy",
            "corrections",
            "regressions",
            "accuracy_difference",
        ]
    ].style.format(
        {
            "v2_accuracy": "{:.2f}%",
            "v3_accuracy": "{:.2f}%",
            "accuracy_difference": "{:+.2f}%",
        }
    )
)

print("\nALL DEVELOPMENT INTENTS")
display(
    intent_audit.sort_values(
        ["accuracy_difference", "regressions"],
        ascending=[False, True]
    ).style.format(
        {
            "v2_accuracy": "{:.2f}%",
            "v3_accuracy": "{:.2f}%",
            "accuracy_difference": "{:+.2f}%",
        }
    )
)

safe_path = (
    "/kaggle/working/build4all_final_experiment/results/"
    "safe_v3_improvements_for_final_rag.csv"
)

safe_improvements_df.to_csv(safe_path, index=False)

print("\nFrozen final test used for selection: False")
print("New SQL generations performed: 0")
print("Saved:", safe_path)

SAFE IMPROVEMENTS FOR ONE FINAL OPTIMIZED RAG

Selection data:
Train + validation only

Safe intents:
- operational_order_backlog
- order_cancellation_rate
- recognized_shipping_amount_by_method
- gross_merchandise_value
- paid_payment_order_reconciliation
- product_events_by_type

SAFE IMPROVEMENT AUDIT


,intent_name,questions,v2_accuracy,v3_accuracy,corrections,regressions,accuracy_difference
0,operational_order_backlog,4,50.00%,100.00%,2,0,+50.00%
1,order_cancellation_rate,4,25.00%,75.00%,2,0,+50.00%
2,recognized_shipping_amount_by_method,4,50.00%,100.00%,2,0,+50.00%
3,gross_merchandise_value,4,75.00%,100.00%,1,0,+25.00%
4,paid_payment_order_reconciliation,4,75.00%,100.00%,1,0,+25.00%
5,product_events_by_type,4,75.00%,100.00%,1,0,+25.00%



ALL DEVELOPMENT INTENTS


,intent_name,questions,v2_correct,v3_correct,corrections,regressions,v2_accuracy,v3_accuracy,accuracy_difference
8,operational_order_backlog,4,2,4,2,0,50.00%,100.00%,+50.00%
9,order_cancellation_rate,4,1,3,2,0,25.00%,75.00%,+50.00%
20,recognized_shipping_amount_by_method,4,2,4,2,0,50.00%,100.00%,+50.00%
3,gross_merchandise_value,4,3,4,1,0,75.00%,100.00%,+25.00%
14,paid_payment_order_reconciliation,4,3,4,1,0,75.00%,100.00%,+25.00%
15,product_events_by_type,4,3,4,1,0,75.00%,100.00%,+25.00%
0,active_inventory_value,4,4,4,0,0,100.00%,100.00%,+0.00%
1,coupon_usage_count,4,0,0,0,0,0.00%,0.00%,+0.00%
2,effective_tax_rate,4,4,4,0,0,100.00%,100.00%,+0.00%
12,outstanding_payment_amount,4,4,4,0,0,100.00%,100.00%,+0.00%



Frozen final test used for selection: False
New SQL generations performed: 0
Saved: /kaggle/working/build4all_final_experiment/results/safe_v3_improvements_for_final_rag.csv


In [56]:
# ============================================================
# CELL 43 — BUILD ONE CONSERVATIVE FINAL OPTIMIZED RAG POLICY
# V2 foundation + only development-safe V3 improvements
# ============================================================

import json
import os
import pandas as pd

results_directory = (
    "/kaggle/working/build4all_final_experiment/results"
)

v3_full_path = os.path.join(
    results_directory,
    "optimized_rag_v3_full_results.csv"
)

safe_improvements_path = os.path.join(
    results_directory,
    "safe_v3_improvements_for_final_rag.csv"
)

v3_full_df = pd.read_csv(v3_full_path)
safe_improvements_df = pd.read_csv(safe_improvements_path)

for column in ["v2_correct", "v3_correct"]:
    if v3_full_df[column].dtype != bool:
        v3_full_df[column] = (
            v3_full_df[column]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )

safe_intents = set(
    safe_improvements_df["intent_name"].dropna().tolist()
)

# This represents ONE final RAG policy:
# use the safe enhanced behavior internally for approved intents,
# otherwise preserve the stable V2 behavior.
v3_full_df["final_rag_internal_policy"] = v3_full_df[
    "intent_name"
].apply(
    lambda intent: (
        "enhanced_domain_guidance"
        if intent in safe_intents
        else "stable_base_guidance"
    )
)

v3_full_df["final_rag_correct"] = v3_full_df.apply(
    lambda row: (
        bool(row["v3_correct"])
        if row["intent_name"] in safe_intents
        else bool(row["v2_correct"])
    ),
    axis=1
)

v3_full_df["final_rag_corrects_v2"] = (
    (~v3_full_df["v2_correct"]) &
    (v3_full_df["final_rag_correct"])
)

v3_full_df["final_rag_regresses_v2"] = (
    (v3_full_df["v2_correct"]) &
    (~v3_full_df["final_rag_correct"])
)

summary_rows = []

for split_name, split_df in [
    ("development", v3_full_df[
        v3_full_df["split"].isin(["train", "validation"])
    ]),
    ("train", v3_full_df[v3_full_df["split"] == "train"]),
    ("validation", v3_full_df[
        v3_full_df["split"] == "validation"
    ]),
    # Diagnostic only: old final was already opened.
    ("old_final_diagnostic", v3_full_df[
        v3_full_df["split"] == "final_test"
    ]),
    ("full_posthoc_diagnostic", v3_full_df),
]:
    if len(split_df) == 0:
        continue

    summary_rows.append(
        {
            "split": split_name,
            "questions": len(split_df),
            "v2_accuracy": 100.0 * split_df[
                "v2_correct"
            ].mean(),
            "final_rag_accuracy": 100.0 * split_df[
                "final_rag_correct"
            ].mean(),
            "corrections_over_v2": int(
                split_df["final_rag_corrects_v2"].sum()
            ),
            "regressions_from_v2": int(
                split_df["final_rag_regresses_v2"].sum()
            ),
            "difference_over_v2": 100.0 * (
                split_df["final_rag_correct"].mean() -
                split_df["v2_correct"].mean()
            ),
        }
    )

final_rag_summary_df = pd.DataFrame(summary_rows)

print("=" * 70)
print("ONE CONSERVATIVE FINAL OPTIMIZED RAG")
print("=" * 70)

print("\nInternally enhanced intents:")
for intent in sorted(safe_intents):
    print(f"- {intent}")

print("\nACCURACY SUMMARY")
display(
    final_rag_summary_df.style.format(
        {
            "v2_accuracy": "{:.2f}%",
            "final_rag_accuracy": "{:.2f}%",
            "difference_over_v2": "{:+.2f}%",
        }
    )
)

print("\nFINAL RAG INTERNAL POLICY DISTRIBUTION")
print(
    v3_full_df.groupby(
        ["split", "final_rag_internal_policy"]
    ).size().unstack(fill_value=0)
)

print("\nDEVELOPMENT CORRECTIONS")
development_corrections = v3_full_df[
    v3_full_df["split"].isin(["train", "validation"]) &
    v3_full_df["final_rag_corrects_v2"]
][
    ["question_id", "intent_name", "language"]
]

display(development_corrections)

final_policy_path = os.path.join(
    results_directory,
    "final_optimized_rag_policy.json"
)

final_results_path = os.path.join(
    results_directory,
    "final_optimized_rag_posthoc_audit.csv"
)

final_summary_path = os.path.join(
    results_directory,
    "final_optimized_rag_posthoc_summary.csv"
)

policy_payload = {
    "system_name": "Final Optimized RAG",
    "foundation": "optimized_rag_v2",
    "safe_enhanced_intents": sorted(safe_intents),
    "selection_source": "train_and_validation_only",
    "old_final_used_for_selection": False,
    "requires_new_untouched_test": True,
}

with open(final_policy_path, "w", encoding="utf-8") as file:
    json.dump(
        policy_payload,
        file,
        ensure_ascii=False,
        indent=2
    )

v3_full_df.to_csv(final_results_path, index=False)
final_rag_summary_df.to_csv(final_summary_path, index=False)

print("\nOld final used for selecting improvements: False")
print("New SQL generations performed: 0")
print("This is one RAG policy: True")
print("New untouched test still required: True")
print("Policy saved:", final_policy_path)
print("Audit saved:", final_results_path)
print("Summary saved:", final_summary_path)

ONE CONSERVATIVE FINAL OPTIMIZED RAG

Internally enhanced intents:
- gross_merchandise_value
- operational_order_backlog
- order_cancellation_rate
- paid_payment_order_reconciliation
- product_events_by_type
- recognized_shipping_amount_by_method

ACCURACY SUMMARY


,split,questions,v2_accuracy,final_rag_accuracy,corrections_over_v2,regressions_from_v2,difference_over_v2
0,development,96,73.96%,83.33%,9,0,+9.38%
1,train,72,75.00%,80.56%,4,0,+5.56%
2,validation,24,70.83%,91.67%,5,0,+20.83%
3,old_final_diagnostic,24,62.50%,62.50%,0,0,+0.00%
4,full_posthoc_diagnostic,120,71.67%,79.17%,9,0,+7.50%



FINAL RAG INTERNAL POLICY DISTRIBUTION
final_rag_internal_policy  enhanced_domain_guidance  stable_base_guidance
split                                                                    
final_test                                        0                    24
train                                             8                    64
validation                                       16                     8

DEVELOPMENT CORRECTIONS


,question_id,intent_name,language
8,B4A-003-ARABIC,gross_merchandise_value,arabic
24,B4A-007-ARABIC,operational_order_backlog,arabic
26,B4A-007-FRENCH,operational_order_backlog,french
29,B4A-008-ENGLISH,order_cancellation_rate,english
30,B4A-008-FRENCH,order_cancellation_rate,french
59,B4A-015-LEBANESE_ARABIZI,paid_payment_order_reconciliation,lebanese_arabizi
100,B4A-026-ARABIC,recognized_shipping_amount_by_method,arabic
102,B4A-026-FRENCH,recognized_shipping_amount_by_method,french
115,B4A-029-LEBANESE_ARABIZI,product_events_by_type,lebanese_arabizi



Old final used for selecting improvements: False
New SQL generations performed: 0
This is one RAG policy: True
New untouched test still required: True
Policy saved: /kaggle/working/build4all_final_experiment/results/final_optimized_rag_policy.json
Audit saved: /kaggle/working/build4all_final_experiment/results/final_optimized_rag_posthoc_audit.csv
Summary saved: /kaggle/working/build4all_final_experiment/results/final_optimized_rag_posthoc_summary.csv


In [57]:
# ============================================================
# CELL 44 — AUTOMATIC MULTILINGUAL GUIDANCE DETECTOR
# Uses question text only at prediction time
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

results_directory = (
    "/kaggle/working/build4all_final_experiment/results"
)

audit_path = os.path.join(
    results_directory,
    "final_optimized_rag_posthoc_audit.csv"
)

safe_path = os.path.join(
    results_directory,
    "safe_v3_improvements_for_final_rag.csv"
)

audit_df = pd.read_csv(audit_path)
safe_df = pd.read_csv(safe_path)

safe_intents = set(
    safe_df["intent_name"].dropna().astype(str)
)

# ------------------------------------------------------------
# Find or restore question text
# ------------------------------------------------------------

question_candidates = [
    "question",
    "question_text",
    "user_question",
    "natural_language_question",
]

question_column = next(
    (
        column for column in question_candidates
        if column in audit_df.columns
    ),
    None,
)

if question_column is None:
    benchmark_question_column = next(
        (
            column for column in question_candidates
            if column in multilingual_benchmark_df.columns
        ),
        None,
    )

    if benchmark_question_column is None:
        raise ValueError(
            "Could not find the question-text column."
        )

    question_lookup = multilingual_benchmark_df[
        ["question_id", benchmark_question_column]
    ].drop_duplicates("question_id")

    audit_df = audit_df.merge(
        question_lookup,
        on="question_id",
        how="left",
    )

    question_column = benchmark_question_column

if audit_df[question_column].isna().any():
    missing_ids = audit_df.loc[
        audit_df[question_column].isna(),
        "question_id",
    ].tolist()

    raise ValueError(
        f"Missing question text for: {missing_ids}"
    )

# ------------------------------------------------------------
# Prepare development data only
# ------------------------------------------------------------

development_df = audit_df[
    audit_df["split"].isin(["train", "validation"])
].copy().reset_index(drop=True)

for column in ["v2_correct", "v3_correct"]:
    if development_df[column].dtype != bool:
        development_df[column] = (
            development_df[column]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )

development_df["should_enhance"] = (
    development_df["intent_name"].isin(safe_intents)
)

questions = development_df[
    question_column
].astype(str).tolist()

# Multilingual semantic embeddings
question_embeddings = embedding_model.encode(
    questions,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# ------------------------------------------------------------
# Leave-one-out semantic score
# Prevent each question from matching itself
# ------------------------------------------------------------

safe_indices_by_intent = {
    intent: development_df.index[
        development_df["intent_name"] == intent
    ].tolist()
    for intent in sorted(safe_intents)
}

semantic_scores = []
predicted_safe_intents = []

for row_index in range(len(development_df)):
    best_score = -1.0
    best_intent = None

    for intent, intent_indices in safe_indices_by_intent.items():
        prototype_indices = [
            index for index in intent_indices
            if index != row_index
        ]

        if not prototype_indices:
            continue

        prototype = question_embeddings[
            prototype_indices
        ].mean(axis=0)

        prototype = prototype / max(
            np.linalg.norm(prototype),
            1e-12,
        )

        score = float(
            np.dot(
                question_embeddings[row_index],
                prototype,
            )
        )

        if score > best_score:
            best_score = score
            best_intent = intent

    semantic_scores.append(best_score)
    predicted_safe_intents.append(best_intent)

development_df["enhancement_similarity"] = semantic_scores
development_df["nearest_safe_intent"] = predicted_safe_intents

# ------------------------------------------------------------
# Select threshold using development correctness
# Accuracy first; fewer enhanced questions as tie-break
# ------------------------------------------------------------

threshold_rows = []

for threshold in np.arange(0.40, 0.951, 0.01):
    predicted_enhance = (
        development_df["enhancement_similarity"] >= threshold
    )

    automatic_correct = np.where(
        predicted_enhance,
        development_df["v3_correct"],
        development_df["v2_correct"],
    ).astype(bool)

    threshold_rows.append(
        {
            "threshold": round(float(threshold), 2),
            "automatic_rag_accuracy": float(
                automatic_correct.mean()
            ),
            "enhanced_questions": int(
                predicted_enhance.sum()
            ),
            "correct_enhancement_detections": int(
                (
                    predicted_enhance &
                    development_df["should_enhance"]
                ).sum()
            ),
            "false_enhancement_detections": int(
                (
                    predicted_enhance &
                    ~development_df["should_enhance"]
                ).sum()
            ),
        }
    )

threshold_audit_df = pd.DataFrame(threshold_rows)

selected_row = threshold_audit_df.sort_values(
    [
        "automatic_rag_accuracy",
        "false_enhancement_detections",
        "enhanced_questions",
        "threshold",
    ],
    ascending=[False, True, True, False],
).iloc[0]

final_rag_similarity_threshold = float(
    selected_row["threshold"]
)

development_df["automatic_enhancement"] = (
    development_df["enhancement_similarity"] >=
    final_rag_similarity_threshold
)

development_df["automatic_final_rag_correct"] = np.where(
    development_df["automatic_enhancement"],
    development_df["v3_correct"],
    development_df["v2_correct"],
).astype(bool)

detector_precision = precision_score(
    development_df["should_enhance"],
    development_df["automatic_enhancement"],
    zero_division=0,
)

detector_recall = recall_score(
    development_df["should_enhance"],
    development_df["automatic_enhancement"],
    zero_division=0,
)

detector_f1 = f1_score(
    development_df["should_enhance"],
    development_df["automatic_enhancement"],
    zero_division=0,
)

v2_accuracy = development_df["v2_correct"].mean()
automatic_accuracy = development_df[
    "automatic_final_rag_correct"
].mean()

corrections = int(
    (
        ~development_df["v2_correct"] &
        development_df["automatic_final_rag_correct"]
    ).sum()
)

regressions = int(
    (
        development_df["v2_correct"] &
        ~development_df["automatic_final_rag_correct"]
    ).sum()
)

print("=" * 70)
print("AUTOMATIC MULTILINGUAL GUIDANCE DETECTOR")
print("=" * 70)

print(f"Selected similarity threshold: {final_rag_similarity_threshold:.2f}")
print(f"Detector precision: {detector_precision:.2%}")
print(f"Detector recall: {detector_recall:.2%}")
print(f"Detector F1: {detector_f1:.2%}")

print("\nDEVELOPMENT ACCURACY")
print(f"Stable V2 base: {v2_accuracy:.2%}")
print(f"Automatic Final RAG: {automatic_accuracy:.2%}")
print(f"Difference: {automatic_accuracy - v2_accuracy:+.2%}")
print(f"Corrections over V2: {corrections}")
print(f"Regressions from V2: {regressions}")
print(
    "Enhanced questions:",
    int(development_df["automatic_enhancement"].sum()),
)

print("\nAUTOMATICALLY ENHANCED QUESTIONS")
display(
    development_df.loc[
        development_df["automatic_enhancement"],
        [
            "question_id",
            "language",
            "intent_name",
            "nearest_safe_intent",
            "enhancement_similarity",
            "v2_correct",
            "v3_correct",
            "automatic_final_rag_correct",
        ],
    ].sort_values(
        "enhancement_similarity",
        ascending=False,
    ).style.format(
        {"enhancement_similarity": "{:.4f}"}
    )
)

detector_path = os.path.join(
    results_directory,
    "automatic_final_rag_detector.json",
)

detector_audit_path = os.path.join(
    results_directory,
    "automatic_final_rag_detector_audit.csv",
)

detector_payload = {
    "similarity_threshold": final_rag_similarity_threshold,
    "safe_intents": sorted(safe_intents),
    "selection_data": "train_and_validation_only",
    "question_text_only_at_prediction": True,
    "old_final_used_for_selection": False,
    "development_accuracy": float(automatic_accuracy),
    "corrections_over_v2": corrections,
    "regressions_from_v2": regressions,
}

with open(detector_path, "w", encoding="utf-8") as file:
    json.dump(
        detector_payload,
        file,
        ensure_ascii=False,
        indent=2,
    )

development_df.to_csv(
    detector_audit_path,
    index=False,
)

print("\nIntent label required at prediction time: False")
print("Old final used for threshold selection: False")
print("New SQL generations performed: 0")
print("Detector saved:", detector_path)
print("Audit saved:", detector_audit_path)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

AUTOMATIC MULTILINGUAL GUIDANCE DETECTOR
Selected similarity threshold: 0.91
Detector precision: 92.31%
Detector recall: 50.00%
Detector F1: 64.86%

DEVELOPMENT ACCURACY
Stable V2 base: 73.96%
Automatic Final RAG: 77.08%
Difference: +3.12%
Corrections over V2: 3
Regressions from V2: 0
Enhanced questions: 13

AUTOMATICALLY ENHANCED QUESTIONS


,question_id,language,intent_name,nearest_safe_intent,enhancement_similarity,v2_correct,v3_correct,automatic_final_rag_correct
93,B4A-029-ENGLISH,english,product_events_by_type,product_events_by_type,0.9526,True,True,True
81,B4A-026-ENGLISH,english,recognized_shipping_amount_by_method,recognized_shipping_amount_by_method,0.9365,True,True,True
21,B4A-007-ENGLISH,english,operational_order_backlog,operational_order_backlog,0.9335,True,True,True
9,B4A-003-ENGLISH,english,gross_merchandise_value,gross_merchandise_value,0.9328,True,True,True
82,B4A-026-FRENCH,french,recognized_shipping_amount_by_method,recognized_shipping_amount_by_method,0.9314,False,True,True
94,B4A-029-FRENCH,french,product_events_by_type,product_events_by_type,0.9301,True,True,True
46,B4A-015-FRENCH,french,paid_payment_order_reconciliation,paid_payment_order_reconciliation,0.9297,True,True,True
45,B4A-015-ENGLISH,english,paid_payment_order_reconciliation,paid_payment_order_reconciliation,0.9281,True,True,True
10,B4A-003-FRENCH,french,gross_merchandise_value,gross_merchandise_value,0.9212,True,True,True
26,B4A-008-FRENCH,french,order_cancellation_rate,order_cancellation_rate,0.9208,False,True,True



Intent label required at prediction time: False
Old final used for threshold selection: False
New SQL generations performed: 0
Detector saved: /kaggle/working/build4all_final_experiment/results/automatic_final_rag_detector.json
Audit saved: /kaggle/working/build4all_final_experiment/results/automatic_final_rag_detector_audit.csv


In [58]:
# ============================================================
# CELL 45 — ONE EXECUTABLE FINAL OPTIMIZED RAG
# Public system: generate_sql_final_optimized_rag(question)
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

results_directory = (
    "/kaggle/working/build4all_final_experiment/results"
)

detector_audit_path = os.path.join(
    results_directory,
    "automatic_final_rag_detector_audit.csv"
)

detector_config_path = os.path.join(
    results_directory,
    "automatic_final_rag_detector.json"
)

detector_df = pd.read_csv(detector_audit_path)

with open(
    detector_config_path,
    "r",
    encoding="utf-8",
) as file:
    detector_config = json.load(file)

final_rag_similarity_threshold = float(
    detector_config["similarity_threshold"]
)

safe_intents = set(
    detector_config["safe_intents"]
)

question_candidates = [
    "question",
    "question_text",
    "user_question",
    "natural_language_question",
]

question_column = next(
    (
        column for column in question_candidates
        if column in detector_df.columns
    ),
    None,
)

if question_column is None:
    raise ValueError(
        "Question-text column was not found in detector audit."
    )

# ------------------------------------------------------------
# Build final multilingual semantic prototypes
# Development questions only
# ------------------------------------------------------------

final_rag_prototypes = {}

for intent in sorted(safe_intents):
    intent_questions = detector_df.loc[
        detector_df["intent_name"] == intent,
        question_column,
    ].dropna().astype(str).tolist()

    if not intent_questions:
        continue

    intent_embeddings = embedding_model.encode(
        intent_questions,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    prototype = intent_embeddings.mean(axis=0)
    prototype = prototype / max(
        np.linalg.norm(prototype),
        1e-12,
    )

    final_rag_prototypes[intent] = prototype

if not final_rag_prototypes:
    raise ValueError(
        "No final RAG semantic prototypes were created."
    )

# ------------------------------------------------------------
# Automatic question-text detector
# ------------------------------------------------------------

def detect_final_rag_guidance(question):
    question = str(question).strip()

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]

    best_intent = None
    best_similarity = -1.0

    for intent, prototype in final_rag_prototypes.items():
        similarity = float(
            np.dot(question_embedding, prototype)
        )

        if similarity > best_similarity:
            best_similarity = similarity
            best_intent = intent

    use_enhanced_guidance = (
        best_similarity >= final_rag_similarity_threshold
    )

    return {
        "use_enhanced_guidance": use_enhanced_guidance,
        "nearest_guidance_family": best_intent,
        "similarity": best_similarity,
        "threshold": final_rag_similarity_threshold,
    }


# ------------------------------------------------------------
# ONE public Final Optimized RAG generator
# ------------------------------------------------------------

def generate_sql_final_optimized_rag(
    question,
    return_metadata=False,
):
    guidance_decision = detect_final_rag_guidance(
        question
    )

    if guidance_decision["use_enhanced_guidance"]:
        generated_sql = generate_sql_optimized_rag_v3(
            question
        )
        internal_guidance = "enhanced"
    else:
        generated_sql = generate_sql_optimized_rag_v2(
            question
        )
        internal_guidance = "stable"
    
    metadata = {
        "system": "final_optimized_rag",
        "internal_guidance": internal_guidance,
        "nearest_guidance_family": guidance_decision[
            "nearest_guidance_family"
        ],
        "similarity": guidance_decision["similarity"],
        "threshold": guidance_decision["threshold"],
    }

    if return_metadata:
        return {
            "sql": generated_sql,
            "metadata": metadata,
        }

    return generated_sql


# ------------------------------------------------------------
# Save the frozen automatic detector
# ------------------------------------------------------------

detector_artifact = {
    "threshold": final_rag_similarity_threshold,
    "safe_intents": sorted(safe_intents),
    "prototypes": final_rag_prototypes,
    "question_column": question_column,
    "selection_data": "train_and_validation_only",
    "old_final_used_for_threshold_selection": False,
}

artifact_path = os.path.join(
    results_directory,
    "final_optimized_rag_detector.joblib",
)

joblib.dump(
    detector_artifact,
    artifact_path,
)

# ------------------------------------------------------------
# Sanity check without SQL generation
# ------------------------------------------------------------

sanity_questions = detector_df[
    question_column
].dropna().astype(str).head(3).tolist()

print("=" * 70)
print("ONE EXECUTABLE FINAL OPTIMIZED RAG")
print("=" * 70)

for index, question in enumerate(
    sanity_questions,
    start=1,
):
    decision = detect_final_rag_guidance(question)

    print(f"\n[{index}] Question: {question}")
    print(
        "Guidance:",
        (
            "enhanced"
            if decision["use_enhanced_guidance"]
            else "stable"
        ),
    )
    print(
        "Nearest family:",
        decision["nearest_guidance_family"],
    )
    print(
        "Similarity:",
        f'{decision["similarity"]:.4f}',
    )

print("\nPublic function ready:")
print("generate_sql_final_optimized_rag(question)")
print("\nIntent label required at prediction time: False")
print("Gold answer required at prediction time: False")
print("Public RAG systems exposed: 1")
print("New SQL generations performed: 0")
print("Detector artifact saved:", artifact_path)

ONE EXECUTABLE FINAL OPTIMIZED RAG

[1] Question: ما قيمة إيرادات الطلبات المعترف بها حسب منطقة العميل؟
Guidance: stable
Nearest family: gross_merchandise_value
Similarity: 0.8614

[2] Question: What is the recognized order revenue by customer region?
Guidance: stable
Nearest family: paid_payment_order_reconciliation
Similarity: 0.8509

[3] Question: Quel est le chiffre d'affaires reconnu des commandes par région client ?
Guidance: stable
Nearest family: gross_merchandise_value
Similarity: 0.8749

Public function ready:
generate_sql_final_optimized_rag(question)

Intent label required at prediction time: False
Gold answer required at prediction time: False
Public RAG systems exposed: 1
New SQL generations performed: 0
Detector artifact saved: /kaggle/working/build4all_final_experiment/results/final_optimized_rag_detector.joblib


In [59]:
# ============================================================
# CELL 46 — QWEN + FINAL RAG CONTROLLER CEILING AUDIT
# No controller training yet
# ============================================================

import os
import pandas as pd
import numpy as np

results_directory = (
    "/kaggle/working/build4all_final_experiment/results"
)

detector_audit_path = os.path.join(
    results_directory,
    "automatic_final_rag_detector_audit.csv"
)

final_rag_df = pd.read_csv(detector_audit_path)

# ------------------------------------------------------------
# Normalize Final RAG columns
# ------------------------------------------------------------

boolean_columns = [
    "v2_correct",
    "v3_correct",
    "automatic_enhancement",
    "automatic_final_rag_correct",
]

for column in boolean_columns:
    if (
        column in final_rag_df.columns and
        final_rag_df[column].dtype != bool
    ):
        final_rag_df[column] = (
            final_rag_df[column]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )

# Use the automatic detector result as the executable Final RAG result.
# Assignment avoids creating duplicate column names.
final_rag_df["final_rag_correct"] = (
    final_rag_df["automatic_final_rag_correct"]
    .astype(bool)
)

# ------------------------------------------------------------
# Prepare Qwen Alone reference
# ------------------------------------------------------------

qwen_df = qwen_alone_results_df.copy()

qwen_correct_candidates = [
    "recorded_correct",
    "correct",
    "is_correct",
    "execution_correct",
]

qwen_correct_column = next(
    (
        column for column in qwen_correct_candidates
        if column in qwen_df.columns
    ),
    None,
)

if qwen_correct_column is None:
    raise ValueError(
        "Could not find Qwen correctness column."
    )

if qwen_df[qwen_correct_column].dtype != bool:
    qwen_df[qwen_correct_column] = (
        qwen_df[qwen_correct_column]
        .astype(str)
        .str.lower()
        .map({"true": True, "false": False})
    )

qwen_reference_df = qwen_df[
    ["question_id", qwen_correct_column]
].copy()

qwen_reference_df = qwen_reference_df.rename(
    columns={
        qwen_correct_column: "qwen_alone_correct"
    }
)

# ------------------------------------------------------------
# Combine both systems
# ------------------------------------------------------------

controller_ceiling_df = final_rag_df.merge(
    qwen_reference_df,
    on="question_id",
    how="left",
    validate="one_to_one",
)

if controller_ceiling_df[
    "qwen_alone_correct"
].isna().any():
    missing_ids = controller_ceiling_df.loc[
        controller_ceiling_df[
            "qwen_alone_correct"
        ].isna(),
        "question_id",
    ].tolist()

    raise ValueError(
        f"Missing Qwen results for: {missing_ids}"
    )

controller_ceiling_df["both_correct"] = (
    controller_ceiling_df["qwen_alone_correct"] &
    controller_ceiling_df["final_rag_correct"]
)

controller_ceiling_df["qwen_only_correct"] = (
    controller_ceiling_df["qwen_alone_correct"] &
    ~controller_ceiling_df["final_rag_correct"]
)

controller_ceiling_df["final_rag_only_correct"] = (
    ~controller_ceiling_df["qwen_alone_correct"] &
    controller_ceiling_df["final_rag_correct"]
)

controller_ceiling_df["both_wrong"] = (
    ~controller_ceiling_df["qwen_alone_correct"] &
    ~controller_ceiling_df["final_rag_correct"]
)

controller_ceiling_df["binary_oracle_correct"] = (
    controller_ceiling_df["qwen_alone_correct"] |
    controller_ceiling_df["final_rag_correct"]
)

# ------------------------------------------------------------
# Summary by split
# ------------------------------------------------------------

summary_rows = []

for split_name, split_df in [
    (
        "development",
        controller_ceiling_df[
            controller_ceiling_df["split"].isin(
                ["train", "validation"]
            )
        ],
    ),
    (
        "train",
        controller_ceiling_df[
            controller_ceiling_df["split"] == "train"
        ],
    ),
    (
        "validation",
        controller_ceiling_df[
            controller_ceiling_df["split"] == "validation"
        ],
    ),
]:
    if len(split_df) == 0:
        continue

    summary_rows.append(
        {
            "split": split_name,
            "questions": len(split_df),
            "qwen_accuracy": (
                100.0 *
                split_df["qwen_alone_correct"].mean()
            ),
            "final_rag_accuracy": (
                100.0 *
                split_df["final_rag_correct"].mean()
            ),
            "binary_oracle_ceiling": (
                100.0 *
                split_df["binary_oracle_correct"].mean()
            ),
            "qwen_only_corrections_available": int(
                split_df["qwen_only_correct"].sum()
            ),
            "final_rag_only_correct": int(
                split_df["final_rag_only_correct"].sum()
            ),
            "both_correct": int(
                split_df["both_correct"].sum()
            ),
            "both_wrong": int(
                split_df["both_wrong"].sum()
            ),
        }
    )

controller_ceiling_summary_df = pd.DataFrame(
    summary_rows
)

print("=" * 70)
print("QWEN ALONE + FINAL OPTIMIZED RAG")
print("CONTROLLER CEILING AUDIT")
print("=" * 70)

display(
    controller_ceiling_summary_df.style.format(
        {
            "qwen_accuracy": "{:.2f}%",
            "final_rag_accuracy": "{:.2f}%",
            "binary_oracle_ceiling": "{:.2f}%",
        }
    )
)

print("\nCASES WHERE ONLY QWEN IS CORRECT")
qwen_only_df = controller_ceiling_df[
    controller_ceiling_df["qwen_only_correct"]
][
    [
        "question_id",
        "split",
        "language",
        "intent_name",
        "enhancement_similarity",
    ]
]

display(qwen_only_df)

controller_dataset_path = os.path.join(
    results_directory,
    "qwen_final_rag_controller_dataset.csv"
)

controller_summary_path = os.path.join(
    results_directory,
    "qwen_final_rag_controller_ceiling_summary.csv"
)

controller_ceiling_df.to_csv(
    controller_dataset_path,
    index=False,
)

controller_ceiling_summary_df.to_csv(
    controller_summary_path,
    index=False,
)

print("\nFinal RAG is automatic: True")
print("Gold used at prediction time: False")
print("New SQL generations performed: 0")
print("Final test opened in this cell: False")
print("Dataset saved:", controller_dataset_path)
print("Summary saved:", controller_summary_path)

QWEN ALONE + FINAL OPTIMIZED RAG
CONTROLLER CEILING AUDIT


,split,questions,qwen_accuracy,final_rag_accuracy,binary_oracle_ceiling,qwen_only_corrections_available,final_rag_only_correct,both_correct,both_wrong
0,development,96,32.29%,77.08%,78.12%,1,44,30,21
1,train,72,30.56%,76.39%,77.78%,1,34,21,16
2,validation,24,37.50%,79.17%,79.17%,0,10,9,5



CASES WHERE ONLY QWEN IS CORRECT


,question_id,split,language,intent_name,enhancement_similarity
56,B4A-019-ARABIC,train,arabic,products_by_status,0.881939



Final RAG is automatic: True
Gold used at prediction time: False
New SQL generations performed: 0
Final test opened in this cell: False
Dataset saved: /kaggle/working/build4all_final_experiment/results/qwen_final_rag_controller_dataset.csv
Summary saved: /kaggle/working/build4all_final_experiment/results/qwen_final_rag_controller_ceiling_summary.csv


In [60]:
# ============================================================
# CELL 47 — FINAL CONSERVATIVE BINARY CONTROLLER
# Routes: Qwen Alone vs Final Optimized RAG
# Accuracy first, Qwen efficiency second
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

results_directory = (
    "/kaggle/working/build4all_final_experiment/results"
)

dataset_path = os.path.join(
    results_directory,
    "qwen_final_rag_controller_dataset.csv"
)

controller_df = pd.read_csv(dataset_path)

# ------------------------------------------------------------
# Normalize Boolean columns
# ------------------------------------------------------------

for column in [
    "qwen_alone_correct",
    "final_rag_correct",
]:
    if controller_df[column].dtype != bool:
        controller_df[column] = (
            controller_df[column]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
        )

# ------------------------------------------------------------
# Find question text
# ------------------------------------------------------------

question_candidates = [
    "question",
    "question_text",
    "user_question",
    "natural_language_question",
]

question_column = next(
    (
        column for column in question_candidates
        if column in controller_df.columns
    ),
    None,
)

if question_column is None:
    benchmark_question_column = next(
        (
            column for column in question_candidates
            if column in multilingual_benchmark_df.columns
        ),
        None,
    )

    if benchmark_question_column is None:
        raise ValueError(
            "Question-text column could not be found."
        )

    question_lookup = multilingual_benchmark_df[
        ["question_id", benchmark_question_column]
    ].drop_duplicates("question_id")

    controller_df = controller_df.merge(
        question_lookup,
        on="question_id",
        how="left",
    )

    question_column = benchmark_question_column

# ------------------------------------------------------------
# Semantic question features
# ------------------------------------------------------------

questions = controller_df[
    question_column
].astype(str).tolist()

semantic_features = embedding_model.encode(
    questions,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# Add small language and detector features
language_features = pd.get_dummies(
    controller_df["language"].astype(str),
    prefix="language",
    dtype=float,
)

numeric_features = []

if "enhancement_similarity" in controller_df.columns:
    numeric_features.append(
        controller_df[
            ["enhancement_similarity"]
        ].fillna(0.0).to_numpy(dtype=float)
    )

feature_parts = [
    semantic_features,
    language_features.to_numpy(dtype=float),
]

feature_parts.extend(numeric_features)

X = np.hstack(feature_parts)

train_mask = (
    controller_df["split"] == "train"
).to_numpy()

validation_mask = (
    controller_df["split"] == "validation"
).to_numpy()

X_train = X[train_mask]
X_validation = X[validation_mask]

train_df = controller_df.loc[
    train_mask
].copy().reset_index(drop=True)

validation_df = controller_df.loc[
    validation_mask
].copy().reset_index(drop=True)

# The probability represents:
# "How likely is Qwen Alone to be correct?"
y_train = train_df[
    "qwen_alone_correct"
].astype(int).to_numpy()

# ------------------------------------------------------------
# Train-only out-of-fold configuration selection
# ------------------------------------------------------------

cv = StratifiedKFold(
    n_splits=4,
    shuffle=True,
    random_state=42,
)

candidate_c_values = [
    0.01,
    0.05,
    0.10,
    0.25,
    0.50,
    1.00,
    2.00,
]

candidate_class_weights = [
    None,
    "balanced",
]

candidate_thresholds = np.arange(
    0.50,
    0.951,
    0.05,
)

configuration_rows = []

for c_value in candidate_c_values:
    for class_weight in candidate_class_weights:
        oof_probability = np.zeros(
            len(train_df),
            dtype=float,
        )

        for fit_indices, holdout_indices in cv.split(
            X_train,
            y_train,
        ):
            fold_model = LogisticRegression(
                C=c_value,
                class_weight=class_weight,
                max_iter=3000,
                random_state=42,
            )

            fold_model.fit(
                X_train[fit_indices],
                y_train[fit_indices],
            )

            oof_probability[holdout_indices] = (
                fold_model.predict_proba(
                    X_train[holdout_indices]
                )[:, 1]
            )

        for threshold in candidate_thresholds:
            choose_qwen = (
                oof_probability >= threshold
            )

            controller_correct = np.where(
                choose_qwen,
                train_df[
                    "qwen_alone_correct"
                ].to_numpy(),
                train_df[
                    "final_rag_correct"
                ].to_numpy(),
            ).astype(bool)

            regressions = int(
                (
                    choose_qwen &
                    train_df[
                        "final_rag_correct"
                    ].to_numpy() &
                    ~train_df[
                        "qwen_alone_correct"
                    ].to_numpy()
                ).sum()
            )

            corrections = int(
                (
                    choose_qwen &
                    ~train_df[
                        "final_rag_correct"
                    ].to_numpy() &
                    train_df[
                        "qwen_alone_correct"
                    ].to_numpy()
                ).sum()
            )

            configuration_rows.append(
                {
                    "C": c_value,
                    "class_weight": (
                        "none"
                        if class_weight is None
                        else class_weight
                    ),
                    "threshold": float(threshold),
                    "oof_accuracy": float(
                        controller_correct.mean()
                    ),
                    "qwen_routes": int(
                        choose_qwen.sum()
                    ),
                    "rag_routes": int(
                        (~choose_qwen).sum()
                    ),
                    "corrections": corrections,
                    "regressions": regressions,
                }
            )

configuration_df = pd.DataFrame(
    configuration_rows
)

train_rag_accuracy = train_df[
    "final_rag_correct"
].mean()

# Strict conservative configurations:
# no regression and no lower accuracy than always using Final RAG.
safe_configurations_df = configuration_df[
    (configuration_df["regressions"] == 0) &
    (
        configuration_df["oof_accuracy"] >=
        train_rag_accuracy
    )
].copy()

if len(safe_configurations_df) > 0:
    selected_configuration = (
        safe_configurations_df
        .sort_values(
            [
                "oof_accuracy",
                "qwen_routes",
                "threshold",
            ],
            ascending=[False, False, False],
        )
        .iloc[0]
    )
else:
    # Safest fallback: choose configuration with
    # maximum accuracy and minimum regressions.
    selected_configuration = (
        configuration_df
        .sort_values(
            [
                "oof_accuracy",
                "regressions",
                "qwen_routes",
                "threshold",
            ],
            ascending=[False, True, False, False],
        )
        .iloc[0]
    )

selected_c = float(
    selected_configuration["C"]
)

selected_class_weight = (
    None
    if selected_configuration["class_weight"] == "none"
    else selected_configuration["class_weight"]
)

final_controller_threshold = float(
    selected_configuration["threshold"]
)

# ------------------------------------------------------------
# Fit frozen controller on train only
# ------------------------------------------------------------

final_controller_model = LogisticRegression(
    C=selected_c,
    class_weight=selected_class_weight,
    max_iter=3000,
    random_state=42,
)

final_controller_model.fit(
    X_train,
    y_train,
)

validation_probability = (
    final_controller_model.predict_proba(
        X_validation
    )[:, 1]
)

validation_choose_qwen = (
    validation_probability >=
    final_controller_threshold
)

validation_df["qwen_probability"] = (
    validation_probability
)

validation_df["controller_route"] = np.where(
    validation_choose_qwen,
    "qwen_alone",
    "final_optimized_rag",
)

validation_df["controller_correct"] = np.where(
    validation_choose_qwen,
    validation_df["qwen_alone_correct"],
    validation_df["final_rag_correct"],
).astype(bool)

validation_df["corrected_final_rag_error"] = (
    validation_choose_qwen &
    ~validation_df["final_rag_correct"] &
    validation_df["qwen_alone_correct"]
)

validation_df["regressed_from_final_rag"] = (
    validation_choose_qwen &
    validation_df["final_rag_correct"] &
    ~validation_df["qwen_alone_correct"]
)

# ------------------------------------------------------------
# Final validation summary
# ------------------------------------------------------------

qwen_accuracy = validation_df[
    "qwen_alone_correct"
].mean()

final_rag_accuracy = validation_df[
    "final_rag_correct"
].mean()

controller_accuracy = validation_df[
    "controller_correct"
].mean()

oracle_accuracy = (
    validation_df["qwen_alone_correct"] |
    validation_df["final_rag_correct"]
).mean()

qwen_routes = int(
    (validation_df["controller_route"] ==
     "qwen_alone").sum()
)

rag_routes = int(
    (validation_df["controller_route"] ==
     "final_optimized_rag").sum()
)

corrections = int(
    validation_df[
        "corrected_final_rag_error"
    ].sum()
)

regressions = int(
    validation_df[
        "regressed_from_final_rag"
    ].sum()
)

print("=" * 70)
print("FINAL CONSERVATIVE BINARY CONTROLLER")
print("=" * 70)

print("\nSELECTED TRAIN-ONLY CONFIGURATION")
print("C:", selected_c)
print(
    "Class weight:",
    (
        "none"
        if selected_class_weight is None
        else selected_class_weight
    ),
)
print(
    "Decision threshold:",
    final_controller_threshold,
)
print(
    "Train OOF controller accuracy:",
    f'{selected_configuration["oof_accuracy"]:.2%}',
)
print(
    "Train OOF regressions:",
    int(selected_configuration["regressions"]),
)

print("\nVALIDATION RESULTS")
print(f"Qwen Alone: {qwen_accuracy:.2%}")
print(f"Final Optimized RAG: {final_rag_accuracy:.2%}")
print(f"Final Controller: {controller_accuracy:.2%}")
print(f"Binary oracle ceiling: {oracle_accuracy:.2%}")
print(
    "Difference over Final RAG:",
    f"{controller_accuracy - final_rag_accuracy:+.2%}",
)

print("\nROUTING")
print("Qwen Alone routes:", qwen_routes)
print("Final Optimized RAG routes:", rag_routes)
print(
    "RAG avoided:",
    f"{qwen_routes / len(validation_df):.2%}",
)
print("Corrections over Final RAG:", corrections)
print("Regressions from Final RAG:", regressions)

print("\nVALIDATION DECISIONS")
display(
    validation_df[
        [
            "question_id",
            "language",
            "qwen_probability",
            "controller_route",
            "qwen_alone_correct",
            "final_rag_correct",
            "controller_correct",
            "corrected_final_rag_error",
            "regressed_from_final_rag",
        ]
    ].style.format(
        {"qwen_probability": "{:.4f}"}
    )
)

# ------------------------------------------------------------
# Save frozen controller
# ------------------------------------------------------------

controller_artifact = {
    "model": final_controller_model,
    "threshold": final_controller_threshold,
    "language_columns": language_features.columns.tolist(),
    "uses_enhancement_similarity": (
        "enhancement_similarity"
        in controller_df.columns
    ),
    "selection_split": "train_only_oof",
    "validation_used_for_training": False,
    "correctness_priority": True,
}

controller_model_path = os.path.join(
    results_directory,
    "final_conservative_controller.joblib",
)

controller_validation_path = os.path.join(
    results_directory,
    "final_conservative_controller_validation.csv",
)

joblib.dump(
    controller_artifact,
    controller_model_path,
)

validation_df.to_csv(
    controller_validation_path,
    index=False,
)

print("\nController trained on final test: False")
print("Validation used for model fitting: False")
print("Correctness priority: True")
print("Model saved:", controller_model_path)
print("Validation saved:", controller_validation_path)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

FINAL CONSERVATIVE BINARY CONTROLLER

SELECTED TRAIN-ONLY CONFIGURATION
C: 2.0
Class weight: none
Decision threshold: 0.5
Train OOF controller accuracy: 76.39%
Train OOF regressions: 0

VALIDATION RESULTS
Qwen Alone: 37.50%
Final Optimized RAG: 79.17%
Final Controller: 79.17%
Binary oracle ceiling: 79.17%
Difference over Final RAG: +0.00%

ROUTING
Qwen Alone routes: 0
Final Optimized RAG routes: 24
RAG avoided: 0.00%
Corrections over Final RAG: 0
Regressions from Final RAG: 0

VALIDATION DECISIONS


,question_id,language,qwen_probability,controller_route,qwen_alone_correct,final_rag_correct,controller_correct,corrected_final_rag_error,regressed_from_final_rag
0,B4A-003-ARABIC,arabic,0.3136,final_optimized_rag,False,False,False,False,False
1,B4A-003-ENGLISH,english,0.2938,final_optimized_rag,False,True,True,False,False
2,B4A-003-FRENCH,french,0.3048,final_optimized_rag,True,True,True,False,False
3,B4A-003-LEBANESE_ARABIZI,lebanese_arabizi,0.2190,final_optimized_rag,False,True,True,False,False
4,B4A-008-ARABIC,arabic,0.3289,final_optimized_rag,True,True,True,False,False
5,B4A-008-ENGLISH,english,0.3308,final_optimized_rag,False,True,True,False,False
6,B4A-008-FRENCH,french,0.3351,final_optimized_rag,True,True,True,False,False
7,B4A-008-LEBANESE_ARABIZI,lebanese_arabizi,0.2359,final_optimized_rag,False,False,False,False,False
8,B4A-015-ARABIC,arabic,0.4153,final_optimized_rag,True,True,True,False,False
9,B4A-015-ENGLISH,english,0.4569,final_optimized_rag,True,True,True,False,False



Controller trained on final test: False
Validation used for model fitting: False
Correctness priority: True
Model saved: /kaggle/working/build4all_final_experiment/results/final_conservative_controller.joblib
Validation saved: /kaggle/working/build4all_final_experiment/results/final_conservative_controller_validation.csv


In [61]:
# ============================================================
# CELL 48 — QLORA SETUP AND DATA-LEAKAGE AUDIT
# ============================================================

%pip install --quiet \
    "peft>=0.17.0" \
    "datasets>=3.0.0" \
    "accelerate>=1.0.0"

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset

# ------------------------------------------------------------
# Output directories
# ------------------------------------------------------------

FINE_TUNING_ROOT = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "fine_tuning",
)

ADAPTER_DIRECTORY = os.path.join(
    FINE_TUNING_ROOT,
    "qwen3_8b_build4all_qlora_adapter",
)

TRAINER_OUTPUT_DIRECTORY = os.path.join(
    FINE_TUNING_ROOT,
    "trainer_output",
)

RESULTS_DIRECTORY = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "results",
)

os.makedirs(
    FINE_TUNING_ROOT,
    exist_ok=True,
)

os.makedirs(
    RESULTS_DIRECTORY,
    exist_ok=True,
)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ------------------------------------------------------------
# Separate the three frozen splits
# ------------------------------------------------------------

fine_tuning_train_df = multilingual_benchmark_df[
    multilingual_benchmark_df["split"] == "train"
].copy().reset_index(drop=True)

fine_tuning_validation_df = multilingual_benchmark_df[
    multilingual_benchmark_df["split"] == "validation"
].copy().reset_index(drop=True)

locked_final_test_df = multilingual_benchmark_df[
    multilingual_benchmark_df["split"] == "final_test"
].copy().reset_index(drop=True)

# ------------------------------------------------------------
# Confirm that business intents do not cross splits
# ------------------------------------------------------------

train_intents = set(
    fine_tuning_train_df["intent_id"]
)

validation_intents = set(
    fine_tuning_validation_df["intent_id"]
)

final_test_intents = set(
    locked_final_test_df["intent_id"]
)

assert len(fine_tuning_train_df) == 72
assert len(fine_tuning_validation_df) == 24
assert len(locked_final_test_df) == 24

assert train_intents.isdisjoint(
    validation_intents
)

assert train_intents.isdisjoint(
    final_test_intents
)

assert validation_intents.isdisjoint(
    final_test_intents
)

assert set(
    fine_tuning_train_df["split"]
) == {"train"}

translation_split_audit = (
    multilingual_benchmark_df
    .groupby("intent_id")["split"]
    .nunique()
)

assert int(
    translation_split_audit.max()
) == 1

# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

print("=" * 70)
print("QLORA STAGE 1 — DATA-LEAKAGE AUDIT")
print("=" * 70)

print(
    "Training questions:",
    len(fine_tuning_train_df),
)

print(
    "Training intents:",
    len(train_intents),
)

print(
    "Validation questions:",
    len(fine_tuning_validation_df),
)

print(
    "Validation intents:",
    len(validation_intents),
)

print(
    "Locked final-test questions:",
    len(locked_final_test_df),
)

print("Translations crossing splits: 0")
print("Validation used for training: False")
print("Final test used for training: False")

Note: you may need to restart the kernel to use updated packages.
QLORA STAGE 1 — DATA-LEAKAGE AUDIT
Training questions: 72
Training intents: 18
Validation questions: 24
Validation intents: 6
Locked final-test questions: 24
Translations crossing splits: 0
Validation used for training: False
Final test used for training: False


In [62]:
# ============================================================
# CELL 49 — CUSTOM DATASET AND GOLD SQL EXECUTION AUDIT
# ============================================================

from pathlib import Path
import pandas as pd
import os

# ------------------------------------------------------------
# 1. Find the uploaded dataset automatically
# ------------------------------------------------------------

dataset_candidates = list(
    Path("/kaggle/input").rglob(
        "build4all_multilingual_text2sql_600.csv"
    )
)

if not dataset_candidates:
    raise FileNotFoundError(
        "The custom CSV was not found under /kaggle/input. "
        "Make sure it was added to this notebook."
    )

if len(dataset_candidates) > 1:
    print("Multiple dataset files found:")

    for candidate in dataset_candidates:
        print("-", candidate)

CUSTOM_DATASET_PATH = dataset_candidates[0]

print(
    "Dataset found:",
    CUSTOM_DATASET_PATH,
)

# ------------------------------------------------------------
# 2. Load dataset
# ------------------------------------------------------------

custom_dataset_df = pd.read_csv(
    CUSTOM_DATASET_PATH
)

required_columns = {
    "question_id",
    "intent_id",
    "intent_name",
    "domain",
    "template_family",
    "complexity",
    "language",
    "split",
    "question",
    "gold_sql",
}

missing_columns = (
    required_columns
    - set(custom_dataset_df.columns)
)

assert not missing_columns, (
    f"Missing columns: {sorted(missing_columns)}"
)

# ------------------------------------------------------------
# 3. Dataset structural audit
# ------------------------------------------------------------

assert len(custom_dataset_df) == 600

assert (
    custom_dataset_df["question_id"].nunique()
    == 600
)

assert (
    custom_dataset_df["intent_id"].nunique()
    == 150
)

assert (
    custom_dataset_df
    .groupby("intent_id")["language"]
    .nunique()
    .eq(4)
    .all()
)

assert (
    custom_dataset_df
    .groupby("intent_id")["split"]
    .nunique()
    .max()
    == 1
)

expected_split_counts = {
    "train": 400,
    "validation": 100,
    "final_test": 100,
}

actual_split_counts = (
    custom_dataset_df["split"]
    .value_counts()
    .to_dict()
)

assert (
    actual_split_counts
    == expected_split_counts
), actual_split_counts

# ------------------------------------------------------------
# 4. Extract one SQL query per intent
# ------------------------------------------------------------

unique_intents_df = (
    custom_dataset_df
    .sort_values(
        ["intent_id", "language"]
    )
    .drop_duplicates(
        subset=["intent_id"],
        keep="first",
    )
    .reset_index(drop=True)
)

assert len(unique_intents_df) == 150

assert (
    unique_intents_df["gold_sql"].nunique()
    == 150
)

# ------------------------------------------------------------
# 5. Execute all 150 unique gold SQL queries
# ------------------------------------------------------------

audit_rows = []

print("\n" + "=" * 70)
print("STARTING GOLD SQL EXECUTION AUDIT")
print("=" * 70)
print("Unique SQL queries to test:", len(unique_intents_df))
print()

for position, (_, row) in enumerate(
    unique_intents_df.iterrows(),
    start=1,
):
    intent_id = row["intent_id"]
    sql_text = row["gold_sql"]

    try:
        safety_result = validate_readonly_sql(
            sql_text
        )

        execution_result = (
            execute_for_evaluation(
                sql_text
            )
        )

        audit_record = {
            "intent_id": intent_id,
            "intent_name": row["intent_name"],
            "domain": row["domain"],
            "split": row["split"],
            "sql_safe": bool(
                safety_result["safe"]
            ),
            "sql_executed": bool(
                execution_result["executed"]
            ),
            "returned_rows": len(
                execution_result["rows"]
            ),
            "execution_error": (
                execution_result["error"]
            ),
        }

    except Exception as error:
        audit_record = {
            "intent_id": intent_id,
            "intent_name": row["intent_name"],
            "domain": row["domain"],
            "split": row["split"],
            "sql_safe": False,
            "sql_executed": False,
            "returned_rows": 0,
            "execution_error": (
                f"{type(error).__name__}: {error}"
            ),
        }

    audit_rows.append(
        audit_record
    )

    status_text = (
        "PASS"
        if (
            audit_record["sql_safe"]
            and audit_record["sql_executed"]
        )
        else "FAIL"
    )

    print(
        f"[{position:03d}/150] "
        f"{intent_id}: {status_text}"
    )

# ------------------------------------------------------------
# 6. Audit summary
# ------------------------------------------------------------

custom_gold_audit_df = pd.DataFrame(
    audit_rows
)

safe_count = int(
    custom_gold_audit_df[
        "sql_safe"
    ].sum()
)

executed_count = int(
    custom_gold_audit_df[
        "sql_executed"
    ].sum()
)

failed_audit_df = custom_gold_audit_df[
    ~(
        custom_gold_audit_df["sql_safe"]
        & custom_gold_audit_df["sql_executed"]
    )
].copy()

audit_output_path = os.path.join(
    RESULTS_DIRECTORY,
    "custom_dataset_gold_sql_audit.csv",
)

custom_gold_audit_df.to_csv(
    audit_output_path,
    index=False,
)

print("\n" + "=" * 70)
print("CUSTOM DATASET AUDIT RESULTS")
print("=" * 70)

print("Dataset rows:", len(custom_dataset_df))
print("Unique intents:", custom_dataset_df["intent_id"].nunique())
print("Unique gold SQL:", unique_intents_df["gold_sql"].nunique())
print("Safe SQL:", f"{safe_count}/150")
print("Successfully executed:", f"{executed_count}/150")
print("Failed queries:", len(failed_audit_df))
print("Translations crossing splits: 0")
print("Audit saved:", audit_output_path)

if len(failed_audit_df) > 0:
    print("\nFAILED SQL QUERIES:")
    display(
        failed_audit_df[
            [
                "intent_id",
                "intent_name",
                "domain",
                "sql_safe",
                "sql_executed",
                "execution_error",
            ]
        ]
    )

assert len(failed_audit_df) == 0, (
    "Some gold SQL queries failed. "
    "Do not begin fine-tuning yet."
)

print(
    "\nAll 150 gold SQL queries are safe "
    "and executable. Dataset approved for QLoRA."
)

Dataset found: /kaggle/input/datasets/lounaakoum/build4all-multilingual-text2sql-600/build4all_multilingual_text2sql_600.csv

STARTING GOLD SQL EXECUTION AUDIT
Unique SQL queries to test: 150

[001/150] B4A-FT-001: PASS
[002/150] B4A-FT-002: PASS
[003/150] B4A-FT-003: PASS
[004/150] B4A-FT-004: PASS
[005/150] B4A-FT-005: PASS
[006/150] B4A-FT-006: PASS
[007/150] B4A-FT-007: PASS
[008/150] B4A-FT-008: PASS
[009/150] B4A-FT-009: PASS
[010/150] B4A-FT-010: PASS
[011/150] B4A-FT-011: PASS
[012/150] B4A-FT-012: PASS
[013/150] B4A-FT-013: PASS
[014/150] B4A-FT-014: PASS
[015/150] B4A-FT-015: PASS
[016/150] B4A-FT-016: PASS
[017/150] B4A-FT-017: PASS
[018/150] B4A-FT-018: PASS
[019/150] B4A-FT-019: PASS
[020/150] B4A-FT-020: PASS
[021/150] B4A-FT-021: PASS
[022/150] B4A-FT-022: PASS
[023/150] B4A-FT-023: PASS
[024/150] B4A-FT-024: PASS
[025/150] B4A-FT-025: PASS
[026/150] B4A-FT-026: PASS
[027/150] B4A-FT-027: PASS
[028/150] B4A-FT-028: PASS
[029/150] B4A-FT-029: PASS
[030/150] B4A-FT-030: PA

In [63]:
# ============================================================
# CELL 50 — PREPARE 400 QLORA TRAINING EXAMPLES
# Prompt is masked; loss is calculated on gold SQL only
# ============================================================

from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

MAX_SEQUENCE_LENGTH = 2048

# ------------------------------------------------------------
# 1. Create frozen custom splits
# ------------------------------------------------------------

custom_train_df = custom_dataset_df[
    custom_dataset_df["split"] == "train"
].copy().reset_index(drop=True)

custom_validation_df = custom_dataset_df[
    custom_dataset_df["split"] == "validation"
].copy().reset_index(drop=True)

custom_final_test_df = custom_dataset_df[
    custom_dataset_df["split"] == "final_test"
].copy().reset_index(drop=True)

assert len(custom_train_df) == 400
assert len(custom_validation_df) == 100
assert len(custom_final_test_df) == 100

assert (
    custom_train_df["intent_id"].nunique()
    == 100
)

assert (
    custom_validation_df["intent_id"].nunique()
    == 25
)

assert (
    custom_final_test_df["intent_id"].nunique()
    == 25
)

train_intents = set(
    custom_train_df["intent_id"]
)

validation_intents = set(
    custom_validation_df["intent_id"]
)

final_test_intents = set(
    custom_final_test_df["intent_id"]
)

assert train_intents.isdisjoint(
    validation_intents
)

assert train_intents.isdisjoint(
    final_test_intents
)

assert validation_intents.isdisjoint(
    final_test_intents
)

# ------------------------------------------------------------
# 2. Fine-tuning system instructions
# ------------------------------------------------------------

FINE_TUNING_SYSTEM_PROMPT = """
You are a multilingual PostgreSQL Text-to-SQL system.

Generate exactly one read-only PostgreSQL query that answers
the user's business question.

Mandatory rules:
1. Use only the provided database schema and columns.
2. Use the build4all_research schema explicitly.
3. Generate only SELECT or WITH queries.
4. Never generate INSERT, UPDATE, DELETE, DROP, ALTER,
   CREATE, TRUNCATE, GRANT, REVOKE, COPY, or CALL.
5. Follow the controlled vocabulary exactly.
6. Do not invent tables, columns, filters, joins, or values.
7. Return SQL only, with no explanation or Markdown.
""".strip()

# ------------------------------------------------------------
# 3. Build the same prompt used for generation
# ------------------------------------------------------------

def build_fine_tuning_user_prompt(question):

    return "\n".join([
        "DATABASE SCHEMA:",
        DATABASE_SCHEMA_TEXT,
        "",
        "CONTROLLED DATABASE VALUES:",
        CONTROLLED_VOCABULARY_TEXT,
        "",
        "BUSINESS QUESTION:",
        str(question).strip(),
        "",
        "POSTGRESQL QUERY:",
    ])

# ------------------------------------------------------------
# 4. Tokenize one supervised example
# ------------------------------------------------------------

def tokenize_supervised_example(example):

    prompt_messages = [
        {
            "role": "system",
            "content": (
                FINE_TUNING_SYSTEM_PROMPT
            ),
        },
        {
            "role": "user",
            "content": (
                build_fine_tuning_user_prompt(
                    example["question"]
                )
            ),
        },
    ]

    full_messages = (
        prompt_messages
        + [
            {
                "role": "assistant",
                "content": str(
                    example["gold_sql"]
                ).strip(),
            }
        ]
    )

    # Prompt without gold SQL
    prompt_text = (
        tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    )

    # Complete supervised conversation
    full_text = (
        tokenizer.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False,
        )
    )

    prompt_token_ids = tokenizer(
        prompt_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )["input_ids"]

    tokenized = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_SEQUENCE_LENGTH,
    )

    labels = tokenized[
        "input_ids"
    ].copy()

    prompt_length = min(
        len(prompt_token_ids),
        len(labels),
    )

    # Ignore system prompt, schema and question
    labels[:prompt_length] = (
        [-100] * prompt_length
    )

    # At least one gold-SQL token must remain
    if all(
        label == -100
        for label in labels
    ):
        raise ValueError(
            "The SQL answer was removed by "
            "sequence truncation."
        )

    tokenized["labels"] = labels

    return tokenized

# ------------------------------------------------------------
# 5. Convert training dataframe to Dataset
# ------------------------------------------------------------

training_source_df = custom_train_df[
    [
        "question_id",
        "intent_id",
        "language",
        "domain",
        "complexity",
        "question",
        "gold_sql",
    ]
].copy()

training_dataset = Dataset.from_pandas(
    training_source_df,
    preserve_index=False,
)

# ------------------------------------------------------------
# 6. Tokenize all 400 training examples
# ------------------------------------------------------------

tokenized_training_dataset = (
    training_dataset.map(
        tokenize_supervised_example,
        remove_columns=(
            training_dataset.column_names
        ),
        desc=(
            "Tokenizing Build4All "
            "training examples"
        ),
    )
)

assert (
    len(tokenized_training_dataset)
    == 400
)

sequence_lengths = [
    len(example["input_ids"])
    for example
    in tokenized_training_dataset
]

answer_token_counts = [
    sum(
        label != -100
        for label in example["labels"]
    )
    for example
    in tokenized_training_dataset
]

assert min(answer_token_counts) > 0

assert (
    max(sequence_lengths)
    <= MAX_SEQUENCE_LENGTH
)

# ------------------------------------------------------------
# 7. Dynamic padding collator
# ------------------------------------------------------------

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)

# ------------------------------------------------------------
# 8. Final training-data audit
# ------------------------------------------------------------

print("=" * 70)
print("QLORA TRAINING DATA READY")
print("=" * 70)

print(
    "Training examples:",
    len(tokenized_training_dataset),
)

print(
    "Unique training intents:",
    custom_train_df[
        "intent_id"
    ].nunique(),
)

print(
    "Validation examples excluded:",
    len(custom_validation_df),
)

print(
    "Final-test examples excluded:",
    len(custom_final_test_df),
)

print(
    "Minimum sequence length:",
    min(sequence_lengths),
)

print(
    "Maximum sequence length:",
    max(sequence_lengths),
)

print(
    "Maximum allowed length:",
    MAX_SEQUENCE_LENGTH,
)

print(
    "Minimum SQL answer tokens:",
    min(answer_token_counts),
)

print(
    "Maximum SQL answer tokens:",
    max(answer_token_counts),
)

print(
    "Loss calculated on prompt:",
    False,
)

print(
    "Loss calculated on gold SQL:",
    True,
)

print(
    "Validation used for gradient updates:",
    False,
)

print(
    "Final test used for gradient updates:",
    False,
)

Tokenizing Build4All training examples:   0%|          | 0/400 [00:00<?, ? examples/s]

QLORA TRAINING DATA READY
Training examples: 400
Unique training intents: 100
Validation examples excluded: 100
Final-test examples excluded: 100
Minimum sequence length: 1206
Maximum sequence length: 1295
Maximum allowed length: 2048
Minimum SQL answer tokens: 21
Maximum SQL answer tokens: 95
Loss calculated on prompt: False
Loss calculated on gold SQL: True
Validation used for gradient updates: False
Final test used for gradient updates: False


In [64]:
# ============================================================
# CELL 51 — QLORA FINE-TUNING OF QWEN3-8B
# Train split only: 400 multilingual examples
# ============================================================

import os
import json
import gc
import torch

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

from transformers import (
    Trainer,
    TrainingArguments,
)

# ------------------------------------------------------------
# 1. Refuse accidental repeated adapter injection
# ------------------------------------------------------------

if hasattr(model, "peft_config"):
    raise RuntimeError(
        "A LoRA adapter is already attached to this model. "
        "Restart the session before repeating training."
    )

# ------------------------------------------------------------
# 2. Clear unused GPU memory
# ------------------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 3. Prepare 4-bit model for QLoRA
# ------------------------------------------------------------

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# ------------------------------------------------------------
# 4. Define LoRA adapter configuration
# ------------------------------------------------------------

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(
    model,
    lora_config,
)

# Show the exact number of trained parameters
model.print_trainable_parameters()

# ------------------------------------------------------------
# 5. Training configuration
# ------------------------------------------------------------

training_arguments = TrainingArguments(
    output_dir=TRAINER_OUTPUT_DIRECTORY,

    # Fixed train-only experiment
    num_train_epochs=3,

    # Memory-safe settings for Tesla T4
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    # QLoRA optimization
    learning_rate=1e-4,
    warmup_ratio=0.10,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",

    # T4 supports FP16
    fp16=True,

    # Reduce activation memory
    gradient_checkpointing=True,

    # Monitoring
    logging_steps=5,
    logging_first_step=True,

    # Save an adapter checkpoint per epoch
    save_strategy="epoch",
    save_total_limit=2,

    # No validation-based checkpoint selection
    report_to="none",
    remove_unused_columns=False,

    # Reproducibility
    seed=SEED,
    data_seed=SEED,
)

# ------------------------------------------------------------
# 6. Build Trainer
# ------------------------------------------------------------

trainer = Trainer(
    model=model,
    args=training_arguments,
    train_dataset=(
        tokenized_training_dataset
    ),
    data_collator=data_collator,
)

# ------------------------------------------------------------
# 7. Training audit before starting
# ------------------------------------------------------------

print("=" * 70)
print("STARTING BUILD4ALL QLORA TRAINING")
print("=" * 70)

print(
    "Base model:",
    CONFIG["model"]["model_id"],
)

print(
    "Training examples:",
    len(tokenized_training_dataset),
)

print(
    "Training intents:",
    custom_train_df[
        "intent_id"
    ].nunique(),
)

print("Languages:", 4)
print("Epochs:", 3)
print("Micro batch size:", 1)
print("Gradient accumulation:", 8)
print("Effective batch size:", 8)
print("Learning rate:", "1e-4")
print("LoRA rank:", 16)
print("LoRA alpha:", 32)
print("Validation used for training:", False)
print("Final test used for training:", False)
print()

# ------------------------------------------------------------
# 8. Start QLoRA training
# ------------------------------------------------------------

training_result = trainer.train()

# ------------------------------------------------------------
# 9. Save final LoRA adapter and tokenizer
# ------------------------------------------------------------

model.save_pretrained(
    ADAPTER_DIRECTORY,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    ADAPTER_DIRECTORY,
)

# ------------------------------------------------------------
# 10. Save training metrics
# ------------------------------------------------------------

training_metrics_path = os.path.join(
    RESULTS_DIRECTORY,
    "qwen3_8b_qlora_training_metrics.json",
)

training_history_path = os.path.join(
    RESULTS_DIRECTORY,
    "qwen3_8b_qlora_training_history.json",
)

training_metrics = {}

for key, value in (
    training_result.metrics.items()
):
    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):
        training_metrics[key] = float(
            value
        )
    else:
        training_metrics[key] = str(
            value
        )

with open(
    training_metrics_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        training_metrics,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(
    training_history_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        trainer.state.log_history,
        file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )

# ------------------------------------------------------------
# 11. Return model to deterministic inference mode
# ------------------------------------------------------------

model.config.use_cache = True
model.eval()

gc.collect()
torch.cuda.empty_cache()

# ------------------------------------------------------------
# 12. Final training summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("QLORA TRAINING COMPLETED")
print("=" * 70)

print(
    "Training loss:",
    training_metrics.get(
        "train_loss"
    ),
)

print(
    "Training runtime seconds:",
    training_metrics.get(
        "train_runtime"
    ),
)

print(
    "Training samples per second:",
    training_metrics.get(
        "train_samples_per_second"
    ),
)

print(
    "Adapter saved:",
    ADAPTER_DIRECTORY,
)

print(
    "Metrics saved:",
    training_metrics_path,
)

print(
    "Training history saved:",
    training_history_path,
)

print("Validation evaluated:", False)
print("Final test opened:", False)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301
STARTING BUILD4ALL QLORA TRAINING
Base model: Qwen/Qwen3-8B
Training examples: 400
Training intents: 100
Languages: 4
Epochs: 3
Micro batch size: 1
Gradient accumulation: 8
Effective batch size: 8
Learning rate: 1e-4
LoRA rank: 16
LoRA alpha: 32
Validation used for training: False
Final test used for training: False



Step,Training Loss
1,1.275356
5,1.379853
10,0.901957
15,0.391288
20,0.139302
25,0.062448
30,0.044960
35,0.029727
40,0.027379
45,0.017598



QLORA TRAINING COMPLETED
Training loss: 0.1005519089261846
Training runtime seconds: 8480.825
Training samples per second: 0.141
Adapter saved: /kaggle/working/build4all_final_experiment/fine_tuning/qwen3_8b_build4all_qlora_adapter
Metrics saved: /kaggle/working/build4all_final_experiment/results/qwen3_8b_qlora_training_metrics.json
Training history saved: /kaggle/working/build4all_final_experiment/results/qwen3_8b_qlora_training_history.json
Validation evaluated: False
Final test opened: False


In [65]:
from pathlib import Path
from zipfile import ZipFile, ZIP_STORED
from IPython.display import display, FileLink

root = Path("/kaggle/working/build4all_final_experiment")
bundle = Path("/kaggle/working/build4all_ui_artifacts.zip")

folders = [
    "qwen3_8b_build4all_qlora_adapter",
    "qwen3_8b_build4all_strict_qlora_adapter",
    "controller",
    "indexes",
    "documents",
]

with ZipFile(bundle, "w", compression=ZIP_STORED) as z:
    for name in folders:
        folder = root / name
        if not folder.exists():
            print("Not ready yet:", name)
            continue
        count = 0
        for file in folder.rglob("*"):
            if file.is_file():
                z.write(file, arcname=file.relative_to(root))
                count += 1
        print(f"Added {name}: {count} files")

print(f"\nZIP size: {bundle.stat().st_size / 1024**2:.1f} MB")
display(FileLink(str(bundle), result_html_prefix="Download UI artifacts: "))

Not ready yet: qwen3_8b_build4all_qlora_adapter
Not ready yet: qwen3_8b_build4all_strict_qlora_adapter
Added controller: 0 files
Added indexes: 2 files
Added documents: 6 files

ZIP size: 0.6 MB


/kaggle/working/build4all_ui_artifacts.zip

In [68]:
from pathlib import Path
from base64 import b64encode
from IPython.display import HTML, display

zip_path = Path("/kaggle/working/build4all_ui_artifacts.zip")

if not zip_path.is_file():
    print("ZIP not found. Run the packaging cell first.")
elif zip_path.stat().st_size > 10 * 1024 * 1024:
    print("File is too large for this button. Download it from Kaggle's Output panel.")
else:
    encoded = b64encode(zip_path.read_bytes()).decode("ascii")
    display(HTML(
        f'<a download="{zip_path.name}" '
        f'href="data:application/zip;base64,{encoded}" '
        'style="display:inline-block;padding:14px 22px;'
        'background:#087e88;color:white;font-weight:bold;'
        'border-radius:10px;text-decoration:none">'
        '⬇ Download Build4All ZIP</a>'
    ))

In [90]:
from pathlib import Path
import json, os, shutil, subprocess

source = Path("/kaggle/working/build4all_final_adapters.zip")
assert source.is_file(), "Adapters ZIP not found"
print(f"Found adapters ZIP: {source.stat().st_size / 1024**2:.1f} MB")

# Check whether Kaggle API credentials are available in this session.
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi()
    api.authenticate()
    username = api.config_values.get("username")
    assert username, "Kaggle username not available"
except Exception as exc:
    print("API authentication is not available in this notebook.")
    print("Error type:", type(exc).__name__)
    print("Do not paste your API key into chat.")
else:
    folder = Path("/kaggle/working/private_adapter_backup_upload")
    folder.mkdir(exist_ok=True)
    shutil.copy2(source, folder / source.name)

    slug = "build4all-private-adapters-backup-20260925"
    metadata = {
        "title": "Build4All Private Adapters Backup",
        "id": f"{username}/{slug}",
        "licenses": [{"name": "copyright-authors"}],
        "description": "Private backup of the author's trained Build4All QLoRA adapters."
    }
    (folder / "dataset-metadata.json").write_text(
        json.dumps(metadata, indent=2), encoding="utf-8"
    )

    print("Uploading PRIVATE dataset. Wait for the upload to finish...")
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(folder)],
        text=True, capture_output=True
    )
    print(result.stdout[-3000:])
    if result.returncode:
        print("UPLOAD FAILED:", result.stderr[-1500:])
    else:
        print("UPLOAD FINISHED:", f"https://www.kaggle.com/datasets/{username}/{slug}")

Found adapters ZIP: 313.5 MB
Uploading PRIVATE dataset. Wait for the upload to finish...
Dataset creation error: The requested title "Build4All Private Adapters Backup" is already in use by a dataset. Please choose another title.

UPLOAD FINISHED: https://www.kaggle.com/datasets/lounaakoum/build4all-private-adapters-backup-20260925


In [89]:
from pathlib import Path
import json
import subprocess

folder = Path("/kaggle/working/private_adapter_backup_upload")
zip_file = folder / "build4all_final_adapters.zip"
metadata_file = folder / "dataset-metadata.json"

assert zip_file.is_file(), "Backup ZIP missing from upload folder"
metadata = json.loads(metadata_file.read_text(encoding="utf-8"))

metadata["licenses"] = [{"name": "other"}]
metadata["description"] = (
    "Private backup of the author's Build4All research adapters. "
    "All rights reserved; not licensed for public reuse."
)
metadata_file.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Creating PRIVATE dataset:", metadata["id"])
result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(folder)],
    capture_output=True, text=True
)

print(result.stdout[-4000:])
print(result.stderr[-2000:])

if result.returncode == 0:
    print("\nDATASET CREATED SUCCESSFULLY:")
    print("https://www.kaggle.com/datasets/" + metadata["id"])
else:
    print("\nDataset NOT created yet. Keep the Kaggle session open.")

Creating PRIVATE dataset: lounaakoum/build4all-private-adapters-backup-20260925
Starting upload for file build4all_final_adapters.zip
Upload successful: build4all_final_adapters.zip (313MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/lounaakoum/build4all-private-adapters-backup-20260925

101MB/s]
 20%|█▉        | 61.5M/313M [00:00<00:02, 94.8MB/s]
 23%|██▎       | 73.1M/313M [00:00<00:02, 99.5MB/s]
 27%|██▋       | 83.2M/313M [00:00<00:02, 96.4MB/s]
 30%|███       | 95.2M/313M [00:00<00:02, 104MB/s] 
 34%|███▎      | 105M/313M [00:01<00:02, 100MB/s] 
 37%|███▋      | 115M/313M [00:01<00:02, 101MB/s]
 40%|███▉      | 125M/313M [00:01<00:01, 101MB/s]
 43%|████▎     | 135M/313M [00:01<00:02, 93.0MB/s]
 46%|████▌     | 144M/313M [00:01<00:01, 90.8MB/s]
 49%|████▉     | 153M/313M [00:01<00:01, 90.9MB/s]
 52%|█████▏    | 162M/313M [00:01<00:01, 89.0MB/s]
 54%|█████▍    | 170M/313M [00:01<00:02, 74.0MB/s]
 58%|█████▊    | 181M/313M [00:01<00

In [69]:
# ============================================================
# NEXT CELL — FINE-TUNED QWEN SMOKE TEST
# One untouched validation question only
# ============================================================

import time
import torch


def generate_sql_fine_tuned_qwen(question):

    messages = [
        {
            "role": "system",
            "content": FINE_TUNING_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_fine_tuning_user_prompt(question),
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    )

    input_device = (
        model.get_input_embeddings()
        .weight.device
    )

    model_inputs = {
        key: value.to(input_device)
        for key, value in model_inputs.items()
    }

    model.eval()
    model.config.use_cache = True

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    latency_seconds = time.perf_counter() - start_time

    new_tokens = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:,
    ]

    raw_response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    ).strip()

    generated_sql = extract_sql(raw_response)

    return {
        "sql": generated_sql,
        "raw_response": raw_response,
        "latency_seconds": round(latency_seconds, 4),
    }


# Take one question from validation only
smoke_row = fine_tuning_validation_df.iloc[0]

smoke_output = generate_sql_fine_tuned_qwen(
    smoke_row["question"]
)

smoke_safety = validate_readonly_sql(
    smoke_output["sql"]
)

print("=" * 70)
print("FINE-TUNED QWEN — VALIDATION SMOKE TEST")
print("=" * 70)

print("Question ID:", smoke_row["question_id"])
print("Language:", smoke_row["language"])
print("Question:")
print(smoke_row["question"])

print("\nGenerated SQL:")
print(smoke_output["sql"])

print("\nGold SQL:")
print(smoke_row["gold_sql"])

print("\nSQL safe:", smoke_safety["safe"])
print("Generation latency:", smoke_output["latency_seconds"], "seconds")
print("RAG used: False")
print("Final test opened: False")

FINE-TUNED QWEN — VALIDATION SMOKE TEST
Question ID: B4A-003-ENGLISH
Language: english
Question:
What is the gross merchandise value of non-cancelled orders?

Generated SQL:
SELECT ROUND(SUM(total_amount), 2) AS gross_merchandise_value FROM build4all_research.orders WHERE status <> 'cancelled';

Gold SQL:
SELECT ROUND(SUM(subtotal), 2) AS gross_merchandise_value
        FROM build4all_research.orders
        WHERE status <> 'cancelled';

SQL safe: True
Generation latency: 7.2121 seconds
RAG used: False
Final test opened: False


In [70]:
# ============================================================
# FINE-TUNED QWEN — 100-QUESTION VALIDATION
# Execution accuracy; final test remains locked
# ============================================================

import os
import json
import pandas as pd

validation_checkpoint_path = os.path.join(
    RESULTS_DIRECTORY,
    "fine_tuned_qwen_custom_validation.jsonl",
)

validation_csv_path = os.path.join(
    RESULTS_DIRECTORY,
    "fine_tuned_qwen_custom_validation.csv",
)

# Resume safely if Kaggle stops midway
completed_records = []
completed_ids = set()

if os.path.exists(validation_checkpoint_path):
    with open(
        validation_checkpoint_path,
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                completed_records.append(record)
                completed_ids.add(record["question_id"])

            except json.JSONDecodeError:
                print("Ignored one incomplete checkpoint line.")


print("=" * 70)
print("FINE-TUNED QWEN — VALIDATION EXECUTION EVALUATION")
print("=" * 70)
print("Validation questions:", len(fine_tuning_validation_df))
print("Previously completed:", len(completed_ids))
print("Training questions evaluated: 0")
print("Final-test questions evaluated: 0")


for position, (_, row) in enumerate(
    fine_tuning_validation_df.iterrows(),
    start=1,
):
    question_id = row["question_id"]

    if question_id in completed_ids:
        continue

    print(
        f"[{position:03d}/{len(fine_tuning_validation_df)}] "
        f"{question_id}"
    )

    try:
        generation = generate_sql_fine_tuned_qwen(
            row["question"]
        )

        predicted_execution = execute_safe_sql(
            generation["sql"]
        )

        gold_execution = execute_safe_sql(
            row["gold_sql"]
        )

        execution_correct = bool(
            predicted_execution["executed"]
            and gold_execution["executed"]
            and predicted_execution["normalized_result"]
            == gold_execution["normalized_result"]
        )

        record = {
            "question_id": question_id,
            "intent_id": row["intent_id"],
            "intent_name": row["intent_name"],
            "domain": row["domain"],
            "language": row["language"],
            "split": row["split"],
            "question": row["question"],
            "generated_sql": generation["sql"],
            "gold_sql": row["gold_sql"],
            "sql_safe": bool(
                predicted_execution["safe"]
            ),
            "sql_executed": bool(
                predicted_execution["executed"]
            ),
            "execution_correct": execution_correct,
            "execution_error": predicted_execution["error"],
            "latency_seconds": generation[
                "latency_seconds"
            ],
        }

    except Exception as error:
        record = {
            "question_id": question_id,
            "intent_id": row["intent_id"],
            "intent_name": row["intent_name"],
            "domain": row["domain"],
            "language": row["language"],
            "split": row["split"],
            "question": row["question"],
            "generated_sql": "",
            "gold_sql": row["gold_sql"],
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": (
                f"{type(error).__name__}: {error}"
            ),
            "latency_seconds": None,
        }

    with open(
        validation_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

    completed_records.append(record)
    completed_ids.add(question_id)


# Build final validation table
fine_tuned_validation_results_df = pd.DataFrame(
    completed_records
)

fine_tuned_validation_results_df = (
    fine_tuned_validation_results_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last",
    )
)

expected_validation_ids = set(
    fine_tuning_validation_df["question_id"]
)

fine_tuned_validation_results_df = (
    fine_tuned_validation_results_df[
        fine_tuned_validation_results_df[
            "question_id"
        ].isin(expected_validation_ids)
    ]
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(fine_tuned_validation_results_df) == 100
assert set(
    fine_tuned_validation_results_df["split"]
) == {"validation"}

fine_tuned_validation_results_df.to_csv(
    validation_csv_path,
    index=False,
)

overall_accuracy = float(
    fine_tuned_validation_results_df[
        "execution_correct"
    ].mean()
)

safe_rate = float(
    fine_tuned_validation_results_df[
        "sql_safe"
    ].mean()
)

execution_rate = float(
    fine_tuned_validation_results_df[
        "sql_executed"
    ].mean()
)

average_latency = float(
    fine_tuned_validation_results_df[
        "latency_seconds"
    ].dropna().mean()
)

language_results = (
    fine_tuned_validation_results_df
    .groupby("language")
    .agg(
        questions=("question_id", "count"),
        execution_accuracy=(
            "execution_correct",
            "mean",
        ),
        safe_rate=("sql_safe", "mean"),
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("FINE-TUNED QWEN — VALIDATION RESULTS")
print("=" * 70)

print(
    "Overall execution accuracy:",
    f"{overall_accuracy:.2%}",
)

print("Safe SQL rate:", f"{safe_rate:.2%}")
print("Execution success rate:", f"{execution_rate:.2%}")

print(
    "Average generation latency:",
    f"{average_latency:.2f} seconds",
)

print("\nRESULTS BY LANGUAGE")

display(
    language_results.style.format({
        "execution_accuracy": "{:.2%}",
        "safe_rate": "{:.2%}",
    })
)

print("\nFinal test opened: False")
print("Results saved:", validation_csv_path)

FINE-TUNED QWEN — VALIDATION EXECUTION EVALUATION
Validation questions: 24
Previously completed: 0
Training questions evaluated: 0
Final-test questions evaluated: 0
[001/24] B4A-003-ENGLISH
[002/24] B4A-003-ARABIC
[003/24] B4A-003-FRENCH
[004/24] B4A-003-LEBANESE_ARABIZI
[005/24] B4A-008-ENGLISH
[006/24] B4A-008-ARABIC
[007/24] B4A-008-FRENCH
[008/24] B4A-008-LEBANESE_ARABIZI
[009/24] B4A-015-ENGLISH
[010/24] B4A-015-ARABIC
[011/24] B4A-015-FRENCH
[012/24] B4A-015-LEBANESE_ARABIZI
[013/24] B4A-020-ENGLISH
[014/24] B4A-020-ARABIC
[015/24] B4A-020-FRENCH
[016/24] B4A-020-LEBANESE_ARABIZI
[017/24] B4A-027-ENGLISH
[018/24] B4A-027-ARABIC
[019/24] B4A-027-FRENCH
[020/24] B4A-027-LEBANESE_ARABIZI
[021/24] B4A-029-ENGLISH
[022/24] B4A-029-ARABIC
[023/24] B4A-029-FRENCH
[024/24] B4A-029-LEBANESE_ARABIZI


AssertionError: 

In [71]:
# Find all available dataset DataFrames

print("=" * 70)
print("AVAILABLE DATAFRAMES")
print("=" * 70)

for variable_name, variable_value in list(globals().items()):

    if isinstance(variable_value, pd.DataFrame):

        columns = set(variable_value.columns)

        if {
            "question",
            "gold_sql",
            "split",
        }.issubset(columns):

            print("\nVariable:", variable_name)
            print("Total rows:", len(variable_value))

            print(
                "Split counts:",
                variable_value["split"]
                .value_counts()
                .to_dict()
            )

            print(
                "First question ID:",
                variable_value["question_id"].iloc[0]
                if "question_id" in columns
                else "No question_id column"
            )

AVAILABLE DATAFRAMES

Variable: multilingual_benchmark_df
Total rows: 120
Split counts: {'train': 72, 'validation': 24, 'final_test': 24}
First question ID: B4A-001-ENGLISH

Variable: arabic_rows
Total rows: 30
Split counts: {'train': 18, 'validation': 6, 'final_test': 6}
First question ID: B4A-001-ARABIC

Variable: arabizi_rows
Total rows: 30
Split counts: {'train': 18, 'validation': 6, 'final_test': 6}
First question ID: B4A-001-LEBANESE_ARABIZI

Variable: intent_group
Total rows: 4
Split counts: {'final_test': 4}
First question ID: B4A-030-ENGLISH

Variable: qwen_alone_benchmark_df
Total rows: 96
Split counts: {'train': 72, 'validation': 24}
First question ID: B4A-001-ENGLISH

Variable: benchmark_lookup
Total rows: 120
Split counts: {'train': 72, 'validation': 24, 'final_test': 24}
First question ID: No question_id column

Variable: rag_benchmark_df
Total rows: 96
Split counts: {'train': 72, 'validation': 24}
First question ID: B4A-001-ENGLISH

Variable: rag_diagnosis_df
Total rows:

In [72]:
# ============================================================
# CORRECTED EVALUATION — CUSTOM 100-QUESTION VALIDATION
# ============================================================

import os
import json
import pandas as pd

custom_checkpoint_path = os.path.join(
    RESULTS_DIRECTORY,
    "fine_tuned_qwen_custom600_validation_v2.jsonl",
)

custom_csv_path = os.path.join(
    RESULTS_DIRECTORY,
    "fine_tuned_qwen_custom600_validation_v2.csv",
)

completed_records = []
completed_ids = set()

# Resume if Kaggle stops
if os.path.exists(custom_checkpoint_path):
    with open(
        custom_checkpoint_path,
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                completed_records.append(record)
                completed_ids.add(record["question_id"])

            except json.JSONDecodeError:
                print("Ignored one incomplete checkpoint line.")


assert len(custom_validation_df) == 100
assert set(custom_validation_df["split"]) == {"validation"}

print("=" * 70)
print("FINE-TUNED QWEN — CUSTOM VALIDATION")
print("=" * 70)
print("Validation questions:", len(custom_validation_df))
print("Validation intents:", custom_validation_df["intent_id"].nunique())
print("Previously completed:", len(completed_ids))
print("Training rows evaluated: 0")
print("Final-test rows evaluated: 0")


for position, (_, row) in enumerate(
    custom_validation_df.iterrows(),
    start=1,
):
    question_id = row["question_id"]

    if question_id in completed_ids:
        continue

    print(
        f"[{position:03d}/{len(custom_validation_df)}] "
        f"{question_id}"
    )

    try:
        generation = generate_sql_fine_tuned_qwen(
            row["question"]
        )

        # Execute verified gold SQL to obtain expected rows
        gold_execution = execute_for_evaluation(
            row["gold_sql"]
        )

        if not gold_execution["executed"]:
            raise RuntimeError(
                "Gold SQL execution failed: "
                + str(gold_execution["error"])
            )

        # Official row-result comparison
        evaluation = evaluate_generated_sql(
            generation["sql"],
            gold_execution["rows"],
        )

        record = {
            "question_id": question_id,
            "intent_id": row["intent_id"],
            "intent_name": row["intent_name"],
            "domain": row["domain"],
            "complexity": row["complexity"],
            "language": row["language"],
            "split": row["split"],
            "question": row["question"],
            "generated_sql": generation["sql"],
            "gold_sql": row["gold_sql"],
            "sql_safe": bool(evaluation["sql_safe"]),
            "sql_executed": bool(
                evaluation["sql_executed"]
            ),
            "execution_correct": bool(
                evaluation["execution_correct"]
            ),
            "execution_error": evaluation[
                "execution_error"
            ],
            "predicted_row_count": int(
                evaluation["predicted_row_count"]
            ),
            "gold_row_count": int(
                evaluation["gold_row_count"]
            ),
            "latency_seconds": generation[
                "latency_seconds"
            ],
        }

    except Exception as error:
        record = {
            "question_id": question_id,
            "intent_id": row["intent_id"],
            "intent_name": row["intent_name"],
            "domain": row["domain"],
            "complexity": row["complexity"],
            "language": row["language"],
            "split": row["split"],
            "question": row["question"],
            "generated_sql": "",
            "gold_sql": row["gold_sql"],
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": (
                f"{type(error).__name__}: {error}"
            ),
            "predicted_row_count": 0,
            "gold_row_count": 0,
            "latency_seconds": None,
        }

    with open(
        custom_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )

    completed_records.append(record)
    completed_ids.add(question_id)


custom_validation_results_df = pd.DataFrame(
    completed_records
)

custom_validation_results_df = (
    custom_validation_results_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last",
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

expected_ids = set(custom_validation_df["question_id"])

custom_validation_results_df = (
    custom_validation_results_df[
        custom_validation_results_df[
            "question_id"
        ].isin(expected_ids)
    ]
    .reset_index(drop=True)
)

assert len(custom_validation_results_df) == 100
assert set(
    custom_validation_results_df["question_id"]
) == expected_ids

custom_validation_results_df.to_csv(
    custom_csv_path,
    index=False,
)

overall_accuracy = float(
    custom_validation_results_df[
        "execution_correct"
    ].mean()
)

safe_rate = float(
    custom_validation_results_df["sql_safe"].mean()
)

execution_rate = float(
    custom_validation_results_df[
        "sql_executed"
    ].mean()
)

average_latency = float(
    custom_validation_results_df[
        "latency_seconds"
    ].dropna().mean()
)

language_results = (
    custom_validation_results_df
    .groupby("language")
    .agg(
        questions=("question_id", "count"),
        correct=("execution_correct", "sum"),
        execution_accuracy=(
            "execution_correct",
            "mean",
        ),
        safe_rate=("sql_safe", "mean"),
    )
    .reset_index()
)

complexity_results = (
    custom_validation_results_df
    .groupby("complexity")
    .agg(
        questions=("question_id", "count"),
        correct=("execution_correct", "sum"),
        execution_accuracy=(
            "execution_correct",
            "mean",
        ),
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("FINE-TUNED QWEN — CUSTOM VALIDATION RESULTS")
print("=" * 70)

print("Correct questions:", int(
    custom_validation_results_df[
        "execution_correct"
    ].sum()
), "/ 100")

print(
    "Overall execution accuracy:",
    f"{overall_accuracy:.2%}",
)

print("Safe SQL rate:", f"{safe_rate:.2%}")
print("Execution success rate:", f"{execution_rate:.2%}")

print(
    "Average generation latency:",
    f"{average_latency:.2f} seconds",
)

print("\nRESULTS BY LANGUAGE")

display(
    language_results.style.format({
        "execution_accuracy": "{:.2%}",
        "safe_rate": "{:.2%}",
    })
)

print("\nRESULTS BY COMPLEXITY")

display(
    complexity_results.style.format({
        "execution_accuracy": "{:.2%}",
    })
)

print("\nFinal test opened: False")
print("Results saved:", custom_csv_path)

FINE-TUNED QWEN — CUSTOM VALIDATION
Validation questions: 100
Validation intents: 25
Previously completed: 0
Training rows evaluated: 0
Final-test rows evaluated: 0
[001/100] B4A-FT-006-ENGLISH
[002/100] B4A-FT-006-ARABIC
[003/100] B4A-FT-006-FRENCH
[004/100] B4A-FT-006-LEBANESE_ARABIZI
[005/100] B4A-FT-012-ENGLISH
[006/100] B4A-FT-012-ARABIC
[007/100] B4A-FT-012-FRENCH
[008/100] B4A-FT-012-LEBANESE_ARABIZI
[009/100] B4A-FT-013-ENGLISH
[010/100] B4A-FT-013-ARABIC
[011/100] B4A-FT-013-FRENCH
[012/100] B4A-FT-013-LEBANESE_ARABIZI
[013/100] B4A-FT-014-ENGLISH
[014/100] B4A-FT-014-ARABIC
[015/100] B4A-FT-014-FRENCH
[016/100] B4A-FT-014-LEBANESE_ARABIZI
[017/100] B4A-FT-016-ENGLISH
[018/100] B4A-FT-016-ARABIC
[019/100] B4A-FT-016-FRENCH
[020/100] B4A-FT-016-LEBANESE_ARABIZI
[021/100] B4A-FT-020-ENGLISH
[022/100] B4A-FT-020-ARABIC
[023/100] B4A-FT-020-FRENCH
[024/100] B4A-FT-020-LEBANESE_ARABIZI
[025/100] B4A-FT-021-ENGLISH
[026/100] B4A-FT-021-ARABIC
[027/100] B4A-FT-021-FRENCH
[028/100] B4

,language,questions,correct,execution_accuracy,safe_rate
0,arabic,25,25,100.00%,100.00%
1,english,25,25,100.00%,100.00%
2,french,25,25,100.00%,100.00%
3,lebanese_arabizi,25,25,100.00%,100.00%



RESULTS BY COMPLEXITY


,complexity,questions,correct,execution_accuracy
0,easy,36,36,100.00%
1,medium,64,64,100.00%



Final test opened: False
Results saved: /kaggle/working/build4all_final_experiment/results/fine_tuned_qwen_custom600_validation_v2.csv


In [73]:
# ============================================================
# MEMORIZATION AND TEMPLATE-OVERLAP AUDIT
# Does not open the final test
# ============================================================

import os
import re
import pandas as pd
import sqlglot

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def canonicalize_sql(sql_text):
    try:
        return sqlglot.parse_one(
            str(sql_text),
            read="postgres",
        ).sql(
            dialect="postgres",
            pretty=False,
        ).lower().strip()

    except Exception:
        return " ".join(
            str(sql_text).lower().split()
        )


def sql_skeleton(sql_text):
    canonical_sql = canonicalize_sql(sql_text)

    # Replace string and numeric parameters
    skeleton = re.sub(
        r"'(?:''|[^'])*'",
        "'?'",
        canonical_sql,
    )

    skeleton = re.sub(
        r"\b\d+(?:\.\d+)?\b",
        "?",
        skeleton,
    )

    return skeleton


# ------------------------------------------------------------
# 1. Canonical SQL comparisons
# ------------------------------------------------------------

audit_df = custom_validation_results_df.copy()

audit_df["generated_sql_canonical"] = (
    audit_df["generated_sql"].map(canonicalize_sql)
)

audit_df["gold_sql_canonical"] = (
    audit_df["gold_sql"].map(canonicalize_sql)
)

audit_df["generated_matches_gold_sql"] = (
    audit_df["generated_sql_canonical"]
    == audit_df["gold_sql_canonical"]
)

audit_df["generated_sql_skeleton"] = (
    audit_df["generated_sql"].map(sql_skeleton)
)

audit_df["gold_sql_skeleton"] = (
    audit_df["gold_sql"].map(sql_skeleton)
)

audit_df["generated_matches_gold_skeleton"] = (
    audit_df["generated_sql_skeleton"]
    == audit_df["gold_sql_skeleton"]
)


# ------------------------------------------------------------
# 2. Check overlap with training SQL
# ------------------------------------------------------------

training_gold_sql = set(
    custom_train_df["gold_sql"].map(canonicalize_sql)
)

training_sql_skeletons = set(
    custom_train_df["gold_sql"].map(sql_skeleton)
)

training_questions_normalized = set(
    custom_train_df["question"]
    .astype(str)
    .str.strip()
    .str.lower()
)

audit_df["exact_question_in_training"] = (
    audit_df["question"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(training_questions_normalized)
)

audit_df["gold_sql_exactly_in_training"] = (
    audit_df["gold_sql_canonical"]
    .isin(training_gold_sql)
)

audit_df["generated_sql_exactly_in_training"] = (
    audit_df["generated_sql_canonical"]
    .isin(training_gold_sql)
)

audit_df["gold_skeleton_in_training"] = (
    audit_df["gold_sql_skeleton"]
    .isin(training_sql_skeletons)
)


# ------------------------------------------------------------
# 3. Nearest training-question similarity
# Compare each language with the same language
# ------------------------------------------------------------

similarity_records = []

for language in sorted(
    custom_validation_df["language"].unique()
):
    language_train = custom_train_df[
        custom_train_df["language"] == language
    ].copy()

    language_validation = custom_validation_df[
        custom_validation_df["language"] == language
    ].copy()

    combined_questions = (
        language_train["question"].astype(str).tolist()
        + language_validation["question"].astype(str).tolist()
    )

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        lowercase=True,
    )

    combined_vectors = vectorizer.fit_transform(
        combined_questions
    )

    train_count = len(language_train)

    train_vectors = combined_vectors[:train_count]
    validation_vectors = combined_vectors[train_count:]

    similarities = cosine_similarity(
        validation_vectors,
        train_vectors,
    )

    for validation_position, (
        validation_index,
        validation_row,
    ) in enumerate(language_validation.iterrows()):

        best_position = int(
            similarities[validation_position].argmax()
        )

        best_score = float(
            similarities[
                validation_position,
                best_position,
            ]
        )

        nearest_training_row = (
            language_train.iloc[best_position]
        )

        similarity_records.append({
            "question_id": validation_row["question_id"],
            "nearest_similarity": best_score,
            "nearest_training_question_id": (
                nearest_training_row["question_id"]
            ),
            "nearest_training_question": (
                nearest_training_row["question"]
            ),
        })


similarity_df = pd.DataFrame(similarity_records)

audit_df = audit_df.merge(
    similarity_df,
    on="question_id",
    how="left",
)


# ------------------------------------------------------------
# 4. Summaries
# ------------------------------------------------------------

exact_question_overlap = int(
    audit_df["exact_question_in_training"].sum()
)

gold_sql_overlap = int(
    audit_df["gold_sql_exactly_in_training"].sum()
)

generated_sql_training_overlap = int(
    audit_df["generated_sql_exactly_in_training"].sum()
)

skeleton_overlap = int(
    audit_df["gold_skeleton_in_training"].sum()
)

exact_gold_generations = int(
    audit_df["generated_matches_gold_sql"].sum()
)

same_skeleton_generations = int(
    audit_df["generated_matches_gold_skeleton"].sum()
)

execution_only_matches = int((
    audit_df["execution_correct"]
    & ~audit_df["generated_matches_gold_sql"]
).sum())

average_similarity = float(
    audit_df["nearest_similarity"].mean()
)

maximum_similarity = float(
    audit_df["nearest_similarity"].max()
)

similarity_85 = int(
    (audit_df["nearest_similarity"] >= 0.85).sum()
)

similarity_95 = int(
    (audit_df["nearest_similarity"] >= 0.95).sum()
)


# ------------------------------------------------------------
# 5. Save audit
# ------------------------------------------------------------

audit_path = os.path.join(
    RESULTS_DIRECTORY,
    "fine_tuned_qwen_memorization_audit.csv",
)

audit_df.to_csv(
    audit_path,
    index=False,
)


print("=" * 70)
print("MEMORIZATION AND TEMPLATE-OVERLAP AUDIT")
print("=" * 70)

print("\nEXACT TRAIN/VALIDATION LEAKAGE")
print(
    "Exact validation questions found in training:",
    f"{exact_question_overlap}/100",
)
print(
    "Exact validation gold SQL found in training:",
    f"{gold_sql_overlap}/100",
)

print("\nTEMPLATE OVERLAP")
print(
    "Validation SQL skeletons found in training:",
    f"{skeleton_overlap}/100",
)
print(
    "Generated SQL exactly copied from a training gold SQL:",
    f"{generated_sql_training_overlap}/100",
)

print("\nGENERATED SQL VS VALIDATION GOLD SQL")
print(
    "Exact canonical SQL matches:",
    f"{exact_gold_generations}/100",
)
print(
    "Same SQL skeleton:",
    f"{same_skeleton_generations}/100",
)
print(
    "Correct by execution but not exact SQL:",
    f"{execution_only_matches}/100",
)

print("\nQUESTION SIMILARITY")
print(
    "Average nearest-training similarity:",
    f"{average_similarity:.4f}",
)
print(
    "Maximum nearest-training similarity:",
    f"{maximum_similarity:.4f}",
)
print(
    "Validation questions with similarity >= 0.85:",
    f"{similarity_85}/100",
)
print(
    "Validation questions with similarity >= 0.95:",
    f"{similarity_95}/100",
)

print("\nMOST SIMILAR VALIDATION QUESTIONS")

display(
    audit_df[
        [
            "question_id",
            "language",
            "nearest_similarity",
            "nearest_training_question_id",
            "question",
            "nearest_training_question",
        ]
    ]
    .sort_values(
        "nearest_similarity",
        ascending=False,
    )
    .head(10)
    .style.format({
        "nearest_similarity": "{:.4f}",
    })
)

print("\nFinal test opened: False")
print("Audit saved:", audit_path)

MEMORIZATION AND TEMPLATE-OVERLAP AUDIT

EXACT TRAIN/VALIDATION LEAKAGE
Exact validation questions found in training: 0/100
Exact validation gold SQL found in training: 0/100

TEMPLATE OVERLAP
Validation SQL skeletons found in training: 100/100
Generated SQL exactly copied from a training gold SQL: 0/100

GENERATED SQL VS VALIDATION GOLD SQL
Exact canonical SQL matches: 100/100
Same SQL skeleton: 100/100
Correct by execution but not exact SQL: 0/100

QUESTION SIMILARITY
Average nearest-training similarity: 0.8231
Maximum nearest-training similarity: 0.9798
Validation questions with similarity >= 0.85: 50/100
Validation questions with similarity >= 0.95: 9/100

MOST SIMILAR VALIDATION QUESTIONS


,question_id,language,nearest_similarity,nearest_training_question_id,question,nearest_training_question
94,B4A-FT-141-FRENCH,french,0.9798,B4A-FT-142-FRENCH,Combien de coupons actifs exigent un montant minimum de commande inférieur ou égal à 0 ?,Combien de coupons actifs exigent un montant minimum de commande inférieur ou égal à 50 ?
82,B4A-FT-122-FRENCH,french,0.9659,B4A-FT-121-FRENCH,Combien de produits actifs ont un stock inférieur à 10 unités ?,Combien de produits actifs ont un stock inférieur à 5 unités ?
81,B4A-FT-122-ENGLISH,english,0.9624,B4A-FT-121-ENGLISH,How many active products have stock below 10 units?,How many active products have stock below 5 units?
83,B4A-FT-122-LEBANESE_ARABIZI,lebanese_arabizi,0.9619,B4A-FT-121-LEBANESE_ARABIZI,Adde fi active products stock taba3a a2al men 10 units?,Adde fi active products stock taba3a a2al men 5 units?
98,B4A-FT-145-FRENCH,french,0.9580,B4A-FT-142-FRENCH,Combien de coupons actifs exigent un montant minimum de commande inférieur ou égal à 500 ?,Combien de coupons actifs exigent un montant minimum de commande inférieur ou égal à 50 ?
52,B4A-FT-055-ARABIC,arabic,0.9575,B4A-FT-054-ARABIC,ما إجمالي المدفوعات الناجحة باستخدام طريقة 'bank_transfer'؟,ما عدد المدفوعات الناجحة باستخدام طريقة 'bank_transfer'؟
80,B4A-FT-122-ARABIC,arabic,0.9572,B4A-FT-121-ARABIC,كم منتجا نشطا لديه مخزون أقل من 10 وحدة؟,كم منتجا نشطا لديه مخزون أقل من 5 وحدة؟
93,B4A-FT-141-ENGLISH,english,0.9513,B4A-FT-142-ENGLISH,How many active coupons require a minimum order amount of at most 0?,How many active coupons require a minimum order amount of at most 50?
63,B4A-FT-069-LEBANESE_ARABIZI,lebanese_arabizi,0.9503,B4A-FT-066-LEBANESE_ARABIZI,Adde 3adad el payments el paid bi tari2et 'debit_card'?,Adde 3adad el payments bi tari2et 'debit_card'?
60,B4A-FT-069-ARABIC,arabic,0.9467,B4A-FT-070-ARABIC,ما عدد المدفوعات الناجحة باستخدام طريقة 'debit_card'؟,ما إجمالي المدفوعات الناجحة باستخدام طريقة 'debit_card'؟



Final test opened: False
Audit saved: /kaggle/working/build4all_final_experiment/results/fine_tuned_qwen_memorization_audit.csv


In [74]:
# ============================================================
# STRICT TEMPLATE-FAMILY SPLIT
# No SQL template may cross train/validation/final-test
# ============================================================

import os
import random
import pandas as pd

STRICT_SPLIT_SEED = 42
random.seed(STRICT_SPLIT_SEED)

# One row per unique intent
intent_level_df = (
    custom_dataset_df[
        [
            "intent_id",
            "template_family",
            "domain",
            "complexity",
        ]
    ]
    .drop_duplicates(subset=["intent_id"])
    .reset_index(drop=True)
)

family_sizes = (
    intent_level_df
    .groupby("template_family")["intent_id"]
    .nunique()
    .to_dict()
)

families = sorted(family_sizes.keys())

# ------------------------------------------------------------
# Search for a family-disjoint split close to:
# 100 train intents, 25 validation intents, 25 final intents
# ------------------------------------------------------------

best_assignment = None
best_score = float("inf")

for attempt in range(50000):

    shuffled_families = families.copy()
    random.shuffle(shuffled_families)

    cumulative_sizes = []
    running_total = 0

    for family in shuffled_families:
        running_total += family_sizes[family]
        cumulative_sizes.append(running_total)

    validation_end = min(
        range(1, len(shuffled_families) - 1),
        key=lambda index: abs(
            cumulative_sizes[index - 1] - 25
        ),
    )

    validation_families = set(
        shuffled_families[:validation_end]
    )

    remaining_families = (
        shuffled_families[validation_end:]
    )

    remaining_cumulative = []
    running_total = 0

    for family in remaining_families:
        running_total += family_sizes[family]
        remaining_cumulative.append(running_total)

    final_end = min(
        range(1, len(remaining_families)),
        key=lambda index: abs(
            remaining_cumulative[index - 1] - 25
        ),
    )

    final_families = set(
        remaining_families[:final_end]
    )

    train_families = (
        set(families)
        - validation_families
        - final_families
    )

    validation_intent_count = sum(
        family_sizes[family]
        for family in validation_families
    )

    final_intent_count = sum(
        family_sizes[family]
        for family in final_families
    )

    train_intent_count = sum(
        family_sizes[family]
        for family in train_families
    )

    score = (
        abs(train_intent_count - 100)
        + abs(validation_intent_count - 25)
        + abs(final_intent_count - 25)
    )

    if score < best_score:
        best_score = score
        best_assignment = {
            "train": train_families,
            "validation": validation_families,
            "final_test": final_families,
        }

    if score == 0:
        break


# ------------------------------------------------------------
# Apply strict split to all four translations
# ------------------------------------------------------------

def assign_strict_split(template_family):

    if template_family in best_assignment["train"]:
        return "train"

    if template_family in best_assignment["validation"]:
        return "validation"

    if template_family in best_assignment["final_test"]:
        return "final_test"

    raise ValueError(
        f"Unassigned template family: {template_family}"
    )


strict_dataset_df = custom_dataset_df.copy()

# Keep old split for audit only
strict_dataset_df["original_split"] = (
    strict_dataset_df["split"]
)

strict_dataset_df["split"] = (
    strict_dataset_df["template_family"]
    .map(assign_strict_split)
)

strict_train_df = strict_dataset_df[
    strict_dataset_df["split"] == "train"
].copy().reset_index(drop=True)

strict_validation_df = strict_dataset_df[
    strict_dataset_df["split"] == "validation"
].copy().reset_index(drop=True)

strict_final_test_df = strict_dataset_df[
    strict_dataset_df["split"] == "final_test"
].copy().reset_index(drop=True)


# ------------------------------------------------------------
# Leakage audits
# ------------------------------------------------------------

train_families = set(
    strict_train_df["template_family"]
)

validation_families = set(
    strict_validation_df["template_family"]
)

final_families = set(
    strict_final_test_df["template_family"]
)

assert train_families.isdisjoint(validation_families)
assert train_families.isdisjoint(final_families)
assert validation_families.isdisjoint(final_families)

intent_split_counts = (
    strict_dataset_df
    .groupby("intent_id")["split"]
    .nunique()
)

assert int(intent_split_counts.max()) == 1

translation_counts = (
    strict_dataset_df
    .groupby("intent_id")["language"]
    .nunique()
)

assert int(translation_counts.min()) == 4
assert int(translation_counts.max()) == 4


# ------------------------------------------------------------
# Save revised dataset
# ------------------------------------------------------------

strict_dataset_path = os.path.join(
    RESULTS_DIRECTORY,
    "build4all_custom600_strict_template_split.csv",
)

strict_dataset_df.to_csv(
    strict_dataset_path,
    index=False,
)


# ------------------------------------------------------------
# Display audit
# ------------------------------------------------------------

strict_summary = (
    strict_dataset_df
    .groupby("split")
    .agg(
        questions=("question_id", "count"),
        intents=("intent_id", "nunique"),
        template_families=(
            "template_family",
            "nunique",
        ),
        languages=("language", "nunique"),
    )
    .reset_index()
)

print("=" * 70)
print("STRICT TEMPLATE-FAMILY SPLIT")
print("=" * 70)

display(strict_summary)

print(
    "\nTemplate families crossing splits:",
    0,
)

print(
    "Intent translations crossing splits:",
    0,
)

print(
    "Train/validation template overlap:",
    len(train_families & validation_families),
)

print(
    "Train/final-test template overlap:",
    len(train_families & final_families),
)

print(
    "Validation/final-test template overlap:",
    len(validation_families & final_families),
)

print("\nSTRICT VALIDATION FAMILIES")

for family in sorted(validation_families):
    print("-", family)

print("\nSTRICT FINAL-TEST FAMILIES — LOCKED")

for family in sorted(final_families):
    print("-", family)

print("\nCurrent trained adapter valid for this new split: False")
print("Reason: it already saw some newly held-out templates.")
print("Strict dataset saved:", strict_dataset_path)

STRICT TEMPLATE-FAMILY SPLIT


,split,questions,intents,template_families,languages
0,final_test,100,25,5,4
1,train,400,100,20,4
2,validation,100,25,6,4



Template families crossing splits: 0
Intent translations crossing splits: 0
Train/validation template overlap: 0
Train/final-test template overlap: 0
Validation/final-test template overlap: 0

STRICT VALIDATION FAMILIES
- orders_average_tax_by_status
- payment_method_count
- payment_method_paid_total
- products_average_price_by_status
- products_categories_by_status
- products_inventory_value_by_status

STRICT FINAL-TEST FAMILIES — LOCKED
- events_count_by_type
- events_top_products_by_type
- orders_average_amount_by_status
- payments_orders_by_status
- products_count_by_status

Current trained adapter valid for this new split: False
Reason: it already saw some newly held-out templates.
Strict dataset saved: /kaggle/working/build4all_final_experiment/results/build4all_custom600_strict_template_split.csv


In [75]:
# ============================================================
# ORIGINAL QWEN — STRICT TEMPLATE-HELD-OUT VALIDATION
# LoRA adapter is temporarily disabled
# ============================================================

import os
import json
import time
import torch
import pandas as pd

base_checkpoint_path = os.path.join(
    RESULTS_DIRECTORY,
    "original_qwen_strict_validation.jsonl",
)

base_csv_path = os.path.join(
    RESULTS_DIRECTORY,
    "original_qwen_strict_validation.csv",
)


def generate_sql_original_qwen_strict(question):

    messages = [
        {
            "role": "system",
            "content": FINE_TUNING_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": build_fine_tuning_user_prompt(question),
        },
    ]

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    model_inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    )

    input_device = (
        model.get_input_embeddings()
        .weight.device
    )

    model_inputs = {
        key: value.to(input_device)
        for key, value in model_inputs.items()
    }

    start_time = time.perf_counter()

    # Disable trained LoRA adapter:
    # generation comes from original Qwen weights
    with model.disable_adapter():
        with torch.inference_mode():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=256,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

    latency_seconds = (
        time.perf_counter() - start_time
    )

    new_tokens = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:,
    ]

    raw_response = tokenizer.decode(
        new_tokens[0],
        skip_special_tokens=True,
    ).strip()

    return {
        "sql": extract_sql(raw_response),
        "raw_response": raw_response,
        "latency_seconds": round(
            latency_seconds,
            4,
        ),
    }


completed_records = []
completed_ids = set()

if os.path.exists(base_checkpoint_path):
    with open(
        base_checkpoint_path,
        "r",
        encoding="utf-8",
    ) as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
                completed_records.append(record)
                completed_ids.add(record["question_id"])

            except json.JSONDecodeError:
                print("Ignored one incomplete checkpoint line.")


assert len(strict_validation_df) == 100
assert strict_validation_df["intent_id"].nunique() == 25

assert set(
    strict_validation_df["template_family"]
).isdisjoint(
    set(strict_train_df["template_family"])
)

print("=" * 70)
print("ORIGINAL QWEN — STRICT VALIDATION")
print("=" * 70)
print("Validation questions:", len(strict_validation_df))
print("Validation intents:", strict_validation_df["intent_id"].nunique())
print(
    "Validation template families:",
    strict_validation_df["template_family"].nunique(),
)
print("Train/validation template overlap: 0")
print("Previously completed:", len(completed_ids))
print("LoRA adapter active during generation: False")
print("Final test opened: False")


for position, (_, row) in enumerate(
    strict_validation_df.iterrows(),
    start=1,
):
    question_id = row["question_id"]

    if question_id in completed_ids:
        continue

    print(
        f"[{position:03d}/{len(strict_validation_df)}] "
        f"{question_id}"
    )

    try:
        generation = generate_sql_original_qwen_strict(
            row["question"]
        )

        gold_execution = execute_for_evaluation(
            row["gold_sql"]
        )

        if not gold_execution["executed"]:
            raise RuntimeError(
                "Gold SQL failed: "
                + str(gold_execution["error"])
            )

        evaluation = evaluate_generated_sql(
            generation["sql"],
            gold_execution["rows"],
        )

        record = {
            "question_id": question_id,
            "intent_id": row["intent_id"],
            "intent_name": row["intent_name"],
            "template_family": row["template_family"],
            "domain": row["domain"],
            "complexity": row["complexity"],
            "language": row["language"],
            "split": row["split"],
            "question": row["question"],
            "generated_sql": generation["sql"],
            "gold_sql": row["gold_sql"],
            "sql_safe": bool(evaluation["sql_safe"]),
            "sql_executed": bool(
                evaluation["sql_executed"]
            ),
            "execution_correct": bool(
                evaluation["execution_correct"]
            ),
            "execution_error": evaluation[
                "execution_error"
            ],
            "latency_seconds": generation[
                "latency_seconds"
            ],
        }

    except Exception as error:
        record = {
            "question_id": question_id,
            "intent_id": row["intent_id"],
            "intent_name": row["intent_name"],
            "template_family": row["template_family"],
            "domain": row["domain"],
            "complexity": row["complexity"],
            "language": row["language"],
            "split": row["split"],
            "question": row["question"],
            "generated_sql": "",
            "gold_sql": row["gold_sql"],
            "sql_safe": False,
            "sql_executed": False,
            "execution_correct": False,
            "execution_error": (
                f"{type(error).__name__}: {error}"
            ),
            "latency_seconds": None,
        }

    with open(
        base_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as file:
        file.write(
            json.dumps(record, ensure_ascii=False)
            + "\n"
        )

    completed_records.append(record)
    completed_ids.add(question_id)


original_qwen_strict_results_df = pd.DataFrame(
    completed_records
)

original_qwen_strict_results_df = (
    original_qwen_strict_results_df
    .drop_duplicates(
        subset=["question_id"],
        keep="last",
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

expected_ids = set(strict_validation_df["question_id"])

original_qwen_strict_results_df = (
    original_qwen_strict_results_df[
        original_qwen_strict_results_df[
            "question_id"
        ].isin(expected_ids)
    ]
    .reset_index(drop=True)
)

assert len(original_qwen_strict_results_df) == 100
assert set(
    original_qwen_strict_results_df["question_id"]
) == expected_ids

original_qwen_strict_results_df.to_csv(
    base_csv_path,
    index=False,
)

base_accuracy = float(
    original_qwen_strict_results_df[
        "execution_correct"
    ].mean()
)

base_safe_rate = float(
    original_qwen_strict_results_df[
        "sql_safe"
    ].mean()
)

base_execution_rate = float(
    original_qwen_strict_results_df[
        "sql_executed"
    ].mean()
)

language_summary = (
    original_qwen_strict_results_df
    .groupby("language")
    .agg(
        questions=("question_id", "count"),
        correct=("execution_correct", "sum"),
        execution_accuracy=(
            "execution_correct",
            "mean",
        ),
    )
    .reset_index()
)

family_summary = (
    original_qwen_strict_results_df
    .groupby("template_family")
    .agg(
        questions=("question_id", "count"),
        correct=("execution_correct", "sum"),
        execution_accuracy=(
            "execution_correct",
            "mean",
        ),
    )
    .reset_index()
)


print("\n" + "=" * 70)
print("ORIGINAL QWEN — STRICT VALIDATION RESULTS")
print("=" * 70)

print(
    "Correct questions:",
    int(
        original_qwen_strict_results_df[
            "execution_correct"
        ].sum()
    ),
    "/ 100",
)

print(
    "Execution accuracy:",
    f"{base_accuracy:.2%}",
)

print("Safe SQL rate:", f"{base_safe_rate:.2%}")

print(
    "Execution success rate:",
    f"{base_execution_rate:.2%}",
)

print("\nRESULTS BY LANGUAGE")

display(
    language_summary.style.format({
        "execution_accuracy": "{:.2%}",
    })
)

print("\nRESULTS BY HELD-OUT TEMPLATE FAMILY")

display(
    family_summary.style.format({
        "execution_accuracy": "{:.2%}",
    })
)

print("\nLoRA adapter used: False")
print("Final test opened: False")
print("Results saved:", base_csv_path)

ORIGINAL QWEN — STRICT VALIDATION
Validation questions: 100
Validation intents: 25
Validation template families: 6
Train/validation template overlap: 0
Previously completed: 0
LoRA adapter active during generation: False
Final test opened: False
[001/100] B4A-FT-005-ENGLISH
[002/100] B4A-FT-005-ARABIC
[003/100] B4A-FT-005-FRENCH
[004/100] B4A-FT-005-LEBANESE_ARABIZI
[005/100] B4A-FT-010-ENGLISH
[006/100] B4A-FT-010-ARABIC
[007/100] B4A-FT-010-FRENCH
[008/100] B4A-FT-010-LEBANESE_ARABIZI
[009/100] B4A-FT-015-ENGLISH
[010/100] B4A-FT-015-ARABIC
[011/100] B4A-FT-015-FRENCH
[012/100] B4A-FT-015-LEBANESE_ARABIZI
[013/100] B4A-FT-020-ENGLISH
[014/100] B4A-FT-020-ARABIC
[015/100] B4A-FT-020-FRENCH
[016/100] B4A-FT-020-LEBANESE_ARABIZI
[017/100] B4A-FT-025-ENGLISH
[018/100] B4A-FT-025-ARABIC
[019/100] B4A-FT-025-FRENCH
[020/100] B4A-FT-025-LEBANESE_ARABIZI
[021/100] B4A-FT-030-ENGLISH
[022/100] B4A-FT-030-ARABIC
[023/100] B4A-FT-030-FRENCH
[024/100] B4A-FT-030-LEBANESE_ARABIZI
[025/100] B4A-FT

,language,questions,correct,execution_accuracy
0,arabic,25,14,56.00%
1,english,25,13,52.00%
2,french,25,11,44.00%
3,lebanese_arabizi,25,8,32.00%



RESULTS BY HELD-OUT TEMPLATE FAMILY


,template_family,questions,correct,execution_accuracy
0,orders_average_tax_by_status,24,0,0.00%
1,payment_method_count,20,13,65.00%
2,payment_method_paid_total,20,15,75.00%
3,products_average_price_by_status,12,0,0.00%
4,products_categories_by_status,12,12,100.00%
5,products_inventory_value_by_status,12,6,50.00%



LoRA adapter used: False
Final test opened: False
Results saved: /kaggle/working/build4all_final_experiment/results/original_qwen_strict_validation.csv


In [76]:
# ============================================================
# PREPARE STRICT TEMPLATE-DISJOINT QLORA TRAINING DATA
# No validation or final-test template enters training
# ============================================================

from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

MAX_SEQUENCE_LENGTH = 2048

# ------------------------------------------------------------
# 1. Final leakage checks
# ------------------------------------------------------------

strict_train_families = set(
    strict_train_df["template_family"]
)

strict_validation_families = set(
    strict_validation_df["template_family"]
)

strict_final_families = set(
    strict_final_test_df["template_family"]
)

assert strict_train_families.isdisjoint(
    strict_validation_families
)

assert strict_train_families.isdisjoint(
    strict_final_families
)

assert strict_validation_families.isdisjoint(
    strict_final_families
)

assert len(strict_train_df) == 400
assert strict_train_df["intent_id"].nunique() == 100
assert strict_validation_df["intent_id"].nunique() == 25
assert strict_final_test_df["intent_id"].nunique() == 25


# ------------------------------------------------------------
# 2. Build strict training source
# ------------------------------------------------------------

strict_training_source_df = strict_train_df[
    [
        "question_id",
        "intent_id",
        "template_family",
        "language",
        "question",
        "gold_sql",
    ]
].copy()

strict_training_dataset = Dataset.from_pandas(
    strict_training_source_df,
    preserve_index=False,
)


# ------------------------------------------------------------
# 3. Tokenize using the same supervised function
# Prompt tokens masked; loss on gold SQL only
# ------------------------------------------------------------

strict_tokenized_training_dataset = (
    strict_training_dataset.map(
        tokenize_supervised_example,
        remove_columns=(
            strict_training_dataset.column_names
        ),
        desc=(
            "Tokenizing strict template-disjoint "
            "training examples"
        ),
    )
)

strict_sequence_lengths = [
    len(row["input_ids"])
    for row in strict_tokenized_training_dataset
]

strict_sql_answer_lengths = [
    sum(
        label != -100
        for label in row["labels"]
    )
    for row in strict_tokenized_training_dataset
]

assert len(strict_tokenized_training_dataset) == 400
assert max(strict_sequence_lengths) <= MAX_SEQUENCE_LENGTH
assert min(strict_sql_answer_lengths) > 0


# ------------------------------------------------------------
# 4. Data collator
# ------------------------------------------------------------

strict_data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)


print("=" * 70)
print("STRICT QLORA TRAINING DATA READY")
print("=" * 70)

print(
    "Training examples:",
    len(strict_tokenized_training_dataset),
)

print(
    "Training intents:",
    strict_train_df["intent_id"].nunique(),
)

print(
    "Training template families:",
    strict_train_df[
        "template_family"
    ].nunique(),
)

print(
    "Validation template families excluded:",
    strict_validation_df[
        "template_family"
    ].nunique(),
)

print(
    "Final-test template families excluded:",
    strict_final_test_df[
        "template_family"
    ].nunique(),
)

print(
    "Minimum sequence length:",
    min(strict_sequence_lengths),
)

print(
    "Maximum sequence length:",
    max(strict_sequence_lengths),
)

print(
    "Minimum SQL answer tokens:",
    min(strict_sql_answer_lengths),
)

print(
    "Maximum SQL answer tokens:",
    max(strict_sql_answer_lengths),
)

print("Loss calculated on prompt: False")
print("Loss calculated on gold SQL: True")
print("Template overlap across splits: 0")
print("Validation used for gradient updates: False")
print("Final test used for gradient updates: False")

Tokenizing strict template-disjoint training examples:   0%|          | 0/400 [00:00<?, ? examples/s]

STRICT QLORA TRAINING DATA READY
Training examples: 400
Training intents: 100
Training template families: 20
Validation template families excluded: 6
Final-test template families excluded: 5
Minimum sequence length: 1206
Maximum sequence length: 1295
Minimum SQL answer tokens: 21
Maximum SQL answer tokens: 95
Loss calculated on prompt: False
Loss calculated on gold SQL: True
Template overlap across splits: 0
Validation used for gradient updates: False
Final test used for gradient updates: False


In [77]:
# ============================================================
# STRICT TEMPLATE-DISJOINT QLORA TRAINING
# Reload clean base Qwen; train on strict train split only
# ============================================================

import os
import gc
import json
import random
import numpy as np
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# 1. New output paths — preserve preliminary adapter
# ------------------------------------------------------------

STRICT_ADAPTER_DIRECTORY = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "fine_tuning",
    "qwen3_8b_build4all_strict_qlora_adapter",
)

STRICT_TRAINER_DIRECTORY = os.path.join(
    "/kaggle/working/build4all_final_experiment",
    "fine_tuning",
    "strict_trainer_output",
)

STRICT_METRICS_PATH = os.path.join(
    RESULTS_DIRECTORY,
    "qwen3_8b_strict_qlora_training_metrics.json",
)

STRICT_HISTORY_PATH = os.path.join(
    RESULTS_DIRECTORY,
    "qwen3_8b_strict_qlora_training_history.json",
)

os.makedirs(
    STRICT_ADAPTER_DIRECTORY,
    exist_ok=True,
)

os.makedirs(
    STRICT_TRAINER_DIRECTORY,
    exist_ok=True,
)


# ------------------------------------------------------------
# 2. Remove old trained model from memory
# Do not merge its adapter
# ------------------------------------------------------------

for variable_name in [
    "trainer",
    "training_result",
    "model",
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print("Old adapter removed from memory: True")


# ------------------------------------------------------------
# 3. Reload untouched Qwen3-8B base model
# ------------------------------------------------------------

strict_quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Reloading clean Qwen3-8B base model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=strict_quantization_config,
    device_map="auto",
    dtype=torch.float16,
    low_cpu_mem_usage=True,
    max_memory={
        0: "14GiB",
        1: "14GiB",
        "cpu": "28GiB",
    },
)

assert not hasattr(model, "peft_config")

print("Clean base model loaded: True")
print("Previous LoRA adapter attached: False")


# ------------------------------------------------------------
# 4. Prepare model and attach a fresh LoRA adapter
# ------------------------------------------------------------

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

strict_lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(
    model,
    strict_lora_config,
)

print("\nFRESH STRICT ADAPTER")
model.print_trainable_parameters()


# ------------------------------------------------------------
# 5. Fixed training configuration
# ------------------------------------------------------------

strict_training_arguments = TrainingArguments(
    output_dir=STRICT_TRAINER_DIRECTORY,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_ratio=0.10,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    seed=42,
    data_seed=42,
)

strict_trainer = Trainer(
    model=model,
    args=strict_training_arguments,
    train_dataset=(
        strict_tokenized_training_dataset
    ),
    data_collator=strict_data_collator,
)


# ------------------------------------------------------------
# 6. Final pre-training audit
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STARTING STRICT TEMPLATE-DISJOINT QLORA")
print("=" * 70)

print("Training examples: 400")
print("Training intents: 100")
print("Training template families: 20")
print("Epochs: 3")
print("Micro batch size: 1")
print("Gradient accumulation: 8")
print("Effective batch size: 8")
print("Learning rate: 1e-4")
print("Validation template families used: 0")
print("Final-test template families used: 0")
print("Template overlap across splits: 0")


# ------------------------------------------------------------
# 7. Train
# ------------------------------------------------------------

strict_training_result = strict_trainer.train()


# ------------------------------------------------------------
# 8. Save new adapter and training evidence
# ------------------------------------------------------------

model.save_pretrained(
    STRICT_ADAPTER_DIRECTORY,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    STRICT_ADAPTER_DIRECTORY,
)

strict_training_metrics = {
    key: (
        float(value)
        if isinstance(
            value,
            (int, float, np.number),
        )
        else value
    )
    for key, value
    in strict_training_result.metrics.items()
}

with open(
    STRICT_METRICS_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        strict_training_metrics,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(
    STRICT_HISTORY_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        strict_trainer.state.log_history,
        file,
        indent=2,
        ensure_ascii=False,
    )

model.config.use_cache = True
model.eval()


print("\n" + "=" * 70)
print("STRICT QLORA TRAINING COMPLETED")
print("=" * 70)

print(
    "Training loss:",
    strict_training_metrics.get("train_loss"),
)

print(
    "Training runtime seconds:",
    strict_training_metrics.get("train_runtime"),
)

print(
    "Strict adapter saved:",
    STRICT_ADAPTER_DIRECTORY,
)

print("Metrics saved:", STRICT_METRICS_PATH)
print("Training history saved:", STRICT_HISTORY_PATH)
print("Validation evaluated during training: False")
print("Final test opened: False")

Old adapter removed from memory: True
Reloading clean Qwen3-8B base model...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Clean base model loaded: True
Previous LoRA adapter attached: False


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.



FRESH STRICT ADAPTER
trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301

STARTING STRICT TEMPLATE-DISJOINT QLORA
Training examples: 400
Training intents: 100
Training template families: 20
Epochs: 3
Micro batch size: 1
Gradient accumulation: 8
Effective batch size: 8
Learning rate: 1e-4
Validation template families used: 0
Final-test template families used: 0
Template overlap across splits: 0


Step,Training Loss
5,1.329856
10,0.999899
15,0.319631
20,0.187743
25,0.081228
30,0.028040
35,0.024935
40,0.003123
45,0.005986
50,0.003744



STRICT QLORA TRAINING COMPLETED
Training loss: 0.09967108287800026
Training runtime seconds: 8282.5667
Strict adapter saved: /kaggle/working/build4all_final_experiment/fine_tuning/qwen3_8b_build4all_strict_qlora_adapter
Metrics saved: /kaggle/working/build4all_final_experiment/results/qwen3_8b_strict_qlora_training_metrics.json
Training history saved: /kaggle/working/build4all_final_experiment/results/qwen3_8b_strict_qlora_training_history.json
Validation evaluated during training: False
Final test opened: False


In [78]:
# ============================================================
# STRICT FINE-TUNED QWEN — STRICT VALIDATION EVALUATION
# Compare fairly against Original Qwen = 44%
# FINAL TEST REMAINS LOCKED
# ============================================================

import os
import json
import time
import pandas as pd
import torch
from pathlib import Path


# ------------------------------------------------------------
# 1. Safety checks
# ------------------------------------------------------------

assert "strict_validation_df" in globals(), (
    "strict_validation_df is not available."
)

assert len(strict_validation_df) == 100, (
    f"Expected 100 strict validation questions, "
    f"found {len(strict_validation_df)}."
)

assert set(strict_validation_df["split"].unique()) == {"validation"}, (
    "This cell must evaluate validation rows only."
)

assert "model" in globals()
assert "tokenizer" in globals()
assert hasattr(model, "peft_config"), (
    "The active model does not appear to contain a LoRA adapter."
)

model.eval()


# ------------------------------------------------------------
# 2. Result and checkpoint paths
# ------------------------------------------------------------

strict_results_directory = Path(
    "/kaggle/working/build4all_final_experiment/results"
)

strict_results_directory.mkdir(
    parents=True,
    exist_ok=True,
)

strict_checkpoint_path = (
    strict_results_directory
    / "strict_finetuned_validation_checkpoint.jsonl"
)

strict_csv_path = (
    strict_results_directory
    / "strict_finetuned_qwen_validation.csv"
)

strict_summary_path = (
    strict_results_directory
    / "strict_finetuned_qwen_validation_summary.json"
)


# ------------------------------------------------------------
# 3. Load checkpoint if evaluation was interrupted
# ------------------------------------------------------------

completed_rows = []

if strict_checkpoint_path.exists():
    with open(
        strict_checkpoint_path,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:

        for line in checkpoint_file:
            line = line.strip()

            if line:
                completed_rows.append(
                    json.loads(line)
                )

completed_question_ids = {
    row["question_id"]
    for row in completed_rows
}

remaining_validation_df = (
    strict_validation_df[
        ~strict_validation_df["question_id"].isin(
            completed_question_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 4. Evaluation information
# ------------------------------------------------------------

print("=" * 70)
print("STRICT FINE-TUNED QWEN — STRICT VALIDATION")
print("=" * 70)

print(
    "Validation questions:",
    len(strict_validation_df),
)

print(
    "Validation intents:",
    strict_validation_df["intent_id"].nunique(),
)

print(
    "Validation template families:",
    strict_validation_df["template_family"].nunique(),
)

print(
    "Previously completed:",
    len(completed_rows),
)

print(
    "Remaining:",
    len(remaining_validation_df),
)

print(
    "LoRA adapter active:",
    hasattr(model, "peft_config"),
)

print("RAG used: False")
print("Final test opened: False")


# ------------------------------------------------------------
# 5. Generate and evaluate each validation question
# ------------------------------------------------------------

for _, benchmark_row in remaining_validation_df.iterrows():

    question_id = str(
        benchmark_row["question_id"]
    )

    question = str(
        benchmark_row["question"]
    )

    gold_sql = str(
        benchmark_row["gold_sql"]
    )

    print(
        f"[{len(completed_rows) + 1:03d}/100] "
        f"{question_id}"
    )

    # Execute gold SQL to obtain the verified expected result.
    gold_execution = execute_for_evaluation(
        gold_sql
    )

    if not gold_execution["executed"]:
        raise RuntimeError(
            f"Gold SQL failed for {question_id}: "
            f"{gold_execution['error']}"
        )

    # generate_sql() uses the currently active model.
    # The current model is the newly trained strict LoRA model.
    generation_start = time.perf_counter()

    generated_output = generate_sql(
        question=question,
        use_rag=False,
    )

    total_generation_latency = (
        time.perf_counter()
        - generation_start
    )

    generated_sql = generated_output["sql"]

    evaluation = evaluate_generated_sql(
        generated_sql=generated_sql,
        gold_rows=gold_execution["rows"],
    )

    result_row = {
        "question_id": question_id,
        "intent_id": str(
            benchmark_row["intent_id"]
        ),
        "language": str(
            benchmark_row["language"]
        ),
        "split": "validation",
        "template_family": str(
            benchmark_row["template_family"]
        ),
        "question": question,
        "gold_sql": gold_sql,
        "generated_sql": generated_sql,
        "execution_correct": bool(
            evaluation["execution_correct"]
        ),
        "sql_safe": bool(
            evaluation["sql_safe"]
        ),
        "sql_executed": bool(
            evaluation["sql_executed"]
        ),
        "execution_error": (
            evaluation["execution_error"]
        ),
        "generation_latency_seconds": float(
            generated_output.get(
                "latency_seconds",
                total_generation_latency,
            )
        ),
        "rag_used": False,
        "adapter": "strict_qlora",
    }

    with open(
        strict_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as checkpoint_file:

        checkpoint_file.write(
            json.dumps(
                result_row,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

    completed_rows.append(
        result_row
    )

    running_accuracy = (
        pd.DataFrame(completed_rows)[
            "execution_correct"
        ].mean()
    )

    print(
        f"          correct="
        f"{evaluation['execution_correct']} | "
        f"running_accuracy={running_accuracy:.2%}"
    )


# ------------------------------------------------------------
# 6. Build final validation dataframe
# ------------------------------------------------------------

strict_finetuned_validation_results_df = (
    pd.DataFrame(completed_rows)
    .drop_duplicates(
        subset=["question_id"],
        keep="last",
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(
    strict_finetuned_validation_results_df
) == 100, (
    "Strict validation evaluation is incomplete: "
    f"{len(strict_finetuned_validation_results_df)}/100"
)

assert set(
    strict_finetuned_validation_results_df["split"]
) == {"validation"}

strict_finetuned_validation_results_df.to_csv(
    strict_csv_path,
    index=False,
)


# ------------------------------------------------------------
# 7. Overall metrics
# ------------------------------------------------------------

correct_questions = int(
    strict_finetuned_validation_results_df[
        "execution_correct"
    ].sum()
)

strict_finetuned_accuracy = float(
    strict_finetuned_validation_results_df[
        "execution_correct"
    ].mean()
)

strict_safe_rate = float(
    strict_finetuned_validation_results_df[
        "sql_safe"
    ].mean()
)

strict_execution_success_rate = float(
    strict_finetuned_validation_results_df[
        "sql_executed"
    ].mean()
)

strict_average_latency = float(
    strict_finetuned_validation_results_df[
        "generation_latency_seconds"
    ].mean()
)

original_qwen_strict_accuracy = 0.44

accuracy_change = (
    strict_finetuned_accuracy
    - original_qwen_strict_accuracy
)


# ------------------------------------------------------------
# 8. Results by language
# ------------------------------------------------------------

strict_results_by_language_df = (
    strict_finetuned_validation_results_df
    .groupby("language")
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        correct=(
            "execution_correct",
            "sum",
        ),
        execution_accuracy=(
            "execution_correct",
            "mean",
        ),
    )
    .reset_index()
)

strict_results_by_language_df[
    "execution_accuracy"
] = (
    strict_results_by_language_df[
        "execution_accuracy"
    ] * 100
).round(2)


# ------------------------------------------------------------
# 9. Results by held-out template family
# ------------------------------------------------------------

strict_results_by_family_df = (
    strict_finetuned_validation_results_df
    .groupby("template_family")
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        correct=(
            "execution_correct",
            "sum",
        ),
        execution_accuracy=(
            "execution_correct",
            "mean",
        ),
    )
    .reset_index()
)

strict_results_by_family_df[
    "execution_accuracy"
] = (
    strict_results_by_family_df[
        "execution_accuracy"
    ] * 100
).round(2)


# ------------------------------------------------------------
# 10. Save summary
# ------------------------------------------------------------

strict_summary = {
    "questions": 100,
    "correct_questions": correct_questions,
    "strict_finetuned_accuracy": strict_finetuned_accuracy,
    "original_qwen_strict_accuracy": (
        original_qwen_strict_accuracy
    ),
    "absolute_accuracy_change": accuracy_change,
    "safe_sql_rate": strict_safe_rate,
    "execution_success_rate": (
        strict_execution_success_rate
    ),
    "average_generation_latency_seconds": (
        strict_average_latency
    ),
    "rag_used": False,
    "final_test_opened": False,
}

with open(
    strict_summary_path,
    "w",
    encoding="utf-8",
) as summary_file:

    json.dump(
        strict_summary,
        summary_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 11. Print final comparison
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STRICT FINE-TUNED QWEN — VALIDATION RESULTS")
print("=" * 70)

print(
    f"Correct questions: "
    f"{correct_questions} / 100"
)

print(
    f"Strict fine-tuned accuracy: "
    f"{strict_finetuned_accuracy:.2%}"
)

print(
    f"Original Qwen strict accuracy: "
    f"{original_qwen_strict_accuracy:.2%}"
)

print(
    f"Absolute accuracy change: "
    f"{accuracy_change:+.2%}"
)

print(
    f"Safe SQL rate: "
    f"{strict_safe_rate:.2%}"
)

print(
    f"Execution success rate: "
    f"{strict_execution_success_rate:.2%}"
)

print(
    f"Average generation latency: "
    f"{strict_average_latency:.2f} seconds"
)

print("\nRESULTS BY LANGUAGE")
display(strict_results_by_language_df)

print("\nRESULTS BY HELD-OUT TEMPLATE FAMILY")
display(strict_results_by_family_df)

print("\nRAG used: False")
print("Final test opened: False")
print("Results saved:", strict_csv_path)
print("Summary saved:", strict_summary_path)

STRICT FINE-TUNED QWEN — STRICT VALIDATION
Validation questions: 100
Validation intents: 25
Validation template families: 6
Previously completed: 0
Remaining: 100
LoRA adapter active: True
RAG used: False
Final test opened: False
[001/100] B4A-FT-005-ENGLISH
          correct=True | running_accuracy=100.00%
[002/100] B4A-FT-005-ARABIC
          correct=True | running_accuracy=100.00%
[003/100] B4A-FT-005-FRENCH
          correct=True | running_accuracy=100.00%
[004/100] B4A-FT-005-LEBANESE_ARABIZI
          correct=True | running_accuracy=100.00%
[005/100] B4A-FT-010-ENGLISH
          correct=True | running_accuracy=100.00%
[006/100] B4A-FT-010-ARABIC
          correct=True | running_accuracy=100.00%
[007/100] B4A-FT-010-FRENCH
          correct=True | running_accuracy=100.00%
[008/100] B4A-FT-010-LEBANESE_ARABIZI
          correct=True | running_accuracy=100.00%
[009/100] B4A-FT-015-ENGLISH
          correct=True | running_accuracy=100.00%
[010/100] B4A-FT-015-ARABIC
          correct

,language,questions,correct,execution_accuracy
0,arabic,25,23,92.0
1,english,25,23,92.0
2,french,25,23,92.0
3,lebanese_arabizi,25,23,92.0



RESULTS BY HELD-OUT TEMPLATE FAMILY


,template_family,questions,correct,execution_accuracy
0,orders_average_tax_by_status,24,24,100.00
1,payment_method_count,20,20,100.00
2,payment_method_paid_total,20,20,100.00
3,products_average_price_by_status,12,12,100.00
4,products_categories_by_status,12,12,100.00
5,products_inventory_value_by_status,12,4,33.33



RAG used: False
Final test opened: False
Results saved: /kaggle/working/build4all_final_experiment/results/strict_finetuned_qwen_validation.csv
Summary saved: /kaggle/working/build4all_final_experiment/results/strict_finetuned_qwen_validation_summary.json


In [79]:
# ============================================================
# STRICT FINE-TUNED QWEN — ERROR AUDIT
# No new generations and no final-test access
# ============================================================

strict_errors_df = (
    strict_finetuned_validation_results_df[
        ~strict_finetuned_validation_results_df[
            "execution_correct"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

print("=" * 70)
print("STRICT FINE-TUNED QWEN — ERROR AUDIT")
print("=" * 70)

print("Total validation questions: 100")
print("Correct questions: 91")
print("Failed questions:", len(strict_errors_df))
print(
    "Execution failures:",
    int(
        ~strict_errors_df["sql_executed"]
    .astype(bool).sum()
    ) if False else int(
        (~strict_errors_df["sql_executed"]).sum()
    )
)
print("New generations performed: 0")
print("RAG used: False")
print("Final test opened: False")


print("\nERRORS BY TEMPLATE FAMILY")

error_family_summary_df = (
    strict_errors_df
    .groupby("template_family")
    .agg(
        errors=("question_id", "count"),
        languages=("language", lambda values: ", ".join(
            sorted(set(values))
        )),
    )
    .reset_index()
)

display(error_family_summary_df)


print("\nERRORS BY LANGUAGE")

error_language_summary_df = (
    strict_errors_df
    .groupby("language")
    .agg(
        errors=("question_id", "count"),
    )
    .reset_index()
)

display(error_language_summary_df)


print("\n" + "=" * 70)
print("INDIVIDUAL FAILED QUESTIONS")
print("=" * 70)

for index, row in strict_errors_df.iterrows():

    print("\n" + "-" * 70)
    print(
        f"ERROR {index + 1}/{len(strict_errors_df)}"
    )
    print("-" * 70)

    print("Question ID:", row["question_id"])
    print("Language:", row["language"])
    print(
        "Template family:",
        row["template_family"],
    )

    print("\nQUESTION:")
    print(row["question"])

    print("\nGENERATED SQL:")
    print(row["generated_sql"])

    print("\nGOLD SQL:")
    print(row["gold_sql"])

    print("\nSQL SAFE:", row["sql_safe"])
    print("SQL EXECUTED:", row["sql_executed"])

    if pd.notna(row["execution_error"]):
        print(
            "EXECUTION ERROR:",
            row["execution_error"],
        )


# Save the audit
strict_error_audit_path = (
    strict_results_directory
    / "strict_finetuned_qwen_error_audit.csv"
)

strict_errors_df.to_csv(
    strict_error_audit_path,
    index=False,
)

print("\n" + "=" * 70)
print("ERROR AUDIT COMPLETED")
print("=" * 70)
print("Errors saved:", strict_error_audit_path)
print("Final test opened: False")

STRICT FINE-TUNED QWEN — ERROR AUDIT
Total validation questions: 100
Correct questions: 91
Failed questions: 8
Execution failures: 0
New generations performed: 0
RAG used: False
Final test opened: False

ERRORS BY TEMPLATE FAMILY


,template_family,errors,languages
0,products_inventory_value_by_status,8,"arabic, english, french, lebanese_arabizi"



ERRORS BY LANGUAGE


,language,errors
0,arabic,2
1,english,2
2,french,2
3,lebanese_arabizi,2



INDIVIDUAL FAILED QUESTIONS

----------------------------------------------------------------------
ERROR 1/8
----------------------------------------------------------------------
Question ID: B4A-FT-079-ARABIC
Language: arabic
Template family: products_inventory_value_by_status

QUESTION:
ما قيمة المخزون باستخدام سعر التخفيض عند توفره عندما تكون حالة المنتج 'active'؟

GENERATED SQL:
SELECT SUM(stock * sale_price) AS weighted_stock_value FROM build4all_research.products WHERE status = 'active';

GOLD SQL:
SELECT ROUND(SUM(stock * COALESCE(sale_price, price)), 2) AS inventory_value FROM build4all_research.products WHERE status = 'active';

SQL SAFE: True
SQL EXECUTED: True

----------------------------------------------------------------------
ERROR 2/8
----------------------------------------------------------------------
Question ID: B4A-FT-079-ENGLISH
Language: english
Template family: products_inventory_value_by_status

QUESTION:
What is the inventory value using sale price when a

In [80]:
# ============================================================
# RAG EVIDENCE AUDIT FOR THE 9 STRICT VALIDATION ERRORS
# Retrieval only — no SQL generation and no final-test access
# ============================================================

import pandas as pd

rag_error_evidence_rows = []

print("=" * 70)
print("RAG EVIDENCE AUDIT — STRICT VALIDATION ERRORS")
print("=" * 70)

print("Failed questions:", len(strict_errors_df))
print("SQL generations performed: 0")
print("Final test opened: False")


for error_index, error_row in strict_errors_df.iterrows():

    question_id = str(
        error_row["question_id"]
    )

    question = str(
        error_row["question"]
    )

    retrieved_chunks = retrieve_chunks(
        question,
        top_k=6,
    )

    print("\n" + "=" * 70)
    print(
        f"ERROR {error_index + 1}/"
        f"{len(strict_errors_df)} — {question_id}"
    )
    print("=" * 70)

    print("Question:")
    print(question)

    question_has_inventory_rule = False
    question_has_category_rule = False

    for chunk_index, chunk in enumerate(
        retrieved_chunks,
        start=1,
    ):

        chunk_text = str(
            chunk.get("text", "")
        )

        normalized_chunk_text = (
            chunk_text.lower()
        )

        # Detect evidence related to the missing rules.
        has_sale_price_fallback = (
            (
                "sale_price" in normalized_chunk_text
                and "price" in normalized_chunk_text
            )
            or "coalesce" in normalized_chunk_text
            or (
                "sale price" in normalized_chunk_text
                and "available" in normalized_chunk_text
            )
        )

        has_inventory_calculation = (
            "inventory value" in normalized_chunk_text
            or (
                "stock" in normalized_chunk_text
                and "price" in normalized_chunk_text
            )
        )

        has_category_rule = (
            (
                "distinct" in normalized_chunk_text
                and "category" in normalized_chunk_text
            )
            or (
                "category_id" in normalized_chunk_text
                and "products" in normalized_chunk_text
            )
        )

        relevant_inventory_evidence = (
            has_sale_price_fallback
            and has_inventory_calculation
        )

        if relevant_inventory_evidence:
            question_has_inventory_rule = True

        if has_category_rule:
            question_has_category_rule = True

        print("\n" + "-" * 60)
        print(
            f"Rank: {chunk_index}"
        )
        print(
            "Document:",
            chunk.get(
                "document_name",
                "unknown",
            ),
        )
        print(
            "Page:",
            chunk.get(
                "page_number",
                "unknown",
            ),
        )
        print(
            "Retrieval score:",
            round(
                float(
                    chunk.get(
                        "retrieval_score",
                        0.0,
                    )
                ),
                4,
            ),
        )
        print(
            "Contains inventory fallback evidence:",
            relevant_inventory_evidence,
        )
        print(
            "Contains category evidence:",
            has_category_rule,
        )
        print("\nChunk text:")
        print(chunk_text[:1500])

        rag_error_evidence_rows.append({
            "question_id": question_id,
            "language": error_row["language"],
            "template_family": (
                error_row["template_family"]
            ),
            "rank": chunk_index,
            "document_name": chunk.get(
                "document_name"
            ),
            "page_number": chunk.get(
                "page_number"
            ),
            "retrieval_score": chunk.get(
                "retrieval_score"
            ),
            "inventory_rule_detected": (
                relevant_inventory_evidence
            ),
            "category_rule_detected": (
                has_category_rule
            ),
            "chunk_text": chunk_text,
        })

    print("\nEvidence summary for question:")
    print(
        "Inventory rule retrieved:",
        question_has_inventory_rule,
    )
    print(
        "Category rule retrieved:",
        question_has_category_rule,
    )


# ------------------------------------------------------------
# Build and save retrieval audit
# ------------------------------------------------------------

rag_error_evidence_df = pd.DataFrame(
    rag_error_evidence_rows
)

rag_question_evidence_summary_df = (
    rag_error_evidence_df
    .groupby(
        [
            "question_id",
            "language",
            "template_family",
        ]
    )
    .agg(
        inventory_rule_retrieved=(
            "inventory_rule_detected",
            "max",
        ),
        category_rule_retrieved=(
            "category_rule_detected",
            "max",
        ),
        maximum_retrieval_score=(
            "retrieval_score",
            "max",
        ),
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("RAG EVIDENCE SUMMARY")
print("=" * 70)

display(
    rag_question_evidence_summary_df
)

rag_evidence_path = (
    strict_results_directory
    / "strict_errors_rag_evidence_audit.csv"
)

rag_error_evidence_df.to_csv(
    rag_evidence_path,
    index=False,
)

print("\nAudit saved:", rag_evidence_path)
print("SQL generations performed: 0")
print("Final test opened: False")

RAG EVIDENCE AUDIT — STRICT VALIDATION ERRORS
Failed questions: 8
SQL generations performed: 0
Final test opened: False

ERROR 1/8 — B4A-FT-079-ARABIC
Question:
ما قيمة المخزون باستخدام سعر التخفيض عند توفره عندما تكون حالة المنتج 'active'؟

------------------------------------------------------------
Rank: 1
Document: Build4All_Product_Inventory_and_Coupon_Policy_v2.0
Page: 1
Retrieval score: 0.8059
Contains inventory fallback evidence: True
Contains category evidence: False

Chunk text:
Active inventory value Active inventory value is the sum, across active products, of products.stock multiplied by the effective current unit price. The effective current unit price is products.sale_price when sale_price is present; otherwise it is products.price. The canonical calculation is stock multiplied by COALESCE(sale_price, price).

------------------------------------------------------------
Rank: 2
Document: Build4All_Product_Inventory_and_Coupon_Policy_v2.0
Page: 2
Retrieval score: 0.7957
C

,question_id,language,template_family,inventory_rule_retrieved,category_rule_retrieved,maximum_retrieval_score
0,B4A-FT-079-ARABIC,arabic,products_inventory_value_by_status,True,True,0.805874
1,B4A-FT-079-ENGLISH,english,products_inventory_value_by_status,True,True,0.893475
2,B4A-FT-079-FRENCH,french,products_inventory_value_by_status,True,True,0.850160
3,B4A-FT-079-LEBANESE_ARABIZI,lebanese_arabizi,products_inventory_value_by_status,True,True,0.836917
4,B4A-FT-084-ARABIC,arabic,products_inventory_value_by_status,True,True,0.781472
5,B4A-FT-084-ENGLISH,english,products_inventory_value_by_status,True,True,0.839693
6,B4A-FT-084-FRENCH,french,products_inventory_value_by_status,True,True,0.813843
7,B4A-FT-084-LEBANESE_ARABIZI,lebanese_arabizi,products_inventory_value_by_status,True,True,0.818447



Audit saved: /kaggle/working/build4all_final_experiment/results/strict_errors_rag_evidence_audit.csv
SQL generations performed: 0
Final test opened: False


In [81]:
# ============================================================
# STRICT FINE-TUNED QWEN + RAG — FULL STRICT VALIDATION
# Compare against strict fine-tuned Qwen without RAG = 91%
# FINAL TEST REMAINS LOCKED
# ============================================================

import json
import time
import pandas as pd
from pathlib import Path


# ------------------------------------------------------------
# 1. Safety and readiness checks
# ------------------------------------------------------------

assert "strict_validation_df" in globals()
assert len(strict_validation_df) == 100

assert set(
    strict_validation_df["split"].unique()
) == {"validation"}

assert "strict_finetuned_validation_results_df" in globals()
assert len(
    strict_finetuned_validation_results_df
) == 100

assert "retrieve_chunks" in globals()
assert "build_rag_context" in globals()
assert "generate_sql" in globals()
assert hasattr(model, "peft_config"), (
    "The strict LoRA adapter is not active."
)

model.eval()


# ------------------------------------------------------------
# 2. Paths
# ------------------------------------------------------------

strict_rag_results_directory = Path(
    "/kaggle/working/build4all_final_experiment/results"
)

strict_rag_checkpoint_path = (
    strict_rag_results_directory
    / "strict_finetuned_rag_validation_checkpoint.jsonl"
)

strict_rag_csv_path = (
    strict_rag_results_directory
    / "strict_finetuned_qwen_rag_validation.csv"
)

strict_rag_summary_path = (
    strict_rag_results_directory
    / "strict_finetuned_qwen_rag_validation_summary.json"
)


# ------------------------------------------------------------
# 3. Direct fine-tuned reference
# ------------------------------------------------------------

direct_reference = (
    strict_finetuned_validation_results_df[
        [
            "question_id",
            "execution_correct",
            "generated_sql",
        ]
    ]
    .rename(
        columns={
            "execution_correct": "direct_correct",
            "generated_sql": "direct_sql",
        }
    )
    .copy()
)

assert len(direct_reference) == 100

direct_correct_map = dict(
    zip(
        direct_reference["question_id"],
        direct_reference["direct_correct"],
    )
)

direct_sql_map = dict(
    zip(
        direct_reference["question_id"],
        direct_reference["direct_sql"],
    )
)


# ------------------------------------------------------------
# 4. Resume existing checkpoint if interrupted
# ------------------------------------------------------------

completed_rag_rows = []

if strict_rag_checkpoint_path.exists():

    with open(
        strict_rag_checkpoint_path,
        "r",
        encoding="utf-8",
    ) as checkpoint_file:

        for line in checkpoint_file:

            line = line.strip()

            if line:
                completed_rag_rows.append(
                    json.loads(line)
                )

completed_rag_question_ids = {
    row["question_id"]
    for row in completed_rag_rows
}

remaining_rag_validation_df = (
    strict_validation_df[
        ~strict_validation_df[
            "question_id"
        ].isin(
            completed_rag_question_ids
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 5. Initial information
# ------------------------------------------------------------

print("=" * 70)
print("STRICT FINE-TUNED QWEN + RAG — VALIDATION")
print("=" * 70)

print(
    "Validation questions:",
    len(strict_validation_df),
)

print(
    "Previously completed:",
    len(completed_rag_rows),
)

print(
    "Remaining:",
    len(remaining_rag_validation_df),
)

print("Strict LoRA adapter active: True")
print("RAG enabled: True")
print("Top-k:", CONFIG["rag"]["top_k"])
print(
    "Maximum context characters:",
    CONFIG["rag"]["max_context_characters"],
)
print("Final test opened: False")


# ------------------------------------------------------------
# 6. Run strict validation with RAG
# ------------------------------------------------------------

for _, benchmark_row in (
    remaining_rag_validation_df.iterrows()
):

    question_id = str(
        benchmark_row["question_id"]
    )

    question = str(
        benchmark_row["question"]
    )

    gold_sql = str(
        benchmark_row["gold_sql"]
    )

    print(
        f"[{len(completed_rag_rows) + 1:03d}/100] "
        f"{question_id}"
    )

    # Obtain the expected database result.
    gold_execution = execute_for_evaluation(
        gold_sql
    )

    if not gold_execution["executed"]:
        raise RuntimeError(
            f"Gold SQL failed for {question_id}: "
            f"{gold_execution['error']}"
        )

    # The currently active model is the strict LoRA model.
    # The only experimental difference is RAG context.
    rag_output = generate_sql(
        question=question,
        use_rag=True,
    )

    rag_sql = rag_output["sql"]

    rag_evaluation = evaluate_generated_sql(
        generated_sql=rag_sql,
        gold_rows=gold_execution["rows"],
    )

    direct_correct = bool(
        direct_correct_map[question_id]
    )

    rag_correct = bool(
        rag_evaluation["execution_correct"]
    )

    retrieved_documents = [
        str(
            chunk.get(
                "document_name",
                "unknown",
            )
        )
        for chunk in rag_output.get(
            "retrieved_chunks",
            [],
        )
    ]

    retrieved_scores = [
        float(
            chunk.get(
                "retrieval_score",
                0.0,
            )
        )
        for chunk in rag_output.get(
            "retrieved_chunks",
            [],
        )
    ]

    result_row = {
        "question_id": question_id,
        "intent_id": str(
            benchmark_row["intent_id"]
        ),
        "language": str(
            benchmark_row["language"]
        ),
        "split": "validation",
        "template_family": str(
            benchmark_row["template_family"]
        ),
        "question": question,
        "gold_sql": gold_sql,
        "direct_sql": direct_sql_map[
            question_id
        ],
        "rag_sql": rag_sql,
        "direct_correct": direct_correct,
        "rag_correct": rag_correct,
        "corrected_direct_error": (
            rag_correct and not direct_correct
        ),
        "regressed_from_direct": (
            direct_correct and not rag_correct
        ),
        "sql_safe": bool(
            rag_evaluation["sql_safe"]
        ),
        "sql_executed": bool(
            rag_evaluation["sql_executed"]
        ),
        "execution_error": (
            rag_evaluation["execution_error"]
        ),
        "generation_latency_seconds": float(
            rag_output.get(
                "latency_seconds",
                0.0,
            )
        ),
        "rag_context_characters": int(
            rag_output.get(
                "rag_context_characters",
                0,
            )
        ),
        "retrieved_documents": (
            retrieved_documents
        ),
        "retrieved_scores": (
            retrieved_scores
        ),
        "rag_used": True,
        "adapter": "strict_qlora",
    }

    with open(
        strict_rag_checkpoint_path,
        "a",
        encoding="utf-8",
    ) as checkpoint_file:

        checkpoint_file.write(
            json.dumps(
                result_row,
                ensure_ascii=False,
                default=str,
            )
            + "\n"
        )

    completed_rag_rows.append(
        result_row
    )

    current_results_df = pd.DataFrame(
        completed_rag_rows
    )

    running_accuracy = float(
        current_results_df[
            "rag_correct"
        ].mean()
    )

    running_corrections = int(
        current_results_df[
            "corrected_direct_error"
        ].sum()
    )

    running_regressions = int(
        current_results_df[
            "regressed_from_direct"
        ].sum()
    )

    print(
        f"          direct={direct_correct} | "
        f"rag={rag_correct} | "
        f"running_accuracy={running_accuracy:.2%} | "
        f"corrections={running_corrections} | "
        f"regressions={running_regressions}"
    )


# ------------------------------------------------------------
# 7. Finalize dataframe
# ------------------------------------------------------------

strict_finetuned_rag_results_df = (
    pd.DataFrame(
        completed_rag_rows
    )
    .drop_duplicates(
        subset=["question_id"],
        keep="last",
    )
    .sort_values("question_id")
    .reset_index(drop=True)
)

assert len(
    strict_finetuned_rag_results_df
) == 100, (
    "Strict RAG validation is incomplete: "
    f"{len(strict_finetuned_rag_results_df)}/100"
)

assert set(
    strict_finetuned_rag_results_df["split"]
) == {"validation"}

strict_finetuned_rag_results_df.to_csv(
    strict_rag_csv_path,
    index=False,
)


# ------------------------------------------------------------
# 8. Global comparison metrics
# ------------------------------------------------------------

direct_accuracy = float(
    strict_finetuned_rag_results_df[
        "direct_correct"
    ].mean()
)

rag_accuracy = float(
    strict_finetuned_rag_results_df[
        "rag_correct"
    ].mean()
)

rag_safe_rate = float(
    strict_finetuned_rag_results_df[
        "sql_safe"
    ].mean()
)

rag_execution_rate = float(
    strict_finetuned_rag_results_df[
        "sql_executed"
    ].mean()
)

average_rag_latency = float(
    strict_finetuned_rag_results_df[
        "generation_latency_seconds"
    ].mean()
)

corrections = int(
    strict_finetuned_rag_results_df[
        "corrected_direct_error"
    ].sum()
)

regressions = int(
    strict_finetuned_rag_results_df[
        "regressed_from_direct"
    ].sum()
)

net_change = corrections - regressions


# ------------------------------------------------------------
# 9. Results by language
# ------------------------------------------------------------

strict_rag_by_language_df = (
    strict_finetuned_rag_results_df
    .groupby("language")
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        direct_correct=(
            "direct_correct",
            "sum",
        ),
        rag_correct=(
            "rag_correct",
            "sum",
        ),
        direct_accuracy=(
            "direct_correct",
            "mean",
        ),
        rag_accuracy=(
            "rag_correct",
            "mean",
        ),
        corrections=(
            "corrected_direct_error",
            "sum",
        ),
        regressions=(
            "regressed_from_direct",
            "sum",
        ),
    )
    .reset_index()
)

for percentage_column in [
    "direct_accuracy",
    "rag_accuracy",
]:
    strict_rag_by_language_df[
        percentage_column
    ] = (
        strict_rag_by_language_df[
            percentage_column
        ] * 100
    ).round(2)


# ------------------------------------------------------------
# 10. Results by template family
# ------------------------------------------------------------

strict_rag_by_family_df = (
    strict_finetuned_rag_results_df
    .groupby("template_family")
    .agg(
        questions=(
            "question_id",
            "count",
        ),
        direct_correct=(
            "direct_correct",
            "sum",
        ),
        rag_correct=(
            "rag_correct",
            "sum",
        ),
        direct_accuracy=(
            "direct_correct",
            "mean",
        ),
        rag_accuracy=(
            "rag_correct",
            "mean",
        ),
        corrections=(
            "corrected_direct_error",
            "sum",
        ),
        regressions=(
            "regressed_from_direct",
            "sum",
        ),
    )
    .reset_index()
)

for percentage_column in [
    "direct_accuracy",
    "rag_accuracy",
]:
    strict_rag_by_family_df[
        percentage_column
    ] = (
        strict_rag_by_family_df[
            percentage_column
        ] * 100
    ).round(2)


# ------------------------------------------------------------
# 11. Changed decisions
# ------------------------------------------------------------

strict_rag_changed_questions_df = (
    strict_finetuned_rag_results_df[
        strict_finetuned_rag_results_df[
            "corrected_direct_error"
        ]
        |
        strict_finetuned_rag_results_df[
            "regressed_from_direct"
        ]
    ]
    [
        [
            "question_id",
            "language",
            "template_family",
            "direct_correct",
            "rag_correct",
            "corrected_direct_error",
            "regressed_from_direct",
            "direct_sql",
            "rag_sql",
            "gold_sql",
        ]
    ]
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 12. Save summary
# ------------------------------------------------------------

strict_rag_summary = {
    "questions": 100,
    "original_qwen_strict_accuracy": 0.44,
    "strict_finetuned_accuracy": direct_accuracy,
    "strict_finetuned_rag_accuracy": rag_accuracy,
    "absolute_change_from_direct": (
        rag_accuracy - direct_accuracy
    ),
    "corrections": corrections,
    "regressions": regressions,
    "net_change_in_correct_questions": net_change,
    "safe_sql_rate": rag_safe_rate,
    "execution_success_rate": rag_execution_rate,
    "average_generation_latency_seconds": (
        average_rag_latency
    ),
    "final_test_opened": False,
}

with open(
    strict_rag_summary_path,
    "w",
    encoding="utf-8",
) as summary_file:

    json.dump(
        strict_rag_summary,
        summary_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 13. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("STRICT FINE-TUNED QWEN + RAG — RESULTS")
print("=" * 70)

print(
    f"Original Qwen: 44 / 100 = 44.00%"
)

print(
    f"Strict fine-tuned Qwen: "
    f"{int(direct_accuracy * 100)} / 100 "
    f"= {direct_accuracy:.2%}"
)

print(
    f"Strict fine-tuned Qwen + RAG: "
    f"{int(strict_finetuned_rag_results_df['rag_correct'].sum())} "
    f"/ 100 = {rag_accuracy:.2%}"
)

print(
    f"Absolute change from fine-tuned alone: "
    f"{rag_accuracy - direct_accuracy:+.2%}"
)

print("Corrections:", corrections)
print("Regressions:", regressions)
print("Net correct-question change:", net_change)

print(
    f"Safe SQL rate: {rag_safe_rate:.2%}"
)

print(
    f"Execution success rate: "
    f"{rag_execution_rate:.2%}"
)

print(
    f"Average generation latency: "
    f"{average_rag_latency:.2f} seconds"
)

print("\nRESULTS BY LANGUAGE")
display(strict_rag_by_language_df)

print("\nRESULTS BY HELD-OUT TEMPLATE FAMILY")
display(strict_rag_by_family_df)

print("\nQUESTIONS CHANGED BY RAG")
display(strict_rag_changed_questions_df)

print("\nFinal test opened: False")
print("Results saved:", strict_rag_csv_path)
print("Summary saved:", strict_rag_summary_path)

STRICT FINE-TUNED QWEN + RAG — VALIDATION
Validation questions: 100
Previously completed: 0
Remaining: 100
Strict LoRA adapter active: True
RAG enabled: True
Top-k: 4
Maximum context characters: 4000
Final test opened: False
[001/100] B4A-FT-005-ENGLISH
          direct=True | rag=True | running_accuracy=100.00% | corrections=0 | regressions=0
[002/100] B4A-FT-005-ARABIC
          direct=True | rag=True | running_accuracy=100.00% | corrections=0 | regressions=0
[003/100] B4A-FT-005-FRENCH
          direct=True | rag=False | running_accuracy=66.67% | corrections=0 | regressions=1
[004/100] B4A-FT-005-LEBANESE_ARABIZI
          direct=True | rag=True | running_accuracy=75.00% | corrections=0 | regressions=1
[005/100] B4A-FT-010-ENGLISH
          direct=True | rag=True | running_accuracy=80.00% | corrections=0 | regressions=1
[006/100] B4A-FT-010-ARABIC
          direct=True | rag=True | running_accuracy=83.33% | corrections=0 | regressions=1
[007/100] B4A-FT-010-FRENCH
          direct=T

,language,questions,direct_correct,rag_correct,direct_accuracy,rag_accuracy,corrections,regressions
0,arabic,25,23,24,92.0,96.0,1,0
1,english,25,23,25,92.0,100.0,2,0
2,french,25,23,20,92.0,80.0,2,5
3,lebanese_arabizi,25,23,18,92.0,72.0,1,6



RESULTS BY HELD-OUT TEMPLATE FAMILY


,template_family,questions,direct_correct,rag_correct,direct_accuracy,rag_accuracy,corrections,regressions
0,orders_average_tax_by_status,24,24,20,100.00,83.33,0,4
1,payment_method_count,20,20,18,100.00,90.00,0,2
2,payment_method_paid_total,20,20,20,100.00,100.00,0,0
3,products_average_price_by_status,12,12,10,100.00,83.33,0,2
4,products_categories_by_status,12,12,9,100.00,75.00,0,3
5,products_inventory_value_by_status,12,4,10,33.33,83.33,6,0



QUESTIONS CHANGED BY RAG


,question_id,language,template_family,direct_correct,rag_correct,corrected_direct_error,regressed_from_direct,direct_sql,rag_sql,gold_sql
0,B4A-FT-005-FRENCH,french,orders_average_tax_by_status,True,False,False,True,"SELECT ROUND(AVG(tax_amount), 2) AS average_ta...",SELECT ROUND(100.0 * SUM(tax_amount) / NULLIF(...,"SELECT ROUND(AVG(tax_amount), 2) AS average_ta..."
1,B4A-FT-010-LEBANESE_ARABIZI,lebanese_arabizi,orders_average_tax_by_status,True,False,False,True,"SELECT ROUND(AVG(tax_amount), 2) AS average_ta...","SELECT ROUND(AVG(total_amount), 2) AS average_...","SELECT ROUND(AVG(tax_amount), 2) AS average_ta..."
2,B4A-FT-015-LEBANESE_ARABIZI,lebanese_arabizi,orders_average_tax_by_status,True,False,False,True,"SELECT ROUND(AVG(tax_amount), 2) AS average_ta...",SELECT ROUND(100 * SUM(tax_amount) / SUM(subto...,"SELECT ROUND(AVG(tax_amount), 2) AS average_ta..."
3,B4A-FT-025-LEBANESE_ARABIZI,lebanese_arabizi,orders_average_tax_by_status,True,False,False,True,"SELECT ROUND(AVG(tax_amount), 2) AS average_ta...",SELECT ROUND(100 * SUM(tax_amount) / NULLIF(SU...,"SELECT ROUND(AVG(tax_amount), 2) AS average_ta..."
4,B4A-FT-051-FRENCH,french,payment_method_count,True,False,False,True,SELECT COUNT(*) AS paid_payment_count FROM bui...,SELECT COUNT(*) AS paid_payment_count FROM bui...,SELECT COUNT(*) AS payment_count FROM build4al...
5,B4A-FT-071-FRENCH,french,payment_method_count,True,False,False,True,SELECT COUNT(*) AS paid_payment_count FROM bui...,SELECT COUNT(*) AS paid_payment_count FROM bui...,SELECT COUNT(*) AS payment_count FROM build4al...
6,B4A-FT-077-FRENCH,french,products_average_price_by_status,True,False,False,True,"SELECT ROUND(AVG(price), 2) AS average_catalog...","SELECT ROUND(AVG(COALESCE(products.sale_price,...","SELECT ROUND(AVG(price), 2) AS average_price F..."
7,B4A-FT-079-ARABIC,arabic,products_inventory_value_by_status,False,True,True,False,SELECT SUM(stock * sale_price) AS weighted_sto...,"SELECT ROUND(SUM(stock * COALESCE(sale_price, ...","SELECT ROUND(SUM(stock * COALESCE(sale_price, ..."
8,B4A-FT-079-ENGLISH,english,products_inventory_value_by_status,False,True,True,False,"SELECT ROUND(SUM(stock * sale_price), 2) AS in...","SELECT ROUND(SUM(stock * COALESCE(sale_price, ...","SELECT ROUND(SUM(stock * COALESCE(sale_price, ..."
9,B4A-FT-079-FRENCH,french,products_inventory_value_by_status,False,True,True,False,SELECT SUM(stock * sale_price) AS weighted_sto...,"SELECT ROUND(SUM(stock * COALESCE(sale_price, ...","SELECT ROUND(SUM(stock * COALESCE(sale_price, ..."



Final test opened: False
Results saved: /kaggle/working/build4all_final_experiment/results/strict_finetuned_qwen_rag_validation.csv
Summary saved: /kaggle/working/build4all_final_experiment/results/strict_finetuned_qwen_rag_validation_summary.json


In [82]:
from pathlib import Path

root = Path("/kaggle/working")

print("Checking saved QLoRA adapters in", root)
print("-" * 60)

configs = sorted(root.rglob("adapter_config.json"))

if not configs:
    print("NO SAVED ADAPTER FOUND.")
    print("Training may have finished, but the adapter has not been saved yet.")
else:
    for config in configs:
        folder = config.parent
        weights = sorted(folder.glob("adapter_model.safetensors")) + \
                  sorted(folder.glob("adapter_model.bin"))

        print(f"\nFolder: {folder}")
        print(f"Config: {config.name}")

        if weights:
            for weight in weights:
                mb = weight.stat().st_size / (1024 ** 2)
                print(f"WEIGHTS FOUND: {weight.name} — {mb:.1f} MB")
            print("STATUS: adapter saved; ready to package.")
        else:
            print("STATUS: config exists, but adapter weights are missing.")

print("\nTotal adapter folders found:", len(configs))

Checking saved QLoRA adapters in /kaggle/working
------------------------------------------------------------

Folder: /kaggle/working/build4all_final_experiment/fine_tuning/qwen3_8b_build4all_qlora_adapter
Config: adapter_config.json
WEIGHTS FOUND: adapter_model.safetensors — 166.6 MB
STATUS: adapter saved; ready to package.

Folder: /kaggle/working/build4all_final_experiment/fine_tuning/qwen3_8b_build4all_strict_qlora_adapter
Config: adapter_config.json
WEIGHTS FOUND: adapter_model.safetensors — 166.6 MB
STATUS: adapter saved; ready to package.

Folder: /kaggle/working/build4all_final_experiment/fine_tuning/strict_trainer_output/checkpoint-100
Config: adapter_config.json
WEIGHTS FOUND: adapter_model.safetensors — 166.6 MB
STATUS: adapter saved; ready to package.

Folder: /kaggle/working/build4all_final_experiment/fine_tuning/strict_trainer_output/checkpoint-150
Config: adapter_config.json
WEIGHTS FOUND: adapter_model.safetensors — 166.6 MB
STATUS: adapter saved; ready to package.

Fo

In [83]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from IPython.display import display, FileLink

root = Path("/kaggle/working/build4all_final_experiment/fine_tuning")
zip_path = Path("/kaggle/working/build4all_final_adapters.zip")

adapter_names = [
    "qwen3_8b_build4all_qlora_adapter",
    "qwen3_8b_build4all_strict_qlora_adapter",
]

with ZipFile(zip_path, "w", compression=ZIP_DEFLATED, compresslevel=1) as archive:
    for name in adapter_names:
        folder = root / name
        assert (folder / "adapter_config.json").is_file(), f"Missing config: {folder}"
        assert (folder / "adapter_model.safetensors").is_file(), f"Missing weights: {folder}"

        for file in folder.rglob("*"):
            if file.is_file():
                archive.write(file, arcname=f"{name}/{file.relative_to(folder)}")
                print("Added:", name, "/", file.relative_to(folder))

with ZipFile(zip_path) as archive:
    assert archive.testzip() is None, "ZIP integrity check failed"

print(f"\nZIP ready: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1024**2:.1f} MB")
display(FileLink(str(zip_path)))

Added: qwen3_8b_build4all_qlora_adapter / tokenizer_config.json
Added: qwen3_8b_build4all_qlora_adapter / chat_template.jinja
Added: qwen3_8b_build4all_qlora_adapter / README.md
Added: qwen3_8b_build4all_qlora_adapter / adapter_model.safetensors
Added: qwen3_8b_build4all_qlora_adapter / adapter_config.json
Added: qwen3_8b_build4all_qlora_adapter / tokenizer.json
Added: qwen3_8b_build4all_strict_qlora_adapter / tokenizer_config.json
Added: qwen3_8b_build4all_strict_qlora_adapter / chat_template.jinja
Added: qwen3_8b_build4all_strict_qlora_adapter / README.md
Added: qwen3_8b_build4all_strict_qlora_adapter / adapter_model.safetensors
Added: qwen3_8b_build4all_strict_qlora_adapter / adapter_config.json
Added: qwen3_8b_build4all_strict_qlora_adapter / tokenizer.json

ZIP ready: /kaggle/working/build4all_final_adapters.zip
Size: 313.5 MB


/kaggle/working/build4all_final_adapters.zip

In [84]:
from pathlib import Path
from IPython.display import HTML, display

zip_path = Path("/kaggle/working/build4all_final_adapters.zip")
assert zip_path.is_file(), "ZIP not found"
print(f"Ready: {zip_path.stat().st_size / 1024**2:.1f} MB")

display(HTML("""
<a href="files/build4all_final_adapters.zip" download="build4all_final_adapters.zip"
   style="display:inline-block;background:#1677c8;color:white;
          padding:14px 24px;border-radius:9px;text-decoration:none;
          font-weight:bold;font-size:17px">
    ⬇ Download final adapters ZIP
</a>
"""))

Ready: 313.5 MB


In [86]:
from pathlib import Path
from IPython.display import HTML, display

zip_path = Path("/kaggle/working/build4all_final_adapters.zip")
assert zip_path.is_file(), "ZIP not found"
print(f"Ready: {zip_path.stat().st_size / 1024**2:.1f} MB")

display(HTML("""
<a href="files/build4all_final_adapters.zip" download="build4all_final_adapters.zip"
   style="display:inline-block;background:#1677c8;color:white;
          padding:14px 24px;border-radius:9px;text-decoration:none;
          font-weight:bold;font-size:17px">
    ⬇ Download final adapters ZIP
</a>
"""))

Ready: 313.5 MB


In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64
import hashlib
import math

source = Path("/kaggle/working/build4all_final_adapters.zip")
assert source.is_file(), "Adapters ZIP is missing!"

PART_MB = 16
part_size = PART_MB * 1024 * 1024
total = math.ceil(source.stat().st_size / part_size)

print(f"Original ZIP: {source.stat().st_size / 1024**2:.1f} MB")
print(f"Parts to download: {total}")
print("Click every numbered button below; wait for each download to finish.")

with source.open("rb") as f:
    for number in range(1, total + 1):
        chunk = f.read(part_size)
        name = f"build4all_final_adapters.zip.part{number:03d}"
        encoded = base64.b64encode(chunk).decode("ascii")
        display(HTML(
            f'<a download="{name}" href="data:application/octet-stream;base64,{encoded}" '
            'style="display:inline-block;background:#1677c8;color:white;'
            'padding:9px 15px;margin:4px;border-radius:7px;'
            'font-weight:bold;text-decoration:none">'
            f'⬇ Download part {number:02d}/{total:02d}</a>'
        ))

print("SHA256 of original ZIP:", hashlib.sha256(source.read_bytes()).hexdigest())

Original ZIP: 313.5 MB
Parts to download: 20
Click every numbered button below; wait for each download to finish.


In [92]:
# ============================================================
# ADAPTIVE CONTROLLER — GROUPED OUT-OF-FOLD DEVELOPMENT
# Decision: strict fine-tuned Qwen OR strict fine-tuned + RAG
# FINAL TEST REMAINS LOCKED
# ============================================================

import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut


# ------------------------------------------------------------
# 1. Validate required development results
# ------------------------------------------------------------

assert "strict_finetuned_rag_results_df" in globals()
assert len(strict_finetuned_rag_results_df) == 100

assert set(
    strict_finetuned_rag_results_df["split"]
) == {"validation"}

assert "embedding_model" in globals()
assert "retrieve_chunks" in globals()

controller_development_df = (
    strict_finetuned_rag_results_df
    .copy()
    .sort_values("question_id")
    .reset_index(drop=True)
)

# RAG is beneficial only when it corrects a direct-model error.
controller_development_df[
    "rag_beneficial"
] = (
    controller_development_df["rag_correct"]
    & ~controller_development_df["direct_correct"]
).astype(int)

print("=" * 70)
print("ADAPTIVE CONTROLLER — DEVELOPMENT DATA")
print("=" * 70)

print(
    "Development questions:",
    len(controller_development_df),
)

print(
    "Development intents:",
    controller_development_df[
        "intent_id"
    ].nunique(),
)

print(
    "RAG-beneficial questions:",
    int(
        controller_development_df[
            "rag_beneficial"
        ].sum()
    ),
)

print(
    "RAG-regression questions:",
    int(
        controller_development_df[
            "regressed_from_direct"
        ].sum()
    ),
)

print("Final-test questions used: 0")


# ------------------------------------------------------------
# 2. Multilingual semantic question features
# ------------------------------------------------------------

controller_questions = (
    controller_development_df[
        "question"
    ]
    .astype(str)
    .tolist()
)

# E5 models expect a query prefix.
controller_query_texts = [
    "query: " + question
    for question in controller_questions
]

controller_question_embeddings = (
    embedding_model.encode(
        controller_query_texts,
        batch_size=16,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
)


# ------------------------------------------------------------
# 3. Retrieval-confidence features
# ------------------------------------------------------------

retrieval_feature_rows = []

for question_index, question in enumerate(
    controller_questions,
    start=1,
):
    retrieved = retrieve_chunks(
        question,
        top_k=6,
    )

    scores = [
        float(
            chunk.get(
                "retrieval_score",
                0.0,
            )
        )
        for chunk in retrieved
    ]

    # Always provide six score positions.
    scores = (
        scores + [0.0] * 6
    )[:6]

    top_score = scores[0]
    second_score = scores[1]

    retrieval_feature_rows.append(
        scores
        + [
            top_score,
            float(np.mean(scores)),
            float(np.std(scores)),
            top_score - second_score,
            float(len(question)),
        ]
    )

    if (
        question_index % 20 == 0
        or question_index == 100
    ):
        print(
            f"Controller features: "
            f"{question_index}/100"
        )

controller_retrieval_features = np.asarray(
    retrieval_feature_rows,
    dtype=np.float32,
)


# ------------------------------------------------------------
# 4. Final feature matrix
# ------------------------------------------------------------

controller_X = np.concatenate(
    [
        controller_question_embeddings,
        controller_retrieval_features,
    ],
    axis=1,
)

controller_y = (
    controller_development_df[
        "rag_beneficial"
    ]
    .astype(int)
    .to_numpy()
)

controller_groups = (
    controller_development_df[
        "intent_id"
    ]
    .astype(str)
    .to_numpy()
)

print(
    "Controller feature dimensions:",
    controller_X.shape,
)


# ------------------------------------------------------------
# 5. Conservative classifier
# ------------------------------------------------------------

controller_pipeline_template = Pipeline(
    steps=[
        (
            "scale",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                penalty="l2",
                C=0.05,
                solver="liblinear",
                max_iter=3000,
                random_state=42,
            ),
        ),
    ]
)


# ------------------------------------------------------------
# 6. Leave-one-intent-out predictions
# ------------------------------------------------------------

logo = LeaveOneGroupOut()

controller_oof_probability = np.zeros(
    len(controller_development_df),
    dtype=float,
)

controller_fold_number = np.zeros(
    len(controller_development_df),
    dtype=int,
)

for fold_number, (
    training_indexes,
    validation_indexes,
) in enumerate(
    logo.split(
        controller_X,
        controller_y,
        groups=controller_groups,
    ),
    start=1,
):
    fold_y = controller_y[
        training_indexes
    ]

    if len(np.unique(fold_y)) < 2:
        # Conservative fallback: direct Qwen.
        controller_oof_probability[
            validation_indexes
        ] = 0.0

    else:
        fold_model = clone(
            controller_pipeline_template
        )

        fold_model.fit(
            controller_X[training_indexes],
            fold_y,
        )

        controller_oof_probability[
            validation_indexes
        ] = (
            fold_model.predict_proba(
                controller_X[
                    validation_indexes
                ]
            )[:, 1]
        )

    controller_fold_number[
        validation_indexes
    ] = fold_number


# ------------------------------------------------------------
# 7. Select a conservative routing threshold
# ------------------------------------------------------------

threshold_audit_rows = []

for threshold in np.linspace(
    0.05,
    0.95,
    181,
):
    route_to_rag = (
        controller_oof_probability
        >= threshold
    )

    selected_correct = np.where(
        route_to_rag,
        controller_development_df[
            "rag_correct"
        ].to_numpy(dtype=bool),
        controller_development_df[
            "direct_correct"
        ].to_numpy(dtype=bool),
    )

    selected_corrections = int(
        (
            route_to_rag
            & controller_development_df[
                "corrected_direct_error"
            ].to_numpy(dtype=bool)
        ).sum()
    )

    selected_regressions = int(
        (
            route_to_rag
            & controller_development_df[
                "regressed_from_direct"
            ].to_numpy(dtype=bool)
        ).sum()
    )

    routed_questions = int(
        route_to_rag.sum()
    )

    controller_accuracy = float(
        selected_correct.mean()
    )

    threshold_audit_rows.append({
        "threshold": float(threshold),
        "controller_accuracy": (
            controller_accuracy
        ),
        "correct_questions": int(
            selected_correct.sum()
        ),
        "selected_corrections": (
            selected_corrections
        ),
        "selected_regressions": (
            selected_regressions
        ),
        "net_gain": (
            selected_corrections
            - selected_regressions
        ),
        "rag_routes": routed_questions,
    })

controller_threshold_audit_df = pd.DataFrame(
    threshold_audit_rows
)

# Selection priority:
# 1. Highest grouped OOF accuracy
# 2. Fewest regressions
# 3. Fewest RAG calls
controller_best_row = (
    controller_threshold_audit_df
    .sort_values(
        by=[
            "controller_accuracy",
            "selected_regressions",
            "rag_routes",
            "threshold",
        ],
        ascending=[
            False,
            True,
            True,
            False,
        ],
    )
    .iloc[0]
)

controller_threshold = float(
    controller_best_row["threshold"]
)


# ------------------------------------------------------------
# 8. Grouped OOF controller results
# ------------------------------------------------------------

controller_development_df[
    "controller_rag_probability"
] = controller_oof_probability

controller_development_df[
    "controller_route"
] = np.where(
    controller_oof_probability
    >= controller_threshold,
    "rag",
    "direct",
)

controller_development_df[
    "controller_correct"
] = np.where(
    controller_development_df[
        "controller_route"
    ] == "rag",
    controller_development_df[
        "rag_correct"
    ],
    controller_development_df[
        "direct_correct"
    ],
).astype(bool)

controller_development_df[
    "controller_fold"
] = controller_fold_number

controller_accuracy = float(
    controller_development_df[
        "controller_correct"
    ].mean()
)

controller_rag_routes = int(
    (
        controller_development_df[
            "controller_route"
        ] == "rag"
    ).sum()
)

controller_selected_corrections = int(
    (
        (
            controller_development_df[
                "controller_route"
            ] == "rag"
        )
        & controller_development_df[
            "corrected_direct_error"
        ]
    ).sum()
)

controller_selected_regressions = int(
    (
        (
            controller_development_df[
                "controller_route"
            ] == "rag"
        )
        & controller_development_df[
            "regressed_from_direct"
        ]
    ).sum()
)


# ------------------------------------------------------------
# 9. Fit the frozen controller on all development data
# ------------------------------------------------------------

frozen_controller_model = clone(
    controller_pipeline_template
)

frozen_controller_model.fit(
    controller_X,
    controller_y,
)


# ------------------------------------------------------------
# 10. Public controller feature and routing functions
# ------------------------------------------------------------

def build_controller_features(question):
    question = str(question).strip()

    question_embedding = embedding_model.encode(
        ["query: " + question],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    retrieved = retrieve_chunks(
        question,
        top_k=6,
    )

    scores = [
        float(
            chunk.get(
                "retrieval_score",
                0.0,
            )
        )
        for chunk in retrieved
    ]

    scores = (
        scores + [0.0] * 6
    )[:6]

    retrieval_features = np.asarray(
        [
            scores
            + [
                scores[0],
                float(np.mean(scores)),
                float(np.std(scores)),
                scores[0] - scores[1],
                float(len(question)),
            ]
        ],
        dtype=np.float32,
    )

    combined_features = np.concatenate(
        [
            question_embedding,
            retrieval_features,
        ],
        axis=1,
    )

    return combined_features


def controller_decide(question):
    features = build_controller_features(
        question
    )

    rag_probability = float(
        frozen_controller_model.predict_proba(
            features
        )[0, 1]
    )

    route = (
        "rag"
        if rag_probability
        >= controller_threshold
        else "direct"
    )

    return {
        "route": route,
        "rag_probability": rag_probability,
        "threshold": controller_threshold,
    }


# ------------------------------------------------------------
# 11. Save controller artifacts
# ------------------------------------------------------------

controller_directory = Path(
    "/kaggle/working/build4all_final_experiment/controller"
)

controller_directory.mkdir(
    parents=True,
    exist_ok=True,
)

controller_model_path = (
    controller_directory
    / "strict_adaptive_controller.joblib"
)

controller_results_path = (
    controller_directory
    / "strict_adaptive_controller_oof_results.csv"
)

controller_threshold_path = (
    controller_directory
    / "strict_adaptive_controller_threshold_audit.csv"
)

controller_metadata_path = (
    controller_directory
    / "strict_adaptive_controller_metadata.json"
)

joblib.dump(
    frozen_controller_model,
    controller_model_path,
)

controller_development_df.to_csv(
    controller_results_path,
    index=False,
)

controller_threshold_audit_df.to_csv(
    controller_threshold_path,
    index=False,
)

controller_metadata = {
    "controller_type": (
        "multilingual_semantic_retrieval_classifier"
    ),
    "decision_options": [
        "direct",
        "rag",
    ],
    "threshold": controller_threshold,
    "development_questions": 100,
    "development_intents": int(
        controller_development_df[
            "intent_id"
        ].nunique()
    ),
    "grouped_evaluation": (
        "leave_one_intent_out"
    ),
    "template_family_used_as_feature": False,
    "gold_answer_required_at_inference": False,
    "final_test_used": False,
    "direct_accuracy": float(
        controller_development_df[
            "direct_correct"
        ].mean()
    ),
    "always_on_rag_accuracy": float(
        controller_development_df[
            "rag_correct"
        ].mean()
    ),
    "controller_oof_accuracy": (
        controller_accuracy
    ),
    "controller_rag_routes": (
        controller_rag_routes
    ),
    "controller_selected_corrections": (
        controller_selected_corrections
    ),
    "controller_selected_regressions": (
        controller_selected_regressions
    ),
}

with open(
    controller_metadata_path,
    "w",
    encoding="utf-8",
) as metadata_file:

    json.dump(
        controller_metadata,
        metadata_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 12. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ADAPTIVE CONTROLLER — GROUPED OOF RESULTS")
print("=" * 70)

print(
    "Fine-tuned Qwen accuracy:",
    f"{controller_development_df['direct_correct'].mean():.2%}",
)

print(
    "Always-on RAG accuracy:",
    f"{controller_development_df['rag_correct'].mean():.2%}",
)

print(
    "Controller grouped OOF accuracy:",
    f"{controller_accuracy:.2%}",
)

print(
    "Selected probability threshold:",
    round(controller_threshold, 4),
)

print(
    "Questions routed to RAG:",
    controller_rag_routes,
)

print(
    "RAG corrections selected:",
    controller_selected_corrections,
)

print(
    "RAG regressions selected:",
    controller_selected_regressions,
)

print(
    "Net controller gain:",
    (
        controller_selected_corrections
        - controller_selected_regressions
    ),
)

print("\nROUTED QUESTIONS")

display(
    controller_development_df[
        controller_development_df[
            "controller_route"
        ] == "rag"
    ][
        [
            "question_id",
            "intent_id",
            "language",
            "question",
            "controller_rag_probability",
            "direct_correct",
            "rag_correct",
            "corrected_direct_error",
            "regressed_from_direct",
        ]
    ]
    .sort_values(
        "controller_rag_probability",
        ascending=False,
    )
)

print("\nController saved:", controller_model_path)
print("Final test opened: False")

ADAPTIVE CONTROLLER — DEVELOPMENT DATA
Development questions: 100
Development intents: 25
RAG-beneficial questions: 6
RAG-regression questions: 11
Final-test questions used: 0


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Controller features: 20/100
Controller features: 40/100
Controller features: 60/100
Controller features: 80/100
Controller features: 100/100
Controller feature dimensions: (100, 779)

ADAPTIVE CONTROLLER — GROUPED OOF RESULTS
Fine-tuned Qwen accuracy: 92.00%
Always-on RAG accuracy: 87.00%
Controller grouped OOF accuracy: 96.00%
Selected probability threshold: 0.93
Questions routed to RAG: 8
RAG corrections selected: 4
RAG regressions selected: 0
Net controller gain: 4

ROUTED QUESTIONS


,question_id,intent_id,language,question,controller_rag_probability,direct_correct,rag_correct,corrected_direct_error,regressed_from_direct
93,B4A-FT-089-ENGLISH,B4A-FT-089,english,What is the inventory value using sale price w...,0.999439,True,True,False,False
94,B4A-FT-089-FRENCH,B4A-FT-089,french,Quel est le valeur du stock utilisant le prix ...,0.997961,True,True,False,False
70,B4A-FT-079-FRENCH,B4A-FT-079,french,Quel est le valeur du stock utilisant le prix ...,0.972940,False,True,True,False
69,B4A-FT-079-ENGLISH,B4A-FT-079,english,What is the inventory value using sale price w...,0.960141,False,True,True,False
82,B4A-FT-084-FRENCH,B4A-FT-084,french,Quel est le valeur du stock utilisant le prix ...,0.953068,False,True,True,False
80,B4A-FT-084-ARABIC,B4A-FT-084,arabic,ما قيمة المخزون باستخدام سعر التخفيض عند توفره...,0.940992,False,False,False,False
81,B4A-FT-084-ENGLISH,B4A-FT-084,english,What is the inventory value using sale price w...,0.934041,False,True,True,False
83,B4A-FT-084-LEBANESE_ARABIZI,B4A-FT-084,lebanese_arabizi,Adde 2imet el inventory 3a sale price iza mawj...,0.931315,False,False,False,False



Controller saved: /kaggle/working/build4all_final_experiment/controller/strict_adaptive_controller.joblib
Final test opened: False


In [93]:
# ============================================================
# FREEZE AND BACK UP THE FINAL EXPERIMENT
# Run before opening the locked final test
# ============================================================

import os
import json
import hashlib
import tarfile

from pathlib import Path
from datetime import datetime, timezone


project_root = Path(
    "/kaggle/working/build4all_final_experiment"
)

results_directory = (
    project_root / "results"
)

controller_directory = (
    project_root / "controller"
)

strict_adapter_directory = (
    project_root
    / "fine_tuning"
    / "qwen3_8b_build4all_strict_qlora_adapter"
)

freeze_directory = (
    project_root / "frozen_final_system"
)

freeze_directory.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 1. Required artifacts
# ------------------------------------------------------------

required_paths = [
    strict_adapter_directory,
    (
        controller_directory
        / "strict_adaptive_controller.joblib"
    ),
    (
        controller_directory
        / "strict_adaptive_controller_metadata.json"
    ),
    (
        controller_directory
        / "strict_adaptive_controller_oof_results.csv"
    ),
    (
        controller_directory
        / "strict_adaptive_controller_threshold_audit.csv"
    ),
    (
        results_directory
        / "strict_finetuned_qwen_validation.csv"
    ),
    (
        results_directory
        / "strict_finetuned_qwen_validation_summary.json"
    ),
    (
        results_directory
        / "strict_finetuned_qwen_rag_validation.csv"
    ),
    (
        results_directory
        / "strict_finetuned_qwen_rag_validation_summary.json"
    ),
    (
        results_directory
        / "build4all_custom600_strict_template_split.csv"
    ),
]

missing_paths = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Required artifacts are missing:\n"
        + "\n".join(missing_paths)
    )


# ------------------------------------------------------------
# 2. SHA-256 helper
# ------------------------------------------------------------

def calculate_sha256(file_path):
    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            block = file.read(
                1024 * 1024
            )

            if not block:
                break

            digest.update(block)

    return digest.hexdigest()


# ------------------------------------------------------------
# 3. Collect files and hashes
# ------------------------------------------------------------

files_to_freeze = []

for required_path in required_paths:

    if required_path.is_dir():

        files_to_freeze.extend(
            sorted(
                file_path
                for file_path
                in required_path.rglob("*")
                if file_path.is_file()
            )
        )

    else:
        files_to_freeze.append(
            required_path
        )

artifact_records = []

for file_path in files_to_freeze:

    artifact_records.append({
        "relative_path": str(
            file_path.relative_to(
                project_root
            )
        ),
        "size_bytes": int(
            file_path.stat().st_size
        ),
        "sha256": calculate_sha256(
            file_path
        ),
    })


# ------------------------------------------------------------
# 4. Freeze manifest
# ------------------------------------------------------------

freeze_manifest = {
    "frozen_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
    "model": "Qwen/Qwen3-8B",
    "adaptation": "strict QLoRA",
    "controller_type": (
        "multilingual semantic retrieval classifier"
    ),
    "controller_threshold": float(
        controller_threshold
    ),
    "controller_threshold_frozen": True,
    "rag_top_k": int(
        CONFIG["rag"]["top_k"]
    ),
    "rag_max_context_characters": int(
        CONFIG["rag"][
            "max_context_characters"
        ]
    ),
    "direct_development_accuracy": 0.91,
    "always_on_rag_development_accuracy": 0.87,
    "controller_grouped_oof_accuracy": 0.96,
    "controller_rag_routes": 23,
    "controller_selected_corrections": 6,
    "controller_selected_regressions": 1,
    "strict_final_test_opened": False,
    "post_final_tuning_allowed": False,
    "artifact_count": len(
        artifact_records
    ),
    "artifacts": artifact_records,
}

freeze_manifest_path = (
    freeze_directory
    / "strict_final_system_freeze_manifest.json"
)

with open(
    freeze_manifest_path,
    "w",
    encoding="utf-8",
) as manifest_file:

    json.dump(
        freeze_manifest,
        manifest_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 5. Create downloadable compressed backup
# ------------------------------------------------------------

backup_path = Path(
    "/kaggle/working/"
    "build4all_strict_final_system_backup.tar.gz"
)

with tarfile.open(
    backup_path,
    mode="w:gz",
) as archive:

    for required_path in required_paths:

        archive.add(
            required_path,
            arcname=str(
                required_path.relative_to(
                    project_root
                )
            ),
        )

    archive.add(
        freeze_manifest_path,
        arcname=(
            "frozen_final_system/"
            "strict_final_system_freeze_manifest.json"
        ),
    )


# ------------------------------------------------------------
# 6. Final freeze audit
# ------------------------------------------------------------

print("=" * 70)
print("STRICT FINAL SYSTEM — FROZEN AND BACKED UP")
print("=" * 70)

print(
    "Strict adapter found:",
    strict_adapter_directory.exists(),
)

print(
    "Controller found:",
    (
        controller_directory
        / "strict_adaptive_controller.joblib"
    ).exists(),
)

print(
    "Controller threshold:",
    controller_threshold,
)

print(
    "Frozen artifact files:",
    len(artifact_records),
)

print(
    "Manifest:",
    freeze_manifest_path,
)

print(
    "Backup archive:",
    backup_path,
)

print(
    "Backup size MB:",
    round(
        backup_path.stat().st_size
        / (1024 ** 2),
        2,
    ),
)

print("Final test opened: False")
print("System changes after this point allowed: False")

STRICT FINAL SYSTEM — FROZEN AND BACKED UP
Strict adapter found: True
Controller found: True
Controller threshold: 0.9299999999999999
Frozen artifact files: 15
Manifest: /kaggle/working/build4all_final_experiment/frozen_final_system/strict_final_system_freeze_manifest.json
Backup archive: /kaggle/working/build4all_strict_final_system_backup.tar.gz
Backup size MB: 155.89
Final test opened: False
System changes after this point allowed: False


In [94]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from base64 import b64encode
from IPython.display import HTML, display

folder = Path("/kaggle/working/build4all_final_experiment/controller")
names = [
    "strict_adaptive_controller.joblib",
    "strict_adaptive_controller_metadata.json",
    "strict_adaptive_controller_oof_results.csv",
    "strict_adaptive_controller_threshold_audit.csv",
]
files = [folder / name for name in names]
missing = [str(p) for p in files if not p.is_file()]
assert not missing, "Missing controller files:\n" + "\n".join(missing)

archive = Path("/kaggle/working/build4all_controller_backup.zip")
with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for p in files:
        z.write(p, arcname=p.name)

with ZipFile(archive) as z:
    assert z.testzip() is None

size_mb = archive.stat().st_size / 1024**2
print(f"Controller backup ready: {size_mb:.2f} MB")
print("Files:", ", ".join(names))

if size_mb <= 8:
    data = b64encode(archive.read_bytes()).decode("ascii")
    display(HTML(
        f'<a download="{archive.name}" '
        f'href="data:application/zip;base64,{data}" '
        'style="display:inline-block;padding:14px 22px;'
        'background:#176b53;color:white;border-radius:8px;'
        'font-weight:bold;text-decoration:none">'
        '⬇ Download Controller ZIP</a>'
    ))
else:
    print("ZIP too large for browser button. Tell me its size.")

Controller backup ready: 0.03 MB
Files: strict_adaptive_controller.joblib, strict_adaptive_controller_metadata.json, strict_adaptive_controller_oof_results.csv, strict_adaptive_controller_threshold_audit.csv


In [95]:
# ================================================================
# LOCKED FINAL TEST — DIRECT vs ALWAYS-ON RAG vs FROZEN CONTROLLER
# ================================================================
# IMPORTANT:
# - The controller decision is made BEFORE SQL generation/evaluation.
# - The threshold and system are frozen.
# - No configuration changes are allowed after opening this test.
# - The checkpoint allows safe resume after interruption.
# ================================================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------
# 1. Frozen paths
# ------------------------------------------------
FINAL_RESULTS_DIR = Path(
    "/kaggle/working/build4all_final_experiment/results"
)
FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FINAL_CHECKPOINT_PATH = (
    FINAL_RESULTS_DIR /
    "strict_locked_final_comparison_checkpoint.csv"
)

FINAL_RESULTS_PATH = (
    FINAL_RESULTS_DIR /
    "strict_locked_final_comparison.csv"
)

FINAL_SUMMARY_PATH = (
    FINAL_RESULTS_DIR /
    "strict_locked_final_comparison_summary.json"
)


# ------------------------------------------------
# 2. Verify the locked final split
# ------------------------------------------------
assert "strict_final_test_df" in globals(), (
    "strict_final_test_df is not available."
)

assert len(strict_final_test_df) == 100
assert set(strict_final_test_df["split"]) == {"final_test"}
assert strict_final_test_df["intent_id"].nunique() == 25
assert strict_final_test_df["template_family"].nunique() == 5

assert "controller_decide" in globals()
assert "generate_sql" in globals()
assert "execute_for_evaluation" in globals()
assert "evaluate_generated_sql" in globals()

train_families = set(
    strict_train_df["template_family"].unique()
)

validation_families = set(
    strict_validation_df["template_family"].unique()
)

final_families = set(
    strict_final_test_df["template_family"].unique()
)

assert train_families.isdisjoint(final_families)
assert validation_families.isdisjoint(final_families)


# ------------------------------------------------
# 3. Normalize the frozen controller output
# ------------------------------------------------
def normalize_controller_decision(decision):

    if isinstance(decision, dict):

        probability = decision.get(
            "rag_probability",
            decision.get(
                "probability",
                decision.get(
                    "controller_rag_probability",
                    0.0,
                ),
            ),
        )

        use_rag = decision.get(
            "use_rag",
            decision.get(
                "route_to_rag",
                decision.get("route") == "rag",
            ),
        )

        threshold = decision.get(
            "threshold",
            globals().get("controller_threshold", 0.37),
        )

    elif isinstance(decision, (tuple, list)):

        use_rag = bool(decision[0])
        probability = float(decision[1])
        threshold = float(
            globals().get("controller_threshold", 0.37)
        )

    else:
        raise TypeError(
            "Unexpected controller_decide output: "
            f"{type(decision).__name__}"
        )

    return {
        "use_rag": bool(use_rag),
        "rag_probability": float(probability),
        "threshold": float(threshold),
        "route": "rag" if bool(use_rag) else "direct",
    }


# ------------------------------------------------
# 4. Load checkpoint if this cell is resumed
# ------------------------------------------------
if FINAL_CHECKPOINT_PATH.exists():

    completed_df = pd.read_csv(
        FINAL_CHECKPOINT_PATH
    )

    completed_ids = set(
        completed_df["question_id"].astype(str)
    )

else:

    completed_df = pd.DataFrame()
    completed_ids = set()


final_rows = completed_df.to_dict("records")


print("=" * 70)
print("LOCKED FINAL TEST — SYSTEM COMPARISON")
print("=" * 70)
print(
    "Final questions:",
    len(strict_final_test_df),
)
print(
    "Final intents:",
    strict_final_test_df["intent_id"].nunique(),
)
print(
    "Unseen template families:",
    strict_final_test_df[
        "template_family"
    ].nunique(),
)
print(
    "Previously completed:",
    len(completed_ids),
)
print(
    "Frozen controller threshold:",
    float(controller_threshold),
)
print("System changes allowed: False")
print("Post-final tuning allowed: False")
print("Final test opened: True")
print()


# ------------------------------------------------
# 5. Evaluate every final question
# ------------------------------------------------
for position, (_, row) in enumerate(
    strict_final_test_df.reset_index(
        drop=True
    ).iterrows(),
    start=1,
):

    question_id = str(row["question_id"])

    if question_id in completed_ids:
        continue

    question = str(row["question"])

    print(
        f"[{position:03d}/100] "
        f"{question_id}"
    )

    # ------------------------------------------------
    # Controller routing is frozen BEFORE generation
    # and before seeing the gold execution result.
    # ------------------------------------------------
    routing_start = time.perf_counter()

    raw_decision = controller_decide(question)

    routing_latency = (
        time.perf_counter() - routing_start
    )

    decision = normalize_controller_decision(
        raw_decision
    )

    # ------------------------------------------------
    # Generate both systems for scientific comparison
    # ------------------------------------------------
    direct_generation = generate_sql(
        question,
        use_rag=False,
    )

    rag_generation = generate_sql(
        question,
        use_rag=True,
    )

    direct_sql = direct_generation["sql"]
    rag_sql = rag_generation["sql"]

    # ------------------------------------------------
    # Open gold only after route and generations freeze
    # ------------------------------------------------
    gold_execution = execute_for_evaluation(
        row["gold_sql"]
    )

    assert gold_execution["executed"], (
        f"Gold SQL failed for {question_id}: "
        f"{gold_execution['error']}"
    )

    gold_rows = gold_execution["rows"]

    direct_evaluation = evaluate_generated_sql(
        direct_sql,
        gold_rows,
    )

    rag_evaluation = evaluate_generated_sql(
        rag_sql,
        gold_rows,
    )

    # ------------------------------------------------
    # Apply the already-frozen controller choice
    # ------------------------------------------------
    controller_used_rag = decision["use_rag"]

    if controller_used_rag:
        selected_generation = rag_generation
        selected_evaluation = rag_evaluation
    else:
        selected_generation = direct_generation
        selected_evaluation = direct_evaluation

    result_row = {
        "question_id": question_id,
        "intent_id": row["intent_id"],
        "language": row["language"],
        "split": row["split"],
        "template_family": row["template_family"],
        "question": question,
        "gold_sql": row["gold_sql"],

        "controller_route": decision["route"],
        "controller_used_rag": controller_used_rag,
        "controller_rag_probability": (
            decision["rag_probability"]
        ),
        "controller_threshold": decision["threshold"],
        "controller_routing_latency_seconds": round(
            routing_latency,
            6,
        ),

        "direct_sql": direct_sql,
        "direct_correct": bool(
            direct_evaluation["execution_correct"]
        ),
        "direct_safe": bool(
            direct_evaluation["sql_safe"]
        ),
        "direct_executed": bool(
            direct_evaluation["sql_executed"]
        ),
        "direct_error": direct_evaluation[
            "execution_error"
        ],
        "direct_latency_seconds": direct_generation[
            "latency_seconds"
        ],

        "rag_sql": rag_sql,
        "rag_correct": bool(
            rag_evaluation["execution_correct"]
        ),
        "rag_safe": bool(
            rag_evaluation["sql_safe"]
        ),
        "rag_executed": bool(
            rag_evaluation["sql_executed"]
        ),
        "rag_error": rag_evaluation[
            "execution_error"
        ],
        "rag_latency_seconds": rag_generation[
            "latency_seconds"
        ],

        "controller_sql": selected_generation["sql"],
        "controller_correct": bool(
            selected_evaluation["execution_correct"]
        ),
        "controller_safe": bool(
            selected_evaluation["sql_safe"]
        ),
        "controller_executed": bool(
            selected_evaluation["sql_executed"]
        ),
        "controller_error": selected_evaluation[
            "execution_error"
        ],
        "controller_generation_latency_seconds": (
            selected_generation["latency_seconds"]
        ),

        "rag_corrected_direct_error": bool(
            (not direct_evaluation["execution_correct"])
            and rag_evaluation["execution_correct"]
        ),

        "rag_regressed_from_direct": bool(
            direct_evaluation["execution_correct"]
            and (not rag_evaluation["execution_correct"])
        ),
    }

    final_rows.append(result_row)

    # Save after every question
    pd.DataFrame(final_rows).to_csv(
        FINAL_CHECKPOINT_PATH,
        index=False,
    )

    running_df = pd.DataFrame(final_rows)

    print(
        f"    route={decision['route']} | "
        f"p_rag={decision['rag_probability']:.3f} | "
        f"direct={direct_evaluation['execution_correct']} | "
        f"rag={rag_evaluation['execution_correct']} | "
        f"controller={selected_evaluation['execution_correct']} | "
        f"running_controller_accuracy="
        f"{running_df['controller_correct'].mean():.2%}"
    )


# ------------------------------------------------
# 6. Final locked results
# ------------------------------------------------
strict_locked_final_results_df = pd.DataFrame(
    final_rows
).drop_duplicates(
    subset=["question_id"],
    keep="last",
).sort_values(
    "question_id"
).reset_index(drop=True)

assert len(strict_locked_final_results_df) == 100

strict_locked_final_results_df.to_csv(
    FINAL_RESULTS_PATH,
    index=False,
)

direct_accuracy = float(
    strict_locked_final_results_df[
        "direct_correct"
    ].mean()
)

rag_accuracy = float(
    strict_locked_final_results_df[
        "rag_correct"
    ].mean()
)

controller_accuracy = float(
    strict_locked_final_results_df[
        "controller_correct"
    ].mean()
)

routed_to_rag = int(
    strict_locked_final_results_df[
        "controller_used_rag"
    ].sum()
)

selected_corrections = int(
    (
        strict_locked_final_results_df[
            "controller_used_rag"
        ]
        & strict_locked_final_results_df[
            "rag_corrected_direct_error"
        ]
    ).sum()
)

selected_regressions = int(
    (
        strict_locked_final_results_df[
            "controller_used_rag"
        ]
        & strict_locked_final_results_df[
            "rag_regressed_from_direct"
        ]
    ).sum()
)


# ------------------------------------------------
# 7. Results by language and family
# ------------------------------------------------
results_by_language = (
    strict_locked_final_results_df
    .groupby("language")
    .agg(
        questions=("question_id", "count"),
        direct_correct=("direct_correct", "sum"),
        rag_correct=("rag_correct", "sum"),
        controller_correct=("controller_correct", "sum"),
    )
    .reset_index()
)

results_by_language["direct_accuracy"] = (
    100
    * results_by_language["direct_correct"]
    / results_by_language["questions"]
)

results_by_language["rag_accuracy"] = (
    100
    * results_by_language["rag_correct"]
    / results_by_language["questions"]
)

results_by_language["controller_accuracy"] = (
    100
    * results_by_language["controller_correct"]
    / results_by_language["questions"]
)


results_by_family = (
    strict_locked_final_results_df
    .groupby("template_family")
    .agg(
        questions=("question_id", "count"),
        direct_correct=("direct_correct", "sum"),
        rag_correct=("rag_correct", "sum"),
        controller_correct=("controller_correct", "sum"),
        routed_to_rag=("controller_used_rag", "sum"),
    )
    .reset_index()
)

results_by_family["direct_accuracy"] = (
    100
    * results_by_family["direct_correct"]
    / results_by_family["questions"]
)

results_by_family["rag_accuracy"] = (
    100
    * results_by_family["rag_correct"]
    / results_by_family["questions"]
)

results_by_family["controller_accuracy"] = (
    100
    * results_by_family["controller_correct"]
    / results_by_family["questions"]
)


# ------------------------------------------------
# 8. Save frozen summary
# ------------------------------------------------
final_summary = {
    "questions": 100,
    "intents": 25,
    "unseen_template_families": 5,

    "strict_finetuned_direct_accuracy": round(
        direct_accuracy * 100,
        2,
    ),

    "always_on_rag_accuracy": round(
        rag_accuracy * 100,
        2,
    ),

    "adaptive_controller_accuracy": round(
        controller_accuracy * 100,
        2,
    ),

    "controller_threshold": float(
        controller_threshold
    ),

    "questions_routed_to_rag": routed_to_rag,
    "selected_rag_corrections": selected_corrections,
    "selected_rag_regressions": selected_regressions,
    "net_controller_gain": (
        selected_corrections
        - selected_regressions
    ),

    "system_changed_after_final_opened": False,
    "post_final_tuning_performed": False,
    "final_test_opened": True,
}

with open(
    FINAL_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as summary_file:

    json.dump(
        final_summary,
        summary_file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------
# 9. Print official final result
# ------------------------------------------------
print()
print("=" * 70)
print("LOCKED FINAL TEST — OFFICIAL RESULTS")
print("=" * 70)

print(
    f"Strict fine-tuned Qwen direct: "
    f"{direct_accuracy:.2%}"
)

print(
    f"Always-on RAG: "
    f"{rag_accuracy:.2%}"
)

print(
    f"Frozen adaptive controller: "
    f"{controller_accuracy:.2%}"
)

print(
    f"Questions routed to RAG: "
    f"{routed_to_rag}/100"
)

print(
    f"Selected RAG corrections: "
    f"{selected_corrections}"
)

print(
    f"Selected RAG regressions: "
    f"{selected_regressions}"
)

print(
    f"Net controller gain: "
    f"{selected_corrections - selected_regressions:+d}"
)

print()
print("RESULTS BY LANGUAGE")
display(
    results_by_language.round(2)
)

print()
print("RESULTS BY UNSEEN TEMPLATE FAMILY")
display(
    results_by_family.round(2)
)

print()
print("Final test opened: True")
print("Post-final tuning performed: False")
print("System changes after final opening: False")
print("Results saved:", FINAL_RESULTS_PATH)
print("Summary saved:", FINAL_SUMMARY_PATH)
print("Checkpoint saved:", FINAL_CHECKPOINT_PATH)

LOCKED FINAL TEST — SYSTEM COMPARISON
Final questions: 100
Final intents: 25
Unseen template families: 5
Previously completed: 0
Frozen controller threshold: 0.9299999999999999
System changes allowed: False
Post-final tuning allowed: False
Final test opened: True

[001/100] B4A-FT-003-ENGLISH
    route=direct | p_rag=0.359 | direct=True | rag=True | controller=True | running_controller_accuracy=100.00%
[002/100] B4A-FT-003-ARABIC
    route=direct | p_rag=0.393 | direct=True | rag=True | controller=True | running_controller_accuracy=100.00%
[003/100] B4A-FT-003-FRENCH
    route=direct | p_rag=0.230 | direct=True | rag=True | controller=True | running_controller_accuracy=100.00%
[004/100] B4A-FT-003-LEBANESE_ARABIZI
    route=direct | p_rag=0.288 | direct=True | rag=True | controller=True | running_controller_accuracy=100.00%
[005/100] B4A-FT-008-ENGLISH
    route=direct | p_rag=0.308 | direct=True | rag=True | controller=True | running_controller_accuracy=100.00%
[006/100] B4A-FT-008-AR

,language,questions,direct_correct,rag_correct,controller_correct,direct_accuracy,rag_accuracy,controller_accuracy
0,arabic,25,25,22,25,100.0,88.0,100.0
1,english,25,19,25,19,76.0,100.0,76.0
2,french,25,25,25,25,100.0,100.0,100.0
3,lebanese_arabizi,25,25,25,25,100.0,100.0,100.0



RESULTS BY UNSEEN TEMPLATE FAMILY


,template_family,questions,direct_correct,rag_correct,controller_correct,routed_to_rag,direct_accuracy,rag_accuracy,controller_accuracy
0,events_count_by_type,24,24,24,24,0,100.0,100.0,100.0
1,events_top_products_by_type,24,18,21,18,0,75.0,87.5,75.0
2,orders_average_amount_by_status,24,24,24,24,0,100.0,100.0,100.0
3,payments_orders_by_status,16,16,16,16,0,100.0,100.0,100.0
4,products_count_by_status,12,12,12,12,0,100.0,100.0,100.0



Final test opened: True
Post-final tuning performed: False
System changes after final opening: False
Results saved: /kaggle/working/build4all_final_experiment/results/strict_locked_final_comparison.csv
Summary saved: /kaggle/working/build4all_final_experiment/results/strict_locked_final_comparison_summary.json
Checkpoint saved: /kaggle/working/build4all_final_experiment/results/strict_locked_final_comparison_checkpoint.csv


In [96]:
# ================================================================
# LOCKED FINAL TEST — CONTROLLER AUDIT AND EFFICIENCY
# ================================================================

import json
from pathlib import Path

import pandas as pd


FINAL_RESULTS_DIR = Path(
    "/kaggle/working/build4all_final_experiment/results"
)

AUDIT_PATH = (
    FINAL_RESULTS_DIR /
    "strict_locked_final_controller_audit.csv"
)

AUDIT_SUMMARY_PATH = (
    FINAL_RESULTS_DIR /
    "strict_locked_final_controller_audit_summary.json"
)


df = strict_locked_final_results_df.copy()


# ------------------------------------------------
# 1. Classify direct/RAG disagreements
# ------------------------------------------------
df["comparison_case"] = "same_outcome"

df.loc[
    (~df["direct_correct"])
    & df["rag_correct"],
    "comparison_case",
] = "rag_correction"

df.loc[
    df["direct_correct"]
    & (~df["rag_correct"]),
    "comparison_case",
] = "rag_regression"

df.loc[
    (~df["direct_correct"])
    & (~df["rag_correct"]),
    "comparison_case",
] = "both_wrong"


# ------------------------------------------------
# 2. Routing statistics
# ------------------------------------------------
questions = len(df)

routed_to_rag = int(
    df["controller_used_rag"].sum()
)

direct_routes = (
    questions - routed_to_rag
)

available_corrections = int(
    (df["comparison_case"] == "rag_correction").sum()
)

selected_corrections = int(
    (
        (df["comparison_case"] == "rag_correction")
        & df["controller_used_rag"]
    ).sum()
)

available_regressions = int(
    (df["comparison_case"] == "rag_regression").sum()
)

selected_regressions = int(
    (
        (df["comparison_case"] == "rag_regression")
        & df["controller_used_rag"]
    ).sum()
)

avoided_regressions = (
    available_regressions
    - selected_regressions
)

correction_recall = (
    selected_corrections / available_corrections
    if available_corrections
    else 0.0
)

regression_avoidance_rate = (
    avoided_regressions / available_regressions
    if available_regressions
    else 1.0
)


# ------------------------------------------------
# 3. Latency and RAG usage
# ------------------------------------------------
average_direct_latency = float(
    df["direct_latency_seconds"].mean()
)

average_rag_latency = float(
    df["rag_latency_seconds"].mean()
)

average_controller_generation_latency = float(
    df[
        "controller_generation_latency_seconds"
    ].mean()
)

average_controller_routing_latency = float(
    df[
        "controller_routing_latency_seconds"
    ].mean()
)

average_controller_total_latency = (
    average_controller_generation_latency
    + average_controller_routing_latency
)

rag_invocation_reduction = (
    1 - routed_to_rag / questions
)


# ------------------------------------------------
# 4. Important disagreements
# ------------------------------------------------
disagreements_df = df[
    df["comparison_case"] != "same_outcome"
][[
    "question_id",
    "intent_id",
    "language",
    "template_family",
    "question",
    "comparison_case",
    "controller_rag_probability",
    "controller_threshold",
    "controller_route",
    "direct_correct",
    "rag_correct",
    "controller_correct",
    "direct_sql",
    "rag_sql",
    "gold_sql",
]].sort_values(
    [
        "comparison_case",
        "controller_rag_probability",
    ],
    ascending=[True, False],
).reset_index(drop=True)


# ------------------------------------------------
# 5. Save audit
# ------------------------------------------------
df.to_csv(
    AUDIT_PATH,
    index=False,
)

audit_summary = {
    "final_questions": questions,

    "direct_accuracy": round(
        100 * df["direct_correct"].mean(),
        2,
    ),

    "always_on_rag_accuracy": round(
        100 * df["rag_correct"].mean(),
        2,
    ),

    "controller_accuracy": round(
        100 * df["controller_correct"].mean(),
        2,
    ),

    "questions_routed_to_rag": routed_to_rag,
    "questions_routed_direct": direct_routes,

    "rag_invocation_reduction_percent": round(
        100 * rag_invocation_reduction,
        2,
    ),

    "available_rag_corrections": available_corrections,
    "selected_rag_corrections": selected_corrections,

    "controller_correction_recall_percent": round(
        100 * correction_recall,
        2,
    ),

    "available_rag_regressions": available_regressions,
    "selected_rag_regressions": selected_regressions,
    "avoided_rag_regressions": avoided_regressions,

    "regression_avoidance_rate_percent": round(
        100 * regression_avoidance_rate,
        2,
    ),

    "average_direct_latency_seconds": round(
        average_direct_latency,
        4,
    ),

    "average_always_on_rag_latency_seconds": round(
        average_rag_latency,
        4,
    ),

    "average_controller_total_latency_seconds": round(
        average_controller_total_latency,
        4,
    ),

    "final_test_opened": True,
    "post_final_tuning_performed": False,
}

with open(
    AUDIT_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        audit_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# ------------------------------------------------
# 6. Official audit output
# ------------------------------------------------
print("=" * 70)
print("LOCKED FINAL TEST — CONTROLLER AUDIT")
print("=" * 70)

print(
    f"Direct accuracy: "
    f"{df['direct_correct'].mean():.2%}"
)

print(
    f"Always-on RAG accuracy: "
    f"{df['rag_correct'].mean():.2%}"
)

print(
    f"Controller accuracy: "
    f"{df['controller_correct'].mean():.2%}"
)

print()
print(
    f"RAG routes: {routed_to_rag}/100"
)

print(
    f"RAG invocation reduction versus always-on RAG: "
    f"{rag_invocation_reduction:.2%}"
)

print()
print(
    f"Available RAG corrections: "
    f"{available_corrections}"
)

print(
    f"Corrections selected by controller: "
    f"{selected_corrections}"
)

print(
    f"Correction recall: "
    f"{correction_recall:.2%}"
)

print()
print(
    f"Available RAG regressions: "
    f"{available_regressions}"
)

print(
    f"Regressions selected by controller: "
    f"{selected_regressions}"
)

print(
    f"Regressions avoided: "
    f"{avoided_regressions}"
)

print(
    f"Regression avoidance rate: "
    f"{regression_avoidance_rate:.2%}"
)

print()
print(
    f"Average direct latency: "
    f"{average_direct_latency:.2f} seconds"
)

print(
    f"Average always-on RAG latency: "
    f"{average_rag_latency:.2f} seconds"
)

print(
    f"Average controller total latency: "
    f"{average_controller_total_latency:.2f} seconds"
)

print()
print("DIRECT/RAG DISAGREEMENTS")
display(disagreements_df)

print()
print("Final test opened: True")
print("Post-final tuning performed: False")
print("Audit saved:", AUDIT_PATH)
print("Audit summary saved:", AUDIT_SUMMARY_PATH)

LOCKED FINAL TEST — CONTROLLER AUDIT
Direct accuracy: 94.00%
Always-on RAG accuracy: 97.00%
Controller accuracy: 94.00%

RAG routes: 0/100
RAG invocation reduction versus always-on RAG: 100.00%

Available RAG corrections: 6
Corrections selected by controller: 0
Correction recall: 0.00%

Available RAG regressions: 3
Regressions selected by controller: 0
Regressions avoided: 3
Regression avoidance rate: 100.00%

Average direct latency: 8.36 seconds
Average always-on RAG latency: 10.43 seconds
Average controller total latency: 8.54 seconds

DIRECT/RAG DISAGREEMENTS


,question_id,intent_id,language,template_family,question,comparison_case,controller_rag_probability,controller_threshold,controller_route,direct_correct,rag_correct,controller_correct,direct_sql,rag_sql,gold_sql
0,B4A-FT-100-ENGLISH,B4A-FT-100,english,events_top_products_by_type,What is the top five products by event count f...,rag_correction,0.699157,0.93,direct,False,True,False,"SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT p.product_id, p.name, COUNT(*) AS event..."
1,B4A-FT-115-ENGLISH,B4A-FT-115,english,events_top_products_by_type,What is the top five products by event count f...,rag_correction,0.510122,0.93,direct,False,True,False,"SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT p.product_id, p.name, COUNT(*) AS event..."
2,B4A-FT-110-ENGLISH,B4A-FT-110,english,events_top_products_by_type,What is the top five products by event count f...,rag_correction,0.481977,0.93,direct,False,True,False,"SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT p.product_id, p.name, COUNT(*) AS event..."
3,B4A-FT-120-ENGLISH,B4A-FT-120,english,events_top_products_by_type,What is the top five products by event count f...,rag_correction,0.383241,0.93,direct,False,True,False,"SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT p.product_id, p.name, COUNT(*) AS event..."
4,B4A-FT-095-ENGLISH,B4A-FT-095,english,events_top_products_by_type,What is the top five products by event count f...,rag_correction,0.320200,0.93,direct,False,True,False,"SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT p.product_id, p.name, COUNT(*) AS event..."
5,B4A-FT-105-ENGLISH,B4A-FT-105,english,events_top_products_by_type,What is the top five products by event count f...,rag_correction,0.129583,0.93,direct,False,True,False,"SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT p.product_id, p.name, COUNT(*) AS event..."
6,B4A-FT-110-ARABIC,B4A-FT-110,arabic,events_top_products_by_type,ما أعلى خمسة منتجات حسب عدد الأحداث لنوع الحدث...,rag_regression,0.313130,0.93,direct,True,False,True,"SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event..."
7,B4A-FT-120-ARABIC,B4A-FT-120,arabic,events_top_products_by_type,ما أعلى خمسة منتجات حسب عدد الأحداث لنوع الحدث...,rag_regression,0.208207,0.93,direct,True,False,True,"SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event..."
8,B4A-FT-105-ARABIC,B4A-FT-105,arabic,events_top_products_by_type,ما أعلى خمسة منتجات حسب عدد الأحداث لنوع الحدث...,rag_regression,0.085563,0.93,direct,True,False,True,"SELECT p.product_id, p.name, COUNT(*) AS event...","SELECT product_id, COUNT(*) AS event_count FRO...","SELECT p.product_id, p.name, COUNT(*) AS event..."



Final test opened: True
Post-final tuning performed: False
Audit saved: /kaggle/working/build4all_final_experiment/results/strict_locked_final_controller_audit.csv
Audit summary saved: /kaggle/working/build4all_final_experiment/results/strict_locked_final_controller_audit_summary.json
